# Smart Home Appliance Identification Using Deep Learning

This notebook contains the reproducible experimental workflow for the COMP700 Honours project on identifying smart-home IoT devices from payload-independent statistical network-flow features.

The workflow covers:

- Google Drive setup and optional Kaggle dataset acquisition;
- exploratory data analysis of the publisher-provided files;
- formal publisher-boundary dataset-shift and class-support testing;
- PCA visualisation of the original source-partition structure;
- pooling and exact deduplication of the source files;
- deterministic stratified reconstruction of the training, validation and held-out test partitions;
- training-only preprocessing and SMOTE;
- hyperparameter tuning and five-seed training of the CNN, LSTM, BiLSTM, Feature-Attention MLP and Attention CNN-BiLSTM;
- five-fold cross-validated Random Forest tuning;
- validation-based model and seed selection;
- saved-artifact held-out evaluation;
- one-flow-at-a-time inference demonstration;
- FP32 LiteRT conversion and equivalence validation;
- controlled x86_64 and ARM64 inference profiling;
- cross-platform predictive-deployment analysis;
- Kernel SHAP and correlation-aware Partition SHAP analysis; and
- post-hoc difficult-class analysis covering confidence, five-seed stability, cross-model errors, real-data coverage, feature-space separability, SMOTE behaviour and contrastive SHAP.

The corrected corpus contains 1,877 unique observations. A deterministic stratified split using `random_state=42` produces 1,200 training observations, 301 validation observations and 376 held-out test observations. Training-derived zero-variance filtering removes 53 of the original 297 predictors, leaving 244 predictors.

The principal architecture is an Attention CNN-BiLSTM. The CNN, LSTM, BiLSTM, Feature-Attention MLP and tuned Random Forest provide comparative evidence. Neural hyperparameters and seed models are selected using validation data only. The held-out test partition is reserved for final predictive evaluation and post-selection analysis.

Training cells save timestamped runs containing models, preprocessing objects, fixed test bundles, selected hyperparameters, training histories, terminal output, source-code hashes, runtime environment information, system diagnostics and package requirements. Evaluation, inference, conversion, profiling and most analysis cells load these saved artifacts and therefore do not require training to occur in the same Colab session.

The SHAP Common Setup cell must be run at least once for the selected CNN-BiLSTM so that the shared background, feature-family mapping and integrity metadata are saved. The full Kernel SHAP cell can then resume in a later Colab session from those saved common artifacts and its checkpoints. The Partition SHAP cell requires the Common Setup cell to have been run in the current runtime, but it reuses completed Kernel and Partition checkpoints rather than recomputing them.

**Authors:** Arjun Ramphal and Darian Robert  
**Supervisors:** Prof. O.P. Kogeda and Prof. M.B. Sinqadu  
**Programme:** BSc Honours in Computer Science  
**Institution:** University of KwaZulu-Natal

## How to Run

This notebook is designed for Google Colab and uses Google Drive for persistent storage.

The required project location is:

```text
/content/drive/MyDrive/Honours_Project/
```

A publicly shared project folder is read-only. Download or copy the complete `Honours_Project` folder into the root of your own `My Drive` before running any cell that creates or updates outputs.

Do not use **Runtime → Run all** unless you intentionally want to retrain every model and recompute the complete SHAP analysis.

### Full Reproduction Order

1. **Google Drive Setup**

   Mount Google Drive and confirm that the notebook can access:

   ```text
   /content/drive/MyDrive/Honours_Project/
   ```

2. **Dataset Acquisition and Storage**

   Run the Kaggle acquisition cell only if the two publisher CSV files are absent. If both files already exist at the required locations, the download is skipped.

3. **Exploratory Data Analysis and Dataset Integrity**

   Run:

   - Dataset Profiling and Exploratory Analysis; and
   - Publisher Split Shift Analysis.

   The PCA and PCA-stabilised t-SNE analyses are included in the Dataset Profiling and Exploratory Analysis cell. There is no separate PCA prerequisite.

   These analyses first examine the original publisher-defined file boundary and then verify the pooled, deduplicated and reconstructed stratified split used for model development.

4. **Model Development**

   Each training cell is self-contained and can be executed independently after the datasets are available:

   - Attention CNN-BiLSTM;
   - Feature-Attention MLP;
   - tuned CNN;
   - tuned LSTM;
   - tuned BiLSTM; and
   - tuned Random Forest.

   Every training cell reconstructs the corrected corpus, split and preprocessing pipeline internally. There are no separate preprocessing or reshaping cells that must be run first.

   Each neural training cell performs its hyperparameter search, trains the selected configuration using seeds `42`, `7`, `123`, `2024` and `99`, and selects its representative model using validation evidence only.

5. **Held-Out Test Evaluation**

   Run the universal evaluator after training, or use it with compatible saved runs.

   For each required architecture, select the intended timestamped run. Evaluate both the validation-selected model and all five seed-specific models when later five-seed analysis is required.

   Canonical metrics are written inside each originating run under:

   ```text
   Test_Evaluation/<model identifier>/metrics.csv
   ```

   Timestamped global summary files are convenience copies and must not be treated as additional experiments.

6. **Result Figures**

   Generate the required figures after their saved inputs exist:

   - **Selected-Model Predictive Comparison Figure** reads one canonical model-specific evaluation from each selected timestamped run and ignores aggregate summary copies.
   - **CNN-BiLSTM Five-Seed Performance Visualisation** requires all five CNN-BiLSTM seed evaluations and an evaluated tuned Random Forest.
   - **Cross-Platform LiteRT Edge-Profiling Summary** requires the selected x86_64 and ARM64 profiling summaries.
   - **Predictive-Deployment Trade-off Figure** requires the cross-platform summary, held-out evaluations and LiteRT profiles.

   The Selected-Model Predictive Comparison Figure does not require `all_model_overall_metrics.csv` or any post-hoc analysis output.

7. **Full Test-Set Inference Demonstration**

   Select one compatible saved model and process the 376 held-out observations one at a time.

   This cell performs actual saved-model inference and displays the predicted class, actual class, confidence, correctness and running accuracy.

   It is an interactive demonstration. The universal evaluator remains the authoritative source of predictive metrics.

8. **LiteRT Model Conversion and Validation**

   Select compatible saved neural models.

   Conversion may occur in a later Colab session because the converter loads the models, test tensors and associated metadata from Google Drive.

   Validation-selected models are the recommended deployment artifacts. Seed-specific models remain available for explicit conversion when required.

   Only converter-validated LiteRT artifacts should be passed to the edge profiler.

9. **Controlled x86_64 Profiling**

   Profile converter-validated `.tflite` artifacts.

   The controlled profiler uses:

   - CPU-only fresh processes;
   - one inference thread;
   - one pinned logical CPU where supported;
   - static batch-one inputs;
   - three fresh processes per model;
   - 50 warm-up inferences per phase; and
   - 10 passes over all 376 held-out inputs per process.

10. **ARM64 Profiling**

    Ensure that the current ARM profiling script exists at:

    ```text
    Honours_Project/ARM_Profiling/comp700_arm_selected_models_v7.py
    ```

    Build the ARM64 bundle in Colab, transfer the generated ZIP to the ARM64 Linux server, extract it and run:

    ```bash
    python3 comp700_arm_selected_models_v7.py profile --auto-install --require-arm64
    ```

    The profiling command creates a private environment when necessary, installs the required dependencies, validates the bundle and performs the controlled ARM64 measurements.

    Archive the complete unchanged `results/` directory under:

    ```text
    Honours_Project/Models/Edge_Profiling_Summaries/ARM64/<timestamp>/results/
    ```

11. **Cross-Platform Analysis**

    Select the intended x86_64 and ARM64 profiling summaries and verify every displayed path before accepting the selection.

    For the retained authoritative evidence, use:

    ```text
    Models/Edge_Profiling_Summaries/
    ├── edge_profile_summary_2026-09-27_20-23-59.csv
    └── ARM64/
        └── 2026-09-27_18-35-58/
            └── results/
                └── arm_edge_profile_summary.csv
    ```

    Then generate the Predictive-Deployment Trade-off Figure using matching held-out, conversion, profiling and cross-platform artifacts.

12. **SHAP Analysis**

    Use this order:

    ```text
    Block 1 — SHAP Common Setup and Integrity Checks
    Block 2 — Kernel SHAP Monte-Carlo Stability Analysis
    Block 3 — Full Held-Out Kernel SHAP Analysis
    Block 4 — Correlation-Aware Partition SHAP
    Block 5 — Full-Test SHAP Post-Processing and Cross-Method Comparison
    ```

    Block 1 must be run once to create the shared saved artifacts.

    Block 2 requires the Block 1 runtime state.

    Block 3 can run in a fresh runtime after Block 1 has created the saved common artifacts. It automatically reuses valid Kernel SHAP checkpoints.

    Block 4 requires the Block 1 runtime state and completed Block 3 Kernel SHAP artifacts. It reuses valid Partition SHAP checkpoints.

    Block 5 is self-contained and loads completed Kernel and Partition SHAP results from Google Drive.

13. **Post-Hoc Difficult-Class Error Analysis**

    Run the required difficult-class analysis cells:

    - Selected CNN-BiLSTM Confusion, Confidence and Case-Level Audit;
    - Five-Seed Difficult-Class Stability;
    - Difficult-Class Comparison Across the Selected Models;
    - Real-Data Coverage and Feature-Space Separability; and
    - Contrastive SHAP Analysis of Difficult-Class Errors.

    The first four cells are independently restartable and load their required artifacts from Google Drive.

    The contrastive SHAP analysis additionally requires the completed full-test Kernel and Partition SHAP tensors, but those tensors do not need to have been computed in the same runtime.

    These analyses explain the behaviour of the locked models. They must not be used to redesign a model and then report performance on the same held-out test set.

### Saved-Artifact Route

Retraining is unnecessary when compatible saved runs already exist.

Use the following shorter route:

1. Google Drive Setup.
2. Held-Out Test Evaluation, only if canonical evaluation outputs are missing.
3. Selected-Model Predictive Comparison Figure.
4. CNN-BiLSTM Five-Seed Performance Visualisation.
5. Full Test-Set Inference Demonstration, if required.
6. LiteRT Model Conversion and Validation, only if validated `.tflite` artifacts are missing.
7. Controlled x86_64 Profiling, only if the required profiles are missing.
8. ARM64 bundle construction and profiling, only if ARM64 results are missing.
9. Cross-Platform LiteRT Edge-Profiling Summary, if it must be regenerated.
10. Predictive-Deployment Trade-off Figure.
11. Full-Test SHAP Post-Processing and Cross-Method Comparison.
12. Any required post-hoc difficult-class analysis cells.

### Integrity Rules

Do not move individual artifacts out of their timestamped run directories.

Do not bypass missing-file, hash, shape, feature-order, class-order, source-provenance or numerical-equivalence errors. These checks prevent artifacts from different model runs or dataset reconstructions from being combined.

Training, preprocessing fitting, SMOTE, hyperparameter tuning and neural seed selection must not use held-out test results.

Held-out metrics and post-hoc diagnostic analyses must be calculated only after the retained models have been selected.

## Environment and Data Setup

### Google Drive Setup

This cell mounts the user's Google Drive in the Colab runtime so that the notebook can access the COMP700 project files stored under:

`/content/drive/MyDrive/`

After mounting, subsequent sections of the notebook can read datasets, load saved models, access archived experimental artifacts and write generated outputs back to Google Drive.

Run this cell before executing any section that depends on files stored in the `Honours_Project` directory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Dataset Acquisition and Storage

This section downloads the public IoT device-identification dataset from Kaggle and places the required CSV files into the exact Google Drive locations expected by the remainder of the notebook.

The dataset source is:

`fanbyprinciple/iot-device-identification`

The required project files are:

```text
/content/drive/MyDrive/Honours_Project/iot_device_train.csv
/content/drive/MyDrive/Honours_Project/iot_device_test.csv
```

#### Kaggle Download Support

The cell installs and imports `kagglehub`, which is used to retrieve the public dataset directly from Kaggle.

Google Drive is mounted if it is not already available, and the project directory:

`/content/drive/MyDrive/Honours_Project`

is created automatically when necessary.

#### Existing Dataset Check

Before downloading anything, the code checks whether both required CSV files already exist in the project directory.

If both files are already present, the download step is skipped.

This avoids unnecessary repeated downloads when the notebook is rerun.

#### Dataset Download

If either required file is missing, the dataset is downloaded into temporary Colab storage using:

`kagglehub.dataset_download(...)`

The downloaded directory is then searched recursively for:

- `iot_device_train.csv`
- `iot_device_test.csv`

The code requires exactly one copy of each file to be found.

If either file is missing or appears more than once, execution is stopped rather than copying an ambiguous source file.

#### Copy to Google Drive

After the two source files are located, they are copied into the fixed Google Drive paths used throughout the project.

This ensures that all later notebook sections can refer to a consistent dataset location regardless of the temporary directory structure created by KaggleHub.

#### Final Verification

After the download and copy step, the code verifies that both required files exist at their expected destinations.

If either file is absent, a `FileNotFoundError` is raised.

When successful, the cell prints the final locations of the training and testing datasets.

#### Methodological Role

This section performs dataset acquisition and project setup only.

It does not:

- modify the dataset contents;
- remove duplicates;
- perform preprocessing;
- create the reconstructed train/validation/test split;
- fit models; or
- evaluate predictive performance.

All dataset cleaning, integrity analysis and reconstructed splitting are handled by later sections of the notebook.

In [ ]:
# DOWNLOAD DATASET DIRECTLY INTO GOOGLE DRIVE
#
# Public Kaggle dataset:
#   fanbyprinciple/iot-device-identification
#
# Final locations expected by the rest of this notebook:
#   /content/drive/MyDrive/Honours_Project/iot_device_train.csv
#   /content/drive/MyDrive/Honours_Project/iot_device_test.csv

!pip install -q kagglehub

import shutil
from pathlib import Path
import kagglehub
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

# CONFIGURATION
DATASET_HANDLE = "fanbyprinciple/iot-device-identification"
PROJECT_DIR = Path("/content/drive/MyDrive/Honours_Project")
TRAIN_DATA_PATH = PROJECT_DIR / "iot_device_train.csv"
TEST_DATA_PATH = PROJECT_DIR / "iot_device_test.csv"

# CREATE PROJECT DIRECTORY IF NECESSARY
PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# CHECK WHETHER DATASET ALREADY EXISTS
if TRAIN_DATA_PATH.exists() and TEST_DATA_PATH.exists():
    print("Dataset already exists in Google Drive.")
else:
    print("Dataset files not found in Google Drive.")
    print("Downloading public Kaggle dataset...")

    # Download into temporary Colab storage first.
    download_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
    print(f"\nKaggle download location:\n{download_path}")

    # Locate required files inside the downloaded dataset.
    train_candidates = list(download_path.rglob("iot_device_train.csv"))
    test_candidates = list(download_path.rglob("iot_device_test.csv"))

    if len(train_candidates) != 1:
        raise FileNotFoundError(
            f"Expected exactly one iot_device_train.csv file, but found {len(train_candidates)}."
        )

    if len(test_candidates) != 1:
        raise FileNotFoundError(
            f"Expected exactly one iot_device_test.csv file, but found {len(test_candidates)}."
        )

    source_train = train_candidates[0]
    source_test = test_candidates[0]

    # Copy the files into the exact Google Drive locations expected by the notebook.
    shutil.copy2(source_train, TRAIN_DATA_PATH)
    shutil.copy2(source_test, TEST_DATA_PATH)
    print("\nDataset copied successfully to Google Drive.")

# FINAL VERIFICATION
if not TRAIN_DATA_PATH.exists():
    raise FileNotFoundError(f"Training dataset was not created:\n{TRAIN_DATA_PATH}")

if not TEST_DATA_PATH.exists():
    raise FileNotFoundError(f"Test dataset was not created:\n{TEST_DATA_PATH}")

print("\nDataset ready.")
print(f"Training dataset:\n{TRAIN_DATA_PATH}")
print(f"\nTest dataset:\n{TEST_DATA_PATH}")

## Exploratory Data Analysis and Dataset Integrity

### Dataset Profiling and Exploratory Analysis

This section performs a comprehensive exploratory analysis of the original publisher-provided training and testing files before model development.

Its main purposes are to:

- characterise the original dataset;
- identify missing values, duplicate observations and class-support differences;
- quantify feature-distribution differences between the original training and testing files;
- inspect correlation structure, dimensionality and distributional overlap;
- verify the reconstructed stratified train/validation/test partition used by the modelling pipeline; and
- generate reproducible CSV reports and figures for the thesis.

The analysis is written to:

`Honours_Project/EDA_Analysis_Results/`

with figures stored under:

`Honours_Project/EDA_Analysis_Results/Figures/`

A consolidated textual report is also written to:

`Dataset_Analysis_Report.txt`

#### Original Publisher Dataset Audit

The original files:

- `iot_device_train.csv`
- `iot_device_test.csv`

are first loaded independently.

The analysis records:

- original dataset dimensions;
- data-type counts;
- missing-value counts;
- duplicate-row counts;
- exact overlap between the original files; and
- class distributions in the original publisher-defined train/test split.

The two source files are also pooled and exact duplicate rows are removed to create the unique combined corpus used later for reconstructed splitting.

#### Zero-Variance Feature Analysis

Numeric predictors with zero variance in the original training file are identified and removed from the working EDA copies of both datasets.

This prevents constant predictors from distorting downstream statistical, correlation and dimensionality-reduction analyses.

The original raw datasets are retained separately so that dataset dimensions, missing values and duplicate statistics can still be reported accurately.

#### Class-Support Analysis

The device-category distribution is examined independently for the original training and testing files.

The analysis identifies:

- classes present in training but absent from testing; and
- classes present in testing but absent from training.

This is important because a publisher-defined test file that does not contain the same class support as the training file cannot provide a complete ten-class evaluation of the final classifier.

The class-distribution results are saved numerically and visualised with side-by-side bar charts.

#### Descriptive Feature Analysis

For every active numeric predictor, the analysis records descriptive statistics for both source files.

Features are also divided into:

- binary features; and
- continuous features.

Additional feature-range information is saved, including:

- training minimum;
- training maximum;
- testing minimum; and
- testing maximum.

These outputs provide a basic description of the scale and behaviour of the network-flow predictors before formal distributional comparison.

#### Statistical Comparison of Original Train and Test Files

Each active numeric feature is compared between the original publisher-defined training and testing datasets using several complementary statistics.

The analysis calculates:

- Kolmogorov-Smirnov statistic and p-value;
- Welch independent-samples t-test;
- Mann-Whitney U test;
- Wasserstein distance;
- standardised Wasserstein distance;
- training and testing means;
- training and testing medians;
- training and testing standard deviations;
- training and testing variances;
- variance ratios; and
- Cohen's \(d\) effect size.

Using multiple measures avoids relying on a single definition of distributional difference.

The Kolmogorov-Smirnov test captures general distributional differences, the t-test focuses on mean differences, the Mann-Whitney test provides a rank-based comparison, Wasserstein distance quantifies distributional displacement, and Cohen's \(d\) provides an interpretable standardised effect-size measure.

#### Multiple-Testing Correction

Because hundreds of predictors are examined simultaneously, raw p-values are supplemented with Benjamini-Hochberg false-discovery-rate correction.

BH-FDR correction is applied separately to:

- Kolmogorov-Smirnov p-values;
- Welch t-test p-values; and
- Mann-Whitney p-values.

For each test, the output records:

- original p-value;
- FDR-adjusted p-value; and
- whether the feature remains significant at an adjusted threshold of 0.05.

This reduces the risk of interpreting large numbers of nominally significant results that arise merely from multiple comparisons.

#### Distribution-Shift Summaries

The code reports the number of features showing:

- raw KS significance;
- raw t-test significance;
- raw Mann-Whitney significance;
- BH-FDR-corrected significance for each test;
- small effect sizes, defined as \(|d| \geq 0.2\);
- medium effect sizes, defined as \(|d| \geq 0.5\); and
- large effect sizes, defined as \(|d| \geq 0.8\).

The largest shifts are also ranked according to:

- KS statistic;
- absolute Cohen's \(d\); and
- standardised Wasserstein distance.

These rankings are saved as CSV files and are used to identify the predictors exhibiting the greatest train/test distribution differences.

The analysis interprets these results specifically as **feature-distribution differences at the original publisher-defined file boundary**. They do not, by themselves, establish pure covariate shift because the analysis does not prove that the conditional relationship \(P(Y \mid X)\) remains unchanged.

#### Distribution Visualisations

Several figures are generated to make the original train/test differences directly inspectable.

These include:

- class-distribution charts;
- top KS-shift bar charts;
- largest Cohen's \(d\) effect sizes;
- largest standardised Wasserstein shifts;
- variance-ratio comparisons;
- histograms for strongly shifted features;
- boxplots;
- violin plots;
- training-versus-testing feature-mean scatter plots; and
- training-versus-testing standard-deviation scatter plots.

The histogram, boxplot and violin figures are generated for the strongest drifting features according to the combined KS/effect-size ranking.

#### Correlation Analysis

Separate Pearson correlation matrices are calculated for the original training and testing datasets.

The analysis saves:

- training correlation matrix;
- testing correlation matrix; and
- absolute train/test correlation-difference matrix.

It also reports the average and maximum absolute correlation difference.

Full correlation heatmaps are produced, together with a more readable heatmap restricted to the 40 highest-variance training features.

These diagnostics are used to inspect whether relationships between predictors differ substantially across the original source-file boundary.

#### Principal Component Analysis

Principal Component Analysis is performed after removing predictors that are constant in the original publisher training file.

The `StandardScaler` and PCA transformation are fitted using the original publisher training file only. The fitted transformations are then applied unchanged to the original publisher testing file.

The analysis saves:

- the complete PCA component-score table;
- a scree plot;
- a cumulative explained-variance curve;
- the number of components needed to explain at least 95% of training variance;
- a two-panel PC1/PC2 source-file projection; and
- a two-panel three-dimensional PCA projection.

The first panel of `PCA_Train_Test` restricts the display to the PC1/PC2 region defined by the training scores. Testing observations inside that region are drawn as hollow markers so that the training observations remain visible.

The second panel displays the complete projection using symmetric-logarithmic axes. This retains the extreme testing observations without compressing the central training cloud into an apparently empty point. The symmetric-log transformation is a display transformation only; it does not alter the fitted PCA scores stored in `PCA_Components.csv`.

The three-dimensional PCA figure follows the same principle: one panel shows the training-defined central region, while the second shows the complete projection using signed-log display coordinates.

These PCA projections are descriptive diagnostics of the original publisher-provided files. They are not used as classifier inputs, inferential tests, feature-selection procedures or evidence that the reconstructed partitions are identical.

#### PCA-Stabilised t-SNE

The t-SNE analysis is performed on the PCA representation rather than directly on the complete high-dimensional feature matrix.

Only the principal components required to explain at least 95% of the training variance are supplied to t-SNE.

The transformation uses:

- two t-SNE dimensions;
- perplexity 30;
- automatic learning-rate selection;
- random initialisation; and
- `random_state=42`.

If more than 2,000 observations are available, a deterministic random subsample of 2,000 observations is used.

The resulting train/test projection is intended as a descriptive visual diagnostic only. It is not used as inferential evidence or as input to any classifier.

#### Dataset Distance, Variance, Outliers and Range Overlap

Additional diagnostics examine how the original source files differ globally and feature by feature.

These include:

- Euclidean distance between standardised train and test centroids;
- training and testing feature variances;
- variance ratios;
- absolute log variance ratios;
- IQR-based outlier counts;
- Z-score outlier counts; and
- feature-range overlap scores.

The range-overlap score compares the shared train/test interval with the union of the observed feature ranges.

These diagnostics provide complementary evidence about how the two source datasets differ beyond simple mean comparisons.

#### Reconstructed Stratified Split Verification

After auditing the original publisher boundary, the two files are pooled and exact duplicate rows are removed.

The unique corpus is then repartitioned using the same deterministic split adopted by the modelling pipeline.

The first stratified split uses:

`test_size=0.20`

with:

`random_state=42`

to create the held-out test set.

The remaining 80% is then stratified again using:

`test_size=0.20`

with the same random state, producing the final approximate:

`64% training / 16% validation / 20% held-out test`

partition.

The reconstructed split is verified by recording:

- unique pooled observations;
- training observations;
- validation observations;
- held-out test observations;
- number of training-derived zero-variance predictors;
- number of retained predictors;
- exact train/validation feature-row overlap;
- exact train/test feature-row overlap;
- exact validation/test feature-row overlap; and
- class distributions in all three reconstructed partitions.

Zero-variance feature removal for this verification is derived from the reconstructed training partition only.

#### Reconstructed Split Distribution Check

For each predictor retained after training-only zero-variance filtering, the reconstructed training partition is compared with:

- the validation partition; and
- the held-out test partition.

Kolmogorov-Smirnov tests are performed for both comparisons, followed by Benjamini-Hochberg FDR correction.

The resulting file:

`Reconstructed_Split_KS_Verification.csv`

records both raw and adjusted KS statistics and significance decisions.

This analysis provides an explicit check that the deterministic reconstructed split does not reproduce the same systematic publisher-boundary separation observed in the original files.

#### Final Dataset Assessment

The final assessment combines:

- class-support differences;
- multiple-testing-corrected feature-distribution differences;
- effect-size evidence; and
- reconstructed-split diagnostics.

Where the original publisher split exhibits class-support mismatch or substantial feature-distribution differences, the analysis recommends:

1. pooling the original source files;
2. removing exact duplicate rows;
3. constructing a new stratified `64/16/20` train/validation/test split; and
4. fitting all learned preprocessing and resampling operations using the training partition only.

This reconstructed partition is the basis used by the subsequent model-development pipeline.

#### Saved Outputs

The EDA section generates a large collection of numerical and graphical artifacts, including:

```text
EDA_Analysis_Results/
├── Dataset_Analysis_Report.txt
├── Basic_Dataset_Information.csv
├── Missing_Value_Summary.csv
├── Duplicate_Summary.csv
├── Class_Distribution.csv
├── Training_Descriptive_Statistics.csv
├── Testing_Descriptive_Statistics.csv
├── Binary_Features.csv
├── Continuous_Features.csv
├── Feature_Ranges.csv
├── Feature_Comparison.csv
├── Top_KS_Shifts.csv
├── Largest_Effect_Sizes.csv
├── Top_Standardized_Wasserstein_Shifts.csv
├── Correlation_Training.csv
├── Correlation_Testing.csv
├── Correlation_Difference.csv
├── PCA_Components.csv
├── PCA_ExplainedVariance.csv
├── FeatureVariance.csv
├── Outliers.csv
├── ZScoreOutliers.csv
├── DistributionOverlap.csv
├── Reconstructed_Class_Distribution.csv
├── Reconstructed_Split_KS_Verification.csv
├── DatasetSummary.csv
└── Figures/
```

The generated figures include the class-distribution, distribution-shift, correlation, PCA, t-SNE, variance and reconstructed-split visualisations used during the dataset-integrity assessment.

#### Methodological Role

This section is a **dataset-analysis and split-validation stage**.

It does not:

- train the final classifiers;
- perform model selection;
- apply SMOTE;
- optimise neural-network hyperparameters;
- evaluate the final saved models; or
- use the held-out reconstructed test partition for model-development decisions.

Its purpose is to establish the empirical justification for replacing the problematic publisher-defined train/test boundary with the deduplicated, stratified and leakage-controlled partition used throughout the final experiments.

In [ ]:
import os
import re
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import (
    ks_2samp,
    zscore,
    ttest_ind,
    mannwhitneyu,
    wasserstein_distance,
)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split

from google.colab import drive

drive.mount('/content/drive')

TRAIN_DATA_PATH = (
    "/content/drive/MyDrive/Honours_Project/"
    "iot_device_train.csv"
)
TEST_DATA_PATH = (
    "/content/drive/MyDrive/Honours_Project/"
    "iot_device_test.csv"
)
OUTPUT_DIR = (
    "/content/drive/MyDrive/Honours_Project/"
    "EDA_Analysis_Results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

REPORT_FILE = os.path.join(
    OUTPUT_DIR,
    "Dataset_Analysis_Report.txt",
)

FIGURE_DIR = os.path.join(
    OUTPUT_DIR,
    "Figures",
)

os.makedirs(FIGURE_DIR, exist_ok=True)

plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300


report = []


def log(text=""):
    print(text)
    report.append(str(text))


def save_report():
    with open(
        REPORT_FILE,
        "w",
        encoding="utf-8",
    ) as file:
        for line in report:
            file.write(str(line) + "\n")


def save_figure(filename):
    plt.tight_layout()

    safe_filename = re.sub(
        r'[\\/*?:"<>|\n\r]',
        "",
        str(filename),
    )

    stem, _ = os.path.splitext(
        safe_filename
    )

    png_path = os.path.join(
        FIGURE_DIR,
        stem + ".png",
    )
    pdf_path = os.path.join(
        FIGURE_DIR,
        stem + ".pdf",
    )

    os.makedirs(
        FIGURE_DIR,
        exist_ok=True,
    )

    plt.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close()


def finite_values(values):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    ).to_numpy(dtype=float)

    return numeric[
        np.isfinite(numeric)
    ]


def signed_log1p(values):
    values = np.asarray(
        values,
        dtype=float,
    )

    return (
        np.sign(values)
        * np.log1p(np.abs(values))
    )


def signed_log10_display(values):
    values = np.asarray(
        values,
        dtype=float,
    )

    return (
        np.sign(values)
        * np.log10(
            1.0 + np.abs(values)
        )
    )


def common_histogram_edges(
    values,
    minimum_bins=20,
    maximum_bins=60,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    if len(values) == 0:
        return np.linspace(
            -1.0,
            1.0,
            minimum_bins + 1,
        )

    minimum = float(
        np.min(values)
    )
    maximum = float(
        np.max(values)
    )

    if minimum == maximum:
        padding = max(
            abs(minimum) * 0.05,
            0.5,
        )

        return np.linspace(
            minimum - padding,
            maximum + padding,
            minimum_bins + 1,
        )

    edges = np.histogram_bin_edges(
        values,
        bins="fd",
    )

    number_of_bins = (
        len(edges) - 1
    )

    if number_of_bins < minimum_bins:
        edges = np.linspace(
            minimum,
            maximum,
            minimum_bins + 1,
        )
    elif number_of_bins > maximum_bins:
        edges = np.linspace(
            minimum,
            maximum,
            maximum_bins + 1,
        )

    return edges


def benjamini_hochberg(
    p_values,
    alpha=0.05,
):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    p_values = np.nan_to_num(
        p_values,
        nan=1.0,
        posinf=1.0,
        neginf=1.0,
    )

    p_values = np.clip(
        p_values,
        0.0,
        1.0,
    )

    order = np.argsort(
        p_values
    )

    ranked_p_values = (
        p_values[order]
    )

    ranks = np.arange(
        1,
        len(p_values) + 1,
    )

    adjusted_ranked = (
        ranked_p_values
        * len(p_values)
        / ranks
    )

    adjusted_ranked = (
        np.minimum.accumulate(
            adjusted_ranked[::-1]
        )[::-1]
    )

    adjusted_ranked = np.clip(
        adjusted_ranked,
        0.0,
        1.0,
    )

    adjusted_p_values = (
        np.empty_like(
            adjusted_ranked
        )
    )

    adjusted_p_values[order] = (
        adjusted_ranked
    )

    rejected = (
        adjusted_p_values <= alpha
    )

    return (
        rejected,
        adjusted_p_values,
    )


log("EXPLORATORY DATA ANALYSIS")
log()
log("Loading datasets...")

train_raw_df = pd.read_csv(
    TRAIN_DATA_PATH
)
test_raw_df = pd.read_csv(
    TEST_DATA_PATH
)

train_df = train_raw_df.copy()
test_df = test_raw_df.copy()

log("Datasets successfully loaded.")
log()


log("=" * 90)
log(
    "SECTION 1 : BASIC INFORMATION "
    "& ZERO-VARIANCE FILTERING"
)
log("=" * 90)

log(
    "Original Training Dataset Shape : "
    f"{train_raw_df.shape}"
)
log(
    "Original Testing Dataset Shape  : "
    f"{test_raw_df.shape}"
)
log()

numeric_cols = (
    train_raw_df
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

train_variances = (
    train_raw_df[numeric_cols]
    .var()
)

zero_var_cols = (
    train_variances[
        train_variances == 0
    ]
    .index
    .tolist()
)

if zero_var_cols:
    log(
        f"Dropping {len(zero_var_cols)} "
        "features with zero variance "
        "(constants)."
    )

    train_df.drop(
        columns=zero_var_cols,
        inplace=True,
    )

    test_df.drop(
        columns=zero_var_cols,
        inplace=True,
        errors="ignore",
    )

    numeric_cols = [
        column
        for column in numeric_cols
        if column not in zero_var_cols
    ]

log(
    f"Active Numeric Features : "
    f"{len(numeric_cols)}"
)
log()

log("Training Data Types")
log(
    train_df
    .dtypes
    .value_counts()
)
log()

log("Testing Data Types")
log(
    test_df
    .dtypes
    .value_counts()
)
log()

basic_stats = pd.DataFrame(
    {
        "Dataset": [
            "Training",
            "Testing",
        ],
        "Rows": [
            train_raw_df.shape[0],
            test_raw_df.shape[0],
        ],
        "Columns": [
            train_raw_df.shape[1],
            test_raw_df.shape[1],
        ],
    }
)

basic_stats.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Basic_Dataset_Information.csv",
    ),
    index=False,
)


log("=" * 90)
log(
    "SECTION 2 & 3 : "
    "MISSING VALUES & DUPLICATES"
)
log("=" * 90)

train_missing = (
    train_raw_df
    .isnull()
    .sum()
)
test_missing = (
    test_raw_df
    .isnull()
    .sum()
)

log(
    "Total Missing Values (Training): "
    f"{train_missing.sum()}"
)
log(
    "Total Missing Values (Testing):  "
    f"{test_missing.sum()}"
)
log()

missing_summary = pd.DataFrame(
    {
        "Feature": (
            train_raw_df.columns
        ),
        "Train Missing": (
            train_missing.values
        ),
        "Test Missing": (
            test_missing.values
        ),
    }
)

missing_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Missing_Value_Summary.csv",
    ),
    index=False,
)

train_duplicates = (
    train_raw_df
    .duplicated()
    .sum()
)
test_duplicates = (
    test_raw_df
    .duplicated()
    .sum()
)

combined_raw_df = pd.concat(
    [
        train_raw_df,
        test_raw_df,
    ],
    axis=0,
    ignore_index=True,
)

combined_unique_df = (
    combined_raw_df
    .drop_duplicates()
    .reset_index(drop=True)
)

pooled_duplicates = (
    len(combined_raw_df)
    - len(combined_unique_df)
)

source_overlap = len(
    pd.merge(
        train_raw_df.drop_duplicates(),
        test_raw_df.drop_duplicates(),
        how="inner",
    )
)

log(
    "Training Duplicate Rows : "
    f"{train_duplicates}"
)
log(
    "Testing Duplicate Rows  : "
    f"{test_duplicates}"
)
log(
    "Duplicate Rows Removed "
    "from Pooled Corpus : "
    f"{pooled_duplicates}"
)
log(
    "Exact Rows Shared Across "
    "Original Files   : "
    f"{source_overlap}"
)
log()

pd.DataFrame(
    {
        "Metric": [
            "Training Duplicate Rows",
            "Testing Duplicate Rows",
            "Pooled Duplicate Rows Removed",
            (
                "Exact Rows Shared Across "
                "Original Files"
            ),
            "Unique Pooled Observations",
        ],
        "Value": [
            train_duplicates,
            test_duplicates,
            pooled_duplicates,
            source_overlap,
            len(combined_unique_df),
        ],
    }
).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Duplicate_Summary.csv",
    ),
    index=False,
)


log("=" * 90)
log("SECTION 4 : CLASS DISTRIBUTION")
log("=" * 90)

train_classes = (
    train_raw_df[
        "device_category"
    ]
    .value_counts()
    .sort_index()
)

test_classes = (
    test_raw_df[
        "device_category"
    ]
    .value_counts()
    .sort_index()
)

log("Training Class Distribution")
log(train_classes)
log()

log("Testing Class Distribution")
log(test_classes)
log()

class_summary = pd.concat(
    [
        train_classes.rename("Train"),
        test_classes.rename("Test"),
    ],
    axis=1,
).fillna(0)

class_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Class_Distribution.csv",
    )
)

train_only = sorted(
    set(train_classes.index)
    - set(test_classes.index)
)

test_only = sorted(
    set(test_classes.index)
    - set(train_classes.index)
)

if len(train_only) > 0:
    log("Classes only in TRAIN")

    for class_name in train_only:
        log(f"   {class_name}")

    log()

if len(test_only) > 0:
    log("Classes only in TEST")

    for class_name in test_only:
        log(f"   {class_name}")

    log()


log("=" * 90)
log(
    "SECTION 5 : DESCRIPTIVE "
    "STATISTICS & FEATURE SUMMARY"
)
log("=" * 90)

train_stats = (
    train_df[numeric_cols]
    .describe()
    .T
)

test_stats = (
    test_df[numeric_cols]
    .describe()
    .T
)

train_stats.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Training_Descriptive_Statistics.csv",
    )
)

test_stats.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Testing_Descriptive_Statistics.csv",
    )
)

binary_features = []
continuous_features = []

for feature in numeric_cols:
    if train_df[feature].nunique() <= 2:
        binary_features.append(feature)
    else:
        continuous_features.append(feature)

log(
    f"Binary Features     : "
    f"{len(binary_features)}"
)
log(
    f"Continuous Features : "
    f"{len(continuous_features)}"
)

pd.DataFrame(
    {
        "Binary Features": pd.Series(
            binary_features
        )
    }
).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Binary_Features.csv",
    ),
    index=False,
)

pd.DataFrame(
    {
        "Continuous Features": pd.Series(
            continuous_features
        )
    }
).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Continuous_Features.csv",
    ),
    index=False,
)

range_summary = []

for feature in numeric_cols:
    range_summary.append(
        {
            "Feature": feature,
            "Train Min": (
                train_df[feature].min()
            ),
            "Train Max": (
                train_df[feature].max()
            ),
            "Test Min": (
                test_df[feature].min()
            ),
            "Test Max": (
                test_df[feature].max()
            ),
        }
    )

pd.DataFrame(
    range_summary
).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Feature_Ranges.csv",
    ),
    index=False,
)

log(
    "Feature range analysis complete.\n"
)
save_report()


log("=" * 90)
log(
    "SECTION 6 : STATISTICAL "
    "FEATURE COMPARISON"
)
log("=" * 90)

comparison_results = []

for feature in numeric_cols:
    train = (
        train_df[feature]
        .dropna()
    )
    test = (
        test_df[feature]
        .dropna()
    )

    ks_stat, ks_p = ks_2samp(
        train,
        test,
    )

    t_stat, t_p = ttest_ind(
        train,
        test,
        equal_var=False,
    )

    u_stat, u_p = mannwhitneyu(
        train,
        test,
        alternative="two-sided",
    )

    w_dist = wasserstein_distance(
        train,
        test,
    )

    train_mean = train.mean()
    test_mean = test.mean()

    train_median = train.median()
    test_median = test.median()

    train_std = train.std()
    test_std = test.std()

    train_var = train.var()
    test_var = test.var()

    pooled_std = np.sqrt(
        (
            train_std ** 2
            + test_std ** 2
        )
        / 2
    )

    cohen_d = (
        0
        if pooled_std == 0
        else (
            train_mean
            - test_mean
        )
        / pooled_std
    )

    standardized_w_dist = (
        0
        if pooled_std == 0
        else w_dist / pooled_std
    )

    comparison_results.append(
        {
            "Feature": feature,
            "Train Mean": train_mean,
            "Test Mean": test_mean,
            "Train Median": train_median,
            "Test Median": test_median,
            "Train Std": train_std,
            "Test Std": test_std,
            "Train Variance": train_var,
            "Test Variance": test_var,
            "Mean Difference": abs(
                train_mean - test_mean
            ),
            "Median Difference": abs(
                train_median - test_median
            ),
            "Std Difference": abs(
                train_std - test_std
            ),
            "Variance Ratio": (
                np.nan
                if train_var == 0
                else test_var / train_var
            ),
            "KS Statistic": ks_stat,
            "KS P Value": ks_p,
            "T Statistic": t_stat,
            "T P Value": t_p,
            "Mann-Whitney P Value": u_p,
            "Wasserstein Distance": w_dist,
            (
                "Standardized "
                "Wasserstein Distance"
            ): standardized_w_dist,
            "Cohens d": cohen_d,
            "Abs Cohens d": abs(cohen_d),
        }
    )

comparison_df = pd.DataFrame(
    comparison_results
)

for (
    source_column,
    adjusted_column,
    rejected_column,
) in [
    (
        "KS P Value",
        "KS FDR P Value",
        "KS FDR Significant",
    ),
    (
        "T P Value",
        "T FDR P Value",
        "T FDR Significant",
    ),
    (
        "Mann-Whitney P Value",
        "Mann-Whitney FDR P Value",
        (
            "Mann-Whitney "
            "FDR Significant"
        ),
    ),
]:
    valid_p_values = (
        comparison_df[source_column]
        .fillna(1.0)
        .clip(0.0, 1.0)
    )

    (
        rejected,
        adjusted_p_values,
    ) = benjamini_hochberg(
        valid_p_values,
        alpha=0.05,
    )

    comparison_df[
        adjusted_column
    ] = adjusted_p_values

    comparison_df[
        rejected_column
    ] = rejected

comparison_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Feature_Comparison.csv",
    ),
    index=False,
)

log(
    "Feature comparison completed.\n"
)


log("=" * 90)
log(
    "SECTION 7 : STATISTICAL "
    "SUMMARY & TOP SHIFTS"
)
log("=" * 90)

total_features = len(
    comparison_df
)

ks_sig = (
    comparison_df["KS P Value"]
    < 0.05
).sum()

ttest_sig = (
    comparison_df["T P Value"]
    < 0.05
).sum()

u_sig = (
    comparison_df[
        "Mann-Whitney P Value"
    ]
    < 0.05
).sum()

ks_fdr_sig = (
    comparison_df[
        "KS FDR Significant"
    ]
    .sum()
)

ttest_fdr_sig = (
    comparison_df[
        "T FDR Significant"
    ]
    .sum()
)

u_fdr_sig = (
    comparison_df[
        "Mann-Whitney "
        "FDR Significant"
    ]
    .sum()
)

large_effect = (
    comparison_df[
        "Abs Cohens d"
    ]
    >= 0.8
).sum()

medium_effect = (
    comparison_df[
        "Abs Cohens d"
    ]
    >= 0.5
).sum()

small_effect = (
    comparison_df[
        "Abs Cohens d"
    ]
    >= 0.2
).sum()

log(
    "Features analysed              : "
    f"{total_features}"
)
log(
    "KS significant                 : "
    f"{ks_sig}"
)
log(
    "T-test significant             : "
    f"{ttest_sig}"
)
log(
    "Mann-Whitney significant       : "
    f"{u_sig}"
)
log(
    "KS significant after BH-FDR    : "
    f"{ks_fdr_sig}"
)
log(
    "T-test significant after BH-FDR: "
    f"{ttest_fdr_sig}"
)
log(
    "Mann-Whitney after BH-FDR      : "
    f"{u_fdr_sig}"
)
log(
    "Large effect size (|d|>=0.8)   : "
    f"{large_effect}"
)
log(
    "Medium effect size (|d|>=0.5)  : "
    f"{medium_effect}"
)
log(
    "Small effect size (|d|>=0.2)   : "
    f"{small_effect}\n"
)

top_ks = (
    comparison_df
    .sort_values(
        "KS Statistic",
        ascending=False,
    )
    .head(25)
)

top_ks.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Top_KS_Shifts.csv",
    ),
    index=False,
)

top_effect = (
    comparison_df
    .sort_values(
        "Abs Cohens d",
        ascending=False,
    )
    .head(25)
)

top_effect.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Largest_Effect_Sizes.csv",
    ),
    index=False,
)

top_w = (
    comparison_df
    .sort_values(
        (
            "Standardized "
            "Wasserstein Distance"
        ),
        ascending=False,
    )
    .head(25)
)

top_w.to_csv(
    os.path.join(
        OUTPUT_DIR,
        (
            "Top_Standardized_"
            "Wasserstein_Shifts.csv"
        ),
    ),
    index=False,
)

log("Top 25 KS Shifts:")
log(
    top_ks[
        [
            "Feature",
            "Train Mean",
            "Test Mean",
            "KS Statistic",
            "KS P Value",
            "Cohens d",
        ]
    ].to_string(index=False)
)

log(
    "\nLargest Effect Sizes "
    "(Cohen's d):"
)

log(
    top_effect[
        [
            "Feature",
            "Cohens d",
            "Train Mean",
            "Test Mean",
        ]
    ].to_string(index=False)
)

log(
    "\nTop 25 Standardized "
    "Wasserstein Shifts:"
)

log(
    top_w[
        [
            "Feature",
            "Train Mean",
            "Test Mean",
            (
                "Standardized "
                "Wasserstein Distance"
            ),
        ]
    ].to_string(index=False)
)

log()

ks_percent = (
    ks_sig
    / total_features
    * 100
)

ks_fdr_percent = (
    ks_fdr_sig
    / total_features
    * 100
)

log("INTERPRETATION:")

if ks_fdr_sig > 0:
    log(
        f"{ks_fdr_sig}/{total_features} "
        "active numeric features "
        f"({ks_fdr_percent:.1f}%) remain"
    )
    log(
        "significantly different after "
        "Benjamini-Hochberg FDR correction."
    )
    log(
        f"{small_effect} features have "
        f"|d|>=0.2, {medium_effect} have "
        f"|d|>=0.5, and"
    )
    log(
        f"{large_effect} have |d|>=0.8. "
        "This is evidence of substantial "
        "feature-"
    )
    log(
        "distribution differences at the "
        "original publisher-defined file "
        "boundary."
    )
else:
    log(
        "No active numeric feature remains "
        "significant after BH-FDR correction."
    )

log(
    "These diagnostics do not establish "
    "pure covariate shift because they do not"
)
log(
    "test whether the conditional "
    "relationship P(Y|X) is unchanged."
)
log()

save_report()


log("=" * 90)
log("SECTION 8 : VISUALISATIONS")
log("=" * 90)

figure = plt.figure(
    figsize=(14, 6)
)

plt.subplot(1, 2, 1)

train_classes.plot(
    kind="bar",
    color="steelblue",
)

plt.title("Training Dataset")
plt.ylabel("Number of Samples")
plt.xticks(
    rotation=45,
    ha="right",
)

plt.subplot(1, 2, 2)

test_classes.plot(
    kind="bar",
    color="darkorange",
)

plt.title("Testing Dataset")
plt.ylabel("Number of Samples")
plt.xticks(
    rotation=45,
    ha="right",
)

save_figure(
    "Class_Distributions.png"
)


for (
    metric,
    filename,
    title,
) in [
    (
        "KS Statistic",
        "Top20_KS_Shift.png",
        (
            "Top 20 Features with "
            "Largest KS Shift"
        ),
    ),
    (
        "Abs Cohens d",
        "Top20_CohensD.png",
        (
            "Largest Effect Sizes "
            "(Cohen's d)"
        ),
    ),
    (
        (
            "Standardized "
            "Wasserstein Distance"
        ),
        (
            "Top20_Standardized_"
            "Wasserstein_Shift.png"
        ),
        (
            "Top 20 Features by "
            "Standardized "
            "Wasserstein Distance"
        ),
    ),
]:
    top20 = (
        comparison_df
        .sort_values(
            metric,
            ascending=False,
        )
        .head(20)
    )

    plt.figure(
        figsize=(12, 8)
    )

    plt.barh(
        top20["Feature"],
        top20[metric],
    )

    plt.gca().invert_yaxis()
    plt.xlabel(metric)
    plt.title(title)

    save_figure(filename)


variance_plot_df = (
    comparison_df[
        [
            "Feature",
            "Variance Ratio",
        ]
    ]
    .copy()
)

variance_plot_df = (
    variance_plot_df[
        np.isfinite(
            variance_plot_df[
                "Variance Ratio"
            ]
        )
        & (
            variance_plot_df[
                "Variance Ratio"
            ]
            > 0
        )
    ]
    .copy()
)

variance_plot_df[
    "Log10 Variance Ratio"
] = np.log10(
    variance_plot_df[
        "Variance Ratio"
    ]
)

variance_plot_df[
    "Absolute Log10 Ratio"
] = abs(
    variance_plot_df[
        "Log10 Variance Ratio"
    ]
)

variance_plot_df = (
    variance_plot_df
    .sort_values(
        "Absolute Log10 Ratio",
        ascending=False,
    )
    .head(20)
    .sort_values(
        "Log10 Variance Ratio"
    )
)

variance_colours = np.where(
    variance_plot_df[
        "Log10 Variance Ratio"
    ]
    >= 0,
    "#E67E22",
    "#2878B5",
)

plt.figure(
    figsize=(12, 8)
)

plt.barh(
    variance_plot_df["Feature"],
    variance_plot_df[
        "Log10 Variance Ratio"
    ],
    color=variance_colours,
    edgecolor="black",
    linewidth=0.4,
)

plt.axvline(
    0,
    color="black",
    linewidth=1.0,
)

plt.xlabel(
    "log10(Test Variance / "
    "Training Variance)"
)
plt.title(
    "Largest Training-Testing "
    "Variance Imbalances"
)
plt.grid(
    axis="x",
    linestyle="--",
    alpha=0.3,
)

save_figure(
    "VarianceRatio.png"
)


top_drift_features = (
    comparison_df
    .sort_values(
        [
            "KS Statistic",
            "Abs Cohens d",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(10)["Feature"]
)

log(
    "Generating scale-aware "
    "distribution plots for top "
    "drifting features..."
)

for feature in top_drift_features:
    train_values = finite_values(
        train_df[feature]
    )
    test_values = finite_values(
        test_df[feature]
    )

    combined_values = np.concatenate(
        [
            train_values,
            test_values,
        ]
    )

    unique_values = np.unique(
        combined_values
    )

    low_cardinality = (
        len(unique_values) <= 10
    )

    if low_cardinality:
        train_percentages = np.array(
            [
                np.mean(
                    train_values == value
                )
                * 100.0
                for value in unique_values
            ]
        )

        test_percentages = np.array(
            [
                np.mean(
                    test_values == value
                )
                * 100.0
                for value in unique_values
            ]
        )

        positions = np.arange(
            len(unique_values)
        )

        width = 0.38

        plt.figure(
            figsize=(10, 5.5)
        )

        plt.bar(
            positions - width / 2,
            train_percentages,
            width=width,
            label="Training",
            color="#2878B5",
            edgecolor="black",
            linewidth=0.5,
        )

        plt.bar(
            positions + width / 2,
            test_percentages,
            width=width,
            label="Testing",
            color="#E67E22",
            edgecolor="black",
            linewidth=0.5,
        )

        plt.xticks(
            positions,
            [
                f"{value:g}"
                for value
                in unique_values
            ],
        )

        plt.xlabel(
            "Observed Value"
        )
        plt.ylabel(
            "Observations (%)"
        )
        plt.title(
            f"{feature}: Empirical "
            "Value Proportions"
        )
        plt.grid(
            axis="y",
            linestyle="--",
            alpha=0.3,
        )
        plt.legend()

        save_figure(
            f"Histogram_{feature}.png"
        )

        plt.figure(
            figsize=(7, 5.5)
        )

        plt.boxplot(
            [
                train_values,
                test_values,
            ],
            labels=[
                "Training",
                "Testing",
            ],
            showmeans=True,
        )

        plt.ylim(
            float(
                np.min(unique_values)
            )
            - 0.05,
            float(
                np.max(unique_values)
            )
            + 0.05,
        )

        plt.ylabel(
            "Observed Value"
        )
        plt.title(
            f"{feature}: Discrete "
            "Distribution Summary"
        )
        plt.grid(
            axis="y",
            linestyle="--",
            alpha=0.3,
        )

        save_figure(
            f"Boxplot_{feature}.png"
        )

        plt.figure(
            figsize=(7, 5.5)
        )

        plt.plot(
            unique_values,
            train_percentages,
            marker="o",
            label="Training",
            color="#2878B5",
        )
        plt.plot(
            unique_values,
            test_percentages,
            marker="s",
            label="Testing",
            color="#E67E22",
        )

        plt.xticks(unique_values)
        plt.xlabel(
            "Observed Value"
        )
        plt.ylabel(
            "Observations (%)"
        )
        plt.title(
            f"{feature}: Discrete "
            "Distribution Profile"
        )
        plt.grid(
            True,
            linestyle="--",
            alpha=0.3,
        )
        plt.legend()

        save_figure(
            f"Violin_{feature}.png"
        )

        continue

    train_display = signed_log1p(
        train_values
    )
    test_display = signed_log1p(
        test_values
    )

    combined_display = np.concatenate(
        [
            train_display,
            test_display,
        ]
    )

    common_bins = (
        common_histogram_edges(
            combined_display
        )
    )

    train_weights = (
        np.ones(
            len(train_display)
        )
        * 100.0
        / len(train_display)
    )

    test_weights = (
        np.ones(
            len(test_display)
        )
        * 100.0
        / len(test_display)
    )

    plt.figure(
        figsize=(10, 5.5)
    )

    plt.hist(
        train_display,
        bins=common_bins,
        weights=train_weights,
        alpha=0.55,
        label="Training",
        color="#2878B5",
        edgecolor="white",
        linewidth=0.4,
    )

    plt.hist(
        test_display,
        bins=common_bins,
        weights=test_weights,
        alpha=0.55,
        label="Testing",
        color="#E67E22",
        edgecolor="white",
        linewidth=0.4,
    )

    plt.xlabel(
        "Signed log1p-transformed value"
    )
    plt.ylabel(
        "Observations per bin (%)"
    )
    plt.title(
        f"{feature}: Scale-Aware "
        "Distribution"
    )
    plt.grid(
        axis="y",
        linestyle="--",
        alpha=0.3,
    )
    plt.legend()

    save_figure(
        f"Histogram_{feature}.png"
    )

    plt.figure(
        figsize=(7, 5.5)
    )

    plt.boxplot(
        [
            train_display,
            test_display,
        ],
        labels=[
            "Training",
            "Testing",
        ],
        showmeans=True,
        showfliers=True,
        flierprops={
            "markersize": 2.5,
            "alpha": 0.35,
        },
    )

    plt.ylabel(
        "Signed log1p-transformed value"
    )
    plt.title(
        f"{feature}: Scale-Aware Boxplot"
    )
    plt.grid(
        axis="y",
        linestyle="--",
        alpha=0.3,
    )

    save_figure(
        f"Boxplot_{feature}.png"
    )

    plt.figure(
        figsize=(7, 5.5)
    )

    violin_parts = plt.violinplot(
        [
            train_display,
            test_display,
        ],
        positions=[1, 2],
        showmeans=True,
        showmedians=True,
        showextrema=True,
        widths=0.8,
    )

    for (
        body,
        colour,
    ) in zip(
        violin_parts["bodies"],
        [
            "#2878B5",
            "#E67E22",
        ],
    ):
        body.set_facecolor(
            colour
        )
        body.set_edgecolor(
            "black"
        )
        body.set_alpha(0.55)

    plt.xticks(
        [1, 2],
        [
            "Training",
            "Testing",
        ],
    )
    plt.ylabel(
        "Signed log1p-transformed value"
    )
    plt.title(
        f"{feature}: Scale-Aware "
        "Violin Plot"
    )
    plt.grid(
        axis="y",
        linestyle="--",
        alpha=0.3,
    )

    save_figure(
        f"Violin_{feature}.png"
    )


for (
    train_column,
    test_column,
    title,
    filename,
    label,
) in [
    (
        "Train Mean",
        "Test Mean",
        (
            "Training and Testing "
            "Feature Means"
        ),
        "FeatureMeans.png",
        "Feature Mean",
    ),
    (
        "Train Std",
        "Test Std",
        (
            "Training and Testing Feature "
            "Standard Deviations"
        ),
        "FeatureStandardDeviation.png",
        "Feature Standard Deviation",
    ),
]:
    x_values = (
        comparison_df[
            train_column
        ]
        .to_numpy(dtype=float)
    )

    y_values = (
        comparison_df[
            test_column
        ]
        .to_numpy(dtype=float)
    )

    valid = (
        np.isfinite(x_values)
        & np.isfinite(y_values)
    )

    x_values = x_values[valid]
    y_values = y_values[valid]

    lower = float(
        min(
            np.min(x_values),
            np.min(y_values),
        )
    )

    upper = float(
        max(
            np.max(x_values),
            np.max(y_values),
        )
    )

    plt.figure(
        figsize=(8, 8)
    )

    plt.scatter(
        x_values,
        y_values,
        s=34,
        alpha=0.65,
        color="#2878B5",
        edgecolor="black",
        linewidth=0.3,
    )

    plt.plot(
        [
            lower,
            upper,
        ],
        [
            lower,
            upper,
        ],
        linestyle="--",
        color="#D62728",
        linewidth=1.4,
        label=(
            "Equal Training/Testing Value"
        ),
    )

    plt.xscale(
        "symlog",
        linthresh=1.0,
    )
    plt.yscale(
        "symlog",
        linthresh=1.0,
    )

    plt.xlabel(
        f"Training {label} "
        "(Symmetric-Log Scale)"
    )
    plt.ylabel(
        f"Testing {label} "
        "(Symmetric-Log Scale)"
    )
    plt.title(title)
    plt.grid(
        True,
        which="both",
        linestyle="--",
        alpha=0.3,
    )
    plt.legend()

    save_figure(filename)


log("=" * 90)
log(
    "SECTION 9 : "
    "CORRELATION ANALYSIS"
)
log("=" * 90)

corr_train = (
    train_df[numeric_cols]
    .corr()
    .fillna(0)
)

corr_test = (
    test_df[numeric_cols]
    .corr()
    .fillna(0)
)

corr_difference = abs(
    corr_train - corr_test
)

corr_train.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Correlation_Training.csv",
    )
)

corr_test.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Correlation_Testing.csv",
    )
)

corr_difference.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Correlation_Difference.csv",
    )
)

log(
    "Average Correlation Difference : "
    f"{corr_difference.values.mean():.4f}"
)
log(
    "Maximum Correlation Difference : "
    f"{corr_difference.values.max():.4f}\n"
)


def plot_legible_heatmap(
    data,
    title,
    filename,
    cmap,
    vmin,
    vmax,
):
    plt.figure(
        figsize=(35, 30)
    )

    mask = np.triu(
        np.ones_like(
            data,
            dtype=bool,
        )
    )

    sns.heatmap(
        data,
        mask=mask,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        xticklabels=data.columns,
        yticklabels=data.columns,
        square=True,
        cbar_kws={
            "shrink": 0.5
        },
        linewidths=0.05,
        linecolor="lightgray",
    )

    plt.title(
        title,
        fontsize=32,
        pad=20,
    )
    plt.xticks(
        rotation=90,
        fontsize=6,
    )
    plt.yticks(
        rotation=0,
        fontsize=6,
    )

    save_figure(filename)


plot_legible_heatmap(
    corr_train,
    "Training Correlation Matrix",
    "Training_Correlation_Matrix.png",
    "viridis",
    -1,
    1,
)

plot_legible_heatmap(
    corr_test,
    "Testing Correlation Matrix",
    "Testing_Correlation_Matrix.png",
    "viridis",
    -1,
    1,
)

plot_legible_heatmap(
    corr_difference,
    "Correlation Difference Matrix",
    "Correlation_Difference.png",
    "magma",
    0,
    1,
)

top_var_cols = (
    train_df[numeric_cols]
    .var()
    .sort_values(
        ascending=False
    )
    .head(40)
    .index
)

corr_train_subset = (
    train_df[
        top_var_cols
    ]
    .corr()
)

plt.figure(
    figsize=(16, 14)
)

mask_subset = np.triu(
    np.ones_like(
        corr_train_subset,
        dtype=bool,
    )
)

sns.heatmap(
    corr_train_subset,
    mask=mask_subset,
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={
        "shrink": 0.5
    },
    xticklabels=(
        corr_train_subset.columns
    ),
    yticklabels=(
        corr_train_subset.columns
    ),
)

plt.title(
    (
        "Correlation Matrix "
        "(Top 40 Highest "
        "Variance Features)"
    ),
    fontsize=16,
    pad=15,
)

plt.xticks(
    rotation=90,
    fontsize=9,
)
plt.yticks(
    rotation=0,
    fontsize=9,
)

save_figure(
    "Training_Correlation_"
    "Subset_Readable.png"
)


log("=" * 90)
log("SECTION 10 : PCA")
log("=" * 90)

scaler = StandardScaler()

train_scaled = scaler.fit_transform(
    train_df[numeric_cols]
)

test_scaled = scaler.transform(
    test_df[numeric_cols]
)

scaled = np.vstack(
    [
        train_scaled,
        test_scaled,
    ]
)

labels = np.array(
    ["Train"] * len(train_df)
    + ["Test"] * len(test_df)
)

pca = PCA()

train_principal_components = (
    pca.fit_transform(
        train_scaled
    )
)

test_principal_components = (
    pca.transform(
        test_scaled
    )
)

principal_components = np.vstack(
    [
        train_principal_components,
        test_principal_components,
    ]
)

explained = (
    pca.explained_variance_ratio_
)

pca_components_df = pd.DataFrame(
    principal_components
)

pca_components_df.insert(
    0,
    "Source Dataset",
    labels,
)

pca_components_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "PCA_Components.csv",
    ),
    index=False,
)


plt.figure(
    figsize=(10, 6)
)

plt.plot(
    np.arange(
        1,
        len(explained) + 1,
    ),
    explained,
    marker="o",
)

plt.xlabel(
    "Principal Component"
)
plt.ylabel(
    "Explained Variance"
)
plt.title(
    "PCA Scree Plot"
)

save_figure(
    "PCA_ScreePlot.png"
)


plt.figure(
    figsize=(10, 6)
)

plt.plot(
    np.cumsum(explained),
    marker="o",
)

plt.axhline(
    y=0.95,
    color="red",
    linestyle="--",
    label=(
        "95% Explained Variance"
    ),
)

plt.xlabel(
    "Principal Components"
)
plt.ylabel(
    "Cumulative Explained Variance"
)
plt.title(
    "Cumulative Explained Variance"
)
plt.legend()
plt.grid(True)

save_figure(
    "PCA_CumulativeVariance.png"
)


mask_train = (
    labels == "Train"
)
mask_test = (
    labels == "Test"
)

training_components = (
    principal_components[
        mask_train
    ]
)

testing_components = (
    principal_components[
        mask_test
    ]
)

training_pc1 = (
    training_components[:, 0]
)
training_pc2 = (
    training_components[:, 1]
)
testing_pc1 = (
    testing_components[:, 0]
)
testing_pc2 = (
    testing_components[:, 1]
)

pc1_min = float(
    np.min(training_pc1)
)
pc1_max = float(
    np.max(training_pc1)
)
pc2_min = float(
    np.min(training_pc2)
)
pc2_max = float(
    np.max(training_pc2)
)

central_testing_mask = (
    (testing_pc1 >= pc1_min)
    & (testing_pc1 <= pc1_max)
    & (testing_pc2 >= pc2_min)
    & (testing_pc2 <= pc2_max)
)

pc1_padding = (
    0.06
    * (pc1_max - pc1_min)
)

pc2_padding = (
    0.06
    * (pc2_max - pc2_min)
)

figure, axes = plt.subplots(
    1,
    2,
    figsize=(12.5, 5.4),
    constrained_layout=True,
)

axes[0].scatter(
    testing_pc1[
        central_testing_mask
    ],
    testing_pc2[
        central_testing_mask
    ],
    s=24,
    facecolors="none",
    edgecolors="#E67E22",
    alpha=0.65,
    linewidths=0.8,
    label=(
        "Testing Inside View "
        f"(n={int(np.sum(central_testing_mask))})"
    ),
    zorder=2,
)

axes[0].scatter(
    training_pc1,
    training_pc2,
    s=13,
    color="#2878B5",
    alpha=0.45,
    linewidths=0,
    label=(
        "Training "
        f"(n={len(training_components)})"
    ),
    zorder=3,
)

axes[0].set_xlim(
    pc1_min - pc1_padding,
    pc1_max + pc1_padding,
)
axes[0].set_ylim(
    pc2_min - pc2_padding,
    pc2_max + pc2_padding,
)
axes[0].set_xlabel(
    "PC1 Score"
)
axes[0].set_ylabel(
    "PC2 Score"
)
axes[0].set_title(
    "A. Central Region Defined "
    "by Training Scores"
)
axes[0].grid(
    alpha=0.2
)
axes[0].legend(
    loc="best"
)

axes[1].scatter(
    testing_pc1,
    testing_pc2,
    s=22,
    facecolors="none",
    edgecolors="#E67E22",
    alpha=0.60,
    linewidths=0.8,
    label=(
        "Testing "
        f"(n={len(testing_components)})"
    ),
    zorder=2,
)

axes[1].scatter(
    training_pc1,
    training_pc2,
    s=13,
    color="#2878B5",
    alpha=0.55,
    linewidths=0,
    label=(
        "Training "
        f"(n={len(training_components)})"
    ),
    zorder=3,
)

axes[1].set_xscale(
    "symlog",
    linthresh=25,
)
axes[1].set_yscale(
    "symlog",
    linthresh=25,
)
axes[1].set_xlabel(
    "PC1 Score "
    "(Symmetric-Log Scale)"
)
axes[1].set_ylabel(
    "PC2 Score "
    "(Symmetric-Log Scale)"
)
axes[1].set_title(
    "B. Full Projection on "
    "Symmetric-Log Axes"
)
axes[1].grid(
    True,
    which="both",
    alpha=0.2,
)
axes[1].legend(
    loc="best"
)

figure.suptitle(
    (
        "Training-Fitted PCA Projection "
        "of the Publisher-Supplied Files"
    ),
    fontsize=13,
    fontweight="bold",
)

save_figure(
    "PCA_Train_Test.png"
)


training_minimums = np.min(
    training_components[:, :3],
    axis=0,
)

training_maximums = np.max(
    training_components[:, :3],
    axis=0,
)

central_testing_3d_mask = np.all(
    (
        testing_components[:, :3]
        >= training_minimums
    )
    & (
        testing_components[:, :3]
        <= training_maximums
    ),
    axis=1,
)

central_testing_3d = (
    testing_components[
        central_testing_3d_mask,
        :3,
    ]
)

display_training_3d = (
    signed_log10_display(
        training_components[:, :3]
    )
)

display_testing_3d = (
    signed_log10_display(
        testing_components[:, :3]
    )
)

figure = plt.figure(
    figsize=(13.5, 6.2)
)

central_axis = figure.add_subplot(
    1,
    2,
    1,
    projection="3d",
)

full_axis = figure.add_subplot(
    1,
    2,
    2,
    projection="3d",
)

central_axis.scatter(
    central_testing_3d[:, 0],
    central_testing_3d[:, 1],
    central_testing_3d[:, 2],
    s=18,
    facecolors="none",
    edgecolors="#E67E22",
    alpha=0.65,
    linewidths=0.7,
    label=(
        "Testing Inside Training Bounds "
        f"(n={len(central_testing_3d)})"
    ),
)

central_axis.scatter(
    training_components[:, 0],
    training_components[:, 1],
    training_components[:, 2],
    s=10,
    color="#2878B5",
    alpha=0.35,
    linewidths=0,
    label=(
        "Training "
        f"(n={len(training_components)})"
    ),
)

central_axis.set_xlim(
    training_minimums[0],
    training_maximums[0],
)
central_axis.set_ylim(
    training_minimums[1],
    training_maximums[1],
)
central_axis.set_zlim(
    training_minimums[2],
    training_maximums[2],
)

central_axis.set_xlabel("PC1")
central_axis.set_ylabel("PC2")
central_axis.set_zlabel("PC3")
central_axis.set_title(
    "A. Training-Defined "
    "Central Region"
)
central_axis.legend(
    loc="upper left",
    fontsize=7,
)

full_axis.scatter(
    display_testing_3d[:, 0],
    display_testing_3d[:, 1],
    display_testing_3d[:, 2],
    s=16,
    facecolors="none",
    edgecolors="#E67E22",
    alpha=0.55,
    linewidths=0.7,
    label=(
        "Testing "
        f"(n={len(testing_components)})"
    ),
)

full_axis.scatter(
    display_training_3d[:, 0],
    display_training_3d[:, 1],
    display_training_3d[:, 2],
    s=10,
    color="#2878B5",
    alpha=0.45,
    linewidths=0,
    label=(
        "Training "
        f"(n={len(training_components)})"
    ),
)

full_axis.set_xlabel(
    "sign(PC1) log10(1 + |PC1|)"
)
full_axis.set_ylabel(
    "sign(PC2) log10(1 + |PC2|)"
)
full_axis.set_zlabel(
    "sign(PC3) log10(1 + |PC3|)"
)
full_axis.set_title(
    "B. Full Projection on "
    "Transformed Display Axes"
)
full_axis.legend(
    loc="upper left",
    fontsize=7,
)

figure.suptitle(
    (
        "Training-Fitted "
        "Three-Dimensional "
        "PCA Projection"
    ),
    fontsize=13,
    fontweight="bold",
)

figure.tight_layout(
    rect=[0, 0, 1, 0.95]
)

save_figure(
    "PCA3D.png"
)


variance_summary = pd.DataFrame(
    {
        "Component": np.arange(
            1,
            len(explained) + 1,
        ),
        "Explained Variance": explained,
        "Cumulative": np.cumsum(
            explained
        ),
    }
)

variance_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "PCA_ExplainedVariance.csv",
    ),
    index=False,
)

log(
    f"PC1 Variance : "
    f"{explained[0]:.4f}"
)
log(
    f"PC2 Variance : "
    f"{explained[1]:.4f}"
)
log(
    f"PC3 Variance : "
    f"{explained[2]:.4f}"
)

components_95 = (
    np.argmax(
        np.cumsum(explained)
        >= 0.95
    )
    + 1
)

log(
    "Components required for "
    "95% variance: "
    f"{components_95}"
)
log(
    "Cumulative variance at "
    f"{components_95} components: "
    f"{np.cumsum(explained)[components_95 - 1]:.4f}\n"
)


log("=" * 90)
log(
    "SECTION 11 : "
    "PCA-STABILIZED t-SNE"
)
log("=" * 90)

MAX_TSNE_SAMPLES = 2000

if (
    scaled.shape[0]
    > MAX_TSNE_SAMPLES
):
    random_generator = (
        np.random.default_rng(42)
    )

    selected_indices = (
        random_generator.choice(
            scaled.shape[0],
            MAX_TSNE_SAMPLES,
            replace=False,
        )
    )

    tsne_data = (
        principal_components[
            selected_indices,
            :components_95,
        ]
    )

    tsne_labels = (
        labels[
            selected_indices
        ]
    )
else:
    tsne_data = (
        principal_components[
            :,
            :components_95,
        ]
    )
    tsne_labels = labels

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="random",
    random_state=42,
)

tsne_embedding = (
    tsne.fit_transform(
        tsne_data
    )
)

plt.figure(
    figsize=(10, 8)
)

train_mask = (
    tsne_labels == "Train"
)
test_mask = (
    tsne_labels == "Test"
)

plt.scatter(
    tsne_embedding[
        train_mask,
        0,
    ],
    tsne_embedding[
        train_mask,
        1,
    ],
    s=20,
    alpha=0.6,
    label="Training",
)

plt.scatter(
    tsne_embedding[
        test_mask,
        0,
    ],
    tsne_embedding[
        test_mask,
        1,
    ],
    s=20,
    alpha=0.6,
    label="Testing",
)

plt.legend()
plt.title(
    "t-SNE Projection "
    f"(PCA-Stabilized Top {components_95})"
)

save_figure(
    "TSNE_Projection.png"
)

log("t-SNE complete.\n")


log("=" * 90)
log(
    "SECTION 12 : DATASET DISTANCE, "
    "VARIANCE, OUTLIERS, OVERLAP"
)
log("=" * 90)

train_centroid = (
    train_scaled.mean(axis=0)
)
test_centroid = (
    test_scaled.mean(axis=0)
)

centroid_distance = np.linalg.norm(
    train_centroid
    - test_centroid
)

log(
    "Euclidean distance between "
    "centroids : "
    f"{centroid_distance:.4f}\n"
)

variance_df = pd.DataFrame(
    {
        "Feature": numeric_cols,
        "Training Variance": (
            train_df[numeric_cols]
            .var()
            .values
        ),
        "Testing Variance": (
            test_df[numeric_cols]
            .var()
            .values
        ),
    }
)

variance_df[
    "Absolute Difference"
] = abs(
    variance_df[
        "Training Variance"
    ]
    - variance_df[
        "Testing Variance"
    ]
)

variance_df[
    "Variance Ratio"
] = (
    variance_df[
        "Testing Variance"
    ]
    / variance_df[
        "Training Variance"
    ]
)

variance_df[
    "Absolute Log Variance Ratio"
] = abs(
    np.log(
        variance_df[
            "Variance Ratio"
        ].replace(0, np.nan)
    )
)

variance_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "FeatureVariance.csv",
    ),
    index=False,
)

top_variance = (
    variance_df
    .sort_values(
        "Absolute Log Variance Ratio",
        ascending=False,
    )
    .head(20)
)

plt.figure(
    figsize=(12, 8)
)

plt.barh(
    top_variance["Feature"],
    top_variance[
        "Absolute Log Variance Ratio"
    ],
)

plt.gca().invert_yaxis()
plt.xlabel(
    "Absolute Natural-Log "
    "Variance Ratio"
)
plt.title(
    "Largest Standardized "
    "Variance Differences"
)

save_figure(
    "StandardizedVarianceDifference.png"
)


outlier_summary = []
z_results = []

for feature in numeric_cols:
    q1_train = (
        train_df[feature]
        .quantile(0.25)
    )
    q3_train = (
        train_df[feature]
        .quantile(0.75)
    )

    iqr_train = (
        q3_train - q1_train
    )

    train_outliers = (
        (
            train_df[feature]
            < q1_train
            - 1.5 * iqr_train
        )
        | (
            train_df[feature]
            > q3_train
            + 1.5 * iqr_train
        )
    ).sum()

    q1_test = (
        test_df[feature]
        .quantile(0.25)
    )
    q3_test = (
        test_df[feature]
        .quantile(0.75)
    )

    iqr_test = (
        q3_test - q1_test
    )

    test_outliers = (
        (
            test_df[feature]
            < q1_test
            - 1.5 * iqr_test
        )
        | (
            test_df[feature]
            > q3_test
            + 1.5 * iqr_test
        )
    ).sum()

    outlier_summary.append(
        {
            "Feature": feature,
            "Training Outliers": (
                train_outliers
            ),
            "Testing Outliers": (
                test_outliers
            ),
        }
    )

    train_z = np.abs(
        zscore(
            train_df[feature]
        )
    )

    test_z = np.abs(
        zscore(
            test_df[feature]
        )
    )

    z_results.append(
        {
            "Feature": feature,
            "Train Z-Outliers": (
                train_z > 3
            ).sum(),
            "Test Z-Outliers": (
                test_z > 3
            ).sum(),
        }
    )

pd.DataFrame(
    outlier_summary
).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Outliers.csv",
    ),
    index=False,
)

pd.DataFrame(
    z_results
).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ZScoreOutliers.csv",
    ),
    index=False,
)

outlier_df = pd.DataFrame(
    outlier_summary
)

log(
    "Top Outlier Features "
    "(by Testing Outliers):"
)

log(
    outlier_df
    .sort_values(
        "Testing Outliers",
        ascending=False,
    )
    .head(20)
    .to_string(index=False)
)

log()

overlap = []

for feature in numeric_cols:
    train_min = (
        train_df[feature]
        .min()
    )
    train_max = (
        train_df[feature]
        .max()
    )
    test_min = (
        test_df[feature]
        .min()
    )
    test_max = (
        test_df[feature]
        .max()
    )

    intersection = max(
        0,
        min(
            train_max,
            test_max,
        )
        - max(
            train_min,
            test_min,
        ),
    )

    union = (
        max(
            train_max,
            test_max,
        )
        - min(
            train_min,
            test_min,
        )
    )

    score = (
        0
        if union == 0
        else intersection / union
    )

    overlap.append(
        {
            "Feature": feature,
            "Overlap": score,
        }
    )

overlap_df = pd.DataFrame(
    overlap
)

overlap_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "DistributionOverlap.csv",
    ),
    index=False,
)

log(
    "Features with Smallest "
    "Distribution Overlap:"
)

log(
    overlap_df
    .sort_values("Overlap")
    .head(20)
    .to_string(index=False)
)

log()

log(
    "Outlier and overlap "
    "calculations complete.\n"
)


log("=" * 90)
log(
    "SECTION 13 : RECONSTRUCTED "
    "SPLIT VERIFICATION"
)
log("=" * 90)

X_reconstructed = (
    combined_unique_df.drop(
        "device_category",
        axis=1,
    )
)

y_reconstructed = (
    combined_unique_df[
        "device_category"
    ]
)

(
    X_temp_reconstructed,
    X_test_reconstructed,
    y_temp_reconstructed,
    y_test_reconstructed,
) = train_test_split(
    X_reconstructed,
    y_reconstructed,
    test_size=0.20,
    random_state=42,
    stratify=y_reconstructed,
)

(
    X_train_reconstructed,
    X_val_reconstructed,
    y_train_reconstructed,
    y_val_reconstructed,
) = train_test_split(
    X_temp_reconstructed,
    y_temp_reconstructed,
    test_size=0.20,
    random_state=42,
    stratify=y_temp_reconstructed,
)

reconstructed_train_variances = (
    X_train_reconstructed.var()
)

reconstructed_zero_var_cols = (
    reconstructed_train_variances[
        reconstructed_train_variances
        == 0
    ]
    .index
    .tolist()
)

reconstructed_active_cols = [
    column
    for column
    in X_train_reconstructed.columns
    if column
    not in reconstructed_zero_var_cols
]


def feature_row_hashes(dataframe):
    return set(
        pd.util.hash_pandas_object(
            dataframe,
            index=False,
        ).tolist()
    )


train_hashes = feature_row_hashes(
    X_train_reconstructed
)

validation_hashes = (
    feature_row_hashes(
        X_val_reconstructed
    )
)

test_hashes = feature_row_hashes(
    X_test_reconstructed
)

train_validation_overlap = len(
    train_hashes.intersection(
        validation_hashes
    )
)

train_test_overlap = len(
    train_hashes.intersection(
        test_hashes
    )
)

validation_test_overlap = len(
    validation_hashes.intersection(
        test_hashes
    )
)

log(
    "Unique pooled observations : "
    f"{len(combined_unique_df)}"
)
log(
    "Training observations      : "
    f"{len(X_train_reconstructed)}"
)
log(
    "Validation observations    : "
    f"{len(X_val_reconstructed)}"
)
log(
    "Testing observations       : "
    f"{len(X_test_reconstructed)}"
)
log(
    "Training-derived zero-variance "
    "features : "
    f"{len(reconstructed_zero_var_cols)}"
)
log(
    "Retained predictors             "
    "       : "
    f"{len(reconstructed_active_cols)}"
)
log(
    "Train/Validation exact feature "
    "overlap  : "
    f"{train_validation_overlap}"
)
log(
    "Train/Test exact feature overlap "
    "       : "
    f"{train_test_overlap}"
)
log(
    "Validation/Test exact feature "
    "overlap   : "
    f"{validation_test_overlap}"
)
log()

reconstructed_classes = pd.concat(
    [
        (
            y_train_reconstructed
            .value_counts()
            .sort_index()
            .rename("Train")
        ),
        (
            y_val_reconstructed
            .value_counts()
            .sort_index()
            .rename("Validation")
        ),
        (
            y_test_reconstructed
            .value_counts()
            .sort_index()
            .rename("Test")
        ),
    ],
    axis=1,
).fillna(0).astype(int)

reconstructed_classes.to_csv(
    os.path.join(
        OUTPUT_DIR,
        (
            "Reconstructed_Class_"
            "Distribution.csv"
        ),
    )
)

log(
    "Reconstructed "
    "Class Distribution"
)
log(reconstructed_classes)
log()

reconstructed_proportions = (
    reconstructed_classes.div(
        reconstructed_classes.sum(
            axis=0
        ),
        axis=1,
    )
)

max_class_proportion_difference = (
    (
        reconstructed_proportions
        .max(axis=1)
        - reconstructed_proportions
        .min(axis=1)
    )
    .max()
)

log(
    "Maximum class-proportion "
    "difference across reconstructed "
    "partitions : "
    f"{max_class_proportion_difference:.4f}"
)
log()

reconstructed_ks_results = []

for feature in reconstructed_active_cols:
    train_values = (
        X_train_reconstructed[
            feature
        ]
        .dropna()
    )

    validation_values = (
        X_val_reconstructed[
            feature
        ]
        .dropna()
    )

    test_values = (
        X_test_reconstructed[
            feature
        ]
        .dropna()
    )

    (
        train_validation_ks,
        train_validation_p,
    ) = ks_2samp(
        train_values,
        validation_values,
    )

    (
        train_test_ks,
        train_test_p,
    ) = ks_2samp(
        train_values,
        test_values,
    )

    reconstructed_ks_results.append(
        {
            "Feature": feature,
            (
                "Train-Validation "
                "KS Statistic"
            ): train_validation_ks,
            (
                "Train-Validation "
                "KS P Value"
            ): train_validation_p,
            (
                "Train-Test "
                "KS Statistic"
            ): train_test_ks,
            (
                "Train-Test "
                "KS P Value"
            ): train_test_p,
        }
    )

reconstructed_ks_df = pd.DataFrame(
    reconstructed_ks_results
)

for (
    source_column,
    adjusted_column,
    rejected_column,
) in [
    (
        (
            "Train-Validation "
            "KS P Value"
        ),
        (
            "Train-Validation "
            "KS FDR P Value"
        ),
        (
            "Train-Validation "
            "KS FDR Significant"
        ),
    ),
    (
        "Train-Test KS P Value",
        "Train-Test KS FDR P Value",
        (
            "Train-Test "
            "KS FDR Significant"
        ),
    ),
]:
    valid_p_values = (
        reconstructed_ks_df[
            source_column
        ]
        .fillna(1.0)
        .clip(0.0, 1.0)
    )

    (
        rejected,
        adjusted_p_values,
    ) = benjamini_hochberg(
        valid_p_values,
        alpha=0.05,
    )

    reconstructed_ks_df[
        adjusted_column
    ] = adjusted_p_values

    reconstructed_ks_df[
        rejected_column
    ] = rejected

reconstructed_ks_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        (
            "Reconstructed_Split_"
            "KS_Verification.csv"
        ),
    ),
    index=False,
)

reconstructed_ks_validation_fdr_sig = (
    reconstructed_ks_df[
        (
            "Train-Validation "
            "KS FDR Significant"
        )
    ]
    .sum()
)

reconstructed_ks_test_fdr_sig = (
    reconstructed_ks_df[
        (
            "Train-Test "
            "KS FDR Significant"
        )
    ]
    .sum()
)

log(
    "Train/Validation KS differences "
    "after BH-FDR : "
    f"{reconstructed_ks_validation_fdr_sig}/"
    f"{len(reconstructed_active_cols)}"
)

log(
    "Train/Test KS differences "
    "after BH-FDR       : "
    f"{reconstructed_ks_test_fdr_sig}/"
    f"{len(reconstructed_active_cols)}"
)

log()

reconstructed_classes.plot(
    kind="bar",
    figsize=(14, 7),
)

plt.title(
    "Class Distribution After "
    "Deduplication and "
    "Stratified Reconstruction"
)
plt.ylabel(
    "Number of Samples"
)
plt.xticks(
    rotation=45,
    ha="right",
)
plt.legend(
    title="Partition"
)

save_figure(
    "Reconstructed_Class_"
    "Distributions.png"
)


log("=" * 90)
log(
    "SECTION 14 : FINAL DATASET "
    "ASSESSMENT & RECOMMENDATION"
)
log("=" * 90)

train_classes_set = set(
    train_classes.index
)
test_classes_set = set(
    test_classes.index
)

missing_test = sorted(
    train_classes_set
    - test_classes_set
)

missing_train = sorted(
    test_classes_set
    - train_classes_set
)

if len(missing_test):
    log(
        "Classes present in TRAIN "
        "but absent from TEST:"
    )

    for class_name in missing_test:
        log(
            f"   - {class_name}"
        )
else:
    log(
        "All training classes "
        "appear in testing."
    )

if len(missing_train):
    log(
        "\nClasses present in TEST "
        "but absent from TRAIN:"
    )

    for class_name in missing_train:
        log(
            f"   - {class_name}"
        )

log()

if (
    len(missing_test) > 0
    or len(missing_train) > 0
):
    assessment = (
        "FEATURE-DISTRIBUTION "
        "DIFFERENCES WITH "
        "CLASS-SUPPORT MISMATCH"
    )
elif ks_fdr_sig > 0:
    assessment = (
        "FEATURE-DISTRIBUTION "
        "DIFFERENCES DETECTED"
    )
else:
    assessment = (
        "NO FEATURE-DISTRIBUTION "
        "DIFFERENCE DETECTED "
        "AFTER BH-FDR"
    )

log("=" * 90)
log("PCA OBSERVATIONS")
log("=" * 90)

log(
    "Review PCA_Train_Test.png "
    "or PCA_Train_Test.pdf."
)
log(
    "Panel A displays the central "
    "region defined by the original "
    "training scores."
)
log(
    "Panel B displays all observations "
    "using symmetric-log axes so that "
    "extreme testing scores do not hide "
    "the training observations."
)
log(
    "The PCA projection is descriptive "
    "evidence of differences at the "
    "original publisher boundary."
)
log(
    "PCA does not independently quantify "
    "shift severity or establish pure "
    "covariate shift."
)
log()

log("=" * 90)
log("t-SNE OBSERVATIONS")
log("=" * 90)

log(
    "Review TSNE_Projection.png "
    "or TSNE_Projection.pdf."
)
log(
    "Distinct Train/Test regions indicate "
    "feature-distribution differences, "
    "but t-SNE is a diagnostic "
    "visualisation only."
)
log(
    "It is not used as inferential "
    "evidence or as model input."
)
log()

log("=" * 90)
log("RECOMMENDATION")
log("=" * 90)

log(
    f"Dataset Assessment : "
    f"{assessment}\n"
)

if (
    len(missing_test) > 0
    or len(missing_train) > 0
    or ks_fdr_sig > 0
):
    log(
        "Based on the statistical analysis, "
        "the original training/testing split"
    )
    log(
        "contains publisher-boundary "
        "feature-distribution differences "
        "and is unsuitable for"
    )
    log(
        "the final ten-class evaluation."
    )
    log(
        "\nThe two source files should be "
        "pooled, exact duplicate rows removed,"
    )
    log(
        "and a new stratified 64/16/20 "
        "train/validation/test split "
        "constructed."
    )
    log(
        "All learned preprocessing and "
        "resampling must then remain "
        "training-only."
    )
else:
    log(
        "The original split appears "
        "appropriate for evaluation."
    )

log()

dataset_summary = pd.DataFrame(
    {
        "Metric": [
            "Original Training Samples",
            "Original Testing Samples",
            (
                "Active Original "
                "Numeric Features"
            ),
            "KS Significant",
            (
                "KS Significant "
                "After BH-FDR"
            ),
            "T-test Significant",
            (
                "T-test Significant "
                "After BH-FDR"
            ),
            "Mann-Whitney Significant",
            (
                "Mann-Whitney Significant "
                "After BH-FDR"
            ),
            "Average KS",
            "Average Absolute Cohen d",
            "Small Effect Count",
            "Medium Effect Count",
            "Large Effect Count",
            "Unique Pooled Observations",
            (
                "Reconstructed "
                "Training Samples"
            ),
            (
                "Reconstructed "
                "Validation Samples"
            ),
            (
                "Reconstructed "
                "Testing Samples"
            ),
            (
                "Reconstructed "
                "Retained Predictors"
            ),
            (
                "Reconstructed Train-Test "
                "Duplicate Overlap"
            ),
            (
                "Reconstructed Train-Test "
                "KS Significant "
                "After BH-FDR"
            ),
            "Dataset Assessment",
        ],
        "Value": [
            len(train_raw_df),
            len(test_raw_df),
            len(numeric_cols),
            ks_sig,
            ks_fdr_sig,
            ttest_sig,
            ttest_fdr_sig,
            u_sig,
            u_fdr_sig,
            (
                comparison_df[
                    "KS Statistic"
                ]
                .mean()
            ),
            (
                comparison_df[
                    "Abs Cohens d"
                ]
                .mean()
            ),
            small_effect,
            medium_effect,
            large_effect,
            len(combined_unique_df),
            len(
                X_train_reconstructed
            ),
            len(
                X_val_reconstructed
            ),
            len(
                X_test_reconstructed
            ),
            len(
                reconstructed_active_cols
            ),
            train_test_overlap,
            reconstructed_ks_test_fdr_sig,
            assessment,
        ],
    }
)

dataset_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "DatasetSummary.csv",
    ),
    index=False,
)

log("Generated Files")

for (
    root,
    directories,
    files,
) in os.walk(OUTPUT_DIR):
    for filename in sorted(files):
        log(
            "  " + filename
        )

log()

log("=" * 90)
log("END OF ANALYSIS")
log("=" * 90)

save_report()

### Publisher Split Shift Analysis

This section performs a focused statistical diagnostic of the **original publisher-defined training and testing files** to determine whether they exhibit meaningful differences in their predictor distributions and class support.

The analysis is saved under:

`Honours_Project/Covariate_Shift_Testing/`

Its purpose is not to train the final classifier. Instead, it provides independent evidence about whether the original source-file boundary is suitable for final model evaluation.

#### Predictor Preparation

The original files:

- `iot_device_train.csv`
- `iot_device_test.csv`

are loaded separately.

The target column:

`device_category`

is excluded from the predictor matrix.

Only predictors that appear in both files are considered. Predictor values are coerced to numeric form, and infinite values are treated as missing.

Features that are entirely missing in both files are excluded.

The diagnostic then distinguishes between:

- predictors that vary in the original training file; and
- predictors that are constant in the original training file but vary in the original testing file.

The latter are saved separately because they provide additional evidence that the two publisher-defined files do not share identical predictor support.

#### Univariate Distribution-Shift Tests

For each active predictor, a two-sample Kolmogorov-Smirnov test compares the original training-file distribution with the original testing-file distribution.

For every tested feature, the code records:

- KS statistic;
- raw KS p-value;
- source-file missing-value rate; and
- target-file missing-value rate.

Because many predictors are tested simultaneously, the p-values are adjusted using the Benjamini-Hochberg false-discovery-rate procedure.

The resulting output records:

- raw KS p-value;
- BH-adjusted p-value; and
- whether the feature remains significant at:

`alpha = 0.05`

The full univariate results are written to:

`univariate_covariate_shift_tests.csv`

Features that are constant in the original training file but variable in the original testing file are written to:

`source_constant_target_variable_features.csv`

#### Multivariate Domain-Classifier Diagnostic

A second diagnostic tests whether the two publisher-defined files can be distinguished jointly from their predictor values.

A binary domain label is created:

- `0` = original publisher training file;
- `1` = original publisher testing file.

A logistic-regression pipeline is then trained to predict this domain label from the predictors.

The pipeline contains:

1. median imputation with missing-value indicators;
2. standard scaling; and
3. class-balanced logistic regression.

The domain classifier is evaluated using repeated stratified cross-validation:

`5 folds × 5 repeats`

with:

`random_state = 42`

The principal metric is ROC-AUC.

If the two files were difficult to distinguish from their predictors, the domain-classifier ROC-AUC would be expected to remain near 0.5.

A materially higher ROC-AUC indicates that the two files occupy distinguishable regions of predictor space.

#### Permutation Significance Test

The domain-classifier result is supplemented with a permutation test.

A stratified five-fold cross-validation scheme is used with:

`200 label permutations`

The permutation procedure compares the observed domain-classifier ROC-AUC against the distribution obtained when the source/test domain labels are randomly reassigned.

The outputs include:

- observed domain-classifier ROC-AUC;
- permutation ROC-AUC scores; and
- permutation p-value.

The repeated cross-validation scores are saved to:

`domain_classifier_auc_folds.csv`

The permutation scores are saved to:

`domain_classifier_permutation_scores.csv`

#### Class-Support Analysis

The target labels from the two publisher-defined files are analysed separately.

For every device class, the code records:

- number of observations in the original training file;
- number of observations in the original testing file;
- training-file class proportion; and
- testing-file class proportion.

The resulting table is saved to:

`class_support_and_proportions.csv`

The diagnostic also explicitly identifies:

- classes present in the training file but absent from the testing file; and
- classes present in the testing file but absent from the training file.

This distinction is important because class-support mismatch is separate from predictor-distribution shift.

#### Summary Visualisation

A two-panel figure is generated and saved in both raster and vector formats:

- `publisher_boundary_shift_summary.png`
- `publisher_boundary_shift_summary.pdf`

The first panel displays the 20 predictors with the largest KS statistics.

Features that remain significant after Benjamini-Hochberg correction are visually distinguished from those that do not.

The second panel compares the class proportions in the original training and testing files.

Together, the two panels summarise:

- feature-distribution differences; and
- class-support/proportion differences.

#### Interpretation

The statistical tests are interpreted conservatively.

The univariate KS tests assess differences in:

`P(X)`

for individual predictors.

The domain classifier assesses whether the two files can be distinguished jointly using their predictors.

A domain-classifier ROC-AUC near 0.5 indicates little detectable multivariate separation between the files.

A materially higher ROC-AUC, together with a significant permutation result, indicates detectable input-distribution shift.

These diagnostics do **not** establish pure covariate shift by themselves.

Pure covariate shift additionally requires the conditional relationship:

`P(Y|X)`

to remain stable between domains, which is not directly tested here.

If one of the original files also lacks one or more device classes, the appropriate description is:

> Dataset shift with feature-distribution and class-support differences.

The complete publisher-defined boundary should therefore not be described as pure covariate shift.

#### Saved Outputs

The section writes the following main artifacts:

```text
Covariate_Shift_Testing/
├── univariate_covariate_shift_tests.csv
├── source_constant_target_variable_features.csv
├── domain_classifier_auc_folds.csv
├── domain_classifier_permutation_scores.csv
├── class_support_and_proportions.csv
├── publisher_boundary_shift_summary.png
├── publisher_boundary_shift_summary.pdf
└── covariate_shift_report.txt
```

The final text report summarises:

- source and target row counts;
- number of active predictors tested;
- number of BH-significant KS tests;
- repeated-CV domain-classifier ROC-AUC;
- observed five-fold ROC-AUC;
- permutation p-value;
- classes missing from the testing file;
- classes missing from the training file; and
- the recommended terminology for describing the observed dataset differences.

#### Methodological Role

This section is a **dataset-shift diagnostic**.

It does not:

- train the final appliance classifier;
- select hyperparameters;
- perform neural model selection;
- evaluate the reconstructed held-out test set; or
- prove pure covariate shift.

Its role is to quantify and document why the original publisher-defined training/testing boundary is problematic and why the project instead uses a pooled, deduplicated and stratified reconstructed split for the final modelling pipeline.

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_score, permutation_test_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from google.colab import drive

drive.mount('/content/drive')
TRAIN_DATA_PATH = "/content/drive/MyDrive/Honours_Project/iot_device_train.csv"
TEST_DATA_PATH = "/content/drive/MyDrive/Honours_Project/iot_device_test.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Honours_Project/Covariate_Shift_Testing"
TARGET = "device_category"
ALPHA = 0.05
RANDOM_STATE = 42
N_PERMUTATIONS = 200

os.makedirs(OUTPUT_DIR, exist_ok=True)


def benjamini_hochberg(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted_ranked = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted_ranked = np.minimum.accumulate(adjusted_ranked[::-1])[::-1]
    adjusted = np.empty_like(adjusted_ranked)
    adjusted[order] = np.clip(adjusted_ranked, 0.0, 1.0)
    return adjusted


source_df = pd.read_csv(TRAIN_DATA_PATH)
target_df = pd.read_csv(TEST_DATA_PATH)

if TARGET not in source_df.columns or TARGET not in target_df.columns:
    raise KeyError(f"Both files must contain the target column '{TARGET}'.")

common_predictors = [col for col in source_df.columns if col != TARGET and col in target_df.columns]

X_source = source_df[common_predictors].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
X_target = target_df[common_predictors].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)

numeric_predictors = [f for f in common_predictors if not (X_source[f].isna().all() and X_target[f].isna().all())]
active_predictors = [f for f in numeric_predictors if X_source[f].nunique(dropna=True) > 1]
source_constant_target_variable = [
    f for f in numeric_predictors if f not in active_predictors and X_target[f].nunique(dropna=True) > 1
]

X_source = X_source[active_predictors]
X_target = X_target[active_predictors]

feature_rows = []
for feature in active_predictors:
    source_values = X_source[feature].dropna().to_numpy(dtype=float)
    target_values = X_target[feature].dropna().to_numpy(dtype=float)

    if len(source_values) < 2 or len(target_values) < 2:
        continue

    ks_statistic, p_value = ks_2samp(source_values, target_values, alternative="two-sided", method="auto")
    feature_rows.append({
        "feature": feature,
        "ks_statistic": float(ks_statistic),
        "ks_p_value": float(p_value),
        "source_missing_rate": float(X_source[feature].isna().mean()),
        "target_missing_rate": float(X_target[feature].isna().mean())
    })

feature_results = pd.DataFrame(feature_rows)
feature_results["ks_p_value_bh"] = benjamini_hochberg(feature_results["ks_p_value"].to_numpy())
feature_results["significant_after_bh"] = feature_results["ks_p_value_bh"] < ALPHA
feature_results = feature_results.sort_values("ks_statistic", ascending=False)

feature_results.to_csv(os.path.join(OUTPUT_DIR, "univariate_covariate_shift_tests.csv"), index=False)
pd.Series(source_constant_target_variable, name="source_constant_but_target_variable").to_csv(
    os.path.join(OUTPUT_DIR, "source_constant_target_variable_features.csv"), index=False
)

# Domain label: 0 = original publisher training file, 1 = original publisher testing file
X_domain = pd.concat([X_source, X_target], axis=0, ignore_index=True)
y_domain = np.concatenate([np.zeros(len(X_source), dtype=int), np.ones(len(X_target), dtype=int)])

domain_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE))
])

repeated_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
domain_auc_scores = cross_val_score(domain_model, X_domain, y_domain, scoring="roc_auc", cv=repeated_cv, n_jobs=-1)

permutation_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
observed_auc, permuted_auc_scores, permutation_p_value = permutation_test_score(
    domain_model, X_domain, y_domain, scoring="roc_auc", cv=permutation_cv,
    n_permutations=N_PERMUTATIONS, n_jobs=-1, random_state=RANDOM_STATE
)

pd.DataFrame({"domain_classifier_roc_auc": domain_auc_scores}).to_csv(
    os.path.join(OUTPUT_DIR, "domain_classifier_auc_folds.csv"), index=False
)
pd.DataFrame({"permuted_roc_auc": permuted_auc_scores}).to_csv(
    os.path.join(OUTPUT_DIR, "domain_classifier_permutation_scores.csv"), index=False
)

source_labels = source_df[TARGET].astype(str)
target_labels = target_df[TARGET].astype(str)

all_classes = sorted(set(source_labels.unique()) | set(target_labels.unique()))
source_counts = source_labels.value_counts()
target_counts = target_labels.value_counts()

class_results = pd.DataFrame({
    "class": all_classes,
    "source_count": [int(source_counts.get(label, 0)) for label in all_classes],
    "target_count": [int(target_counts.get(label, 0)) for label in all_classes]
})
class_results["source_proportion"] = class_results["source_count"] / len(source_df)
class_results["target_proportion"] = class_results["target_count"] / len(target_df)
class_results.to_csv(os.path.join(OUTPUT_DIR, "class_support_and_proportions.csv"), index=False)

missing_from_target = class_results.loc[class_results["target_count"] == 0, "class"].tolist()
missing_from_source = class_results.loc[class_results["source_count"] == 0, "class"].tolist()

top_features = feature_results.head(20).sort_values("ks_statistic")
fig, axes = plt.subplots(1, 2, figsize=(17, 7))

feature_colours = np.where(top_features["significant_after_bh"], "#d95f02", "#7570b3")
axes[0].barh(top_features["feature"], top_features["ks_statistic"], color=feature_colours)
axes[0].set_title("Largest Publisher-Boundary Feature Differences")
axes[0].set_xlabel("Two-Sample KS Statistic")
axes[0].grid(axis="x", linestyle="--", alpha=0.35)

class_positions = np.arange(len(class_results))
axes[1].bar(class_positions - 0.2, class_results["source_proportion"], width=0.4, label="Training File")
axes[1].bar(class_positions + 0.2, class_results["target_proportion"], width=0.4, label="Testing File")
axes[1].set_xticks(class_positions)
axes[1].set_xticklabels(class_results["class"], rotation=45, ha="right")
axes[1].set_ylabel("Class Proportion")
axes[1].set_title("Class Support and Proportions")
axes[1].legend()
axes[1].grid(axis="y", linestyle="--", alpha=0.35)

fig.tight_layout()

publisher_shift_png_path = os.path.join(
    OUTPUT_DIR,
    "publisher_boundary_shift_summary.png",
)

publisher_shift_pdf_path = os.path.join(
    OUTPUT_DIR,
    "publisher_boundary_shift_summary.pdf",
)

fig.savefig(
    publisher_shift_png_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

fig.savefig(
    publisher_shift_pdf_path,
    format="pdf",
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(fig)

print(f"Saved PNG: {publisher_shift_png_path}")
print(f"Saved PDF: {publisher_shift_pdf_path}")

significant_count = int(feature_results["significant_after_bh"].sum())

report = f"""PUBLISHER-BOUNDARY SHIFT DIAGNOSTIC

Source rows: {len(source_df)}
Target rows: {len(target_df)}
Active numeric predictors tested: {len(feature_results)}
BH-significant KS tests at alpha={ALPHA}: {significant_count}/{len(feature_results)}

Repeated-CV domain-classifier ROC-AUC:
{domain_auc_scores.mean():.4f} +/- {domain_auc_scores.std(ddof=1):.4f}

Five-fold observed domain ROC-AUC:
{observed_auc:.4f}

Domain-classifier permutation p-value:
{permutation_p_value:.6f}

Classes missing from target:
{missing_from_target}

Classes missing from source:
{missing_from_source}

INTERPRETATION

The KS tests assess univariate differences in P(X).

The domain classifier assesses whether the files are distinguishable using
their predictors jointly. A domain-classifier ROC-AUC close to 0.5 indicates
little detectable multivariate input shift. A materially higher ROC-AUC with
a significant permutation result indicates input-distribution shift.

These tests do not, by themselves, prove pure covariate shift. Pure covariate
shift additionally requires P(Y|X) to remain stable.

If either file lacks a class, report:

"Dataset shift with feature-distribution and class-support differences."

Do not describe the complete publisher boundary as pure covariate shift."""

with open(os.path.join(OUTPUT_DIR, "covariate_shift_report.txt"), "w", encoding="utf-8") as report_file:
    report_file.write(report)

print(report)
print(f"\nResults saved to: {OUTPUT_DIR}")

## Model Development

### Attention CNN-BiLSTM

This self-contained cell performs hyperparameter tuning, five-seed final training, validation-based champion selection and complete artifact archiving for the principal Attention CNN-BiLSTM.

The cell pools the two publisher CSV files, removes exact duplicate rows and reconstructs the deterministic stratified 64/16/20 split using `random_state=42`. It obtains 1,200 training, 301 validation and 376 held-out observations.

Preprocessing is learned from the training partition only:

1. infinite values are converted to missing values;
2. 53 training-zero-variance predictors are removed;
3. mean imputation is fitted;
4. standardisation is fitted;
5. SMOTE is applied only to the processed training partition; and
6. targets are encoded for ten-class classification.

This leaves 244 predictors. Each observation is reshaped to `(244, 1)`. The feature axis is a fixed engineered-feature ordering, not packet time.

The architecture is:

```text
Input (244, 1)
→ Conv1D
→ Batch Normalisation
→ Max Pooling
→ Bidirectional LSTM with sequence output
→ Batch Normalisation
→ Dropout
→ Learned attention pooling
→ Softmax output (10 classes)
```

The custom `SelfAttention` layer performs learned attention pooling. The historical name is retained for Keras serialization compatibility; it is not query-key-value self-attention.

Keras Tuner GridSearch evaluates the complete 24-configuration grid:

```text
filters:                     32 or 64
LSTM units per direction:    32 or 64
dropout:                     0.3, 0.4 or 0.5
Adam learning rate:          0.001 or 0.0005
```

Each search trial uses at most 50 epochs, batch size 32, validation-accuracy optimisation and early stopping on validation accuracy with patience 8.

The selected configuration is then trained with seeds:

```text
42, 7, 123, 2024, 99
```

Final training uses at most 100 epochs and batch size 32. Early stopping monitors validation loss with patience 15 and restores the best weights. `ReduceLROnPlateau` monitors validation accuracy with patience 5, factor 0.5 and minimum learning rate `1e-5`.

The champion seed is selected using the highest restored validation accuracy, with lower restored validation loss as the tie-breaker. Held-out test performance does not participate in tuning or seed selection.

Successful runs are finalized under:

```text
Honours_Project/Models/Tuned/CNN_BiLSTM/<timestamp>/
```

The directory contains the five seed models, `best_validation_cnn_bilstm.keras`, preprocessing objects, split indices, held-out arrays, feature and class names, tuner outputs, training histories, figures, validation summaries, source code and SHA-256 hash, terminal output, runtime environment, system diagnostics, package requirements, metadata and a file inventory.

The held-out test bundle is archived but not evaluated in this cell.

In [ ]:
# ATTENTION CNN-BILSTM: SELF-CONTAINED TUNING, FIVE-SEED TRAINING AND ARCHIVING

try:
    get_ipython().run_line_magic("pip", "install -q keras-tuner imbalanced-learn psutil")
except NameError:
    pass

import contextlib
import gc
import hashlib
import importlib.metadata as metadata
import io
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

import imblearn
import joblib
import keras
import keras_tuner as kt
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import sklearn
import tensorflow as tf
from google.colab import drive
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils import shuffle
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import backend as K
from tensorflow.keras.layers import BatchNormalization, Bidirectional, Conv1D, Dense, Dropout, Input, Layer, LSTM, MaxPooling1D
from tensorflow.keras.models import Model

try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print(f"TensorFlow deterministic operations could not be enabled: {exc}")

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

MODEL_NAME = 'Attention CNN-BiLSTM'
MODEL_KEY = 'cnn_bilstm'
PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
TRAIN_CSV = PROJECT_ROOT / "iot_device_train.csv"
TEST_CSV = PROJECT_ROOT / "iot_device_test.csv"
OUTPUT_PARENT = PROJECT_ROOT / "Models" / "Tuned" / 'CNN_BiLSTM'
TIMEZONE = ZoneInfo("Africa/Johannesburg")
SEEDS = [42, 7, 123, 2024, 99]
SEARCH_SPACE = {"filters": [32, 64], "lstm_units_per_direction": [32, 64], "dropout": [0.3, 0.4, 0.5], "learning_rate": [0.001, 0.0005]}
MAX_TRIALS = 24
SEARCH_EPOCHS = 50
FINAL_EPOCHS = 100
SEARCH_BATCH_SIZE = 32
FINAL_BATCH_SIZE = 32
PLOT_ALL_SEEDS = True
SEARCH_PATIENCE = 8
FINAL_PATIENCE = 15
LR_PATIENCE = 5
LR_FACTOR = 0.5
MIN_LR = 1e-5
CHAMPION_FILENAME = 'best_validation_cnn_bilstm.keras'


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def command_record(command):
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=False)
        return {"command": command, "returncode": result.returncode, "stdout": result.stdout, "stderr": result.stderr}
    except Exception as exc:
        return {"command": command, "returncode": None, "stdout": "", "stderr": repr(exc)}


class TeeStream:
    def __init__(self, original_stream, log_handle):
        self.original_stream = original_stream
        self.log_handle = log_handle

    def write(self, message):
        self.original_stream.write(message)
        self.original_stream.flush()
        self.log_handle.write(message)
        self.log_handle.flush()

    def flush(self):
        self.original_stream.flush()
        self.log_handle.flush()

    def isatty(self):
        return self.original_stream.isatty()

    def fileno(self):
        return self.original_stream.fileno()


terminal_capture = {}


def stop_terminal_capture(result=None):
    if not terminal_capture:
        return
    sys.stdout = terminal_capture["stdout"]
    sys.stderr = terminal_capture["stderr"]
    terminal_capture["handle"].flush()
    terminal_capture["handle"].close()
    failed = bool(
        result is not None
        and (getattr(result, "error_before_exec", None) is not None or getattr(result, "error_in_exec", None) is not None)
    )
    if failed:
        error = getattr(result, "error_before_exec", None) or getattr(result, "error_in_exec", None)
        terminal_capture["run_dir"].joinpath("RUN_FAILED.txt").write_text(
            f"Run failed before finalisation.\n{type(error).__name__}: {error}\n",
            encoding="utf-8",
        )
    try:
        get_ipython().events.unregister("post_run_cell", stop_terminal_capture)
    except Exception:
        pass
    terminal_capture.clear()


def start_terminal_capture(log_path, run_dir):
    log_handle = Path(log_path).open("w", encoding="utf-8")
    terminal_capture.update({
        "stdout": sys.stdout,
        "stderr": sys.stderr,
        "handle": log_handle,
        "run_dir": Path(run_dir),
    })
    sys.stdout = TeeStream(terminal_capture["stdout"], log_handle)
    sys.stderr = TeeStream(terminal_capture["stderr"], log_handle)
    try:
        get_ipython().events.register("post_run_cell", stop_terminal_capture)
    except Exception:
        pass


def version(distribution):
    try:
        return metadata.version(distribution)
    except metadata.PackageNotFoundError:
        return "not installed"


def current_cell_source():
    try:
        history = get_ipython().history_manager.input_hist_raw
        if history and str(history[-1]).strip():
            return str(history[-1])
    except Exception:
        pass
    return "# Current notebook-cell source was unavailable.\n"


for required_path in (TRAIN_CSV, TEST_CSV):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

run_started = datetime.now(TIMEZONE)
RUN_TIMESTAMP = run_started.strftime("%Y-%m-%d_%H-%M-%S")
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
FINAL_RUN_DIR = OUTPUT_PARENT / RUN_TIMESTAMP
RUN_DIR = OUTPUT_PARENT / f"_TEMP_{MODEL_KEY.upper()}_{RUN_TIMESTAMP}"
if RUN_DIR.exists() or FINAL_RUN_DIR.exists():
    raise FileExistsError(f"Run directory already exists for timestamp {RUN_TIMESTAMP}")
FIGURES_DIR = RUN_DIR / "Figures"
TUNER_DIR = RUN_DIR / "Tuner"
FIGURES_DIR.mkdir(parents=True, exist_ok=False)
start_terminal_capture(RUN_DIR / "terminal_output.txt", RUN_DIR)

runtime_environment = {
    "captured_at": run_started.isoformat(),
    "timezone": "Africa/Johannesburg",
    "python": {"version": sys.version, "executable": sys.executable, "implementation": platform.python_implementation()},
    "operating_system": {"platform": platform.platform(), "system": platform.system(), "release": platform.release(), "machine": platform.machine(), "processor": platform.processor()},
    "hardware": {
        "physical_cpu_count": psutil.cpu_count(logical=False),
        "logical_cpu_count": psutil.cpu_count(logical=True),
        "memory_total_bytes": int(psutil.virtual_memory().total),
        "tensorflow_visible_cpus": [device.name for device in tf.config.list_physical_devices("CPU")],
        "tensorflow_visible_gpus": [device.name for device in tf.config.list_physical_devices("GPU")],
    },
    "colab": {"release_tag": os.environ.get("COLAB_RELEASE_TAG"), "gpu_type": os.environ.get("COLAB_GPU")},
    "software": {
        "tensorflow": version("tensorflow"), "keras": version("keras"), "keras_tuner": version("keras-tuner"),
        "numpy": version("numpy"), "pandas": version("pandas"), "scikit_learn": version("scikit-learn"),
        "imbalanced_learn": version("imbalanced-learn"), "joblib": version("joblib"),
        "matplotlib": version("matplotlib"), "psutil": version("psutil"),
    },
    "tensorflow_build": tf.sysconfig.get_build_info(),
    "determinism": {"TF_DETERMINISTIC_OPS": os.environ.get("TF_DETERMINISTIC_OPS"), "TF_CUDNN_DETERMINISTIC": os.environ.get("TF_CUDNN_DETERMINISTIC")},
}
(RUN_DIR / "runtime_environment.json").write_text(json.dumps(runtime_environment, indent=2, default=str), encoding="utf-8")
(RUN_DIR / "system_diagnostics.json").write_text(
    json.dumps({
        "uname": command_record(["uname", "-a"]), "lscpu": command_record(["lscpu"]),
        "memory": command_record(["free", "-h"]), "nvidia_smi": command_record(["nvidia-smi"]),
    }, indent=2, default=str), encoding="utf-8"
)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=False)
(RUN_DIR / "requirements.txt").write_text(freeze.stdout, encoding="utf-8")
source_path = RUN_DIR / "training_cell_source.py"
source_path.write_text(current_cell_source(), encoding="utf-8")
(RUN_DIR / "training_cell_source.sha256").write_text(sha256_file(source_path), encoding="utf-8")

dataset_manifest = {
    "publisher_train": {"path": str(TRAIN_CSV), "size_bytes": TRAIN_CSV.stat().st_size, "sha256": sha256_file(TRAIN_CSV)},
    "publisher_test": {"path": str(TEST_CSV), "size_bytes": TEST_CSV.stat().st_size, "sha256": sha256_file(TEST_CSV)},
}
(RUN_DIR / "dataset_manifest.json").write_text(json.dumps(dataset_manifest, indent=2), encoding="utf-8")

publisher_train = pd.read_csv(TRAIN_CSV)
publisher_test = pd.read_csv(TEST_CSV)
combined = pd.concat([publisher_train, publisher_test], ignore_index=True).drop_duplicates().reset_index(drop=True)
X_all = combined.drop(columns="device_category").replace([np.inf, -np.inf], np.nan)
y_all = combined["device_category"].astype(str)

label_encoder = LabelEncoder()
y_all_int = label_encoder.fit_transform(y_all)
all_indices = np.arange(len(combined), dtype=np.int64)
X_development, X_test, y_development, y_test, index_development, index_test = train_test_split(
    X_all, y_all_int, all_indices, test_size=0.20, random_state=42, stratify=y_all_int
)
X_train, X_val, y_train, y_val, index_train, index_val = train_test_split(
    X_development, y_development, index_development, test_size=0.20, random_state=42, stratify=y_development
)

zero_variance_columns = X_train.columns[X_train.var() == 0].tolist()
X_train = X_train.drop(columns=zero_variance_columns)
X_val = X_val.drop(columns=zero_variance_columns)
X_test = X_test.drop(columns=zero_variance_columns)
feature_names = X_train.columns.tolist()

train_hashes = set(pd.util.hash_pandas_object(X_train, index=False).astype(str))
val_hashes = set(pd.util.hash_pandas_object(X_val, index=False).astype(str))
test_hashes = set(pd.util.hash_pandas_object(X_test, index=False).astype(str))
if train_hashes & val_hashes or train_hashes & test_hashes or val_hashes & test_hashes:
    raise RuntimeError("An exact feature vector occurs in more than one partition.")

imputer = SimpleImputer(strategy="mean")
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed).astype(np.float32)
X_val_scaled = scaler.transform(X_val_imputed).astype(np.float32)
X_test_scaled = scaler.transform(X_test_imputed).astype(np.float32)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)
X_train_balanced, y_train_balanced = shuffle(X_train_balanced, y_train_balanced, random_state=42)
X_train_balanced = np.asarray(X_train_balanced, dtype=np.float32)
NUM_CLASSES = len(label_encoder.classes_)
NUM_FEATURES = X_train_balanced.shape[1]
y_train_onehot = tf.keras.utils.to_categorical(y_train_balanced, NUM_CLASSES).astype(np.float32)
y_val_onehot = tf.keras.utils.to_categorical(y_val, NUM_CLASSES).astype(np.float32)
y_test_onehot = tf.keras.utils.to_categorical(y_test, NUM_CLASSES).astype(np.float32)
try:
    one_hot_encoder = OneHotEncoder(
        categories=[label_encoder.classes_], sparse_output=False, handle_unknown="error"
    )
except TypeError:
    one_hot_encoder = OneHotEncoder(
        categories=[label_encoder.classes_], sparse=False, handle_unknown="error"
    )
one_hot_encoder.fit(np.asarray(label_encoder.classes_).reshape(-1, 1))

if (len(combined), len(X_train), len(X_val), len(X_test)) != (1877, 1200, 301, 376):
    raise RuntimeError(f"Unexpected corpus/split sizes: {len(combined)}, {len(X_train)}, {len(X_val)}, {len(X_test)}")
if (len(zero_variance_columns), NUM_FEATURES, len(X_train_balanced), NUM_CLASSES) != (53, 244, 1280, 10):
    raise RuntimeError(
        f"Unexpected preprocessing result: removed={len(zero_variance_columns)}, retained={NUM_FEATURES}, "
        f"balanced={len(X_train_balanced)}, classes={NUM_CLASSES}"
    )

X_train_3d = X_train_balanced.reshape(-1, NUM_FEATURES, 1)
X_val_3d = X_val_scaled.reshape(-1, NUM_FEATURES, 1)
X_test_3d = X_test_scaled.reshape(-1, NUM_FEATURES, 1)
X_train_model = X_train_3d
X_val_model = X_val_3d
X_test_model = X_test_3d

np.save(RUN_DIR / "X_test_3d.npy", X_test_3d)
np.save(RUN_DIR / "y_test_1d.npy", np.asarray(y_test, dtype=np.int64))
np.save(RUN_DIR / "device_classes.npy", np.asarray(label_encoder.classes_))
np.save(RUN_DIR / f"{MODEL_KEY}_device_classes.npy", np.asarray(label_encoder.classes_))
np.save(RUN_DIR / "feature_names.npy", np.asarray(feature_names, dtype=object))
np.savez_compressed(RUN_DIR / "split_indices.npz", train=index_train, validation=index_val, test=index_test)
np.savez_compressed(
    RUN_DIR / f"{MODEL_KEY}_dataset.npz",
    X_train=X_train_model,
    y_train=y_train_onehot,
    X_val=X_val_model,
    y_val=y_val_onehot,
    X_test=X_test_model,
    y_test=y_test_onehot,
)
(RUN_DIR / "zero_variance_columns.txt").write_text("".join(f"{column}\n" for column in zero_variance_columns), encoding="utf-8")
joblib.dump(imputer, RUN_DIR / "imputer.joblib")
joblib.dump(scaler, RUN_DIR / "scaler.joblib")
joblib.dump(label_encoder, RUN_DIR / "label_encoder.joblib")
joblib.dump(one_hot_encoder, RUN_DIR / "one_hot_encoder.joblib")

@tf.keras.utils.register_keras_serializable(package="COMP700")
class SelfAttention(Layer):
    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1), initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1), initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        scores = K.tanh(K.dot(inputs, self.W) + self.b)
        weights = K.softmax(scores, axis=1)
        return K.sum(inputs * weights, axis=1)

def build_model(hp):
    inputs = Input(shape=(NUM_FEATURES, 1), name="input")
    x = Conv1D(hp.Choice("filters", [32, 64]), 3, padding="same", activation="relu", name="convolution")(inputs)
    x = BatchNormalization(name="convolution_batch_normalisation")(x)
    x = MaxPooling1D(2, name="max_pooling")(x)
    x = Bidirectional(LSTM(hp.Choice("lstm_units", [32, 64]), return_sequences=True), name="bidirectional_lstm")(x)
    x = BatchNormalization(name="recurrent_batch_normalisation")(x)
    x = Dropout(hp.Choice("dropout", [0.3, 0.4, 0.5]), name="dropout")(x)
    x = SelfAttention(name="attention_pooling")(x)
    outputs = Dense(NUM_CLASSES, activation="softmax", name="output")(x)
    model = Model(inputs, outputs, name="attention_cnn_bilstm")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(hp.Choice("learning_rate", [1e-3, 5e-4])),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)
tuner = kt.GridSearch(
    build_model,
    objective=kt.Objective("val_accuracy", direction="max"),
    max_trials=MAX_TRIALS,
    directory=str(TUNER_DIR),
    project_name=MODEL_KEY,
    overwrite=True,
    seed=42,
)
tuner.search(
    X_train_model,
    y_train_onehot,
    validation_data=(X_val_model, y_val_onehot),
    epochs=SEARCH_EPOCHS,
    batch_size=SEARCH_BATCH_SIZE,
    shuffle=True,
    callbacks=[EarlyStopping(monitor="val_accuracy", mode="max", patience=SEARCH_PATIENCE, restore_best_weights=True, verbose=1)],
    verbose=2,
)

best_hp = tuner.get_best_hyperparameters(1)[0]
(RUN_DIR / "best_hyperparameters.json").write_text(json.dumps(best_hp.values, indent=2, default=str), encoding="utf-8")
with io.StringIO() as buffer, contextlib.redirect_stdout(buffer):
    tuner.results_summary(num_trials=MAX_TRIALS)
    (RUN_DIR / "tuner_results_summary.txt").write_text(buffer.getvalue(), encoding="utf-8")

architecture_model = build_model(best_hp)
with (RUN_DIR / "model_architecture.txt").open("w", encoding="utf-8") as handle:
    architecture_model.summary(print_fn=lambda line: handle.write(line + "\n"))
(RUN_DIR / "model_architecture.json").write_text(architecture_model.to_json(), encoding="utf-8")
parameter_count = int(architecture_model.count_params())
del architecture_model
gc.collect()

seed_results = []
for seed in SEEDS:
    print("\n" + "=" * 100)
    print(f"{MODEL_NAME} — FINAL TRAINING SEED {seed}")
    print("=" * 100)
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(seed)
    model = build_model(best_hp)
    history = model.fit(
        X_train_model,
        y_train_onehot,
        validation_data=(X_val_model, y_val_onehot),
        epochs=FINAL_EPOCHS,
        batch_size=FINAL_BATCH_SIZE,
        shuffle=True,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=FINAL_PATIENCE, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_accuracy", factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=MIN_LR, verbose=1),
        ],
        verbose=2,
    )
    validation_probabilities = model.predict(X_val_model, verbose=0)
    validation_accuracy = float(np.mean(np.argmax(validation_probabilities, axis=1) == y_val))
    validation_loss = float(model.evaluate(X_val_model, y_val_onehot, verbose=0)[0])
    restored_epoch = int(np.argmin(history.history["val_loss"])) + 1
    model_path = RUN_DIR / f"cnn_bilstm_seed_{seed}.keras"
    model.save(model_path)
    (RUN_DIR / f"{MODEL_KEY}_seed_{seed}_model_config.json").write_text(
        model.to_json(), encoding="utf-8"
    )
    history_frame = pd.DataFrame(history.history)
    history_frame.insert(0, "epoch", np.arange(1, len(history_frame) + 1))
    history_path = RUN_DIR / f"{MODEL_KEY}_seed_{seed}_training_history.csv"
    history_frame.to_csv(history_path, index=False)
    seed_results.append({
        "model": MODEL_NAME,
        "seed": int(seed),
        "validation_accuracy": validation_accuracy,
        "validation_loss": validation_loss,
        "restored_epoch": restored_epoch,
        "model_file": model_path.name,
        "history_file": history_path.name,
    })
    del model
    gc.collect()

champion_index = min(
    range(len(seed_results)),
    key=lambda index: (-seed_results[index]["validation_accuracy"], seed_results[index]["validation_loss"]),
)
champion = seed_results[champion_index]
shutil.copy2(RUN_DIR / champion["model_file"], RUN_DIR / CHAMPION_FILENAME)
seed_frame = pd.DataFrame(seed_results)
seed_frame["selected_champion"] = False
seed_frame.loc[champion_index, "selected_champion"] = True
seed_frame.to_csv(RUN_DIR / "validation_seed_results.csv", index=False)
seed_frame.to_csv(RUN_DIR / f"{MODEL_KEY}_five_seed_validation_results.csv", index=False)

validation_summary = pd.DataFrame({
    "Metric": [
        "Mean Restored Validation Accuracy (%)",
        "Validation Accuracy Std (%)",
        "Mean Restored Validation Loss",
        "Validation Loss Std",
        "Best Validation Accuracy (%)",
        "Best Validation Loss",
        "Best Validation Seed",
        "Restored Epoch",
    ],
    "Value": [
        seed_frame["validation_accuracy"].mean() * 100,
        seed_frame["validation_accuracy"].std(ddof=1) * 100,
        seed_frame["validation_loss"].mean(),
        seed_frame["validation_loss"].std(ddof=1),
        champion["validation_accuracy"] * 100,
        champion["validation_loss"],
        champion["seed"],
        champion["restored_epoch"],
    ],
})
validation_summary.to_csv(
    RUN_DIR / f"{MODEL_KEY}_validation_summary_statistics.csv", index=False
)


def plot_training_history(result_row, filename_prefix, title):
    history_data = pd.read_csv(RUN_DIR / result_row["history_file"])
    for metric, validation_metric, y_label in [
        ("accuracy", "val_accuracy", "Accuracy"),
        ("loss", "val_loss", "Categorical cross-entropy"),
    ]:
        figure, axis = plt.subplots(figsize=(9, 6))
        axis.plot(history_data["epoch"], history_data[metric], label=f"Training {metric}")
        axis.plot(history_data["epoch"], history_data[validation_metric], label=f"Validation {metric}")
        axis.axvline(
            result_row["restored_epoch"], linestyle="--", color="black",
            label=f"Restored epoch {result_row['restored_epoch']}"
        )
        axis.set_title(title)
        axis.set_xlabel("Epoch")
        axis.set_ylabel(y_label)
        axis.grid(True, linestyle="--", alpha=0.35)
        axis.legend()
        figure.tight_layout()
        figure.savefig(FIGURES_DIR / f"{filename_prefix}_{metric}.png", dpi=300, bbox_inches="tight")
        figure.savefig(FIGURES_DIR / f"{filename_prefix}_{metric}.pdf", bbox_inches="tight")
        plt.show()
        plt.close(figure)


if PLOT_ALL_SEEDS:
    for result_row in seed_results:
        plot_training_history(
            result_row,
            f"{MODEL_KEY}_seed_{result_row['seed']}",
            f"{MODEL_NAME}: Seed {result_row['seed']}",
        )

plot_training_history(
    champion,
    f"{MODEL_KEY}_selected_seed",
    f"{MODEL_NAME}: Validation-Selected Seed {champion['seed']}",
)

run_metadata = {
    "project": "Smart Home Appliance Identification Using Deep Learning",
    "model_name": MODEL_NAME,
    "run_started": run_started.isoformat(),
    "run_completed": datetime.now(TIMEZONE).isoformat(),
    "dataset": {
        "unique_rows": len(combined), "training_rows_before_smote": len(X_train),
        "training_rows_after_smote": len(X_train_balanced), "validation_rows": len(X_val),
        "held_out_test_rows": len(X_test), "removed_zero_variance_features": len(zero_variance_columns),
        "retained_features": NUM_FEATURES, "classes": label_encoder.classes_.tolist(),
        "split_random_state": 42, "smote_random_state": 42,
    },
    "search": {
        "method": "Keras Tuner GridSearch", "objective": "val_accuracy", "tuner_seed": 42,
        "search_space": SEARCH_SPACE, "maximum_trials": MAX_TRIALS,
        "maximum_epochs_per_trial": SEARCH_EPOCHS, "batch_size": SEARCH_BATCH_SIZE,
        "early_stopping_monitor": "val_accuracy", "early_stopping_patience": SEARCH_PATIENCE,
        "selected_hyperparameters": best_hp.values,
    },
    "final_training": {
        "seeds": SEEDS, "maximum_epochs": FINAL_EPOCHS, "batch_size": FINAL_BATCH_SIZE,
        "early_stopping_monitor": "val_loss", "early_stopping_patience": FINAL_PATIENCE,
        "reduce_lr_monitor": "val_accuracy", "reduce_lr_patience": LR_PATIENCE,
        "reduce_lr_factor": LR_FACTOR, "minimum_learning_rate": MIN_LR,
        "training_figures_for_all_seeds": PLOT_ALL_SEEDS,
    },
    "selection": {
        "rule": "highest restored validation accuracy; lower restored validation loss as tie-breaker",
        "held_out_test_used": False, "champion_seed": int(champion["seed"]),
        "champion_validation_accuracy": float(champion["validation_accuracy"]),
        "champion_validation_loss": float(champion["validation_loss"]),
        "champion_filename": CHAMPION_FILENAME,
    },
    "model_parameter_count": parameter_count,
}
(RUN_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2, default=str), encoding="utf-8")

print("\n" + "=" * 100)
print(f"{MODEL_NAME} RUN COMPLETE")
print("=" * 100)
print(seed_frame.to_string(index=False))
print(f"Selected hyperparameters: {best_hp.values}")
print(f"Validation-selected seed: {champion['seed']}")
print(f"Champion model: {RUN_DIR / CHAMPION_FILENAME}")
print(f"Temporary run directory: {RUN_DIR}")
print(f"Final run directory after successful finalisation: {FINAL_RUN_DIR}")
print("The held-out test set was archived but was not evaluated or used for selection in this cell.")
stop_terminal_capture()

try:
    shutil.move(str(RUN_DIR), str(FINAL_RUN_DIR))
except Exception as exc:
    (RUN_DIR / "RUN_FAILED.txt").write_text(
        f"Training completed, but final directory creation failed.\n{type(exc).__name__}: {exc}\n",
        encoding="utf-8",
    )
    raise
RUN_DIR = FINAL_RUN_DIR
FIGURES_DIR = RUN_DIR / "Figures"
TUNER_DIR = RUN_DIR / "Tuner"
try:
    inventory_rows = []
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file() and path.name != "file_inventory.csv":
            inventory_rows.append({"relative_path": str(path.relative_to(RUN_DIR)), "size_bytes": path.stat().st_size, "sha256": sha256_file(path)})
    pd.DataFrame(inventory_rows).to_csv(RUN_DIR / "file_inventory.csv", index=False)
    (RUN_DIR / "RUN_COMPLETE.txt").write_text(
        "Run completed and finalised successfully.\n", encoding="utf-8"
    )
except Exception as exc:
    (RUN_DIR / "RUN_FAILED.txt").write_text(
        f"Training completed, but archive inventory finalisation failed.\n{type(exc).__name__}: {exc}\n",
        encoding="utf-8",
    )
    raise
print(f"Final run directory: {RUN_DIR}")
print(f"Terminal output: {RUN_DIR / 'terminal_output.txt'}")

### Feature-Attention MLP

This self-contained cell tunes and trains the Feature-Attention MLP using the same corrected corpus, deterministic split and training-only preprocessing policy as the other neural models.

The model receives the 244 retained predictors as a two-dimensional feature vector. A learned 244-unit softmax layer assigns a weight to each input predictor, and these weights are multiplied element-wise with the original input. This produces a feature-weighted representation before dense classification.

The architecture is:

```text
Input (244)
→ feature softmax weights
→ element-wise feature weighting
→ Dense layer 1
→ Batch Normalisation
→ Dropout
→ Dense layer 2
→ Batch Normalisation
→ Dropout
→ Dense (64, ReLU)
→ Batch Normalisation
→ Softmax output (10 classes)
```

Keras Tuner GridSearch evaluates 16 configurations:

```text
first hidden width:      256 or 512
second hidden width:     128 or 256
shared dropout rate:     0.3 or 0.4
Adam learning rate:      0.001 or 0.0005
```

Each search trial uses at most 50 epochs, batch size 64, validation-accuracy optimisation and early stopping on validation accuracy with patience 8.

The selected configuration is trained using seeds `42, 7, 123, 2024, 99` for at most 100 epochs with batch size 64. Early stopping monitors validation loss with patience 15, while learning-rate reduction monitors validation accuracy with patience 5.

The retained seed is selected by restored validation accuracy, with restored validation loss as the tie-breaker. The held-out test set is excluded from tuning and seed selection.

Successful runs are finalized under:

```text
Honours_Project/Models/Tuned/Feature_Attention_MLP/<timestamp>/
```

The validation-selected model is saved as:

```text
best_validation_mlp.keras
```

The run also preserves every seed model, preprocessing objects, held-out arrays, selected hyperparameters, histories, validation results, terminal output, source provenance, runtime environment, system diagnostics, requirements and a file inventory.

In [ ]:
# FEATURE-ATTENTION MLP: SELF-CONTAINED TUNING, FIVE-SEED TRAINING AND ARCHIVING

try:
    get_ipython().run_line_magic("pip", "install -q keras-tuner imbalanced-learn psutil")
except NameError:
    pass

import contextlib
import gc
import hashlib
import importlib.metadata as metadata
import io
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

import imblearn
import joblib
import keras
import keras_tuner as kt
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import sklearn
import tensorflow as tf
from google.colab import drive
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils import shuffle
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout, Input, Multiply
from tensorflow.keras.models import Model

try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print(f"TensorFlow deterministic operations could not be enabled: {exc}")

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

MODEL_NAME = 'Feature-Attention MLP'
MODEL_KEY = 'mlp'
PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
TRAIN_CSV = PROJECT_ROOT / "iot_device_train.csv"
TEST_CSV = PROJECT_ROOT / "iot_device_test.csv"
OUTPUT_PARENT = PROJECT_ROOT / "Models" / "Tuned" / 'Feature_Attention_MLP'
TIMEZONE = ZoneInfo("Africa/Johannesburg")
SEEDS = [42, 7, 123, 2024, 99]
SEARCH_SPACE = {"first_hidden_units": [256, 512], "second_hidden_units": [128, 256], "dropout": [0.3, 0.4], "learning_rate": [0.001, 0.0005]}
MAX_TRIALS = 16
SEARCH_EPOCHS = 50
FINAL_EPOCHS = 100
SEARCH_BATCH_SIZE = 64
FINAL_BATCH_SIZE = 64
PLOT_ALL_SEEDS = True
SEARCH_PATIENCE = 8
FINAL_PATIENCE = 15
LR_PATIENCE = 5
LR_FACTOR = 0.5
MIN_LR = 1e-5
CHAMPION_FILENAME = 'best_validation_mlp.keras'


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def command_record(command):
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=False)
        return {"command": command, "returncode": result.returncode, "stdout": result.stdout, "stderr": result.stderr}
    except Exception as exc:
        return {"command": command, "returncode": None, "stdout": "", "stderr": repr(exc)}


class TeeStream:
    def __init__(self, original_stream, log_handle):
        self.original_stream = original_stream
        self.log_handle = log_handle

    def write(self, message):
        self.original_stream.write(message)
        self.original_stream.flush()
        self.log_handle.write(message)
        self.log_handle.flush()

    def flush(self):
        self.original_stream.flush()
        self.log_handle.flush()

    def isatty(self):
        return self.original_stream.isatty()

    def fileno(self):
        return self.original_stream.fileno()


terminal_capture = {}


def stop_terminal_capture(result=None):
    if not terminal_capture:
        return
    sys.stdout = terminal_capture["stdout"]
    sys.stderr = terminal_capture["stderr"]
    terminal_capture["handle"].flush()
    terminal_capture["handle"].close()
    failed = bool(
        result is not None
        and (getattr(result, "error_before_exec", None) is not None or getattr(result, "error_in_exec", None) is not None)
    )
    if failed:
        error = getattr(result, "error_before_exec", None) or getattr(result, "error_in_exec", None)
        terminal_capture["run_dir"].joinpath("RUN_FAILED.txt").write_text(
            f"Run failed before finalisation.\n{type(error).__name__}: {error}\n",
            encoding="utf-8",
        )
    try:
        get_ipython().events.unregister("post_run_cell", stop_terminal_capture)
    except Exception:
        pass
    terminal_capture.clear()


def start_terminal_capture(log_path, run_dir):
    log_handle = Path(log_path).open("w", encoding="utf-8")
    terminal_capture.update({
        "stdout": sys.stdout,
        "stderr": sys.stderr,
        "handle": log_handle,
        "run_dir": Path(run_dir),
    })
    sys.stdout = TeeStream(terminal_capture["stdout"], log_handle)
    sys.stderr = TeeStream(terminal_capture["stderr"], log_handle)
    try:
        get_ipython().events.register("post_run_cell", stop_terminal_capture)
    except Exception:
        pass


def version(distribution):
    try:
        return metadata.version(distribution)
    except metadata.PackageNotFoundError:
        return "not installed"


def current_cell_source():
    try:
        history = get_ipython().history_manager.input_hist_raw
        if history and str(history[-1]).strip():
            return str(history[-1])
    except Exception:
        pass
    return "# Current notebook-cell source was unavailable.\n"


for required_path in (TRAIN_CSV, TEST_CSV):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

run_started = datetime.now(TIMEZONE)
RUN_TIMESTAMP = run_started.strftime("%Y-%m-%d_%H-%M-%S")
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
FINAL_RUN_DIR = OUTPUT_PARENT / RUN_TIMESTAMP
RUN_DIR = OUTPUT_PARENT / f"_TEMP_{MODEL_KEY.upper()}_{RUN_TIMESTAMP}"
if RUN_DIR.exists() or FINAL_RUN_DIR.exists():
    raise FileExistsError(f"Run directory already exists for timestamp {RUN_TIMESTAMP}")
FIGURES_DIR = RUN_DIR / "Figures"
TUNER_DIR = RUN_DIR / "Tuner"
FIGURES_DIR.mkdir(parents=True, exist_ok=False)
start_terminal_capture(RUN_DIR / "terminal_output.txt", RUN_DIR)

runtime_environment = {
    "captured_at": run_started.isoformat(),
    "timezone": "Africa/Johannesburg",
    "python": {"version": sys.version, "executable": sys.executable, "implementation": platform.python_implementation()},
    "operating_system": {"platform": platform.platform(), "system": platform.system(), "release": platform.release(), "machine": platform.machine(), "processor": platform.processor()},
    "hardware": {
        "physical_cpu_count": psutil.cpu_count(logical=False),
        "logical_cpu_count": psutil.cpu_count(logical=True),
        "memory_total_bytes": int(psutil.virtual_memory().total),
        "tensorflow_visible_cpus": [device.name for device in tf.config.list_physical_devices("CPU")],
        "tensorflow_visible_gpus": [device.name for device in tf.config.list_physical_devices("GPU")],
    },
    "colab": {"release_tag": os.environ.get("COLAB_RELEASE_TAG"), "gpu_type": os.environ.get("COLAB_GPU")},
    "software": {
        "tensorflow": version("tensorflow"), "keras": version("keras"), "keras_tuner": version("keras-tuner"),
        "numpy": version("numpy"), "pandas": version("pandas"), "scikit_learn": version("scikit-learn"),
        "imbalanced_learn": version("imbalanced-learn"), "joblib": version("joblib"),
        "matplotlib": version("matplotlib"), "psutil": version("psutil"),
    },
    "tensorflow_build": tf.sysconfig.get_build_info(),
    "determinism": {"TF_DETERMINISTIC_OPS": os.environ.get("TF_DETERMINISTIC_OPS"), "TF_CUDNN_DETERMINISTIC": os.environ.get("TF_CUDNN_DETERMINISTIC")},
}
(RUN_DIR / "runtime_environment.json").write_text(json.dumps(runtime_environment, indent=2, default=str), encoding="utf-8")
(RUN_DIR / "system_diagnostics.json").write_text(
    json.dumps({
        "uname": command_record(["uname", "-a"]), "lscpu": command_record(["lscpu"]),
        "memory": command_record(["free", "-h"]), "nvidia_smi": command_record(["nvidia-smi"]),
    }, indent=2, default=str), encoding="utf-8"
)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=False)
(RUN_DIR / "requirements.txt").write_text(freeze.stdout, encoding="utf-8")
source_path = RUN_DIR / "training_cell_source.py"
source_path.write_text(current_cell_source(), encoding="utf-8")
(RUN_DIR / "training_cell_source.sha256").write_text(sha256_file(source_path), encoding="utf-8")

dataset_manifest = {
    "publisher_train": {"path": str(TRAIN_CSV), "size_bytes": TRAIN_CSV.stat().st_size, "sha256": sha256_file(TRAIN_CSV)},
    "publisher_test": {"path": str(TEST_CSV), "size_bytes": TEST_CSV.stat().st_size, "sha256": sha256_file(TEST_CSV)},
}
(RUN_DIR / "dataset_manifest.json").write_text(json.dumps(dataset_manifest, indent=2), encoding="utf-8")

publisher_train = pd.read_csv(TRAIN_CSV)
publisher_test = pd.read_csv(TEST_CSV)
combined = pd.concat([publisher_train, publisher_test], ignore_index=True).drop_duplicates().reset_index(drop=True)
X_all = combined.drop(columns="device_category").replace([np.inf, -np.inf], np.nan)
y_all = combined["device_category"].astype(str)

label_encoder = LabelEncoder()
y_all_int = label_encoder.fit_transform(y_all)
all_indices = np.arange(len(combined), dtype=np.int64)
X_development, X_test, y_development, y_test, index_development, index_test = train_test_split(
    X_all, y_all_int, all_indices, test_size=0.20, random_state=42, stratify=y_all_int
)
X_train, X_val, y_train, y_val, index_train, index_val = train_test_split(
    X_development, y_development, index_development, test_size=0.20, random_state=42, stratify=y_development
)

zero_variance_columns = X_train.columns[X_train.var() == 0].tolist()
X_train = X_train.drop(columns=zero_variance_columns)
X_val = X_val.drop(columns=zero_variance_columns)
X_test = X_test.drop(columns=zero_variance_columns)
feature_names = X_train.columns.tolist()

train_hashes = set(pd.util.hash_pandas_object(X_train, index=False).astype(str))
val_hashes = set(pd.util.hash_pandas_object(X_val, index=False).astype(str))
test_hashes = set(pd.util.hash_pandas_object(X_test, index=False).astype(str))
if train_hashes & val_hashes or train_hashes & test_hashes or val_hashes & test_hashes:
    raise RuntimeError("An exact feature vector occurs in more than one partition.")

imputer = SimpleImputer(strategy="mean")
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed).astype(np.float32)
X_val_scaled = scaler.transform(X_val_imputed).astype(np.float32)
X_test_scaled = scaler.transform(X_test_imputed).astype(np.float32)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)
X_train_balanced, y_train_balanced = shuffle(X_train_balanced, y_train_balanced, random_state=42)
X_train_balanced = np.asarray(X_train_balanced, dtype=np.float32)
NUM_CLASSES = len(label_encoder.classes_)
NUM_FEATURES = X_train_balanced.shape[1]
y_train_onehot = tf.keras.utils.to_categorical(y_train_balanced, NUM_CLASSES).astype(np.float32)
y_val_onehot = tf.keras.utils.to_categorical(y_val, NUM_CLASSES).astype(np.float32)
y_test_onehot = tf.keras.utils.to_categorical(y_test, NUM_CLASSES).astype(np.float32)
try:
    one_hot_encoder = OneHotEncoder(
        categories=[label_encoder.classes_], sparse_output=False, handle_unknown="error"
    )
except TypeError:
    one_hot_encoder = OneHotEncoder(
        categories=[label_encoder.classes_], sparse=False, handle_unknown="error"
    )
one_hot_encoder.fit(np.asarray(label_encoder.classes_).reshape(-1, 1))

if (len(combined), len(X_train), len(X_val), len(X_test)) != (1877, 1200, 301, 376):
    raise RuntimeError(f"Unexpected corpus/split sizes: {len(combined)}, {len(X_train)}, {len(X_val)}, {len(X_test)}")
if (len(zero_variance_columns), NUM_FEATURES, len(X_train_balanced), NUM_CLASSES) != (53, 244, 1280, 10):
    raise RuntimeError(
        f"Unexpected preprocessing result: removed={len(zero_variance_columns)}, retained={NUM_FEATURES}, "
        f"balanced={len(X_train_balanced)}, classes={NUM_CLASSES}"
    )

X_train_3d = X_train_balanced.reshape(-1, NUM_FEATURES, 1)
X_val_3d = X_val_scaled.reshape(-1, NUM_FEATURES, 1)
X_test_3d = X_test_scaled.reshape(-1, NUM_FEATURES, 1)
X_train_model = X_train_balanced
X_val_model = X_val_scaled
X_test_model = X_test_scaled

np.save(RUN_DIR / "X_test_3d.npy", X_test_3d)
np.save(RUN_DIR / "y_test_1d.npy", np.asarray(y_test, dtype=np.int64))
np.save(RUN_DIR / "device_classes.npy", np.asarray(label_encoder.classes_))
np.save(RUN_DIR / f"{MODEL_KEY}_device_classes.npy", np.asarray(label_encoder.classes_))
np.save(RUN_DIR / "feature_names.npy", np.asarray(feature_names, dtype=object))
np.savez_compressed(RUN_DIR / "split_indices.npz", train=index_train, validation=index_val, test=index_test)
np.savez_compressed(
    RUN_DIR / f"{MODEL_KEY}_dataset.npz",
    X_train=X_train_model,
    y_train=y_train_onehot,
    X_val=X_val_model,
    y_val=y_val_onehot,
    X_test=X_test_model,
    y_test=y_test_onehot,
)
(RUN_DIR / "zero_variance_columns.txt").write_text("".join(f"{column}\n" for column in zero_variance_columns), encoding="utf-8")
joblib.dump(imputer, RUN_DIR / "imputer.joblib")
joblib.dump(scaler, RUN_DIR / "scaler.joblib")
joblib.dump(label_encoder, RUN_DIR / "label_encoder.joblib")
joblib.dump(one_hot_encoder, RUN_DIR / "one_hot_encoder.joblib")



def build_model(hp):
    inputs = Input(shape=(NUM_FEATURES,), name="input")
    weights = Dense(NUM_FEATURES, activation="softmax", name="feature_attention")(inputs)
    x = Multiply(name="feature_weighting")([inputs, weights])
    x = Dense(hp.Choice("first_hidden_units", [256, 512]), activation="relu", name="dense_1")(x)
    x = BatchNormalization(name="batch_normalisation_1")(x)
    dropout = hp.Choice("dropout", [0.3, 0.4])
    x = Dropout(dropout, name="dropout_1")(x)
    x = Dense(hp.Choice("second_hidden_units", [128, 256]), activation="relu", name="dense_2")(x)
    x = BatchNormalization(name="batch_normalisation_2")(x)
    x = Dropout(dropout, name="dropout_2")(x)
    x = Dense(64, activation="relu", name="dense_3")(x)
    x = BatchNormalization(name="batch_normalisation_3")(x)
    outputs = Dense(NUM_CLASSES, activation="softmax", name="output")(x)
    model = Model(inputs, outputs, name="feature_attention_mlp")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(hp.Choice("learning_rate", [1e-3, 5e-4])),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)
tuner = kt.GridSearch(
    build_model,
    objective=kt.Objective("val_accuracy", direction="max"),
    max_trials=MAX_TRIALS,
    directory=str(TUNER_DIR),
    project_name=MODEL_KEY,
    overwrite=True,
    seed=42,
)
tuner.search(
    X_train_model,
    y_train_onehot,
    validation_data=(X_val_model, y_val_onehot),
    epochs=SEARCH_EPOCHS,
    batch_size=SEARCH_BATCH_SIZE,
    shuffle=True,
    callbacks=[EarlyStopping(monitor="val_accuracy", mode="max", patience=SEARCH_PATIENCE, restore_best_weights=True, verbose=1)],
    verbose=2,
)

best_hp = tuner.get_best_hyperparameters(1)[0]
(RUN_DIR / "best_hyperparameters.json").write_text(json.dumps(best_hp.values, indent=2, default=str), encoding="utf-8")
with io.StringIO() as buffer, contextlib.redirect_stdout(buffer):
    tuner.results_summary(num_trials=MAX_TRIALS)
    (RUN_DIR / "tuner_results_summary.txt").write_text(buffer.getvalue(), encoding="utf-8")

architecture_model = build_model(best_hp)
with (RUN_DIR / "model_architecture.txt").open("w", encoding="utf-8") as handle:
    architecture_model.summary(print_fn=lambda line: handle.write(line + "\n"))
(RUN_DIR / "model_architecture.json").write_text(architecture_model.to_json(), encoding="utf-8")
parameter_count = int(architecture_model.count_params())
del architecture_model
gc.collect()

seed_results = []
for seed in SEEDS:
    print("\n" + "=" * 100)
    print(f"{MODEL_NAME} — FINAL TRAINING SEED {seed}")
    print("=" * 100)
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(seed)
    model = build_model(best_hp)
    history = model.fit(
        X_train_model,
        y_train_onehot,
        validation_data=(X_val_model, y_val_onehot),
        epochs=FINAL_EPOCHS,
        batch_size=FINAL_BATCH_SIZE,
        shuffle=True,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=FINAL_PATIENCE, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_accuracy", factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=MIN_LR, verbose=1),
        ],
        verbose=2,
    )
    validation_probabilities = model.predict(X_val_model, verbose=0)
    validation_accuracy = float(np.mean(np.argmax(validation_probabilities, axis=1) == y_val))
    validation_loss = float(model.evaluate(X_val_model, y_val_onehot, verbose=0)[0])
    restored_epoch = int(np.argmin(history.history["val_loss"])) + 1
    model_path = RUN_DIR / f"mlp_seed_{seed}.keras"
    model.save(model_path)
    (RUN_DIR / f"{MODEL_KEY}_seed_{seed}_model_config.json").write_text(
        model.to_json(), encoding="utf-8"
    )
    history_frame = pd.DataFrame(history.history)
    history_frame.insert(0, "epoch", np.arange(1, len(history_frame) + 1))
    history_path = RUN_DIR / f"{MODEL_KEY}_seed_{seed}_training_history.csv"
    history_frame.to_csv(history_path, index=False)
    seed_results.append({
        "model": MODEL_NAME,
        "seed": int(seed),
        "validation_accuracy": validation_accuracy,
        "validation_loss": validation_loss,
        "restored_epoch": restored_epoch,
        "model_file": model_path.name,
        "history_file": history_path.name,
    })
    del model
    gc.collect()

champion_index = min(
    range(len(seed_results)),
    key=lambda index: (-seed_results[index]["validation_accuracy"], seed_results[index]["validation_loss"]),
)
champion = seed_results[champion_index]
shutil.copy2(RUN_DIR / champion["model_file"], RUN_DIR / CHAMPION_FILENAME)
seed_frame = pd.DataFrame(seed_results)
seed_frame["selected_champion"] = False
seed_frame.loc[champion_index, "selected_champion"] = True
seed_frame.to_csv(RUN_DIR / "validation_seed_results.csv", index=False)
seed_frame.to_csv(RUN_DIR / f"{MODEL_KEY}_five_seed_validation_results.csv", index=False)

validation_summary = pd.DataFrame({
    "Metric": [
        "Mean Restored Validation Accuracy (%)",
        "Validation Accuracy Std (%)",
        "Mean Restored Validation Loss",
        "Validation Loss Std",
        "Best Validation Accuracy (%)",
        "Best Validation Loss",
        "Best Validation Seed",
        "Restored Epoch",
    ],
    "Value": [
        seed_frame["validation_accuracy"].mean() * 100,
        seed_frame["validation_accuracy"].std(ddof=1) * 100,
        seed_frame["validation_loss"].mean(),
        seed_frame["validation_loss"].std(ddof=1),
        champion["validation_accuracy"] * 100,
        champion["validation_loss"],
        champion["seed"],
        champion["restored_epoch"],
    ],
})
validation_summary.to_csv(
    RUN_DIR / f"{MODEL_KEY}_validation_summary_statistics.csv", index=False
)


def plot_training_history(result_row, filename_prefix, title):
    history_data = pd.read_csv(RUN_DIR / result_row["history_file"])
    for metric, validation_metric, y_label in [
        ("accuracy", "val_accuracy", "Accuracy"),
        ("loss", "val_loss", "Categorical cross-entropy"),
    ]:
        figure, axis = plt.subplots(figsize=(9, 6))
        axis.plot(history_data["epoch"], history_data[metric], label=f"Training {metric}")
        axis.plot(history_data["epoch"], history_data[validation_metric], label=f"Validation {metric}")
        axis.axvline(
            result_row["restored_epoch"], linestyle="--", color="black",
            label=f"Restored epoch {result_row['restored_epoch']}"
        )
        axis.set_title(title)
        axis.set_xlabel("Epoch")
        axis.set_ylabel(y_label)
        axis.grid(True, linestyle="--", alpha=0.35)
        axis.legend()
        figure.tight_layout()
        figure.savefig(FIGURES_DIR / f"{filename_prefix}_{metric}.png", dpi=300, bbox_inches="tight")
        figure.savefig(FIGURES_DIR / f"{filename_prefix}_{metric}.pdf", bbox_inches="tight")
        plt.show()
        plt.close(figure)


if PLOT_ALL_SEEDS:
    for result_row in seed_results:
        plot_training_history(
            result_row,
            f"{MODEL_KEY}_seed_{result_row['seed']}",
            f"{MODEL_NAME}: Seed {result_row['seed']}",
        )

plot_training_history(
    champion,
    f"{MODEL_KEY}_selected_seed",
    f"{MODEL_NAME}: Validation-Selected Seed {champion['seed']}",
)

run_metadata = {
    "project": "Smart Home Appliance Identification Using Deep Learning",
    "model_name": MODEL_NAME,
    "run_started": run_started.isoformat(),
    "run_completed": datetime.now(TIMEZONE).isoformat(),
    "dataset": {
        "unique_rows": len(combined), "training_rows_before_smote": len(X_train),
        "training_rows_after_smote": len(X_train_balanced), "validation_rows": len(X_val),
        "held_out_test_rows": len(X_test), "removed_zero_variance_features": len(zero_variance_columns),
        "retained_features": NUM_FEATURES, "classes": label_encoder.classes_.tolist(),
        "split_random_state": 42, "smote_random_state": 42,
    },
    "search": {
        "method": "Keras Tuner GridSearch", "objective": "val_accuracy", "tuner_seed": 42,
        "search_space": SEARCH_SPACE, "maximum_trials": MAX_TRIALS,
        "maximum_epochs_per_trial": SEARCH_EPOCHS, "batch_size": SEARCH_BATCH_SIZE,
        "early_stopping_monitor": "val_accuracy", "early_stopping_patience": SEARCH_PATIENCE,
        "selected_hyperparameters": best_hp.values,
    },
    "final_training": {
        "seeds": SEEDS, "maximum_epochs": FINAL_EPOCHS, "batch_size": FINAL_BATCH_SIZE,
        "early_stopping_monitor": "val_loss", "early_stopping_patience": FINAL_PATIENCE,
        "reduce_lr_monitor": "val_accuracy", "reduce_lr_patience": LR_PATIENCE,
        "reduce_lr_factor": LR_FACTOR, "minimum_learning_rate": MIN_LR,
        "training_figures_for_all_seeds": PLOT_ALL_SEEDS,
    },
    "selection": {
        "rule": "highest restored validation accuracy; lower restored validation loss as tie-breaker",
        "held_out_test_used": False, "champion_seed": int(champion["seed"]),
        "champion_validation_accuracy": float(champion["validation_accuracy"]),
        "champion_validation_loss": float(champion["validation_loss"]),
        "champion_filename": CHAMPION_FILENAME,
    },
    "model_parameter_count": parameter_count,
}
(RUN_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2, default=str), encoding="utf-8")

print("\n" + "=" * 100)
print(f"{MODEL_NAME} RUN COMPLETE")
print("=" * 100)
print(seed_frame.to_string(index=False))
print(f"Selected hyperparameters: {best_hp.values}")
print(f"Validation-selected seed: {champion['seed']}")
print(f"Champion model: {RUN_DIR / CHAMPION_FILENAME}")
print(f"Temporary run directory: {RUN_DIR}")
print(f"Final run directory after successful finalisation: {FINAL_RUN_DIR}")
print("The held-out test set was archived but was not evaluated or used for selection in this cell.")
stop_terminal_capture()

try:
    shutil.move(str(RUN_DIR), str(FINAL_RUN_DIR))
except Exception as exc:
    (RUN_DIR / "RUN_FAILED.txt").write_text(
        f"Training completed, but final directory creation failed.\n{type(exc).__name__}: {exc}\n",
        encoding="utf-8",
    )
    raise
RUN_DIR = FINAL_RUN_DIR
FIGURES_DIR = RUN_DIR / "Figures"
TUNER_DIR = RUN_DIR / "Tuner"
try:
    inventory_rows = []
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file() and path.name != "file_inventory.csv":
            inventory_rows.append({"relative_path": str(path.relative_to(RUN_DIR)), "size_bytes": path.stat().st_size, "sha256": sha256_file(path)})
    pd.DataFrame(inventory_rows).to_csv(RUN_DIR / "file_inventory.csv", index=False)
    (RUN_DIR / "RUN_COMPLETE.txt").write_text(
        "Run completed and finalised successfully.\n", encoding="utf-8"
    )
except Exception as exc:
    (RUN_DIR / "RUN_FAILED.txt").write_text(
        f"Training completed, but archive inventory finalisation failed.\n{type(exc).__name__}: {exc}\n",
        encoding="utf-8",
    )
    raise
print(f"Final run directory: {RUN_DIR}")
print(f"Terminal output: {RUN_DIR / 'terminal_output.txt'}")

### Baseline Models

#### CNN

The CNN is a tuned neural comparator that tests whether local interactions among adjacent positions in the fixed engineered-feature order provide useful discriminatory information. The feature axis is not chronological packet time, so its convolution must be interpreted as operating over neighbouring engineered predictors rather than successive packets.

This training cell is self-contained. It reconstructs the corrected corpus, deterministic split, training-derived preprocessing and SMOTE-balanced training data internally.

Each observation is supplied as:

```text
(samples, 244, 1)
```

The architecture is:

```text
Input (244, 1)
→ Conv1D
→ MaxPooling1D
→ Flatten
→ Dense(ReLU)
→ Dropout
→ Softmax output (10 classes)
```

Keras Tuner GridSearch evaluates all 24 configurations:

```text
convolutional filters:  32 or 64
dense units:            32 or 64
dropout:                0.2, 0.3 or 0.4
Adam learning rate:     0.001 or 0.0005
kernel size:            fixed at 3
padding:                fixed as valid
```

Search trials use at most 50 epochs with batch size 32. Validation accuracy is the tuning objective, and early stopping monitors validation accuracy with patience 8 and restores the best weights.

The selected hyperparameter tuple is trained independently using seeds:

```text
42, 7, 123, 2024, 99
```

Final training uses at most 100 epochs and batch size 32. Early stopping monitors validation loss with patience 15 and restores the best weights. `ReduceLROnPlateau` monitors validation accuracy with patience 5, reduction factor 0.5 and minimum learning rate `1e-5`.

The retained seed is selected by highest restored validation accuracy, with lower restored validation loss as the tie-breaker. Held-out test results are not used for tuning or seed selection.

Because `PLOT_ALL_SEEDS=False`, the cell stores every seed history but generates training/validation accuracy and loss figures only for the validation-selected seed.

Successful runs are finalised under:

```text
Honours_Project/Models/Tuned/CNN_Tuned/<timestamp>/
```

The retained model is saved as:

```text
best_validation_cnn.keras
```

The run also preserves all five seed models, fitted preprocessing objects, split indices, held-out arrays, selected hyperparameters, validation summaries, histories, terminal output, executable source, source hash, runtime environment, system diagnostics, package requirements and a file inventory.

In [ ]:
# TUNED CNN BASELINE: SELF-CONTAINED TUNING, FIVE-SEED TRAINING AND ARCHIVING

try:
    get_ipython().run_line_magic("pip", "install -q keras-tuner imbalanced-learn psutil")
except NameError:
    pass

import contextlib
import gc
import hashlib
import importlib.metadata as metadata
import io
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

import imblearn
import joblib
import keras
import keras_tuner as kt
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import sklearn
import tensorflow as tf
from google.colab import drive
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils import shuffle
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Conv1D, Dense, Dropout, Flatten, Input, MaxPooling1D
from tensorflow.keras.models import Model

try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print(f"TensorFlow deterministic operations could not be enabled: {exc}")

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

MODEL_NAME = 'Tuned CNN Baseline'
MODEL_KEY = 'cnn'
PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
TRAIN_CSV = PROJECT_ROOT / "iot_device_train.csv"
TEST_CSV = PROJECT_ROOT / "iot_device_test.csv"
OUTPUT_PARENT = PROJECT_ROOT / "Models" / "Tuned" / 'CNN_Tuned'
TIMEZONE = ZoneInfo("Africa/Johannesburg")
SEEDS = [42, 7, 123, 2024, 99]
SEARCH_SPACE = {"filters": [32, 64], "dense_units": [32, 64], "dropout": [0.2, 0.3, 0.4], "learning_rate": [0.001, 0.0005], "fixed_kernel_size": 3}
MAX_TRIALS = 24
SEARCH_EPOCHS = 50
FINAL_EPOCHS = 100
SEARCH_BATCH_SIZE = 32
FINAL_BATCH_SIZE = 32
PLOT_ALL_SEEDS = False
SEARCH_PATIENCE = 8
FINAL_PATIENCE = 15
LR_PATIENCE = 5
LR_FACTOR = 0.5
MIN_LR = 1e-5
CHAMPION_FILENAME = 'best_validation_cnn.keras'


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def command_record(command):
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=False)
        return {"command": command, "returncode": result.returncode, "stdout": result.stdout, "stderr": result.stderr}
    except Exception as exc:
        return {"command": command, "returncode": None, "stdout": "", "stderr": repr(exc)}


class TeeStream:
    def __init__(self, original_stream, log_handle):
        self.original_stream = original_stream
        self.log_handle = log_handle

    def write(self, message):
        self.original_stream.write(message)
        self.original_stream.flush()
        self.log_handle.write(message)
        self.log_handle.flush()

    def flush(self):
        self.original_stream.flush()
        self.log_handle.flush()

    def isatty(self):
        return self.original_stream.isatty()

    def fileno(self):
        return self.original_stream.fileno()


terminal_capture = {}


def stop_terminal_capture(result=None):
    if not terminal_capture:
        return
    sys.stdout = terminal_capture["stdout"]
    sys.stderr = terminal_capture["stderr"]
    terminal_capture["handle"].flush()
    terminal_capture["handle"].close()
    failed = bool(
        result is not None
        and (getattr(result, "error_before_exec", None) is not None or getattr(result, "error_in_exec", None) is not None)
    )
    if failed:
        error = getattr(result, "error_before_exec", None) or getattr(result, "error_in_exec", None)
        terminal_capture["run_dir"].joinpath("RUN_FAILED.txt").write_text(
            f"Run failed before finalisation.\n{type(error).__name__}: {error}\n",
            encoding="utf-8",
        )
    try:
        get_ipython().events.unregister("post_run_cell", stop_terminal_capture)
    except Exception:
        pass
    terminal_capture.clear()


def start_terminal_capture(log_path, run_dir):
    log_handle = Path(log_path).open("w", encoding="utf-8")
    terminal_capture.update({
        "stdout": sys.stdout,
        "stderr": sys.stderr,
        "handle": log_handle,
        "run_dir": Path(run_dir),
    })
    sys.stdout = TeeStream(terminal_capture["stdout"], log_handle)
    sys.stderr = TeeStream(terminal_capture["stderr"], log_handle)
    try:
        get_ipython().events.register("post_run_cell", stop_terminal_capture)
    except Exception:
        pass


def version(distribution):
    try:
        return metadata.version(distribution)
    except metadata.PackageNotFoundError:
        return "not installed"


def current_cell_source():
    try:
        history = get_ipython().history_manager.input_hist_raw
        if history and str(history[-1]).strip():
            return str(history[-1])
    except Exception:
        pass
    return "# Current notebook-cell source was unavailable.\n"


for required_path in (TRAIN_CSV, TEST_CSV):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

run_started = datetime.now(TIMEZONE)
RUN_TIMESTAMP = run_started.strftime("%Y-%m-%d_%H-%M-%S")
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
FINAL_RUN_DIR = OUTPUT_PARENT / RUN_TIMESTAMP
RUN_DIR = OUTPUT_PARENT / f"_TEMP_{MODEL_KEY.upper()}_{RUN_TIMESTAMP}"
if RUN_DIR.exists() or FINAL_RUN_DIR.exists():
    raise FileExistsError(f"Run directory already exists for timestamp {RUN_TIMESTAMP}")
FIGURES_DIR = RUN_DIR / "Figures"
TUNER_DIR = RUN_DIR / "Tuner"
FIGURES_DIR.mkdir(parents=True, exist_ok=False)
start_terminal_capture(RUN_DIR / "terminal_output.txt", RUN_DIR)

runtime_environment = {
    "captured_at": run_started.isoformat(),
    "timezone": "Africa/Johannesburg",
    "python": {"version": sys.version, "executable": sys.executable, "implementation": platform.python_implementation()},
    "operating_system": {"platform": platform.platform(), "system": platform.system(), "release": platform.release(), "machine": platform.machine(), "processor": platform.processor()},
    "hardware": {
        "physical_cpu_count": psutil.cpu_count(logical=False),
        "logical_cpu_count": psutil.cpu_count(logical=True),
        "memory_total_bytes": int(psutil.virtual_memory().total),
        "tensorflow_visible_cpus": [device.name for device in tf.config.list_physical_devices("CPU")],
        "tensorflow_visible_gpus": [device.name for device in tf.config.list_physical_devices("GPU")],
    },
    "colab": {"release_tag": os.environ.get("COLAB_RELEASE_TAG"), "gpu_type": os.environ.get("COLAB_GPU")},
    "software": {
        "tensorflow": version("tensorflow"), "keras": version("keras"), "keras_tuner": version("keras-tuner"),
        "numpy": version("numpy"), "pandas": version("pandas"), "scikit_learn": version("scikit-learn"),
        "imbalanced_learn": version("imbalanced-learn"), "joblib": version("joblib"),
        "matplotlib": version("matplotlib"), "psutil": version("psutil"),
    },
    "tensorflow_build": tf.sysconfig.get_build_info(),
    "determinism": {"TF_DETERMINISTIC_OPS": os.environ.get("TF_DETERMINISTIC_OPS"), "TF_CUDNN_DETERMINISTIC": os.environ.get("TF_CUDNN_DETERMINISTIC")},
}
(RUN_DIR / "runtime_environment.json").write_text(json.dumps(runtime_environment, indent=2, default=str), encoding="utf-8")
(RUN_DIR / "system_diagnostics.json").write_text(
    json.dumps({
        "uname": command_record(["uname", "-a"]), "lscpu": command_record(["lscpu"]),
        "memory": command_record(["free", "-h"]), "nvidia_smi": command_record(["nvidia-smi"]),
    }, indent=2, default=str), encoding="utf-8"
)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=False)
(RUN_DIR / "requirements.txt").write_text(freeze.stdout, encoding="utf-8")
source_path = RUN_DIR / "training_cell_source.py"
source_path.write_text(current_cell_source(), encoding="utf-8")
(RUN_DIR / "training_cell_source.sha256").write_text(sha256_file(source_path), encoding="utf-8")

dataset_manifest = {
    "publisher_train": {"path": str(TRAIN_CSV), "size_bytes": TRAIN_CSV.stat().st_size, "sha256": sha256_file(TRAIN_CSV)},
    "publisher_test": {"path": str(TEST_CSV), "size_bytes": TEST_CSV.stat().st_size, "sha256": sha256_file(TEST_CSV)},
}
(RUN_DIR / "dataset_manifest.json").write_text(json.dumps(dataset_manifest, indent=2), encoding="utf-8")

publisher_train = pd.read_csv(TRAIN_CSV)
publisher_test = pd.read_csv(TEST_CSV)
combined = pd.concat([publisher_train, publisher_test], ignore_index=True).drop_duplicates().reset_index(drop=True)
X_all = combined.drop(columns="device_category").replace([np.inf, -np.inf], np.nan)
y_all = combined["device_category"].astype(str)

label_encoder = LabelEncoder()
y_all_int = label_encoder.fit_transform(y_all)
all_indices = np.arange(len(combined), dtype=np.int64)
X_development, X_test, y_development, y_test, index_development, index_test = train_test_split(
    X_all, y_all_int, all_indices, test_size=0.20, random_state=42, stratify=y_all_int
)
X_train, X_val, y_train, y_val, index_train, index_val = train_test_split(
    X_development, y_development, index_development, test_size=0.20, random_state=42, stratify=y_development
)

zero_variance_columns = X_train.columns[X_train.var() == 0].tolist()
X_train = X_train.drop(columns=zero_variance_columns)
X_val = X_val.drop(columns=zero_variance_columns)
X_test = X_test.drop(columns=zero_variance_columns)
feature_names = X_train.columns.tolist()

train_hashes = set(pd.util.hash_pandas_object(X_train, index=False).astype(str))
val_hashes = set(pd.util.hash_pandas_object(X_val, index=False).astype(str))
test_hashes = set(pd.util.hash_pandas_object(X_test, index=False).astype(str))
if train_hashes & val_hashes or train_hashes & test_hashes or val_hashes & test_hashes:
    raise RuntimeError("An exact feature vector occurs in more than one partition.")

imputer = SimpleImputer(strategy="mean")
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed).astype(np.float32)
X_val_scaled = scaler.transform(X_val_imputed).astype(np.float32)
X_test_scaled = scaler.transform(X_test_imputed).astype(np.float32)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)
X_train_balanced, y_train_balanced = shuffle(X_train_balanced, y_train_balanced, random_state=42)
X_train_balanced = np.asarray(X_train_balanced, dtype=np.float32)
NUM_CLASSES = len(label_encoder.classes_)
NUM_FEATURES = X_train_balanced.shape[1]
y_train_onehot = tf.keras.utils.to_categorical(y_train_balanced, NUM_CLASSES).astype(np.float32)
y_val_onehot = tf.keras.utils.to_categorical(y_val, NUM_CLASSES).astype(np.float32)
y_test_onehot = tf.keras.utils.to_categorical(y_test, NUM_CLASSES).astype(np.float32)
try:
    one_hot_encoder = OneHotEncoder(
        categories=[label_encoder.classes_], sparse_output=False, handle_unknown="error"
    )
except TypeError:
    one_hot_encoder = OneHotEncoder(
        categories=[label_encoder.classes_], sparse=False, handle_unknown="error"
    )
one_hot_encoder.fit(np.asarray(label_encoder.classes_).reshape(-1, 1))

if (len(combined), len(X_train), len(X_val), len(X_test)) != (1877, 1200, 301, 376):
    raise RuntimeError(f"Unexpected corpus/split sizes: {len(combined)}, {len(X_train)}, {len(X_val)}, {len(X_test)}")
if (len(zero_variance_columns), NUM_FEATURES, len(X_train_balanced), NUM_CLASSES) != (53, 244, 1280, 10):
    raise RuntimeError(
        f"Unexpected preprocessing result: removed={len(zero_variance_columns)}, retained={NUM_FEATURES}, "
        f"balanced={len(X_train_balanced)}, classes={NUM_CLASSES}"
    )

X_train_3d = X_train_balanced.reshape(-1, NUM_FEATURES, 1)
X_val_3d = X_val_scaled.reshape(-1, NUM_FEATURES, 1)
X_test_3d = X_test_scaled.reshape(-1, NUM_FEATURES, 1)
X_train_model = X_train_3d
X_val_model = X_val_3d
X_test_model = X_test_3d

np.save(RUN_DIR / "X_test_3d.npy", X_test_3d)
np.save(RUN_DIR / "y_test_1d.npy", np.asarray(y_test, dtype=np.int64))
np.save(RUN_DIR / "device_classes.npy", np.asarray(label_encoder.classes_))
np.save(RUN_DIR / f"{MODEL_KEY}_device_classes.npy", np.asarray(label_encoder.classes_))
np.save(RUN_DIR / "feature_names.npy", np.asarray(feature_names, dtype=object))
np.savez_compressed(RUN_DIR / "split_indices.npz", train=index_train, validation=index_val, test=index_test)
np.savez_compressed(
    RUN_DIR / f"{MODEL_KEY}_dataset.npz",
    X_train=X_train_model,
    y_train=y_train_onehot,
    X_val=X_val_model,
    y_val=y_val_onehot,
    X_test=X_test_model,
    y_test=y_test_onehot,
)
(RUN_DIR / "zero_variance_columns.txt").write_text("".join(f"{column}\n" for column in zero_variance_columns), encoding="utf-8")
joblib.dump(imputer, RUN_DIR / "imputer.joblib")
joblib.dump(scaler, RUN_DIR / "scaler.joblib")
joblib.dump(label_encoder, RUN_DIR / "label_encoder.joblib")
joblib.dump(one_hot_encoder, RUN_DIR / "one_hot_encoder.joblib")



def build_model(hp):
    inputs = Input(shape=(NUM_FEATURES, 1), name="input")
    x = Conv1D(hp.Choice("filters", [32, 64]), 3, padding="valid", activation="relu", name="convolution")(inputs)
    x = MaxPooling1D(2, name="max_pooling")(x)
    x = Flatten(name="flatten")(x)
    x = Dense(hp.Choice("dense_units", [32, 64]), activation="relu", name="dense")(x)
    x = Dropout(hp.Choice("dropout", [0.2, 0.3, 0.4]), name="dropout")(x)
    outputs = Dense(NUM_CLASSES, activation="softmax", name="output")(x)
    model = Model(inputs, outputs, name="tuned_cnn_baseline")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(hp.Choice("learning_rate", [1e-3, 5e-4])),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)
tuner = kt.GridSearch(
    build_model,
    objective=kt.Objective("val_accuracy", direction="max"),
    max_trials=MAX_TRIALS,
    directory=str(TUNER_DIR),
    project_name=MODEL_KEY,
    overwrite=True,
    seed=42,
)
tuner.search(
    X_train_model,
    y_train_onehot,
    validation_data=(X_val_model, y_val_onehot),
    epochs=SEARCH_EPOCHS,
    batch_size=SEARCH_BATCH_SIZE,
    shuffle=True,
    callbacks=[EarlyStopping(monitor="val_accuracy", mode="max", patience=SEARCH_PATIENCE, restore_best_weights=True, verbose=1)],
    verbose=2,
)

best_hp = tuner.get_best_hyperparameters(1)[0]
(RUN_DIR / "best_hyperparameters.json").write_text(json.dumps(best_hp.values, indent=2, default=str), encoding="utf-8")
with io.StringIO() as buffer, contextlib.redirect_stdout(buffer):
    tuner.results_summary(num_trials=MAX_TRIALS)
    (RUN_DIR / "tuner_results_summary.txt").write_text(buffer.getvalue(), encoding="utf-8")

architecture_model = build_model(best_hp)
with (RUN_DIR / "model_architecture.txt").open("w", encoding="utf-8") as handle:
    architecture_model.summary(print_fn=lambda line: handle.write(line + "\n"))
(RUN_DIR / "model_architecture.json").write_text(architecture_model.to_json(), encoding="utf-8")
parameter_count = int(architecture_model.count_params())
del architecture_model
gc.collect()

seed_results = []
for seed in SEEDS:
    print("\n" + "=" * 100)
    print(f"{MODEL_NAME} — FINAL TRAINING SEED {seed}")
    print("=" * 100)
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(seed)
    model = build_model(best_hp)
    history = model.fit(
        X_train_model,
        y_train_onehot,
        validation_data=(X_val_model, y_val_onehot),
        epochs=FINAL_EPOCHS,
        batch_size=FINAL_BATCH_SIZE,
        shuffle=True,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=FINAL_PATIENCE, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_accuracy", factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=MIN_LR, verbose=1),
        ],
        verbose=2,
    )
    validation_probabilities = model.predict(X_val_model, verbose=0)
    validation_accuracy = float(np.mean(np.argmax(validation_probabilities, axis=1) == y_val))
    validation_loss = float(model.evaluate(X_val_model, y_val_onehot, verbose=0)[0])
    restored_epoch = int(np.argmin(history.history["val_loss"])) + 1
    model_path = RUN_DIR / f"cnn_seed_{seed}.keras"
    model.save(model_path)
    (RUN_DIR / f"{MODEL_KEY}_seed_{seed}_model_config.json").write_text(
        model.to_json(), encoding="utf-8"
    )
    history_frame = pd.DataFrame(history.history)
    history_frame.insert(0, "epoch", np.arange(1, len(history_frame) + 1))
    history_path = RUN_DIR / f"{MODEL_KEY}_seed_{seed}_training_history.csv"
    history_frame.to_csv(history_path, index=False)
    seed_results.append({
        "model": MODEL_NAME,
        "seed": int(seed),
        "validation_accuracy": validation_accuracy,
        "validation_loss": validation_loss,
        "restored_epoch": restored_epoch,
        "model_file": model_path.name,
        "history_file": history_path.name,
    })
    del model
    gc.collect()

champion_index = min(
    range(len(seed_results)),
    key=lambda index: (-seed_results[index]["validation_accuracy"], seed_results[index]["validation_loss"]),
)
champion = seed_results[champion_index]
shutil.copy2(RUN_DIR / champion["model_file"], RUN_DIR / CHAMPION_FILENAME)
seed_frame = pd.DataFrame(seed_results)
seed_frame["selected_champion"] = False
seed_frame.loc[champion_index, "selected_champion"] = True
seed_frame.to_csv(RUN_DIR / "validation_seed_results.csv", index=False)
seed_frame.to_csv(RUN_DIR / f"{MODEL_KEY}_five_seed_validation_results.csv", index=False)

validation_summary = pd.DataFrame({
    "Metric": [
        "Mean Restored Validation Accuracy (%)",
        "Validation Accuracy Std (%)",
        "Mean Restored Validation Loss",
        "Validation Loss Std",
        "Best Validation Accuracy (%)",
        "Best Validation Loss",
        "Best Validation Seed",
        "Restored Epoch",
    ],
    "Value": [
        seed_frame["validation_accuracy"].mean() * 100,
        seed_frame["validation_accuracy"].std(ddof=1) * 100,
        seed_frame["validation_loss"].mean(),
        seed_frame["validation_loss"].std(ddof=1),
        champion["validation_accuracy"] * 100,
        champion["validation_loss"],
        champion["seed"],
        champion["restored_epoch"],
    ],
})
validation_summary.to_csv(
    RUN_DIR / f"{MODEL_KEY}_validation_summary_statistics.csv", index=False
)


def plot_training_history(result_row, filename_prefix, title):
    history_data = pd.read_csv(RUN_DIR / result_row["history_file"])
    for metric, validation_metric, y_label in [
        ("accuracy", "val_accuracy", "Accuracy"),
        ("loss", "val_loss", "Categorical cross-entropy"),
    ]:
        figure, axis = plt.subplots(figsize=(9, 6))
        axis.plot(history_data["epoch"], history_data[metric], label=f"Training {metric}")
        axis.plot(history_data["epoch"], history_data[validation_metric], label=f"Validation {metric}")
        axis.axvline(
            result_row["restored_epoch"], linestyle="--", color="black",
            label=f"Restored epoch {result_row['restored_epoch']}"
        )
        axis.set_title(title)
        axis.set_xlabel("Epoch")
        axis.set_ylabel(y_label)
        axis.grid(True, linestyle="--", alpha=0.35)
        axis.legend()
        figure.tight_layout()
        figure.savefig(FIGURES_DIR / f"{filename_prefix}_{metric}.png", dpi=300, bbox_inches="tight")
        figure.savefig(FIGURES_DIR / f"{filename_prefix}_{metric}.pdf", bbox_inches="tight")
        plt.show()
        plt.close(figure)


if PLOT_ALL_SEEDS:
    for result_row in seed_results:
        plot_training_history(
            result_row,
            f"{MODEL_KEY}_seed_{result_row['seed']}",
            f"{MODEL_NAME}: Seed {result_row['seed']}",
        )

plot_training_history(
    champion,
    f"{MODEL_KEY}_selected_seed",
    f"{MODEL_NAME}: Validation-Selected Seed {champion['seed']}",
)

run_metadata = {
    "project": "Smart Home Appliance Identification Using Deep Learning",
    "model_name": MODEL_NAME,
    "run_started": run_started.isoformat(),
    "run_completed": datetime.now(TIMEZONE).isoformat(),
    "dataset": {
        "unique_rows": len(combined), "training_rows_before_smote": len(X_train),
        "training_rows_after_smote": len(X_train_balanced), "validation_rows": len(X_val),
        "held_out_test_rows": len(X_test), "removed_zero_variance_features": len(zero_variance_columns),
        "retained_features": NUM_FEATURES, "classes": label_encoder.classes_.tolist(),
        "split_random_state": 42, "smote_random_state": 42,
    },
    "search": {
        "method": "Keras Tuner GridSearch", "objective": "val_accuracy", "tuner_seed": 42,
        "search_space": SEARCH_SPACE, "maximum_trials": MAX_TRIALS,
        "maximum_epochs_per_trial": SEARCH_EPOCHS, "batch_size": SEARCH_BATCH_SIZE,
        "early_stopping_monitor": "val_accuracy", "early_stopping_patience": SEARCH_PATIENCE,
        "selected_hyperparameters": best_hp.values,
    },
    "final_training": {
        "seeds": SEEDS, "maximum_epochs": FINAL_EPOCHS, "batch_size": FINAL_BATCH_SIZE,
        "early_stopping_monitor": "val_loss", "early_stopping_patience": FINAL_PATIENCE,
        "reduce_lr_monitor": "val_accuracy", "reduce_lr_patience": LR_PATIENCE,
        "reduce_lr_factor": LR_FACTOR, "minimum_learning_rate": MIN_LR,
        "training_figures_for_all_seeds": PLOT_ALL_SEEDS,
    },
    "selection": {
        "rule": "highest restored validation accuracy; lower restored validation loss as tie-breaker",
        "held_out_test_used": False, "champion_seed": int(champion["seed"]),
        "champion_validation_accuracy": float(champion["validation_accuracy"]),
        "champion_validation_loss": float(champion["validation_loss"]),
        "champion_filename": CHAMPION_FILENAME,
    },
    "model_parameter_count": parameter_count,
}
(RUN_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2, default=str), encoding="utf-8")

print("\n" + "=" * 100)
print(f"{MODEL_NAME} RUN COMPLETE")
print("=" * 100)
print(seed_frame.to_string(index=False))
print(f"Selected hyperparameters: {best_hp.values}")
print(f"Validation-selected seed: {champion['seed']}")
print(f"Champion model: {RUN_DIR / CHAMPION_FILENAME}")
print(f"Temporary run directory: {RUN_DIR}")
print(f"Final run directory after successful finalisation: {FINAL_RUN_DIR}")
print("The held-out test set was archived but was not evaluated or used for selection in this cell.")
stop_terminal_capture()

try:
    shutil.move(str(RUN_DIR), str(FINAL_RUN_DIR))
except Exception as exc:
    (RUN_DIR / "RUN_FAILED.txt").write_text(
        f"Training completed, but final directory creation failed.\n{type(exc).__name__}: {exc}\n",
        encoding="utf-8",
    )
    raise
RUN_DIR = FINAL_RUN_DIR
FIGURES_DIR = RUN_DIR / "Figures"
TUNER_DIR = RUN_DIR / "Tuner"
try:
    inventory_rows = []
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file() and path.name != "file_inventory.csv":
            inventory_rows.append({"relative_path": str(path.relative_to(RUN_DIR)), "size_bytes": path.stat().st_size, "sha256": sha256_file(path)})
    pd.DataFrame(inventory_rows).to_csv(RUN_DIR / "file_inventory.csv", index=False)
    (RUN_DIR / "RUN_COMPLETE.txt").write_text(
        "Run completed and finalised successfully.\n", encoding="utf-8"
    )
except Exception as exc:
    (RUN_DIR / "RUN_FAILED.txt").write_text(
        f"Training completed, but archive inventory finalisation failed.\n{type(exc).__name__}: {exc}\n",
        encoding="utf-8",
    )
    raise
print(f"Final run directory: {RUN_DIR}")
print(f"Terminal output: {RUN_DIR / 'terminal_output.txt'}")

#### LSTM

The LSTM is a tuned recurrent comparator that tests whether traversing the fixed engineered-feature order in one direction provides useful classification evidence.

Each observation is reshaped from 244 retained predictors to:

```text
(samples, 244, 1)
```

This sequence axis represents the fixed predictor ordering and not packet time. The model must therefore not be described as directly learning packet-level temporal behaviour.

The architecture is:

```text
Input (244, 1)
→ LSTM(return_sequences=False)
→ Dense(ReLU)
→ Dropout
→ Softmax output (10 classes)
```

Keras Tuner GridSearch evaluates all 24 configurations:

```text
LSTM units:          32 or 64
dense units:         32 or 64
dropout:             0.2, 0.3 or 0.4
Adam learning rate:  0.001 or 0.0005
```

Search trials use at most 50 epochs and batch size 32. Validation accuracy is the tuning objective. Early stopping monitors validation accuracy with patience 8 and restores the best weights.

The selected hyperparameter tuple is trained independently using seeds:

```text
42, 7, 123, 2024, 99
```

Final training uses at most 100 epochs and batch size 32. Early stopping monitors validation loss with patience 15 and restores the best weights. `ReduceLROnPlateau` monitors validation accuracy with patience 5, reduction factor 0.5 and minimum learning rate `1e-5`.

The retained seed is selected by highest restored validation accuracy, with lower restored validation loss as the tie-breaker. The held-out test set does not participate in tuning or seed selection.

Because `PLOT_ALL_SEEDS=False`, every seed history is archived, but training/validation figures are generated only for the validation-selected seed:

```text
Figures/lstm_selected_seed_accuracy.png
Figures/lstm_selected_seed_accuracy.pdf
Figures/lstm_selected_seed_loss.png
Figures/lstm_selected_seed_loss.pdf
```

Successful runs are finalised under:

```text
Honours_Project/Models/Tuned/LSTM_Tuned/<timestamp>/
```

The retained model is saved as:

```text
best_validation_lstm.keras
```

The run also preserves all five seed models, preprocessing objects, split indices, held-out arrays, tuner outputs, selected hyperparameters, validation results, histories, terminal output, source provenance, runtime environment, system diagnostics, requirements and a file inventory.

In [ ]:
# TUNED LSTM BASELINE: SELF-CONTAINED TUNING, FIVE-SEED TRAINING AND ARCHIVING

try:
    get_ipython().run_line_magic("pip", "install -q keras-tuner imbalanced-learn psutil")
except NameError:
    pass

import contextlib
import gc
import hashlib
import importlib.metadata as metadata
import io
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

import imblearn
import joblib
import keras
import keras_tuner as kt
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import sklearn
import tensorflow as tf
from google.colab import drive
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils import shuffle
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
from tensorflow.keras.models import Model

try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print(f"TensorFlow deterministic operations could not be enabled: {exc}")

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

MODEL_NAME = 'Tuned LSTM Baseline'
MODEL_KEY = 'lstm'
PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
TRAIN_CSV = PROJECT_ROOT / "iot_device_train.csv"
TEST_CSV = PROJECT_ROOT / "iot_device_test.csv"
OUTPUT_PARENT = PROJECT_ROOT / "Models" / "Tuned" / 'LSTM_Tuned'
TIMEZONE = ZoneInfo("Africa/Johannesburg")
SEEDS = [42, 7, 123, 2024, 99]
SEARCH_SPACE = {"lstm_units": [32, 64], "dense_units": [32, 64], "dropout": [0.2, 0.3, 0.4], "learning_rate": [0.001, 0.0005]}
MAX_TRIALS = 24
SEARCH_EPOCHS = 50
FINAL_EPOCHS = 100
SEARCH_BATCH_SIZE = 32
FINAL_BATCH_SIZE = 32
PLOT_ALL_SEEDS = False
SEARCH_PATIENCE = 8
FINAL_PATIENCE = 15
LR_PATIENCE = 5
LR_FACTOR = 0.5
MIN_LR = 1e-5
CHAMPION_FILENAME = 'best_validation_lstm.keras'


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def command_record(command):
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=False)
        return {"command": command, "returncode": result.returncode, "stdout": result.stdout, "stderr": result.stderr}
    except Exception as exc:
        return {"command": command, "returncode": None, "stdout": "", "stderr": repr(exc)}


class TeeStream:
    def __init__(self, original_stream, log_handle):
        self.original_stream = original_stream
        self.log_handle = log_handle

    def write(self, message):
        self.original_stream.write(message)
        self.original_stream.flush()
        self.log_handle.write(message)
        self.log_handle.flush()

    def flush(self):
        self.original_stream.flush()
        self.log_handle.flush()

    def isatty(self):
        return self.original_stream.isatty()

    def fileno(self):
        return self.original_stream.fileno()


terminal_capture = {}


def stop_terminal_capture(result=None):
    if not terminal_capture:
        return
    sys.stdout = terminal_capture["stdout"]
    sys.stderr = terminal_capture["stderr"]
    terminal_capture["handle"].flush()
    terminal_capture["handle"].close()
    failed = bool(
        result is not None
        and (getattr(result, "error_before_exec", None) is not None or getattr(result, "error_in_exec", None) is not None)
    )
    if failed:
        error = getattr(result, "error_before_exec", None) or getattr(result, "error_in_exec", None)
        terminal_capture["run_dir"].joinpath("RUN_FAILED.txt").write_text(
            f"Run failed before finalisation.\n{type(error).__name__}: {error}\n",
            encoding="utf-8",
        )
    try:
        get_ipython().events.unregister("post_run_cell", stop_terminal_capture)
    except Exception:
        pass
    terminal_capture.clear()


def start_terminal_capture(log_path, run_dir):
    log_handle = Path(log_path).open("w", encoding="utf-8")
    terminal_capture.update({
        "stdout": sys.stdout,
        "stderr": sys.stderr,
        "handle": log_handle,
        "run_dir": Path(run_dir),
    })
    sys.stdout = TeeStream(terminal_capture["stdout"], log_handle)
    sys.stderr = TeeStream(terminal_capture["stderr"], log_handle)
    try:
        get_ipython().events.register("post_run_cell", stop_terminal_capture)
    except Exception:
        pass


def version(distribution):
    try:
        return metadata.version(distribution)
    except metadata.PackageNotFoundError:
        return "not installed"


def current_cell_source():
    try:
        history = get_ipython().history_manager.input_hist_raw
        if history and str(history[-1]).strip():
            return str(history[-1])
    except Exception:
        pass
    return "# Current notebook-cell source was unavailable.\n"


for required_path in (TRAIN_CSV, TEST_CSV):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

run_started = datetime.now(TIMEZONE)
RUN_TIMESTAMP = run_started.strftime("%Y-%m-%d_%H-%M-%S")
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
FINAL_RUN_DIR = OUTPUT_PARENT / RUN_TIMESTAMP
RUN_DIR = OUTPUT_PARENT / f"_TEMP_{MODEL_KEY.upper()}_{RUN_TIMESTAMP}"
if RUN_DIR.exists() or FINAL_RUN_DIR.exists():
    raise FileExistsError(f"Run directory already exists for timestamp {RUN_TIMESTAMP}")
FIGURES_DIR = RUN_DIR / "Figures"
TUNER_DIR = RUN_DIR / "Tuner"
FIGURES_DIR.mkdir(parents=True, exist_ok=False)
start_terminal_capture(RUN_DIR / "terminal_output.txt", RUN_DIR)

runtime_environment = {
    "captured_at": run_started.isoformat(),
    "timezone": "Africa/Johannesburg",
    "python": {"version": sys.version, "executable": sys.executable, "implementation": platform.python_implementation()},
    "operating_system": {"platform": platform.platform(), "system": platform.system(), "release": platform.release(), "machine": platform.machine(), "processor": platform.processor()},
    "hardware": {
        "physical_cpu_count": psutil.cpu_count(logical=False),
        "logical_cpu_count": psutil.cpu_count(logical=True),
        "memory_total_bytes": int(psutil.virtual_memory().total),
        "tensorflow_visible_cpus": [device.name for device in tf.config.list_physical_devices("CPU")],
        "tensorflow_visible_gpus": [device.name for device in tf.config.list_physical_devices("GPU")],
    },
    "colab": {"release_tag": os.environ.get("COLAB_RELEASE_TAG"), "gpu_type": os.environ.get("COLAB_GPU")},
    "software": {
        "tensorflow": version("tensorflow"), "keras": version("keras"), "keras_tuner": version("keras-tuner"),
        "numpy": version("numpy"), "pandas": version("pandas"), "scikit_learn": version("scikit-learn"),
        "imbalanced_learn": version("imbalanced-learn"), "joblib": version("joblib"),
        "matplotlib": version("matplotlib"), "psutil": version("psutil"),
    },
    "tensorflow_build": tf.sysconfig.get_build_info(),
    "determinism": {"TF_DETERMINISTIC_OPS": os.environ.get("TF_DETERMINISTIC_OPS"), "TF_CUDNN_DETERMINISTIC": os.environ.get("TF_CUDNN_DETERMINISTIC")},
}
(RUN_DIR / "runtime_environment.json").write_text(json.dumps(runtime_environment, indent=2, default=str), encoding="utf-8")
(RUN_DIR / "system_diagnostics.json").write_text(
    json.dumps({
        "uname": command_record(["uname", "-a"]), "lscpu": command_record(["lscpu"]),
        "memory": command_record(["free", "-h"]), "nvidia_smi": command_record(["nvidia-smi"]),
    }, indent=2, default=str), encoding="utf-8"
)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=False)
(RUN_DIR / "requirements.txt").write_text(freeze.stdout, encoding="utf-8")
source_path = RUN_DIR / "training_cell_source.py"
source_path.write_text(current_cell_source(), encoding="utf-8")
(RUN_DIR / "training_cell_source.sha256").write_text(sha256_file(source_path), encoding="utf-8")

dataset_manifest = {
    "publisher_train": {"path": str(TRAIN_CSV), "size_bytes": TRAIN_CSV.stat().st_size, "sha256": sha256_file(TRAIN_CSV)},
    "publisher_test": {"path": str(TEST_CSV), "size_bytes": TEST_CSV.stat().st_size, "sha256": sha256_file(TEST_CSV)},
}
(RUN_DIR / "dataset_manifest.json").write_text(json.dumps(dataset_manifest, indent=2), encoding="utf-8")

publisher_train = pd.read_csv(TRAIN_CSV)
publisher_test = pd.read_csv(TEST_CSV)
combined = pd.concat([publisher_train, publisher_test], ignore_index=True).drop_duplicates().reset_index(drop=True)
X_all = combined.drop(columns="device_category").replace([np.inf, -np.inf], np.nan)
y_all = combined["device_category"].astype(str)

label_encoder = LabelEncoder()
y_all_int = label_encoder.fit_transform(y_all)
all_indices = np.arange(len(combined), dtype=np.int64)
X_development, X_test, y_development, y_test, index_development, index_test = train_test_split(
    X_all, y_all_int, all_indices, test_size=0.20, random_state=42, stratify=y_all_int
)
X_train, X_val, y_train, y_val, index_train, index_val = train_test_split(
    X_development, y_development, index_development, test_size=0.20, random_state=42, stratify=y_development
)

zero_variance_columns = X_train.columns[X_train.var() == 0].tolist()
X_train = X_train.drop(columns=zero_variance_columns)
X_val = X_val.drop(columns=zero_variance_columns)
X_test = X_test.drop(columns=zero_variance_columns)
feature_names = X_train.columns.tolist()

train_hashes = set(pd.util.hash_pandas_object(X_train, index=False).astype(str))
val_hashes = set(pd.util.hash_pandas_object(X_val, index=False).astype(str))
test_hashes = set(pd.util.hash_pandas_object(X_test, index=False).astype(str))
if train_hashes & val_hashes or train_hashes & test_hashes or val_hashes & test_hashes:
    raise RuntimeError("An exact feature vector occurs in more than one partition.")

imputer = SimpleImputer(strategy="mean")
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed).astype(np.float32)
X_val_scaled = scaler.transform(X_val_imputed).astype(np.float32)
X_test_scaled = scaler.transform(X_test_imputed).astype(np.float32)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)
X_train_balanced, y_train_balanced = shuffle(X_train_balanced, y_train_balanced, random_state=42)
X_train_balanced = np.asarray(X_train_balanced, dtype=np.float32)
NUM_CLASSES = len(label_encoder.classes_)
NUM_FEATURES = X_train_balanced.shape[1]
y_train_onehot = tf.keras.utils.to_categorical(y_train_balanced, NUM_CLASSES).astype(np.float32)
y_val_onehot = tf.keras.utils.to_categorical(y_val, NUM_CLASSES).astype(np.float32)
y_test_onehot = tf.keras.utils.to_categorical(y_test, NUM_CLASSES).astype(np.float32)
try:
    one_hot_encoder = OneHotEncoder(
        categories=[label_encoder.classes_], sparse_output=False, handle_unknown="error"
    )
except TypeError:
    one_hot_encoder = OneHotEncoder(
        categories=[label_encoder.classes_], sparse=False, handle_unknown="error"
    )
one_hot_encoder.fit(np.asarray(label_encoder.classes_).reshape(-1, 1))

if (len(combined), len(X_train), len(X_val), len(X_test)) != (1877, 1200, 301, 376):
    raise RuntimeError(f"Unexpected corpus/split sizes: {len(combined)}, {len(X_train)}, {len(X_val)}, {len(X_test)}")
if (len(zero_variance_columns), NUM_FEATURES, len(X_train_balanced), NUM_CLASSES) != (53, 244, 1280, 10):
    raise RuntimeError(
        f"Unexpected preprocessing result: removed={len(zero_variance_columns)}, retained={NUM_FEATURES}, "
        f"balanced={len(X_train_balanced)}, classes={NUM_CLASSES}"
    )

X_train_3d = X_train_balanced.reshape(-1, NUM_FEATURES, 1)
X_val_3d = X_val_scaled.reshape(-1, NUM_FEATURES, 1)
X_test_3d = X_test_scaled.reshape(-1, NUM_FEATURES, 1)
X_train_model = X_train_3d
X_val_model = X_val_3d
X_test_model = X_test_3d

np.save(RUN_DIR / "X_test_3d.npy", X_test_3d)
np.save(RUN_DIR / "y_test_1d.npy", np.asarray(y_test, dtype=np.int64))
np.save(RUN_DIR / "device_classes.npy", np.asarray(label_encoder.classes_))
np.save(RUN_DIR / f"{MODEL_KEY}_device_classes.npy", np.asarray(label_encoder.classes_))
np.save(RUN_DIR / "feature_names.npy", np.asarray(feature_names, dtype=object))
np.savez_compressed(RUN_DIR / "split_indices.npz", train=index_train, validation=index_val, test=index_test)
np.savez_compressed(
    RUN_DIR / f"{MODEL_KEY}_dataset.npz",
    X_train=X_train_model,
    y_train=y_train_onehot,
    X_val=X_val_model,
    y_val=y_val_onehot,
    X_test=X_test_model,
    y_test=y_test_onehot,
)
(RUN_DIR / "zero_variance_columns.txt").write_text("".join(f"{column}\n" for column in zero_variance_columns), encoding="utf-8")
joblib.dump(imputer, RUN_DIR / "imputer.joblib")
joblib.dump(scaler, RUN_DIR / "scaler.joblib")
joblib.dump(label_encoder, RUN_DIR / "label_encoder.joblib")
joblib.dump(one_hot_encoder, RUN_DIR / "one_hot_encoder.joblib")



def build_model(hp):
    inputs = Input(shape=(NUM_FEATURES, 1), name="input")
    x = LSTM(hp.Choice("lstm_units", [32, 64]), return_sequences=False, name="lstm")(inputs)
    x = Dense(hp.Choice("dense_units", [32, 64]), activation="relu", name="dense")(x)
    x = Dropout(hp.Choice("dropout", [0.2, 0.3, 0.4]), name="dropout")(x)
    outputs = Dense(NUM_CLASSES, activation="softmax", name="output")(x)
    model = Model(inputs, outputs, name="tuned_lstm_baseline")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(hp.Choice("learning_rate", [1e-3, 5e-4])),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)
tuner = kt.GridSearch(
    build_model,
    objective=kt.Objective("val_accuracy", direction="max"),
    max_trials=MAX_TRIALS,
    directory=str(TUNER_DIR),
    project_name=MODEL_KEY,
    overwrite=True,
    seed=42,
)
tuner.search(
    X_train_model,
    y_train_onehot,
    validation_data=(X_val_model, y_val_onehot),
    epochs=SEARCH_EPOCHS,
    batch_size=SEARCH_BATCH_SIZE,
    shuffle=True,
    callbacks=[EarlyStopping(monitor="val_accuracy", mode="max", patience=SEARCH_PATIENCE, restore_best_weights=True, verbose=1)],
    verbose=2,
)

best_hp = tuner.get_best_hyperparameters(1)[0]
(RUN_DIR / "best_hyperparameters.json").write_text(json.dumps(best_hp.values, indent=2, default=str), encoding="utf-8")
with io.StringIO() as buffer, contextlib.redirect_stdout(buffer):
    tuner.results_summary(num_trials=MAX_TRIALS)
    (RUN_DIR / "tuner_results_summary.txt").write_text(buffer.getvalue(), encoding="utf-8")

architecture_model = build_model(best_hp)
with (RUN_DIR / "model_architecture.txt").open("w", encoding="utf-8") as handle:
    architecture_model.summary(print_fn=lambda line: handle.write(line + "\n"))
(RUN_DIR / "model_architecture.json").write_text(architecture_model.to_json(), encoding="utf-8")
parameter_count = int(architecture_model.count_params())
del architecture_model
gc.collect()

seed_results = []
for seed in SEEDS:
    print("\n" + "=" * 100)
    print(f"{MODEL_NAME} — FINAL TRAINING SEED {seed}")
    print("=" * 100)
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(seed)
    model = build_model(best_hp)
    history = model.fit(
        X_train_model,
        y_train_onehot,
        validation_data=(X_val_model, y_val_onehot),
        epochs=FINAL_EPOCHS,
        batch_size=FINAL_BATCH_SIZE,
        shuffle=True,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=FINAL_PATIENCE, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_accuracy", factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=MIN_LR, verbose=1),
        ],
        verbose=2,
    )
    validation_probabilities = model.predict(X_val_model, verbose=0)
    validation_accuracy = float(np.mean(np.argmax(validation_probabilities, axis=1) == y_val))
    validation_loss = float(model.evaluate(X_val_model, y_val_onehot, verbose=0)[0])
    restored_epoch = int(np.argmin(history.history["val_loss"])) + 1
    model_path = RUN_DIR / f"lstm_seed_{seed}.keras"
    model.save(model_path)
    (RUN_DIR / f"{MODEL_KEY}_seed_{seed}_model_config.json").write_text(
        model.to_json(), encoding="utf-8"
    )
    history_frame = pd.DataFrame(history.history)
    history_frame.insert(0, "epoch", np.arange(1, len(history_frame) + 1))
    history_path = RUN_DIR / f"{MODEL_KEY}_seed_{seed}_training_history.csv"
    history_frame.to_csv(history_path, index=False)
    seed_results.append({
        "model": MODEL_NAME,
        "seed": int(seed),
        "validation_accuracy": validation_accuracy,
        "validation_loss": validation_loss,
        "restored_epoch": restored_epoch,
        "model_file": model_path.name,
        "history_file": history_path.name,
    })
    del model
    gc.collect()

champion_index = min(
    range(len(seed_results)),
    key=lambda index: (-seed_results[index]["validation_accuracy"], seed_results[index]["validation_loss"]),
)
champion = seed_results[champion_index]
shutil.copy2(RUN_DIR / champion["model_file"], RUN_DIR / CHAMPION_FILENAME)
seed_frame = pd.DataFrame(seed_results)
seed_frame["selected_champion"] = False
seed_frame.loc[champion_index, "selected_champion"] = True
seed_frame.to_csv(RUN_DIR / "validation_seed_results.csv", index=False)
seed_frame.to_csv(RUN_DIR / f"{MODEL_KEY}_five_seed_validation_results.csv", index=False)

validation_summary = pd.DataFrame({
    "Metric": [
        "Mean Restored Validation Accuracy (%)",
        "Validation Accuracy Std (%)",
        "Mean Restored Validation Loss",
        "Validation Loss Std",
        "Best Validation Accuracy (%)",
        "Best Validation Loss",
        "Best Validation Seed",
        "Restored Epoch",
    ],
    "Value": [
        seed_frame["validation_accuracy"].mean() * 100,
        seed_frame["validation_accuracy"].std(ddof=1) * 100,
        seed_frame["validation_loss"].mean(),
        seed_frame["validation_loss"].std(ddof=1),
        champion["validation_accuracy"] * 100,
        champion["validation_loss"],
        champion["seed"],
        champion["restored_epoch"],
    ],
})
validation_summary.to_csv(
    RUN_DIR / f"{MODEL_KEY}_validation_summary_statistics.csv", index=False
)


def plot_training_history(result_row, filename_prefix, title):
    history_data = pd.read_csv(RUN_DIR / result_row["history_file"])
    for metric, validation_metric, y_label in [
        ("accuracy", "val_accuracy", "Accuracy"),
        ("loss", "val_loss", "Categorical cross-entropy"),
    ]:
        figure, axis = plt.subplots(figsize=(9, 6))
        axis.plot(history_data["epoch"], history_data[metric], label=f"Training {metric}")
        axis.plot(history_data["epoch"], history_data[validation_metric], label=f"Validation {metric}")
        axis.axvline(
            result_row["restored_epoch"], linestyle="--", color="black",
            label=f"Restored epoch {result_row['restored_epoch']}"
        )
        axis.set_title(title)
        axis.set_xlabel("Epoch")
        axis.set_ylabel(y_label)
        axis.grid(True, linestyle="--", alpha=0.35)
        axis.legend()
        figure.tight_layout()
        figure.savefig(FIGURES_DIR / f"{filename_prefix}_{metric}.png", dpi=300, bbox_inches="tight")
        figure.savefig(FIGURES_DIR / f"{filename_prefix}_{metric}.pdf", bbox_inches="tight")
        plt.show()
        plt.close(figure)


if PLOT_ALL_SEEDS:
    for result_row in seed_results:
        plot_training_history(
            result_row,
            f"{MODEL_KEY}_seed_{result_row['seed']}",
            f"{MODEL_NAME}: Seed {result_row['seed']}",
        )

plot_training_history(
    champion,
    f"{MODEL_KEY}_selected_seed",
    f"{MODEL_NAME}: Validation-Selected Seed {champion['seed']}",
)

run_metadata = {
    "project": "Smart Home Appliance Identification Using Deep Learning",
    "model_name": MODEL_NAME,
    "run_started": run_started.isoformat(),
    "run_completed": datetime.now(TIMEZONE).isoformat(),
    "dataset": {
        "unique_rows": len(combined), "training_rows_before_smote": len(X_train),
        "training_rows_after_smote": len(X_train_balanced), "validation_rows": len(X_val),
        "held_out_test_rows": len(X_test), "removed_zero_variance_features": len(zero_variance_columns),
        "retained_features": NUM_FEATURES, "classes": label_encoder.classes_.tolist(),
        "split_random_state": 42, "smote_random_state": 42,
    },
    "search": {
        "method": "Keras Tuner GridSearch", "objective": "val_accuracy", "tuner_seed": 42,
        "search_space": SEARCH_SPACE, "maximum_trials": MAX_TRIALS,
        "maximum_epochs_per_trial": SEARCH_EPOCHS, "batch_size": SEARCH_BATCH_SIZE,
        "early_stopping_monitor": "val_accuracy", "early_stopping_patience": SEARCH_PATIENCE,
        "selected_hyperparameters": best_hp.values,
    },
    "final_training": {
        "seeds": SEEDS, "maximum_epochs": FINAL_EPOCHS, "batch_size": FINAL_BATCH_SIZE,
        "early_stopping_monitor": "val_loss", "early_stopping_patience": FINAL_PATIENCE,
        "reduce_lr_monitor": "val_accuracy", "reduce_lr_patience": LR_PATIENCE,
        "reduce_lr_factor": LR_FACTOR, "minimum_learning_rate": MIN_LR,
        "training_figures_for_all_seeds": PLOT_ALL_SEEDS,
    },
    "selection": {
        "rule": "highest restored validation accuracy; lower restored validation loss as tie-breaker",
        "held_out_test_used": False, "champion_seed": int(champion["seed"]),
        "champion_validation_accuracy": float(champion["validation_accuracy"]),
        "champion_validation_loss": float(champion["validation_loss"]),
        "champion_filename": CHAMPION_FILENAME,
    },
    "model_parameter_count": parameter_count,
}
(RUN_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2, default=str), encoding="utf-8")

print("\n" + "=" * 100)
print(f"{MODEL_NAME} RUN COMPLETE")
print("=" * 100)
print(seed_frame.to_string(index=False))
print(f"Selected hyperparameters: {best_hp.values}")
print(f"Validation-selected seed: {champion['seed']}")
print(f"Champion model: {RUN_DIR / CHAMPION_FILENAME}")
print(f"Temporary run directory: {RUN_DIR}")
print(f"Final run directory after successful finalisation: {FINAL_RUN_DIR}")
print("The held-out test set was archived but was not evaluated or used for selection in this cell.")
stop_terminal_capture()

try:
    shutil.move(str(RUN_DIR), str(FINAL_RUN_DIR))
except Exception as exc:
    (RUN_DIR / "RUN_FAILED.txt").write_text(
        f"Training completed, but final directory creation failed.\n{type(exc).__name__}: {exc}\n",
        encoding="utf-8",
    )
    raise
RUN_DIR = FINAL_RUN_DIR
FIGURES_DIR = RUN_DIR / "Figures"
TUNER_DIR = RUN_DIR / "Tuner"
try:
    inventory_rows = []
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file() and path.name != "file_inventory.csv":
            inventory_rows.append({"relative_path": str(path.relative_to(RUN_DIR)), "size_bytes": path.stat().st_size, "sha256": sha256_file(path)})
    pd.DataFrame(inventory_rows).to_csv(RUN_DIR / "file_inventory.csv", index=False)
    (RUN_DIR / "RUN_COMPLETE.txt").write_text(
        "Run completed and finalised successfully.\n", encoding="utf-8"
    )
except Exception as exc:
    (RUN_DIR / "RUN_FAILED.txt").write_text(
        f"Training completed, but archive inventory finalisation failed.\n{type(exc).__name__}: {exc}\n",
        encoding="utf-8",
    )
    raise
print(f"Final run directory: {RUN_DIR}")
print(f"Terminal output: {RUN_DIR / 'terminal_output.txt'}")

#### BiLSTM

The BiLSTM is a tuned recurrent comparator that processes the fixed engineered-feature order in both forward and reverse directions. Its comparison with the LSTM investigates whether bidirectional traversal of this ordering provides an advantage.

Each observation is represented as:

```text
(samples, 244, 1)
```

The 244 positions are engineered predictors rather than chronological packet observations.

The architecture is:

```text
Input (244, 1)
→ Bidirectional LSTM(return_sequences=False)
→ concatenated forward and backward final states
→ Dense(ReLU)
→ Dropout
→ Softmax output (10 classes)
```

Keras Tuner GridSearch evaluates all 24 configurations:

```text
LSTM units per direction:  32 or 64
dense units:               32 or 64
dropout:                   0.2, 0.3 or 0.4
Adam learning rate:        0.001 or 0.0005
```

Search trials use at most 50 epochs and batch size 32. Validation accuracy is the tuning objective. Early stopping monitors validation accuracy with patience 8 and restores the best weights.

The selected hyperparameter tuple is trained independently using seeds:

```text
42, 7, 123, 2024, 99
```

Final training uses at most 100 epochs and batch size 32. Early stopping monitors validation loss with patience 15 and restores the best weights. `ReduceLROnPlateau` monitors validation accuracy with patience 5, reduction factor 0.5 and minimum learning rate `1e-5`.

The retained seed is selected by highest restored validation accuracy, with lower restored validation loss as the tie-breaker. Held-out test performance is excluded from tuning and seed selection.

Because `PLOT_ALL_SEEDS=False`, every seed history is archived, but training/validation figures are generated only for the validation-selected seed:

```text
Figures/bilstm_selected_seed_accuracy.png
Figures/bilstm_selected_seed_accuracy.pdf
Figures/bilstm_selected_seed_loss.png
Figures/bilstm_selected_seed_loss.pdf
```

Successful runs are finalised under:

```text
Honours_Project/Models/Tuned/BiLSTM_Tuned/<timestamp>/
```

The retained model is saved as:

```text
best_validation_bilstm.keras
```

The run also preserves all five seed models, preprocessing objects, split indices, held-out arrays, tuner outputs, selected hyperparameters, validation results, histories, terminal output, source provenance, runtime environment, system diagnostics, requirements and a file inventory.

In [ ]:
# TUNED BILSTM BASELINE: SELF-CONTAINED TUNING, FIVE-SEED TRAINING AND ARCHIVING

try:
    get_ipython().run_line_magic("pip", "install -q keras-tuner imbalanced-learn psutil")
except NameError:
    pass

import contextlib
import gc
import hashlib
import importlib.metadata as metadata
import io
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

import imblearn
import joblib
import keras
import keras_tuner as kt
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import sklearn
import tensorflow as tf
from google.colab import drive
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils import shuffle
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Bidirectional, Dense, Dropout, Input, LSTM
from tensorflow.keras.models import Model

try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print(f"TensorFlow deterministic operations could not be enabled: {exc}")

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

MODEL_NAME = 'Tuned BiLSTM Baseline'
MODEL_KEY = 'bilstm'
PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
TRAIN_CSV = PROJECT_ROOT / "iot_device_train.csv"
TEST_CSV = PROJECT_ROOT / "iot_device_test.csv"
OUTPUT_PARENT = PROJECT_ROOT / "Models" / "Tuned" / 'BiLSTM_Tuned'
TIMEZONE = ZoneInfo("Africa/Johannesburg")
SEEDS = [42, 7, 123, 2024, 99]
SEARCH_SPACE = {"lstm_units_per_direction": [32, 64], "dense_units": [32, 64], "dropout": [0.2, 0.3, 0.4], "learning_rate": [0.001, 0.0005]}
MAX_TRIALS = 24
SEARCH_EPOCHS = 50
FINAL_EPOCHS = 100
SEARCH_BATCH_SIZE = 32
FINAL_BATCH_SIZE = 32
PLOT_ALL_SEEDS = False
SEARCH_PATIENCE = 8
FINAL_PATIENCE = 15
LR_PATIENCE = 5
LR_FACTOR = 0.5
MIN_LR = 1e-5
CHAMPION_FILENAME = 'best_validation_bilstm.keras'


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def command_record(command):
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=False)
        return {"command": command, "returncode": result.returncode, "stdout": result.stdout, "stderr": result.stderr}
    except Exception as exc:
        return {"command": command, "returncode": None, "stdout": "", "stderr": repr(exc)}


class TeeStream:
    def __init__(self, original_stream, log_handle):
        self.original_stream = original_stream
        self.log_handle = log_handle

    def write(self, message):
        self.original_stream.write(message)
        self.original_stream.flush()
        self.log_handle.write(message)
        self.log_handle.flush()

    def flush(self):
        self.original_stream.flush()
        self.log_handle.flush()

    def isatty(self):
        return self.original_stream.isatty()

    def fileno(self):
        return self.original_stream.fileno()


terminal_capture = {}


def stop_terminal_capture(result=None):
    if not terminal_capture:
        return
    sys.stdout = terminal_capture["stdout"]
    sys.stderr = terminal_capture["stderr"]
    terminal_capture["handle"].flush()
    terminal_capture["handle"].close()
    failed = bool(
        result is not None
        and (getattr(result, "error_before_exec", None) is not None or getattr(result, "error_in_exec", None) is not None)
    )
    if failed:
        error = getattr(result, "error_before_exec", None) or getattr(result, "error_in_exec", None)
        terminal_capture["run_dir"].joinpath("RUN_FAILED.txt").write_text(
            f"Run failed before finalisation.\n{type(error).__name__}: {error}\n",
            encoding="utf-8",
        )
    try:
        get_ipython().events.unregister("post_run_cell", stop_terminal_capture)
    except Exception:
        pass
    terminal_capture.clear()


def start_terminal_capture(log_path, run_dir):
    log_handle = Path(log_path).open("w", encoding="utf-8")
    terminal_capture.update({
        "stdout": sys.stdout,
        "stderr": sys.stderr,
        "handle": log_handle,
        "run_dir": Path(run_dir),
    })
    sys.stdout = TeeStream(terminal_capture["stdout"], log_handle)
    sys.stderr = TeeStream(terminal_capture["stderr"], log_handle)
    try:
        get_ipython().events.register("post_run_cell", stop_terminal_capture)
    except Exception:
        pass


def version(distribution):
    try:
        return metadata.version(distribution)
    except metadata.PackageNotFoundError:
        return "not installed"


def current_cell_source():
    try:
        history = get_ipython().history_manager.input_hist_raw
        if history and str(history[-1]).strip():
            return str(history[-1])
    except Exception:
        pass
    return "# Current notebook-cell source was unavailable.\n"


for required_path in (TRAIN_CSV, TEST_CSV):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

run_started = datetime.now(TIMEZONE)
RUN_TIMESTAMP = run_started.strftime("%Y-%m-%d_%H-%M-%S")
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
FINAL_RUN_DIR = OUTPUT_PARENT / RUN_TIMESTAMP
RUN_DIR = OUTPUT_PARENT / f"_TEMP_{MODEL_KEY.upper()}_{RUN_TIMESTAMP}"
if RUN_DIR.exists() or FINAL_RUN_DIR.exists():
    raise FileExistsError(f"Run directory already exists for timestamp {RUN_TIMESTAMP}")
FIGURES_DIR = RUN_DIR / "Figures"
TUNER_DIR = RUN_DIR / "Tuner"
FIGURES_DIR.mkdir(parents=True, exist_ok=False)
start_terminal_capture(RUN_DIR / "terminal_output.txt", RUN_DIR)

runtime_environment = {
    "captured_at": run_started.isoformat(),
    "timezone": "Africa/Johannesburg",
    "python": {"version": sys.version, "executable": sys.executable, "implementation": platform.python_implementation()},
    "operating_system": {"platform": platform.platform(), "system": platform.system(), "release": platform.release(), "machine": platform.machine(), "processor": platform.processor()},
    "hardware": {
        "physical_cpu_count": psutil.cpu_count(logical=False),
        "logical_cpu_count": psutil.cpu_count(logical=True),
        "memory_total_bytes": int(psutil.virtual_memory().total),
        "tensorflow_visible_cpus": [device.name for device in tf.config.list_physical_devices("CPU")],
        "tensorflow_visible_gpus": [device.name for device in tf.config.list_physical_devices("GPU")],
    },
    "colab": {"release_tag": os.environ.get("COLAB_RELEASE_TAG"), "gpu_type": os.environ.get("COLAB_GPU")},
    "software": {
        "tensorflow": version("tensorflow"), "keras": version("keras"), "keras_tuner": version("keras-tuner"),
        "numpy": version("numpy"), "pandas": version("pandas"), "scikit_learn": version("scikit-learn"),
        "imbalanced_learn": version("imbalanced-learn"), "joblib": version("joblib"),
        "matplotlib": version("matplotlib"), "psutil": version("psutil"),
    },
    "tensorflow_build": tf.sysconfig.get_build_info(),
    "determinism": {"TF_DETERMINISTIC_OPS": os.environ.get("TF_DETERMINISTIC_OPS"), "TF_CUDNN_DETERMINISTIC": os.environ.get("TF_CUDNN_DETERMINISTIC")},
}
(RUN_DIR / "runtime_environment.json").write_text(json.dumps(runtime_environment, indent=2, default=str), encoding="utf-8")
(RUN_DIR / "system_diagnostics.json").write_text(
    json.dumps({
        "uname": command_record(["uname", "-a"]), "lscpu": command_record(["lscpu"]),
        "memory": command_record(["free", "-h"]), "nvidia_smi": command_record(["nvidia-smi"]),
    }, indent=2, default=str), encoding="utf-8"
)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=False)
(RUN_DIR / "requirements.txt").write_text(freeze.stdout, encoding="utf-8")
source_path = RUN_DIR / "training_cell_source.py"
source_path.write_text(current_cell_source(), encoding="utf-8")
(RUN_DIR / "training_cell_source.sha256").write_text(sha256_file(source_path), encoding="utf-8")

dataset_manifest = {
    "publisher_train": {"path": str(TRAIN_CSV), "size_bytes": TRAIN_CSV.stat().st_size, "sha256": sha256_file(TRAIN_CSV)},
    "publisher_test": {"path": str(TEST_CSV), "size_bytes": TEST_CSV.stat().st_size, "sha256": sha256_file(TEST_CSV)},
}
(RUN_DIR / "dataset_manifest.json").write_text(json.dumps(dataset_manifest, indent=2), encoding="utf-8")

publisher_train = pd.read_csv(TRAIN_CSV)
publisher_test = pd.read_csv(TEST_CSV)
combined = pd.concat([publisher_train, publisher_test], ignore_index=True).drop_duplicates().reset_index(drop=True)
X_all = combined.drop(columns="device_category").replace([np.inf, -np.inf], np.nan)
y_all = combined["device_category"].astype(str)

label_encoder = LabelEncoder()
y_all_int = label_encoder.fit_transform(y_all)
all_indices = np.arange(len(combined), dtype=np.int64)
X_development, X_test, y_development, y_test, index_development, index_test = train_test_split(
    X_all, y_all_int, all_indices, test_size=0.20, random_state=42, stratify=y_all_int
)
X_train, X_val, y_train, y_val, index_train, index_val = train_test_split(
    X_development, y_development, index_development, test_size=0.20, random_state=42, stratify=y_development
)

zero_variance_columns = X_train.columns[X_train.var() == 0].tolist()
X_train = X_train.drop(columns=zero_variance_columns)
X_val = X_val.drop(columns=zero_variance_columns)
X_test = X_test.drop(columns=zero_variance_columns)
feature_names = X_train.columns.tolist()

train_hashes = set(pd.util.hash_pandas_object(X_train, index=False).astype(str))
val_hashes = set(pd.util.hash_pandas_object(X_val, index=False).astype(str))
test_hashes = set(pd.util.hash_pandas_object(X_test, index=False).astype(str))
if train_hashes & val_hashes or train_hashes & test_hashes or val_hashes & test_hashes:
    raise RuntimeError("An exact feature vector occurs in more than one partition.")

imputer = SimpleImputer(strategy="mean")
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed).astype(np.float32)
X_val_scaled = scaler.transform(X_val_imputed).astype(np.float32)
X_test_scaled = scaler.transform(X_test_imputed).astype(np.float32)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)
X_train_balanced, y_train_balanced = shuffle(X_train_balanced, y_train_balanced, random_state=42)
X_train_balanced = np.asarray(X_train_balanced, dtype=np.float32)
NUM_CLASSES = len(label_encoder.classes_)
NUM_FEATURES = X_train_balanced.shape[1]
y_train_onehot = tf.keras.utils.to_categorical(y_train_balanced, NUM_CLASSES).astype(np.float32)
y_val_onehot = tf.keras.utils.to_categorical(y_val, NUM_CLASSES).astype(np.float32)
y_test_onehot = tf.keras.utils.to_categorical(y_test, NUM_CLASSES).astype(np.float32)
try:
    one_hot_encoder = OneHotEncoder(
        categories=[label_encoder.classes_], sparse_output=False, handle_unknown="error"
    )
except TypeError:
    one_hot_encoder = OneHotEncoder(
        categories=[label_encoder.classes_], sparse=False, handle_unknown="error"
    )
one_hot_encoder.fit(np.asarray(label_encoder.classes_).reshape(-1, 1))

if (len(combined), len(X_train), len(X_val), len(X_test)) != (1877, 1200, 301, 376):
    raise RuntimeError(f"Unexpected corpus/split sizes: {len(combined)}, {len(X_train)}, {len(X_val)}, {len(X_test)}")
if (len(zero_variance_columns), NUM_FEATURES, len(X_train_balanced), NUM_CLASSES) != (53, 244, 1280, 10):
    raise RuntimeError(
        f"Unexpected preprocessing result: removed={len(zero_variance_columns)}, retained={NUM_FEATURES}, "
        f"balanced={len(X_train_balanced)}, classes={NUM_CLASSES}"
    )

X_train_3d = X_train_balanced.reshape(-1, NUM_FEATURES, 1)
X_val_3d = X_val_scaled.reshape(-1, NUM_FEATURES, 1)
X_test_3d = X_test_scaled.reshape(-1, NUM_FEATURES, 1)
X_train_model = X_train_3d
X_val_model = X_val_3d
X_test_model = X_test_3d

np.save(RUN_DIR / "X_test_3d.npy", X_test_3d)
np.save(RUN_DIR / "y_test_1d.npy", np.asarray(y_test, dtype=np.int64))
np.save(RUN_DIR / "device_classes.npy", np.asarray(label_encoder.classes_))
np.save(RUN_DIR / f"{MODEL_KEY}_device_classes.npy", np.asarray(label_encoder.classes_))
np.save(RUN_DIR / "feature_names.npy", np.asarray(feature_names, dtype=object))
np.savez_compressed(RUN_DIR / "split_indices.npz", train=index_train, validation=index_val, test=index_test)
np.savez_compressed(
    RUN_DIR / f"{MODEL_KEY}_dataset.npz",
    X_train=X_train_model,
    y_train=y_train_onehot,
    X_val=X_val_model,
    y_val=y_val_onehot,
    X_test=X_test_model,
    y_test=y_test_onehot,
)
(RUN_DIR / "zero_variance_columns.txt").write_text("".join(f"{column}\n" for column in zero_variance_columns), encoding="utf-8")
joblib.dump(imputer, RUN_DIR / "imputer.joblib")
joblib.dump(scaler, RUN_DIR / "scaler.joblib")
joblib.dump(label_encoder, RUN_DIR / "label_encoder.joblib")
joblib.dump(one_hot_encoder, RUN_DIR / "one_hot_encoder.joblib")



def build_model(hp):
    inputs = Input(shape=(NUM_FEATURES, 1), name="input")
    x = Bidirectional(LSTM(hp.Choice("lstm_units", [32, 64]), return_sequences=False), name="bidirectional_lstm")(inputs)
    x = Dense(hp.Choice("dense_units", [32, 64]), activation="relu", name="dense")(x)
    x = Dropout(hp.Choice("dropout", [0.2, 0.3, 0.4]), name="dropout")(x)
    outputs = Dense(NUM_CLASSES, activation="softmax", name="output")(x)
    model = Model(inputs, outputs, name="tuned_bilstm_baseline")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(hp.Choice("learning_rate", [1e-3, 5e-4])),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)
tuner = kt.GridSearch(
    build_model,
    objective=kt.Objective("val_accuracy", direction="max"),
    max_trials=MAX_TRIALS,
    directory=str(TUNER_DIR),
    project_name=MODEL_KEY,
    overwrite=True,
    seed=42,
)
tuner.search(
    X_train_model,
    y_train_onehot,
    validation_data=(X_val_model, y_val_onehot),
    epochs=SEARCH_EPOCHS,
    batch_size=SEARCH_BATCH_SIZE,
    shuffle=True,
    callbacks=[EarlyStopping(monitor="val_accuracy", mode="max", patience=SEARCH_PATIENCE, restore_best_weights=True, verbose=1)],
    verbose=2,
)

best_hp = tuner.get_best_hyperparameters(1)[0]
(RUN_DIR / "best_hyperparameters.json").write_text(json.dumps(best_hp.values, indent=2, default=str), encoding="utf-8")
with io.StringIO() as buffer, contextlib.redirect_stdout(buffer):
    tuner.results_summary(num_trials=MAX_TRIALS)
    (RUN_DIR / "tuner_results_summary.txt").write_text(buffer.getvalue(), encoding="utf-8")

architecture_model = build_model(best_hp)
with (RUN_DIR / "model_architecture.txt").open("w", encoding="utf-8") as handle:
    architecture_model.summary(print_fn=lambda line: handle.write(line + "\n"))
(RUN_DIR / "model_architecture.json").write_text(architecture_model.to_json(), encoding="utf-8")
parameter_count = int(architecture_model.count_params())
del architecture_model
gc.collect()

seed_results = []
for seed in SEEDS:
    print("\n" + "=" * 100)
    print(f"{MODEL_NAME} — FINAL TRAINING SEED {seed}")
    print("=" * 100)
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(seed)
    model = build_model(best_hp)
    history = model.fit(
        X_train_model,
        y_train_onehot,
        validation_data=(X_val_model, y_val_onehot),
        epochs=FINAL_EPOCHS,
        batch_size=FINAL_BATCH_SIZE,
        shuffle=True,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=FINAL_PATIENCE, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_accuracy", factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=MIN_LR, verbose=1),
        ],
        verbose=2,
    )
    validation_probabilities = model.predict(X_val_model, verbose=0)
    validation_accuracy = float(np.mean(np.argmax(validation_probabilities, axis=1) == y_val))
    validation_loss = float(model.evaluate(X_val_model, y_val_onehot, verbose=0)[0])
    restored_epoch = int(np.argmin(history.history["val_loss"])) + 1
    model_path = RUN_DIR / f"bilstm_seed_{seed}.keras"
    model.save(model_path)
    (RUN_DIR / f"{MODEL_KEY}_seed_{seed}_model_config.json").write_text(
        model.to_json(), encoding="utf-8"
    )
    history_frame = pd.DataFrame(history.history)
    history_frame.insert(0, "epoch", np.arange(1, len(history_frame) + 1))
    history_path = RUN_DIR / f"{MODEL_KEY}_seed_{seed}_training_history.csv"
    history_frame.to_csv(history_path, index=False)
    seed_results.append({
        "model": MODEL_NAME,
        "seed": int(seed),
        "validation_accuracy": validation_accuracy,
        "validation_loss": validation_loss,
        "restored_epoch": restored_epoch,
        "model_file": model_path.name,
        "history_file": history_path.name,
    })
    del model
    gc.collect()

champion_index = min(
    range(len(seed_results)),
    key=lambda index: (-seed_results[index]["validation_accuracy"], seed_results[index]["validation_loss"]),
)
champion = seed_results[champion_index]
shutil.copy2(RUN_DIR / champion["model_file"], RUN_DIR / CHAMPION_FILENAME)
seed_frame = pd.DataFrame(seed_results)
seed_frame["selected_champion"] = False
seed_frame.loc[champion_index, "selected_champion"] = True
seed_frame.to_csv(RUN_DIR / "validation_seed_results.csv", index=False)
seed_frame.to_csv(RUN_DIR / f"{MODEL_KEY}_five_seed_validation_results.csv", index=False)

validation_summary = pd.DataFrame({
    "Metric": [
        "Mean Restored Validation Accuracy (%)",
        "Validation Accuracy Std (%)",
        "Mean Restored Validation Loss",
        "Validation Loss Std",
        "Best Validation Accuracy (%)",
        "Best Validation Loss",
        "Best Validation Seed",
        "Restored Epoch",
    ],
    "Value": [
        seed_frame["validation_accuracy"].mean() * 100,
        seed_frame["validation_accuracy"].std(ddof=1) * 100,
        seed_frame["validation_loss"].mean(),
        seed_frame["validation_loss"].std(ddof=1),
        champion["validation_accuracy"] * 100,
        champion["validation_loss"],
        champion["seed"],
        champion["restored_epoch"],
    ],
})
validation_summary.to_csv(
    RUN_DIR / f"{MODEL_KEY}_validation_summary_statistics.csv", index=False
)


def plot_training_history(result_row, filename_prefix, title):
    history_data = pd.read_csv(RUN_DIR / result_row["history_file"])
    for metric, validation_metric, y_label in [
        ("accuracy", "val_accuracy", "Accuracy"),
        ("loss", "val_loss", "Categorical cross-entropy"),
    ]:
        figure, axis = plt.subplots(figsize=(9, 6))
        axis.plot(history_data["epoch"], history_data[metric], label=f"Training {metric}")
        axis.plot(history_data["epoch"], history_data[validation_metric], label=f"Validation {metric}")
        axis.axvline(
            result_row["restored_epoch"], linestyle="--", color="black",
            label=f"Restored epoch {result_row['restored_epoch']}"
        )
        axis.set_title(title)
        axis.set_xlabel("Epoch")
        axis.set_ylabel(y_label)
        axis.grid(True, linestyle="--", alpha=0.35)
        axis.legend()
        figure.tight_layout()
        figure.savefig(FIGURES_DIR / f"{filename_prefix}_{metric}.png", dpi=300, bbox_inches="tight")
        figure.savefig(FIGURES_DIR / f"{filename_prefix}_{metric}.pdf", bbox_inches="tight")
        plt.show()
        plt.close(figure)


if PLOT_ALL_SEEDS:
    for result_row in seed_results:
        plot_training_history(
            result_row,
            f"{MODEL_KEY}_seed_{result_row['seed']}",
            f"{MODEL_NAME}: Seed {result_row['seed']}",
        )

plot_training_history(
    champion,
    f"{MODEL_KEY}_selected_seed",
    f"{MODEL_NAME}: Validation-Selected Seed {champion['seed']}",
)

run_metadata = {
    "project": "Smart Home Appliance Identification Using Deep Learning",
    "model_name": MODEL_NAME,
    "run_started": run_started.isoformat(),
    "run_completed": datetime.now(TIMEZONE).isoformat(),
    "dataset": {
        "unique_rows": len(combined), "training_rows_before_smote": len(X_train),
        "training_rows_after_smote": len(X_train_balanced), "validation_rows": len(X_val),
        "held_out_test_rows": len(X_test), "removed_zero_variance_features": len(zero_variance_columns),
        "retained_features": NUM_FEATURES, "classes": label_encoder.classes_.tolist(),
        "split_random_state": 42, "smote_random_state": 42,
    },
    "search": {
        "method": "Keras Tuner GridSearch", "objective": "val_accuracy", "tuner_seed": 42,
        "search_space": SEARCH_SPACE, "maximum_trials": MAX_TRIALS,
        "maximum_epochs_per_trial": SEARCH_EPOCHS, "batch_size": SEARCH_BATCH_SIZE,
        "early_stopping_monitor": "val_accuracy", "early_stopping_patience": SEARCH_PATIENCE,
        "selected_hyperparameters": best_hp.values,
    },
    "final_training": {
        "seeds": SEEDS, "maximum_epochs": FINAL_EPOCHS, "batch_size": FINAL_BATCH_SIZE,
        "early_stopping_monitor": "val_loss", "early_stopping_patience": FINAL_PATIENCE,
        "reduce_lr_monitor": "val_accuracy", "reduce_lr_patience": LR_PATIENCE,
        "reduce_lr_factor": LR_FACTOR, "minimum_learning_rate": MIN_LR,
        "training_figures_for_all_seeds": PLOT_ALL_SEEDS,
    },
    "selection": {
        "rule": "highest restored validation accuracy; lower restored validation loss as tie-breaker",
        "held_out_test_used": False, "champion_seed": int(champion["seed"]),
        "champion_validation_accuracy": float(champion["validation_accuracy"]),
        "champion_validation_loss": float(champion["validation_loss"]),
        "champion_filename": CHAMPION_FILENAME,
    },
    "model_parameter_count": parameter_count,
}
(RUN_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2, default=str), encoding="utf-8")

print("\n" + "=" * 100)
print(f"{MODEL_NAME} RUN COMPLETE")
print("=" * 100)
print(seed_frame.to_string(index=False))
print(f"Selected hyperparameters: {best_hp.values}")
print(f"Validation-selected seed: {champion['seed']}")
print(f"Champion model: {RUN_DIR / CHAMPION_FILENAME}")
print(f"Temporary run directory: {RUN_DIR}")
print(f"Final run directory after successful finalisation: {FINAL_RUN_DIR}")
print("The held-out test set was archived but was not evaluated or used for selection in this cell.")
stop_terminal_capture()

try:
    shutil.move(str(RUN_DIR), str(FINAL_RUN_DIR))
except Exception as exc:
    (RUN_DIR / "RUN_FAILED.txt").write_text(
        f"Training completed, but final directory creation failed.\n{type(exc).__name__}: {exc}\n",
        encoding="utf-8",
    )
    raise
RUN_DIR = FINAL_RUN_DIR
FIGURES_DIR = RUN_DIR / "Figures"
TUNER_DIR = RUN_DIR / "Tuner"
try:
    inventory_rows = []
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file() and path.name != "file_inventory.csv":
            inventory_rows.append({"relative_path": str(path.relative_to(RUN_DIR)), "size_bytes": path.stat().st_size, "sha256": sha256_file(path)})
    pd.DataFrame(inventory_rows).to_csv(RUN_DIR / "file_inventory.csv", index=False)
    (RUN_DIR / "RUN_COMPLETE.txt").write_text(
        "Run completed and finalised successfully.\n", encoding="utf-8"
    )
except Exception as exc:
    (RUN_DIR / "RUN_FAILED.txt").write_text(
        f"Training completed, but archive inventory finalisation failed.\n{type(exc).__name__}: {exc}\n",
        encoding="utf-8",
    )
    raise
print(f"Final run directory: {RUN_DIR}")
print(f"Terminal output: {RUN_DIR / 'terminal_output.txt'}")

#### Tuned Random Forest Comparator

This self-contained cell trains the classical Random Forest comparator using a leakage-controlled `imblearn` pipeline.

The pipeline is:

```text
Mean imputation
→ VarianceThreshold(0)
→ StandardScaler
→ SMOTE(random_state=42)
→ RandomForestClassifier
```

`RandomizedSearchCV` evaluates 30 sampled configurations using five-fold stratified cross-validation on the 1,200-observation training partition. Every preprocessing and SMOTE operation is refitted inside the corresponding cross-validation training fold. Macro-F1 is the refit and model-selection metric; accuracy and balanced accuracy are also recorded.

The separate 301-observation validation partition is used only after the cross-validation-selected pipeline has been refitted. The 376-observation held-out test partition is archived without being evaluated in this training cell.

Because a Random Forest has no training epochs, the cell creates descriptive cumulative-tree diagnostics. It rebuilds the selected configuration using increasing numbers of trees and records training and validation accuracy and log loss. These curves do not select the tree count; the number of trees remains the value selected by cross-validation.

Successful runs are finalized under:

```text
Honours_Project/Models/Baselines/Tuned_RF/<timestamp>/
```

The run preserves:

- `tuned_random_forest_pipeline.joblib`;
- the complete `RandomizedSearchCV` object;
- cross-validation results;
- selected hyperparameters;
- validation predictions, probabilities and metrics;
- cumulative-tree diagnostic data and figures;
- raw held-out inputs and labels;
- source code and SHA-256 hash;
- terminal output;
- `runtime_environment.json`;
- `system_diagnostics.json`;
- `requirements.txt`; and
- the final file inventory.

The recorded environment identifies the actual CPU, memory allocation, operating system, Python version and relevant package versions used by that execution.

In [ ]:
import hashlib
import json
import os
import platform
import shutil
import socket
import subprocess
import sys
import warnings
from datetime import datetime
from importlib.metadata import PackageNotFoundError, version as package_version
from pathlib import Path
from zoneinfo import ZoneInfo

try:
    get_ipython().run_line_magic("pip", "install -q imbalanced-learn psutil")
except NameError:
    pass

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
from google.colab import drive
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    log_loss,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
CV_FOLDS = 5
RF_SEARCH_ITERATIONS = 30

EXPECTED_UNIQUE_ROWS = 1877
EXPECTED_TRAIN_ROWS = 1200
EXPECTED_VAL_ROWS = 301
EXPECTED_TEST_ROWS = 376
EXPECTED_ORIGINAL_FEATURES = 297
EXPECTED_RETAINED_FEATURES = 244
EXPECTED_ZERO_VARIANCE_FEATURES = 53
EXPECTED_CLASSES = 10

TRAIN_PATH = Path(
    "/content/drive/MyDrive/Honours_Project/iot_device_train.csv"
)
TEST_PATH = Path(
    "/content/drive/MyDrive/Honours_Project/iot_device_test.csv"
)
MODELS_ROOT = Path(
    "/content/drive/MyDrive/Honours_Project/Models"
)
DRIVE_PARENT = MODELS_ROOT / "Baselines" / "Tuned_RF"

drive.mount("/content/drive")

GOOGLE_DRIVE_ROOT = Path("/content/drive/MyDrive")

if not GOOGLE_DRIVE_ROOT.exists():
    raise FileNotFoundError(
        "Google Drive is not mounted.\n\n"
        "Mount Google Drive before running this cell."
    )

DRIVE_PARENT.mkdir(parents=True, exist_ok=True)

print(f"[System] Tuned RF directory ready:\n{DRIVE_PARENT}")

RUN_TIMEZONE = ZoneInfo("Africa/Johannesburg")
RUN_DATETIME = datetime.now(RUN_TIMEZONE)
RUN_TIMESTAMP = RUN_DATETIME.strftime("%Y-%m-%d_%H-%M-%S")

TEMP_RUN_DIR = DRIVE_PARENT / f"_TEMP_TUNED_RF_RUN_{RUN_TIMESTAMP}"
FINAL_RUN_DIR = DRIVE_PARENT / RUN_TIMESTAMP

if TEMP_RUN_DIR.exists():
    shutil.rmtree(TEMP_RUN_DIR)

TEMP_RUN_DIR.mkdir(parents=True, exist_ok=False)

FIGURES_DIR = TEMP_RUN_DIR / "Figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=False)


class TeeStream:
    def __init__(self, original_stream, log_file):
        self.original_stream = original_stream
        self.log_file = log_file

    def write(self, message):
        self.original_stream.write(message)
        self.original_stream.flush()
        self.log_file.write(message)
        self.log_file.flush()

    def flush(self):
        self.original_stream.flush()
        self.log_file.flush()

    def isatty(self):
        return self.original_stream.isatty()

    def fileno(self):
        return self.original_stream.fileno()


terminal_log_path = TEMP_RUN_DIR / "terminal_output.txt"
terminal_log_file = open(
    terminal_log_path,
    "w",
    encoding="utf-8",
)

original_stdout = sys.stdout
original_stderr = sys.stderr

sys.stdout = TeeStream(original_stdout, terminal_log_file)
sys.stderr = TeeStream(original_stderr, terminal_log_file)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def command_record(command):
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=False,
        )

        return {
            "command": command,
            "returncode": int(result.returncode),
            "stdout": result.stdout,
            "stderr": result.stderr,
        }

    except Exception as exc:
        return {
            "command": command,
            "returncode": None,
            "stdout": "",
            "stderr": repr(exc),
        }


def installed_version(distribution_name):
    try:
        return package_version(distribution_name)

    except PackageNotFoundError:
        return None


def capture_runtime_environment(output_directory, captured_at):
    diagnostics = {
        "uname": command_record(["uname", "-a"]),
        "lscpu": command_record(["lscpu"]),
        "memory": command_record(["free", "-h"]),
        "nvidia_smi": command_record(["nvidia-smi"]),
        "nvidia_smi_query": command_record(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,driver_version",
                "--format=csv,noheader,nounits",
            ]
        ),
    }

    gpu_query_lines = [
        line.strip()
        for line in diagnostics[
            "nvidia_smi_query"
        ]["stdout"].splitlines()
        if line.strip()
    ]

    detected_gpus = []

    for line in gpu_query_lines:
        fields = [
            field.strip()
            for field in line.split(",")
        ]

        try:
            memory_total_mib = (
                float(fields[1])
                if len(fields) > 1
                else None
            )
        except (TypeError, ValueError):
            memory_total_mib = None

        detected_gpus.append(
            {
                "name": (
                    fields[0]
                    if len(fields) > 0
                    else None
                ),
                "memory_total_mib": memory_total_mib,
                "driver_version": (
                    fields[2]
                    if len(fields) > 2
                    else None
                ),
            }
        )

    runtime_environment = {
        "captured_at": captured_at.isoformat(),
        "timezone": "Africa/Johannesburg",
        "hostname": socket.gethostname(),
        "python": {
            "version": sys.version,
            "executable": sys.executable,
            "implementation": (
                platform.python_implementation()
            ),
        },
        "operating_system": {
            "platform": platform.platform(),
            "system": platform.system(),
            "release": platform.release(),
            "version": platform.version(),
            "machine": platform.machine(),
            "processor": platform.processor(),
        },
        "hardware": {
            "physical_cpu_count": (
                psutil.cpu_count(logical=False)
            ),
            "logical_cpu_count": (
                psutil.cpu_count(logical=True)
            ),
            "memory_total_bytes": int(
                psutil.virtual_memory().total
            ),
            "gpus": detected_gpus,
        },
        "colab": {
            "release_tag": os.environ.get(
                "COLAB_RELEASE_TAG"
            ),
            "gpu_type": os.environ.get(
                "COLAB_GPU"
            ),
        },
        "software": {
            "numpy": installed_version("numpy"),
            "pandas": installed_version("pandas"),
            "scikit_learn": installed_version(
                "scikit-learn"
            ),
            "imbalanced_learn": installed_version(
                "imbalanced-learn"
            ),
            "joblib": installed_version("joblib"),
            "matplotlib": installed_version(
                "matplotlib"
            ),
            "scipy": installed_version("scipy"),
            "psutil": installed_version("psutil"),
            "google_colab": installed_version(
                "google-colab"
            ),
        },
        "threading_environment": {
            key: os.environ.get(key)
            for key in [
                "OMP_NUM_THREADS",
                "OPENBLAS_NUM_THREADS",
                "MKL_NUM_THREADS",
                "NUMEXPR_NUM_THREADS",
            ]
        },
        "randomness": {
            "random_state": RANDOM_STATE,
            "cv_folds": CV_FOLDS,
            "search_iterations": RF_SEARCH_ITERATIONS,
        },
        "workload": {
            "estimator": "RandomForestClassifier",
            "training_device": "CPU",
            "allocated_gpu_used_for_random_forest_training": False,
            "randomized_search_n_jobs": -1,
            "random_forest_n_jobs": 1,
        },
    }

    runtime_environment_path = (
        output_directory
        / "runtime_environment.json"
    )

    diagnostics_path = (
        output_directory
        / "system_diagnostics.json"
    )

    requirements_path = (
        output_directory
        / "requirements.txt"
    )

    runtime_environment_path.write_text(
        json.dumps(
            runtime_environment,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )

    diagnostics_path.write_text(
        json.dumps(
            diagnostics,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )

    freeze = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "freeze",
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    requirements_path.write_text(
        freeze.stdout,
        encoding="utf-8",
    )

    return runtime_environment, diagnostics


def calculate_metrics(
    y_true,
    y_pred,
    probabilities,
    labels,
):
    (
        macro_precision,
        macro_recall,
        macro_f1,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0,
    )

    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        average="weighted",
        zero_division=0,
    )

    y_binary = label_binarize(
        y_true,
        classes=labels,
    )

    try:
        macro_roc_auc = roc_auc_score(
            y_binary,
            probabilities,
            average="macro",
            multi_class="ovr",
        )
    except Exception:
        macro_roc_auc = np.nan

    try:
        macro_average_precision = (
            average_precision_score(
                y_binary,
                probabilities,
                average="macro",
            )
        )
    except Exception:
        macro_average_precision = np.nan

    return {
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_precision": float(
            macro_precision
        ),
        "macro_recall": float(
            macro_recall
        ),
        "macro_f1": float(
            macro_f1
        ),
        "weighted_precision": float(
            weighted_precision
        ),
        "weighted_recall": float(
            weighted_recall
        ),
        "weighted_f1": float(
            weighted_f1
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=labels,
            )
        ),
        "macro_roc_auc_ovr": (
            None
            if np.isnan(macro_roc_auc)
            else float(macro_roc_auc)
        ),
        "macro_average_precision_ovr": (
            None
            if np.isnan(
                macro_average_precision
            )
            else float(
                macro_average_precision
            )
        ),
    }


try:
    (
        runtime_environment_rf,
        system_diagnostics_rf,
    ) = capture_runtime_environment(
        TEMP_RUN_DIR,
        RUN_DATETIME,
    )

    print("\n" + "=" * 100)
    print(
        "TUNED RANDOM FOREST — "
        "LEAKAGE-CONTROLLED TRAINING "
        "AND VALIDATION"
    )
    print("=" * 100)

    print(f"\nRun timestamp: {RUN_TIMESTAMP}")
    print(
        "Run timezone: "
        "Africa/Johannesburg (UTC+2)"
    )
    print(
        f"Temporary run directory:\n"
        f"{TEMP_RUN_DIR}"
    )

    print(
        "\nRuntime environment "
        "captured before training:"
    )

    print(
        "  CPU: "
        f"{runtime_environment_rf['hardware']['physical_cpu_count']} "
        "physical / "
        f"{runtime_environment_rf['hardware']['logical_cpu_count']} "
        "logical processors"
    )

    print(
        "  RAM: "
        f"{runtime_environment_rf['hardware']['memory_total_bytes'] / (1024 ** 3):.2f} "
        "GiB"
    )

    if runtime_environment_rf[
        "hardware"
    ]["gpus"]:
        for gpu_index, gpu in enumerate(
            runtime_environment_rf[
                "hardware"
            ]["gpus"],
            start=1,
        ):
            gpu_memory_text = (
                f"{gpu['memory_total_mib']:.0f} MiB"
                if gpu[
                    "memory_total_mib"
                ] is not None
                else "memory unavailable"
            )

            print(
                f"  GPU {gpu_index}: "
                f"{gpu['name']} "
                f"({gpu_memory_text}, "
                f"driver "
                f"{gpu['driver_version']})"
            )
    else:
        print(
            "  GPU: none detected "
            "by nvidia-smi"
        )

    print(
        "  Random Forest training "
        "device: CPU"
    )
    print(
        "  Saved: "
        "runtime_environment.json"
    )
    print(
        "  Saved: "
        "system_diagnostics.json"
    )
    print(
        "  Saved: requirements.txt"
    )

    print("\n" + "=" * 100)
    print("LOADING SOURCE DATA")
    print("=" * 100)

    train_df_rf = pd.read_csv(
        TRAIN_PATH
    )
    test_df_rf = pd.read_csv(
        TEST_PATH
    )

    print(
        f"Original train file: "
        f"{train_df_rf.shape}"
    )
    print(
        f"Original test file : "
        f"{test_df_rf.shape}"
    )

    combined_df_rf = (
        pd.concat(
            [train_df_rf, test_df_rf],
            axis=0,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

    print(
        "Combined deduplicated dataset: "
        f"{combined_df_rf.shape}"
    )

    if (
        len(combined_df_rf)
        != EXPECTED_UNIQUE_ROWS
    ):
        raise RuntimeError(
            f"Expected "
            f"{EXPECTED_UNIQUE_ROWS} "
            "unique observations; found "
            f"{len(combined_df_rf)}."
        )

    X_full_rf = combined_df_rf.drop(
        columns=["device_category"]
    )
    y_full_rf = combined_df_rf[
        "device_category"
    ]

    if (
        X_full_rf.shape[1]
        != EXPECTED_ORIGINAL_FEATURES
    ):
        raise RuntimeError(
            f"Expected "
            f"{EXPECTED_ORIGINAL_FEATURES} "
            "predictors before filtering; "
            f"found {X_full_rf.shape[1]}."
        )

    (
        X_temp_rf,
        X_test_rf,
        y_temp_rf,
        y_test_rf,
    ) = train_test_split(
        X_full_rf,
        y_full_rf,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y_full_rf,
    )

    (
        X_train_rf,
        X_val_rf,
        y_train_rf,
        y_val_rf,
    ) = train_test_split(
        X_temp_rf,
        y_temp_rf,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y_temp_rf,
    )

    print("\nReconstructed split:")
    print(
        f"Training   : "
        f"{X_train_rf.shape}"
    )
    print(
        f"Validation : "
        f"{X_val_rf.shape}"
    )
    print(
        f"Held-out   : "
        f"{X_test_rf.shape}"
    )

    if (
        len(X_train_rf)
        != EXPECTED_TRAIN_ROWS
    ):
        raise RuntimeError(
            f"Expected "
            f"{EXPECTED_TRAIN_ROWS} "
            "training rows; found "
            f"{len(X_train_rf)}."
        )

    if (
        len(X_val_rf)
        != EXPECTED_VAL_ROWS
    ):
        raise RuntimeError(
            f"Expected "
            f"{EXPECTED_VAL_ROWS} "
            "validation rows; found "
            f"{len(X_val_rf)}."
        )

    if (
        len(X_test_rf)
        != EXPECTED_TEST_ROWS
    ):
        raise RuntimeError(
            f"Expected "
            f"{EXPECTED_TEST_ROWS} "
            "held-out rows; found "
            f"{len(X_test_rf)}."
        )

    label_encoder_rf = LabelEncoder()

    y_train_1d_rf = (
        label_encoder_rf.fit_transform(
            y_train_rf
        )
    )

    y_val_1d_rf = (
        label_encoder_rf.transform(
            y_val_rf
        )
    )

    y_test_1d_rf = (
        label_encoder_rf.transform(
            y_test_rf
        )
    )

    device_classes_rf = np.asarray(
        label_encoder_rf.classes_
    )

    if (
        len(device_classes_rf)
        != EXPECTED_CLASSES
    ):
        raise RuntimeError(
            f"Expected {EXPECTED_CLASSES} "
            "device classes; found "
            f"{len(device_classes_rf)}."
        )

    labels_rf = np.arange(
        len(device_classes_rf)
    )

    print(
        "\nNumber of device classes: "
        f"{len(device_classes_rf)}"
    )
    print("Device classes:")

    for index, class_name in enumerate(
        device_classes_rf
    ):
        print(
            f"  {index}: {class_name}"
        )

    rf_pipeline = ImbPipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="mean"
                ),
            ),
            (
                "variance",
                VarianceThreshold(
                    threshold=0.0
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "smote",
                SMOTE(
                    random_state=RANDOM_STATE
                ),
            ),
            (
                "rf",
                RandomForestClassifier(
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
        ]
    )

    param_distributions_rf = {
        "rf__n_estimators": [
            100,
            200,
            300,
            500,
            800,
        ],
        "rf__max_depth": [
            None,
            5,
            10,
            15,
            20,
            30,
        ],
        "rf__min_samples_split": [
            2,
            5,
            10,
            15,
        ],
        "rf__min_samples_leaf": [
            1,
            2,
            4,
            8,
        ],
        "rf__max_features": [
            "sqrt",
            "log2",
            0.25,
            0.50,
            0.75,
            1.0,
        ],
        "rf__criterion": [
            "gini",
            "entropy",
            "log_loss",
        ],
        "rf__class_weight": [
            None,
            "balanced",
            "balanced_subsample",
        ],
    }

    cv_rf = StratifiedKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    rf_search = RandomizedSearchCV(
        estimator=rf_pipeline,
        param_distributions=(
            param_distributions_rf
        ),
        n_iter=RF_SEARCH_ITERATIONS,
        scoring={
            "macro_f1": "f1_macro",
            "accuracy": "accuracy",
            "balanced_accuracy": (
                "balanced_accuracy"
            ),
        },
        refit="macro_f1",
        cv=cv_rf,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=2,
        return_train_score=True,
        error_score="raise",
    )

    print("\n" + "=" * 100)
    print(
        "STARTING RANDOM FOREST "
        "RANDOMIZED SEARCH"
    )
    print("=" * 100)

    print(
        "\nSearch candidates: "
        f"{RF_SEARCH_ITERATIONS}"
    )
    print(
        f"CV folds: {CV_FOLDS}"
    )
    print(
        "Total candidate-fold fits: "
        f"{RF_SEARCH_ITERATIONS * CV_FOLDS}"
    )

    rf_search.fit(
        X_train_rf,
        y_train_1d_rf,
    )

    print("\n" + "=" * 100)
    print(
        "RANDOM FOREST SEARCH COMPLETE"
    )
    print("=" * 100)

    print(
        "\nBest cross-validation "
        "macro-F1: "
        f"{rf_search.best_score_:.6f}"
    )

    print("\nBest hyperparameters:")

    best_params_clean = {}

    for parameter, value in (
        rf_search.best_params_.items()
    ):
        clean_parameter = parameter.replace(
            "rf__",
            "",
        )

        best_params_clean[
            clean_parameter
        ] = value

        print(
            f"  {clean_parameter}: "
            f"{value}"
        )

    rf_cv_results_df = pd.DataFrame(
        rf_search.cv_results_
    ).sort_values(
        "rank_test_macro_f1"
    )

    rf_cv_results_df.to_csv(
        TEMP_RUN_DIR
        / "rf_search_results.csv",
        index=False,
    )

    tuned_rf_pipeline = (
        rf_search.best_estimator_
    )

    variance_selector_rf = (
        tuned_rf_pipeline.named_steps[
            "variance"
        ]
    )

    final_feature_mask_rf = (
        variance_selector_rf.get_support()
    )

    retained_features_rf = np.asarray(
        X_train_rf.columns
    )[final_feature_mask_rf]

    zero_variance_features_rf = np.asarray(
        X_train_rf.columns
    )[~final_feature_mask_rf]

    print("\n" + "=" * 100)
    print(
        "FINAL TRAINING-FITTED "
        "FEATURE AUDIT"
    )
    print("=" * 100)

    print(
        "Original predictor count: "
        f"{X_train_rf.shape[1]}"
    )
    print(
        "Zero-variance predictors "
        "removed: "
        f"{len(zero_variance_features_rf)}"
    )
    print(
        "Predictors retained: "
        f"{len(retained_features_rf)}"
    )

    if (
        len(zero_variance_features_rf)
        != EXPECTED_ZERO_VARIANCE_FEATURES
    ):
        raise RuntimeError(
            f"Expected "
            f"{EXPECTED_ZERO_VARIANCE_FEATURES} "
            "training-zero-variance "
            "predictors; found "
            f"{len(zero_variance_features_rf)}."
        )

    if (
        len(retained_features_rf)
        != EXPECTED_RETAINED_FEATURES
    ):
        raise RuntimeError(
            f"Expected "
            f"{EXPECTED_RETAINED_FEATURES} "
            "retained predictors; found "
            f"{len(retained_features_rf)}."
        )

    with open(
        TEMP_RUN_DIR
        / "retained_features.txt",
        "w",
        encoding="utf-8",
    ) as handle:
        for feature in retained_features_rf:
            handle.write(
                f"{feature}\n"
            )

    with open(
        TEMP_RUN_DIR
        / "zero_variance_columns.txt",
        "w",
        encoding="utf-8",
    ) as handle:
        for feature in (
            zero_variance_features_rf
        ):
            handle.write(
                f"{feature}\n"
            )

    training_class_counts = np.bincount(
        y_train_1d_rf,
        minlength=EXPECTED_CLASSES,
    )

    expected_balanced_class_size = int(
        np.max(training_class_counts)
    )

    expected_smote_training_rows = int(
        expected_balanced_class_size
        * EXPECTED_CLASSES
    )

    print(
        "\nTraining class counts "
        "before SMOTE:"
    )

    for (
        class_index,
        class_count,
    ) in enumerate(
        training_class_counts
    ):
        print(
            "  "
            f"{device_classes_rf[class_index]}: "
            f"{class_count}"
        )

    print(
        "\nExpected training rows after "
        "final SMOTE balancing: "
        f"{expected_smote_training_rows}"
    )

    validation_predictions_rf = (
        tuned_rf_pipeline.predict(
            X_val_rf
        )
    )

    validation_probabilities_rf = (
        tuned_rf_pipeline.predict_proba(
            X_val_rf
        )
    )

    if not np.all(
        np.isfinite(
            validation_probabilities_rf
        )
    ):
        raise RuntimeError(
            "Validation probabilities "
            "contain NaN or Inf."
        )

    validation_metrics_rf = (
        calculate_metrics(
            y_val_1d_rf,
            validation_predictions_rf,
            validation_probabilities_rf,
            labels_rf,
        )
    )

    print("\n" + "=" * 100)
    print(
        "TUNED RANDOM FOREST — "
        "VALIDATION RESULTS"
    )
    print("=" * 100)

    print(
        "\nAccuracy:                 "
        f"{validation_metrics_rf['accuracy'] * 100:.2f}%"
    )
    print(
        "Balanced accuracy:        "
        f"{validation_metrics_rf['balanced_accuracy'] * 100:.2f}%"
    )
    print(
        "Macro precision:          "
        f"{validation_metrics_rf['macro_precision']:.4f}"
    )
    print(
        "Macro recall:             "
        f"{validation_metrics_rf['macro_recall']:.4f}"
    )
    print(
        "Macro-F1:                 "
        f"{validation_metrics_rf['macro_f1']:.4f}"
    )
    print(
        "Log loss:                 "
        f"{validation_metrics_rf['log_loss']:.6f}"
    )
    print(
        "Macro ROC-AUC (OvR):      "
        f"{validation_metrics_rf['macro_roc_auc_ovr']:.4f}"
    )
    print(
        "Macro average precision:  "
        f"{validation_metrics_rf['macro_average_precision_ovr']:.4f}"
    )

    np.save(
        TEMP_RUN_DIR
        / "validation_predictions.npy",
        np.asarray(
            validation_predictions_rf,
            dtype=np.int64,
        ),
    )

    np.save(
        TEMP_RUN_DIR
        / "validation_probabilities.npy",
        np.asarray(
            validation_probabilities_rf,
            dtype=np.float64,
        ),
    )

    with open(
        TEMP_RUN_DIR
        / "validation_metrics.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            validation_metrics_rf,
            handle,
            indent=4,
            allow_nan=False,
        )

    print("\n" + "=" * 100)
    print(
        "GENERATING RANDOM FOREST "
        "TRAINING/VALIDATION "
        "DIAGNOSTIC FIGURES"
    )
    print("=" * 100)

    fitted_imputer_rf = (
        tuned_rf_pipeline.named_steps[
            "imputer"
        ]
    )
    fitted_variance_rf = (
        tuned_rf_pipeline.named_steps[
            "variance"
        ]
    )
    fitted_scaler_rf = (
        tuned_rf_pipeline.named_steps[
            "scaler"
        ]
    )

    X_train_diag_rf = (
        fitted_imputer_rf.transform(
            X_train_rf
        )
    )
    X_train_diag_rf = (
        fitted_variance_rf.transform(
            X_train_diag_rf
        )
    )
    X_train_diag_rf = (
        fitted_scaler_rf.transform(
            X_train_diag_rf
        )
    )

    X_val_diag_rf = (
        fitted_imputer_rf.transform(
            X_val_rf
        )
    )
    X_val_diag_rf = (
        fitted_variance_rf.transform(
            X_val_diag_rf
        )
    )
    X_val_diag_rf = (
        fitted_scaler_rf.transform(
            X_val_diag_rf
        )
    )

    diagnostic_smote_rf = SMOTE(
        random_state=RANDOM_STATE
    )

    (
        X_train_balanced_diag_rf,
        y_train_balanced_diag_rf,
    ) = diagnostic_smote_rf.fit_resample(
        X_train_diag_rf,
        y_train_1d_rf,
    )

    retained_rf = (
        tuned_rf_pipeline.named_steps["rf"]
    )

    retained_rf_params = (
        retained_rf.get_params(
            deep=False
        ).copy()
    )

    selected_n_estimators_rf = int(
        retained_rf_params.pop(
            "n_estimators"
        )
    )

    retained_rf_params.pop(
        "warm_start",
        None,
    )

    if selected_n_estimators_rf < 1:
        raise RuntimeError(
            "Invalid selected n_estimators "
            f"value: "
            f"{selected_n_estimators_rf}"
        )

    diagnostic_checkpoint_count_rf = min(
        50,
        selected_n_estimators_rf,
    )

    tree_checkpoints_rf = np.unique(
        np.linspace(
            1,
            selected_n_estimators_rf,
            diagnostic_checkpoint_count_rf,
            dtype=int,
        )
    )

    if (
        tree_checkpoints_rf[-1]
        != selected_n_estimators_rf
    ):
        tree_checkpoints_rf = np.append(
            tree_checkpoints_rf,
            selected_n_estimators_rf,
        )

    diagnostic_rf = RandomForestClassifier(
        **retained_rf_params,
        n_estimators=1,
        warm_start=True,
    )

    diagnostic_rows_rf = []

    for tree_count_rf in (
        tree_checkpoints_rf
    ):
        diagnostic_rf.set_params(
            n_estimators=int(
                tree_count_rf
            )
        )

        diagnostic_rf.fit(
            X_train_balanced_diag_rf,
            y_train_balanced_diag_rf,
        )

        train_probabilities_diag_rf = (
            diagnostic_rf.predict_proba(
                X_train_diag_rf
            )
        )

        validation_probabilities_diag_rf = (
            diagnostic_rf.predict_proba(
                X_val_diag_rf
            )
        )

        train_predictions_diag_rf = np.argmax(
            train_probabilities_diag_rf,
            axis=1,
        )

        validation_predictions_diag_rf = (
            np.argmax(
                validation_probabilities_diag_rf,
                axis=1,
            )
        )

        diagnostic_rows_rf.append(
            {
                "n_estimators": int(
                    tree_count_rf
                ),
                "training_accuracy": float(
                    accuracy_score(
                        y_train_1d_rf,
                        train_predictions_diag_rf,
                    )
                ),
                "validation_accuracy": float(
                    accuracy_score(
                        y_val_1d_rf,
                        validation_predictions_diag_rf,
                    )
                ),
                "training_log_loss": float(
                    log_loss(
                        y_train_1d_rf,
                        train_probabilities_diag_rf,
                        labels=labels_rf,
                    )
                ),
                "validation_log_loss": float(
                    log_loss(
                        y_val_1d_rf,
                        validation_probabilities_diag_rf,
                        labels=labels_rf,
                    )
                ),
            }
        )

    rf_training_curve_df = (
        pd.DataFrame(
            diagnostic_rows_rf
        )
    )

    RF_TRAINING_CURVE_CSV = (
        TEMP_RUN_DIR
        / "rf_cv_selected_training_validation_curve.csv"
    )

    rf_training_curve_df.to_csv(
        RF_TRAINING_CURVE_CSV,
        index=False,
    )

    retained_validation_probabilities_rf = (
        tuned_rf_pipeline.predict_proba(
            X_val_rf
        )
    )

    diagnostic_validation_probabilities_rf = (
        diagnostic_rf.predict_proba(
            X_val_diag_rf
        )
    )

    diagnostic_probability_max_abs_diff_rf = float(
        np.max(
            np.abs(
                retained_validation_probabilities_rf
                - diagnostic_validation_probabilities_rf
            )
        )
    )

    diagnostic_class_agreement_rf = float(
        np.mean(
            np.argmax(
                retained_validation_probabilities_rf,
                axis=1,
            )
            == np.argmax(
                diagnostic_validation_probabilities_rf,
                axis=1,
            )
        )
    )

    print(
        "\nSelected number of trees: "
        f"{selected_n_estimators_rf}"
    )
    print(
        "Diagnostic checkpoints: "
        f"{len(rf_training_curve_df)}"
    )
    print(
        "Final checkpoint class agreement "
        "with retained pipeline: "
        f"{diagnostic_class_agreement_rf * 100.0:.3f}%"
    )
    print(
        "Final checkpoint maximum "
        "probability difference: "
        f"{diagnostic_probability_max_abs_diff_rf:.12g}"
    )

    if not np.isclose(
        diagnostic_class_agreement_rf,
        1.0,
        atol=0.0,
        rtol=0.0,
    ):
        raise RuntimeError(
            "The cumulative-tree diagnostic "
            "did not reproduce the retained "
            "Random Forest's validation "
            "predictions."
        )

    if not np.allclose(
        retained_validation_probabilities_rf,
        diagnostic_validation_probabilities_rf,
        atol=1e-12,
        rtol=1e-12,
    ):
        raise RuntimeError(
            "The cumulative-tree diagnostic "
            "did not reproduce the retained "
            "Random Forest's validation "
            "probabilities within tolerance."
        )

    RF_ACCURACY_PNG = (
        FIGURES_DIR
        / "rf_training_validation_accuracy_cv_selected_configuration.png"
    )
    RF_ACCURACY_PDF = (
        FIGURES_DIR
        / "rf_training_validation_accuracy_cv_selected_configuration.pdf"
    )
    RF_LOSS_PNG = (
        FIGURES_DIR
        / "rf_training_validation_log_loss_cv_selected_configuration.png"
    )
    RF_LOSS_PDF = (
        FIGURES_DIR
        / "rf_training_validation_log_loss_cv_selected_configuration.pdf"
    )

    fig, ax = plt.subplots(
        figsize=(10, 7)
    )

    ax.plot(
        rf_training_curve_df[
            "n_estimators"
        ],
        rf_training_curve_df[
            "training_accuracy"
        ],
        linewidth=2,
        label="Training Accuracy",
    )

    ax.plot(
        rf_training_curve_df[
            "n_estimators"
        ],
        rf_training_curve_df[
            "validation_accuracy"
        ],
        linewidth=2,
        label="Validation Accuracy",
    )

    ax.axvline(
        selected_n_estimators_rf,
        linestyle="--",
        linewidth=1.5,
        label=(
            "CV-selected forest "
            f"({selected_n_estimators_rf} trees)"
        ),
    )

    ax.set_title(
        "Tuned Random Forest - "
        "Training and Validation Accuracy",
        fontweight="bold",
    )
    ax.set_xlabel(
        "Number of Trees",
        fontweight="bold",
    )
    ax.set_ylabel(
        "Accuracy",
        fontweight="bold",
    )
    ax.legend()
    ax.grid(
        True,
        linestyle="--",
        alpha=0.6,
    )

    fig.tight_layout()
    fig.savefig(
        RF_ACCURACY_PNG,
        dpi=300,
        bbox_inches="tight",
    )
    fig.savefig(
        RF_ACCURACY_PDF,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(
        figsize=(10, 7)
    )

    ax.plot(
        rf_training_curve_df[
            "n_estimators"
        ],
        rf_training_curve_df[
            "training_log_loss"
        ],
        linewidth=2,
        label="Training Log Loss",
    )

    ax.plot(
        rf_training_curve_df[
            "n_estimators"
        ],
        rf_training_curve_df[
            "validation_log_loss"
        ],
        linewidth=2,
        label="Validation Log Loss",
    )

    ax.axvline(
        selected_n_estimators_rf,
        linestyle="--",
        linewidth=1.5,
        label=(
            "CV-selected forest "
            f"({selected_n_estimators_rf} trees)"
        ),
    )

    ax.set_title(
        "Tuned Random Forest - "
        "Training and Validation Log Loss",
        fontweight="bold",
    )
    ax.set_xlabel(
        "Number of Trees",
        fontweight="bold",
    )
    ax.set_ylabel(
        "Log Loss",
        fontweight="bold",
    )
    ax.legend()
    ax.grid(
        True,
        linestyle="--",
        alpha=0.6,
    )

    fig.tight_layout()
    fig.savefig(
        RF_LOSS_PNG,
        dpi=300,
        bbox_inches="tight",
    )
    fig.savefig(
        RF_LOSS_PDF,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    print(
        "\nSaved Random Forest "
        "diagnostic figures:"
    )
    print(f"  - {RF_ACCURACY_PNG}")
    print(f"  - {RF_ACCURACY_PDF}")
    print(f"  - {RF_LOSS_PNG}")
    print(f"  - {RF_LOSS_PDF}")
    print(
        f"\nSaved curve data:\n"
        f"{RF_TRAINING_CURVE_CSV}"
    )

    np.save(
        TEMP_RUN_DIR
        / "X_test_raw.npy",
        np.asarray(
            X_test_rf,
            dtype=np.float32,
        ),
    )

    np.save(
        TEMP_RUN_DIR
        / "y_test_1d.npy",
        np.asarray(
            y_test_1d_rf,
            dtype=np.int64,
        ),
    )

    np.save(
        TEMP_RUN_DIR
        / "device_classes.npy",
        device_classes_rf,
    )

    np.save(
        TEMP_RUN_DIR
        / "raw_feature_names.npy",
        np.asarray(
            X_test_rf.columns,
            dtype=object,
        ),
    )

    test_bundle_manifest = {
        "input_file": "X_test_raw.npy",
        "input_space": (
            "raw_reconstructed_predictors"
        ),
        "input_shape": list(
            X_test_rf.shape
        ),
        "feature_count": int(
            X_test_rf.shape[1]
        ),
        "feature_names_file": (
            "raw_feature_names.npy"
        ),
        "labels_file": (
            "y_test_1d.npy"
        ),
        "classes_file": (
            "device_classes.npy"
        ),
        "held_out_prediction_performed_here": False,
        "held_out_evaluation_block": (
            "Universal Held-Out Test Evaluation"
        ),
    }

    with open(
        TEMP_RUN_DIR
        / "test_bundle_manifest.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            test_bundle_manifest,
            handle,
            indent=4,
            allow_nan=False,
        )

    tuned_rf_path = (
        TEMP_RUN_DIR
        / "tuned_random_forest_pipeline.joblib"
    )

    joblib.dump(
        tuned_rf_pipeline,
        tuned_rf_path,
    )

    joblib.dump(
        rf_search,
        TEMP_RUN_DIR
        / "rf_randomized_search.joblib",
    )

    joblib.dump(
        label_encoder_rf,
        TEMP_RUN_DIR
        / "label_encoder.joblib",
    )

    print(
        "\nSaved fitted tuned RF pipeline:\n"
        f"{tuned_rf_path}"
    )

    tuned_rf_results = {
        "run_timestamp": (
            RUN_DATETIME.isoformat()
        ),
        "random_state": RANDOM_STATE,
        "search": {
            "method": (
                "RandomizedSearchCV"
            ),
            "selection_metric": (
                "macro_f1"
            ),
            "best_cv_macro_f1": float(
                rf_search.best_score_
            ),
            "cv_folds": CV_FOLDS,
            "search_iterations": (
                RF_SEARCH_ITERATIONS
            ),
            "best_hyperparameters": (
                best_params_clean
            ),
        },
        "preprocessing": {
            "imputation": "mean",
            "variance_threshold": 0.0,
            "features_before_variance_filter": int(
                X_train_rf.shape[1]
            ),
            "zero_variance_features_removed": int(
                len(
                    zero_variance_features_rf
                )
            ),
            "features_retained": int(
                len(retained_features_rf)
            ),
            "standardisation": (
                "StandardScaler"
            ),
            "smote_random_state": (
                RANDOM_STATE
            ),
            "smote_location": (
                "inside each CV training fold "
                "and inside final training refit"
            ),
            "validation_oversampled": False,
            "test_oversampled": False,
        },
        "dataset": {
            "unique_rows": int(
                len(combined_df_rf)
            ),
            "training_rows": int(
                len(X_train_rf)
            ),
            "validation_rows": int(
                len(X_val_rf)
            ),
            "held_out_rows": int(
                len(X_test_rf)
            ),
            "expected_final_smote_rows": (
                expected_smote_training_rows
            ),
            "class_count": int(
                len(device_classes_rf)
            ),
        },
        "validation": (
            validation_metrics_rf
        ),
        "training_validation_diagnostic": {
            "x_axis": "number_of_trees",
            "descriptive_only": True,
            "used_for_model_selection": False,
            "selected_n_estimators": int(
                selected_n_estimators_rf
            ),
            "checkpoint_count": int(
                len(rf_training_curve_df)
            ),
            "curve_csv": (
                RF_TRAINING_CURVE_CSV.name
            ),
            "accuracy_png": str(
                Path("Figures")
                / RF_ACCURACY_PNG.name
            ),
            "accuracy_pdf": str(
                Path("Figures")
                / RF_ACCURACY_PDF.name
            ),
            "log_loss_png": str(
                Path("Figures")
                / RF_LOSS_PNG.name
            ),
            "log_loss_pdf": str(
                Path("Figures")
                / RF_LOSS_PDF.name
            ),
            "final_checkpoint_class_agreement": (
                diagnostic_class_agreement_rf
            ),
            "final_checkpoint_max_probability_difference": (
                diagnostic_probability_max_abs_diff_rf
            ),
        },
        "held_out_test": {
            "prediction_performed": False,
            "evaluation_performed": False,
            "evaluation_deferred_to": (
                "Universal Held-Out "
                "Test Evaluation"
            ),
        },
    }

    with open(
        TEMP_RUN_DIR
        / "tuned_rf_results.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            tuned_rf_results,
            handle,
            indent=4,
            allow_nan=False,
        )

    tuned_rf_results_df = pd.DataFrame(
        [
            {
                "best_cv_macro_f1": (
                    rf_search.best_score_
                ),
                "validation_accuracy": (
                    validation_metrics_rf[
                        "accuracy"
                    ]
                ),
                "validation_balanced_accuracy": (
                    validation_metrics_rf[
                        "balanced_accuracy"
                    ]
                ),
                "validation_macro_f1": (
                    validation_metrics_rf[
                        "macro_f1"
                    ]
                ),
                "validation_log_loss": (
                    validation_metrics_rf[
                        "log_loss"
                    ]
                ),
                "features_retained": len(
                    retained_features_rf
                ),
                "n_estimators": (
                    best_params_clean.get(
                        "n_estimators"
                    )
                ),
                "max_depth": (
                    best_params_clean.get(
                        "max_depth"
                    )
                ),
                "min_samples_split": (
                    best_params_clean.get(
                        "min_samples_split"
                    )
                ),
                "min_samples_leaf": (
                    best_params_clean.get(
                        "min_samples_leaf"
                    )
                ),
                "max_features": (
                    best_params_clean.get(
                        "max_features"
                    )
                ),
                "criterion": (
                    best_params_clean.get(
                        "criterion"
                    )
                ),
                "class_weight": (
                    best_params_clean.get(
                        "class_weight"
                    )
                ),
                "held_out_test_evaluated": False,
            }
        ]
    )

    tuned_rf_results_df.to_csv(
        TEMP_RUN_DIR
        / "tuned_rf_results.csv",
        index=False,
    )

    print("\n" + "=" * 100)
    print("SAVING SOURCE CODE")
    print("=" * 100)

    try:
        notebook_history = (
            get_ipython().user_ns.get(
                "In",
                [],
            )
        )

        current_cell_code = (
            notebook_history[-1]
            if len(notebook_history) > 0
            else ""
        )

    except Exception as exc:
        current_cell_code = (
            "Error retrieving cell source: "
            f"{repr(exc)}"
        )

    code_path = (
        TEMP_RUN_DIR
        / "tuned_random_forest_experiment.py"
    )

    with open(
        code_path,
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write(
            current_cell_code
        )

    code_sha256 = sha256_file(
        code_path
    )

    with open(
        TEMP_RUN_DIR
        / "tuned_random_forest_experiment.sha256",
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write(
            code_sha256
        )

    print(
        f"Code SHA-256: {code_sha256}"
    )

    metadata = {
        "project": (
            "Smart Home Appliance Identification"
        ),
        "experiment": (
            "Tuned Random Forest"
        ),
        "run_timestamp": (
            RUN_DATETIME.isoformat()
        ),
        "run_timestamp_folder": (
            RUN_TIMESTAMP
        ),
        "timezone": (
            "Africa/Johannesburg"
        ),
        "random_state": RANDOM_STATE,
        "runtime_environment": {
            "runtime_environment_file": (
                "runtime_environment.json"
            ),
            "system_diagnostics_file": (
                "system_diagnostics.json"
            ),
            "requirements_file": (
                "requirements.txt"
            ),
            "captured_before_training": True,
        },
        "dataset": {
            "original_train_shape": list(
                train_df_rf.shape
            ),
            "original_test_shape": list(
                test_df_rf.shape
            ),
            "deduplicated_shape": list(
                combined_df_rf.shape
            ),
            "train_shape": list(
                X_train_rf.shape
            ),
            "validation_shape": list(
                X_val_rf.shape
            ),
            "held_out_shape": list(
                X_test_rf.shape
            ),
        },
        "pipeline_order": [
            "SimpleImputer(mean)",
            "VarianceThreshold(0)",
            "StandardScaler",
            "SMOTE(random_state=42)",
            "RandomForestClassifier",
        ],
        "cross_validation": {
            "type": (
                "StratifiedKFold"
            ),
            "folds": CV_FOLDS,
            "shuffle": True,
            "random_state": RANDOM_STATE,
            "preprocessing_fitted_inside_folds": True,
            "smote_fitted_inside_folds": True,
        },
        "search": {
            "type": (
                "RandomizedSearchCV"
            ),
            "iterations": (
                RF_SEARCH_ITERATIONS
            ),
            "selection_metric": (
                "macro_f1"
            ),
            "best_cv_macro_f1": float(
                rf_search.best_score_
            ),
            "best_params": (
                best_params_clean
            ),
        },
        "final_feature_count": int(
            len(retained_features_rf)
        ),
        "zero_variance_feature_count": int(
            len(
                zero_variance_features_rf
            )
        ),
        "validation_metrics": (
            validation_metrics_rf
        ),
        "training_validation_figures": {
            "interpretation": (
                "Random Forest has no "
                "training epochs. These "
                "descriptive curves show "
                "training and validation "
                "performance as the cumulative "
                "number of trees grows for the "
                "CV-selected configuration."
            ),
            "x_axis": "number_of_trees",
            "used_for_model_selection": False,
            "selected_n_estimators": int(
                selected_n_estimators_rf
            ),
            "curve_csv": (
                RF_TRAINING_CURVE_CSV.name
            ),
            "accuracy_png": str(
                Path("Figures")
                / RF_ACCURACY_PNG.name
            ),
            "accuracy_pdf": str(
                Path("Figures")
                / RF_ACCURACY_PDF.name
            ),
            "log_loss_png": str(
                Path("Figures")
                / RF_LOSS_PNG.name
            ),
            "log_loss_pdf": str(
                Path("Figures")
                / RF_LOSS_PDF.name
            ),
        },
        "held_out_test": {
            "prediction_performed": False,
            "evaluation_performed": False,
            "evaluation_deferred_to": (
                "Universal Held-Out "
                "Test Evaluation"
            ),
        },
        "code": {
            "filename": code_path.name,
            "sha256": code_sha256,
        },
    }

    with open(
        TEMP_RUN_DIR
        / "run_metadata.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            metadata,
            handle,
            indent=4,
            allow_nan=False,
        )

    print("\n" + "=" * 100)
    print("CREATING FILE INVENTORY")
    print("=" * 100)

    inventory = []

    for file_path in sorted(
        TEMP_RUN_DIR.rglob("*")
    ):
        if file_path.is_file():
            relative_path = (
                file_path.relative_to(
                    TEMP_RUN_DIR
                )
            )

            inventory.append(
                {
                    "File": str(
                        relative_path
                    ),
                    "Size_Bytes": int(
                        file_path.stat().st_size
                    ),
                    "SHA256": sha256_file(
                        file_path
                    ),
                }
            )

    inventory_df = pd.DataFrame(
        inventory
    )

    inventory_df.to_csv(
        TEMP_RUN_DIR
        / "file_inventory.csv",
        index=False,
    )

    if FINAL_RUN_DIR.exists():
        raise FileExistsError(
            "A final tuned-RF run "
            "directory with this timestamp "
            "already exists:\n"
            f"{FINAL_RUN_DIR}"
        )

    print("\n" + "=" * 100)
    print(
        "FINALISING GOOGLE DRIVE "
        "RUN DIRECTORY"
    )
    print("=" * 100)

    sys.stdout.flush()
    sys.stderr.flush()
    terminal_log_file.flush()
    terminal_log_file.close()

    sys.stdout = original_stdout
    sys.stderr = original_stderr

    shutil.move(
        str(TEMP_RUN_DIR),
        str(FINAL_RUN_DIR),
    )

    print("\n" + "=" * 100)
    print(
        "TUNED RANDOM FOREST "
        "TRAINING COMPLETE"
    )
    print("=" * 100)

    print(
        "\nBest CV macro-F1: "
        f"{rf_search.best_score_:.4f}"
    )

    print("\nBest hyperparameters:")

    for parameter, value in (
        best_params_clean.items()
    ):
        print(
            f"  {parameter}: {value}"
        )

    print("\nValidation:")
    print(
        "  Accuracy: "
        f"{validation_metrics_rf['accuracy'] * 100:.2f}%"
    )
    print(
        "  Macro-F1: "
        f"{validation_metrics_rf['macro_f1']:.4f}"
    )

    print("\nHeld-out test:")
    print(
        "  NOT evaluated in this block."
    )
    print(
        "  Raw held-out bundle saved "
        "for the Universal Held-Out "
        "Test Evaluation block."
    )

    print(
        "\nFinal retained predictors:"
    )
    print(
        f"  {len(retained_features_rf)}"
    )

    print(
        f"\nFinal run directory:\n"
        f"{FINAL_RUN_DIR}"
    )
    print(
        f"\nSaved fitted pipeline:\n"
        f"{FINAL_RUN_DIR / 'tuned_random_forest_pipeline.joblib'}"
    )

    print(
        "\nRecorded execution environment:"
    )
    print(
        "  - "
        f"{FINAL_RUN_DIR / 'runtime_environment.json'}"
    )
    print(
        "  - "
        f"{FINAL_RUN_DIR / 'system_diagnostics.json'}"
    )
    print(
        "  - "
        f"{FINAL_RUN_DIR / 'requirements.txt'}"
    )

    print(
        "\nTraining/validation "
        "diagnostic figures:"
    )
    print(
        "  - Figures/"
        f"{RF_ACCURACY_PNG.name}"
    )
    print(
        "  - Figures/"
        f"{RF_LOSS_PNG.name}"
    )
    print(
        "  - "
        f"{RF_TRAINING_CURVE_CSV.name}"
    )
    print(
        "  Note: Random Forest has no "
        "epochs; these curves use "
        "cumulative number of trees for "
        "the CV-selected configuration."
    )

except Exception as exc:
    try:
        terminal_log_file.flush()
        terminal_log_file.close()
    except Exception:
        pass

    sys.stdout = original_stdout
    sys.stderr = original_stderr

    print("\n" + "=" * 100)
    print(
        "TUNED RANDOM FOREST "
        "TRAINING FAILED"
    )
    print("=" * 100)

    print(
        f"\nError: {repr(exc)}"
    )
    print(
        "\nPartial artifacts remain in:\n"
        f"{TEMP_RUN_DIR}"
    )

    raise

## Model Evaluation and Deployment

### Held-Out Test Evaluation

This universal saved-artifact evaluator calculates the authoritative held-out results without retraining or modifying any model.

It discovers compatible timestamped runs beneath:

```text
Honours_Project/Models/
```

Supported artifacts include:

- validation-selected CNN, LSTM, BiLSTM, Feature-Attention MLP and CNN-BiLSTM models;
- seed-specific neural models; and
- `tuned_random_forest_pipeline.joblib`.

The user may select the newest compatible run, every compatible run, numbered runs, a range of runs or an explicit run path.

Neural models are loaded without compilation. The custom `SelfAttention` class is registered for CNN-BiLSTM deserialization. Inputs are automatically adapted to either `(376, 244, 1)` or `(376, 244)`. The Random Forest uses its saved raw 297-feature test matrix and fitted preprocessing pipeline.

For each model, the evaluator records:

- accuracy and balanced accuracy;
- macro and weighted precision, recall and F1;
- test log loss;
- macro one-vs-rest ROC-AUC;
- macro average precision;
- per-class metrics;
- predicted classes and probabilities;
- confusion matrix;
- ROC curves; and
- precision-recall curves.

Outputs are written beneath the corresponding run:

```text
<run>/
└── Test_Evaluation/
    ├── test_evaluation_summary.csv
    ├── <model identifier>/
    │   ├── metrics.csv
    │   ├── metrics.json
    │   ├── classification_report.csv
    │   ├── per_class_metrics.csv
    │   ├── predicted_classes.npy
    │   ├── prediction_probabilities.npy
    │   └── Figures/
    └── Figures/
```

When all five seed-specific models for an architecture are available, the evaluator also produces per-seed tables, mean and sample-standard-deviation summaries, and across-seed accuracy and loss figures.

A timestamped global summary is written under:

```text
Honours_Project/Models/Test_Evaluation_Summaries/
```

Global summaries duplicate selected rows for convenient discovery. The canonical model-specific result remains the `metrics.csv` stored inside the originating run. Downstream cells should deduplicate candidates by run directory rather than treating these summary copies as separate experiments.

In [ ]:
import json
from datetime import datetime
from pathlib import Path
import re
from zoneinfo import ZoneInfo

from google.colab import drive
from IPython.display import display
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    auc,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    log_loss,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import label_binarize
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.layers import Layer
from tensorflow.keras.models import load_model

drive.mount("/content/drive")

MODELS_ROOT = Path("/content/drive/MyDrive/Honours_Project/Models")
EXPECTED_TEST_ROWS = 376
EXPECTED_CLASS_COUNT = 10

FINAL_SELECTED_KERAS_FILENAMES = (
    "best_validation_cnn_bilstm.keras",
    "best_validation_mlp.keras",
    "best_validation_cnn.keras",
    "best_validation_lstm.keras",
    "best_validation_bilstm.keras",
)

FINAL_SEED_KERAS_PATTERNS = (
    "cnn_bilstm_seed_*.keras",
    "mlp_seed_*.keras",
    "cnn_seed_*.keras",
    "lstm_seed_*.keras",
    "bilstm_seed_*.keras",
)

FINAL_TUNED_RF_FILENAME = "tuned_random_forest_pipeline.joblib"
RETAINED_KERAS_PATTERNS = (
    *FINAL_SELECTED_KERAS_FILENAMES,
    *FINAL_SEED_KERAS_PATTERNS,
)
TUNED_RF_FILENAME = FINAL_TUNED_RF_FILENAME


@tf.keras.utils.register_keras_serializable(package="COMP700")
class SelfAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="att_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True,
        )
        self.b = self.add_weight(
            name="att_bias",
            shape=(input_shape[1], 1),
            initializer="zeros",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, inputs):
        e = K.tanh(K.dot(inputs, self.W) + self.b)
        a = K.softmax(e, axis=1)
        return K.sum(inputs * a, axis=1)


SELF_ATTENTION_CUSTOM_OBJECTS = {
    "SelfAttention": SelfAttention,
    "Custom>SelfAttention": SelfAttention,
    "COMP700>SelfAttention": SelfAttention,
}


def safe_name(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("_")


def save_both(fig, base_path):
    base_path = Path(base_path)
    base_path.parent.mkdir(parents=True, exist_ok=True)

    png_path = base_path.with_suffix(".png")
    pdf_path = base_path.with_suffix(".pdf")

    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white",
    )

    if not png_path.exists():
        raise RuntimeError(f"PNG figure was not saved:\n{png_path}")
    if not pdf_path.exists():
        raise RuntimeError(f"PDF figure was not saved:\n{pdf_path}")

    print(f"[Saved Figure] {png_path}")
    print(f"[Saved Figure] {pdf_path}")


def parse_selection(text, total):
    selected = set()

    for token in text.replace(" ", "").split(","):
        if not token:
            continue

        if "-" in token:
            start, end = token.split("-", 1)
            selected.update(range(int(start), int(end) + 1))
        else:
            selected.add(int(token))

    invalid = [
        index
        for index in selected
        if index < 1 or index > total
    ]

    if invalid:
        raise ValueError(f"Invalid selection numbers: {invalid}")

    return sorted(selected)


def extract_run_timestamp(path):
    match = re.match(
        r"^(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})",
        path.name,
    )

    if match:
        return datetime.strptime(
            match.group(1),
            "%Y-%m-%d_%H-%M-%S",
        )

    return datetime.fromtimestamp(path.stat().st_mtime)


def run_has_retained_model(run_dir):
    run_dir = Path(run_dir)

    if any(
        (run_dir / filename).exists()
        for filename in FINAL_SELECTED_KERAS_FILENAMES
    ):
        return True

    if any(
        list(run_dir.glob(pattern))
        for pattern in FINAL_SEED_KERAS_PATTERNS
    ):
        return True

    if (run_dir / FINAL_TUNED_RF_FILENAME).exists():
        return True

    return False


def run_has_test_bundle(run_dir):
    run_dir = Path(run_dir)

    common_files_exist = (
        (run_dir / "y_test_1d.npy").exists()
        and (run_dir / "device_classes.npy").exists()
    )

    input_exists = (
        (run_dir / "X_test_3d.npy").exists()
        or (run_dir / "X_test_raw.npy").exists()
    )

    return bool(
        common_files_exist
        and input_exists
        and run_has_retained_model(run_dir)
    )


def find_runs():
    candidate_directories = set()

    for filename in ("X_test_3d.npy", "X_test_raw.npy"):
        for input_path in MODELS_ROOT.rglob(filename):
            run_dir = input_path.parent

            if "Test_Evaluation" in run_dir.parts:
                continue

            if any(
                part.startswith("_TEMP_")
                for part in run_dir.parts
            ):
                continue

            if run_has_test_bundle(run_dir):
                candidate_directories.add(run_dir)

    return sorted(
        candidate_directories,
        key=extract_run_timestamp,
        reverse=True,
    )


def describe_run_bundle(run_dir):
    bundle_types = []

    if (run_dir / "X_test_3d.npy").exists():
        bundle_types.append("neural")

    if (run_dir / "X_test_raw.npy").exists():
        bundle_types.append("tuned-RF")

    return "+".join(bundle_types)


def choose_runs(runs):
    if not runs:
        raise RuntimeError(
            "No compatible run folders found.\n"
            "Each run must contain y_test_1d.npy, "
            "device_classes.npy and either X_test_3d.npy "
            "or X_test_raw.npy."
        )

    print(
        "\n"
        + "=" * 110
        + "\nCOMPATIBLE SAVED RUNS\n"
        + "=" * 110
    )

    for index, run_dir in enumerate(runs, start=1):
        try:
            shown = run_dir.relative_to(MODELS_ROOT)
        except ValueError:
            shown = run_dir

        print(
            f"[{index:>3}] {shown} "
            f"[{describe_run_bundle(run_dir)}]"
        )

    print(
        "\nSelection commands:\n"
        "  latest     = newest compatible run\n"
        "  all        = every compatible run\n"
        "  1,3,5-8    = selected numbered runs\n"
        "  path       = explicit run folder"
    )

    choice = (
        input("\nSelect run(s) [latest]: ")
        .strip()
        .lower()
        or "latest"
    )

    if choice == "latest":
        return [runs[0]]

    if choice == "all":
        return runs

    if choice == "path":
        run_dir = Path(
            input("Paste run folder: ").strip()
        )

        if not run_has_test_bundle(run_dir):
            raise FileNotFoundError(
                "The selected folder does not contain a "
                "compatible held-out test bundle:\n"
                f"{run_dir}"
            )

        return [run_dir]

    indexes = parse_selection(choice, len(runs))
    return [runs[index - 1] for index in indexes]


def load_test_bundle(run_dir):
    run_dir = Path(run_dir)

    y = np.asarray(
        np.load(run_dir / "y_test_1d.npy"),
        dtype=np.int64,
    ).reshape(-1)

    classes = np.asarray(
        np.load(
            run_dir / "device_classes.npy",
            allow_pickle=True,
        )
    )

    X_keras = None
    X_rf = None

    keras_path = run_dir / "X_test_3d.npy"
    rf_path = run_dir / "X_test_raw.npy"

    if keras_path.exists():
        X_keras = np.asarray(
            np.load(keras_path),
            dtype=np.float32,
        )

        if len(X_keras) != len(y):
            raise RuntimeError(
                f"Keras X/y size mismatch in {run_dir}"
            )

    if rf_path.exists():
        X_rf = np.asarray(
            np.load(rf_path),
            dtype=np.float32,
        )

        if len(X_rf) != len(y):
            raise RuntimeError(
                f"RF X/y size mismatch in {run_dir}"
            )

    if X_keras is None and X_rf is None:
        raise RuntimeError(
            "No compatible held-out input tensor found in "
            f"{run_dir}"
        )

    if (
        EXPECTED_TEST_ROWS is not None
        and len(y) != EXPECTED_TEST_ROWS
    ):
        raise RuntimeError(
            f"Expected {EXPECTED_TEST_ROWS} test rows; "
            f"found {len(y)} in {run_dir}"
        )

    if (
        EXPECTED_CLASS_COUNT is not None
        and len(classes) != EXPECTED_CLASS_COUNT
    ):
        raise RuntimeError(
            f"Expected {EXPECTED_CLASS_COUNT} classes; "
            f"found {len(classes)} in {run_dir}"
        )

    return {
        "X_keras": X_keras,
        "X_rf": X_rf,
        "y": y,
        "classes": classes,
    }


def discover_models(run_dir):
    run_dir = Path(run_dir)

    keras_paths = []
    seen = set()

    for pattern in RETAINED_KERAS_PATTERNS:
        for path in sorted(run_dir.glob(pattern)):
            if path not in seen:
                keras_paths.append(path)
                seen.add(path)

    rf_paths = []
    tuned_rf_path = run_dir / TUNED_RF_FILENAME

    if tuned_rf_path.exists():
        rf_paths.append(tuned_rf_path)

    return keras_paths, rf_paths


def adapt_keras_input(X_test_3d, model):
    input_shape = model.input_shape

    if isinstance(input_shape, list):
        input_shape = input_shape[0]

    if len(input_shape) == 3:
        X_model = X_test_3d

    elif len(input_shape) == 2:
        if (
            X_test_3d.ndim == 3
            and X_test_3d.shape[-1] == 1
        ):
            X_model = X_test_3d[..., 0]
        else:
            X_model = X_test_3d

    else:
        raise RuntimeError(
            f"Unsupported Keras input shape: {input_shape}"
        )

    X_model = np.asarray(
        X_model,
        dtype=np.float32,
    )

    if (
        input_shape[1] is not None
        and X_model.shape[1] != int(input_shape[1])
    ):
        raise RuntimeError(
            "Feature mismatch. "
            f"Model expects {input_shape}; "
            f"got {X_model.shape}"
        )

    return X_model


def prepare_rf_input(model, X_test_raw):
    if X_test_raw is None:
        raise RuntimeError(
            "The selected tuned Random Forest run does not "
            "contain X_test_raw.npy."
        )

    X_array = np.asarray(
        X_test_raw,
        dtype=np.float32,
    )

    expected_features = getattr(
        model,
        "n_features_in_",
        None,
    )

    if (
        expected_features is not None
        and X_array.shape[1] != int(expected_features)
    ):
        raise RuntimeError(
            "Tuned RF raw-feature mismatch. "
            f"Model expects {expected_features}; "
            "held-out bundle contains "
            f"{X_array.shape[1]}."
        )

    feature_names = getattr(
        model,
        "feature_names_in_",
        None,
    )

    if feature_names is not None:
        if len(feature_names) != X_array.shape[1]:
            raise RuntimeError(
                "Tuned RF feature-name count does not match "
                "the held-out raw-feature matrix."
            )

        return pd.DataFrame(
            X_array,
            columns=[
                str(value)
                for value in feature_names
            ],
        )

    return X_array


def keras_probabilities(model_path, X_test_3d):
    tf.keras.backend.clear_session()

    model = load_model(
        model_path,
        custom_objects=SELF_ATTENTION_CUSTOM_OBJECTS,
        compile=False,
    )

    X_model = adapt_keras_input(
        X_test_3d,
        model,
    )

    return np.asarray(
        model.predict(
            X_model,
            verbose=0,
        ),
        dtype=np.float64,
    )


def rf_probabilities(
    model_path,
    X_test_raw,
    class_count,
):
    model = joblib.load(model_path)

    X_model = prepare_rf_input(
        model,
        X_test_raw,
    )

    raw_probabilities = np.asarray(
        model.predict_proba(X_model),
        dtype=np.float64,
    )

    probabilities = np.zeros(
        (len(X_model), class_count),
        dtype=np.float64,
    )

    model_classes = getattr(
        model,
        "classes_",
        None,
    )

    if (
        model_classes is None
        and hasattr(model, "named_steps")
        and "rf" in model.named_steps
    ):
        model_classes = (
            model.named_steps["rf"].classes_
        )

    if model_classes is None:
        raise RuntimeError(
            "Could not determine tuned RF class-column order."
        )

    for column, encoded_class in enumerate(model_classes):
        probabilities[:, int(encoded_class)] = (
            raw_probabilities[:, column]
        )

    return probabilities


def evaluate_probabilities(
    run_dir,
    model_name,
    model_source,
    model_type,
    y_true,
    probabilities,
    classes,
):
    labels = np.arange(len(classes))

    if probabilities.shape != (
        len(y_true),
        len(classes),
    ):
        raise RuntimeError(
            f"{model_name}: unexpected probability shape "
            f"{probabilities.shape}"
        )

    y_pred = np.argmax(
        probabilities,
        axis=1,
    )

    (
        macro_precision,
        macro_recall,
        macro_f1,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0,
    )

    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        average="weighted",
        zero_division=0,
    )

    y_binary = label_binarize(
        y_true,
        classes=labels,
    )

    try:
        macro_roc_auc = roc_auc_score(
            y_binary,
            probabilities,
            average="macro",
            multi_class="ovr",
        )
    except Exception:
        macro_roc_auc = np.nan

    try:
        macro_average_precision = (
            average_precision_score(
                y_binary,
                probabilities,
                average="macro",
            )
        )
    except Exception:
        macro_average_precision = np.nan

    metrics = {
        "evaluated_at": datetime.now(
            ZoneInfo("Africa/Johannesburg")
        ).isoformat(),
        "model_name": model_name,
        "model_type": model_type,
        "model_source": str(model_source),
        "test_rows": int(len(y_true)),
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_precision": float(
            macro_precision
        ),
        "macro_recall": float(
            macro_recall
        ),
        "macro_f1": float(
            macro_f1
        ),
        "weighted_precision": float(
            weighted_precision
        ),
        "weighted_recall": float(
            weighted_recall
        ),
        "weighted_f1": float(
            weighted_f1
        ),
        "test_log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=labels,
            )
        ),
        "macro_roc_auc_ovr": (
            None
            if np.isnan(macro_roc_auc)
            else float(macro_roc_auc)
        ),
        "macro_average_precision_ovr": (
            None
            if np.isnan(macro_average_precision)
            else float(macro_average_precision)
        ),
    }

    clean_name = safe_name(model_name)

    result_dir = (
        run_dir
        / "Test_Evaluation"
        / clean_name
    )

    figure_dir = result_dir / "Figures"

    result_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    figure_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.save(
        result_dir / "predicted_classes.npy",
        y_pred,
    )

    np.save(
        result_dir / "prediction_probabilities.npy",
        probabilities,
    )

    with open(
        result_dir / "metrics.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            metrics,
            handle,
            indent=2,
            allow_nan=False,
        )

    pd.DataFrame(
        [metrics]
    ).to_csv(
        result_dir / "metrics.csv",
        index=False,
    )

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=[
            str(class_name)
            for class_name in classes
        ],
        zero_division=0,
        output_dict=True,
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=[
            str(class_name)
            for class_name in classes
        ],
        zero_division=0,
    )

    pd.DataFrame(
        report_dict
    ).transpose().to_csv(
        result_dir / "classification_report.csv"
    )

    with open(
        result_dir / "classification_report.txt",
        "w",
        encoding="utf-8",
    ) as handle:
        handle.write(report_text)

    per_class_dataframe = pd.DataFrame(
        {
            "class": [
                str(class_name)
                for class_name in classes
            ],
            "precision": [
                report_dict[str(class_name)]["precision"]
                for class_name in classes
            ],
            "recall": [
                report_dict[str(class_name)]["recall"]
                for class_name in classes
            ],
            "f1_score": [
                report_dict[str(class_name)]["f1-score"]
                for class_name in classes
            ],
            "support": [
                int(report_dict[str(class_name)]["support"])
                for class_name in classes
            ],
        }
    )

    per_class_dataframe.to_csv(
        result_dir / "per_class_metrics.csv",
        index=False,
    )

    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=labels,
    )

    np.save(
        result_dir / "confusion_matrix.npy",
        confusion,
    )

    figure, axis = plt.subplots(
        figsize=(11, 9)
    )

    sns.heatmap(
        confusion,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=classes,
        yticklabels=classes,
        ax=axis,
    )

    axis.set_title(
        f"{model_name} - Confusion Matrix",
        fontweight="bold",
    )
    axis.set_xlabel(
        "Predicted Device Class",
        fontweight="bold",
    )
    axis.set_ylabel(
        "True Device Class",
        fontweight="bold",
    )
    axis.tick_params(
        axis="x",
        rotation=45,
    )
    axis.tick_params(
        axis="y",
        rotation=0,
    )

    figure.tight_layout()
    save_both(
        figure,
        figure_dir / "confusion_matrix",
    )
    plt.show()
    plt.close(figure)

    figure, axis = plt.subplots(
        figsize=(10, 6)
    )

    sns.barplot(
        data=per_class_dataframe,
        x="class",
        y="f1_score",
        ax=axis,
    )

    axis.set_title(
        f"{model_name} - F1 Score per Class",
        fontweight="bold",
    )
    axis.set_xlabel(
        "Device Class",
        fontweight="bold",
    )
    axis.set_ylabel(
        "F1 Score",
        fontweight="bold",
    )
    axis.set_ylim(0, 1.05)
    axis.tick_params(
        axis="x",
        rotation=45,
    )
    axis.grid(
        axis="y",
        linestyle="--",
        alpha=0.6,
    )

    figure.tight_layout()
    save_both(
        figure,
        figure_dir / "per_class_f1",
    )
    plt.show()
    plt.close(figure)

    figure, axis = plt.subplots(
        figsize=(10, 8)
    )

    for class_index, class_name in enumerate(classes):
        if (
            len(
                np.unique(
                    y_binary[:, class_index]
                )
            )
            < 2
        ):
            continue

        false_positive_rate, true_positive_rate, _ = (
            roc_curve(
                y_binary[:, class_index],
                probabilities[:, class_index],
            )
        )

        class_auc = auc(
            false_positive_rate,
            true_positive_rate,
        )

        axis.plot(
            false_positive_rate,
            true_positive_rate,
            linewidth=2,
            label=(
                f"{class_name} "
                f"(AUC={class_auc:.3f})"
            ),
        )

    axis.plot(
        [0, 1],
        [0, 1],
        "k--",
        linewidth=1.5,
        label="Random Chance",
    )

    axis.set_title(
        f"{model_name} - ROC Curves (One-vs-Rest)",
        fontweight="bold",
    )
    axis.set_xlabel(
        "False Positive Rate",
        fontweight="bold",
    )
    axis.set_ylabel(
        "True Positive Rate",
        fontweight="bold",
    )
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1.05)
    axis.legend(
        loc="lower right",
        fontsize="small",
        ncol=2,
    )
    axis.grid(
        True,
        linestyle="--",
        alpha=0.6,
    )

    figure.tight_layout()
    save_both(
        figure,
        figure_dir / "roc_curves_ovr",
    )
    plt.show()
    plt.close(figure)

    figure, axis = plt.subplots(
        figsize=(10, 8)
    )

    for class_index, class_name in enumerate(classes):
        if np.sum(y_binary[:, class_index]) == 0:
            continue

        precision, recall, _ = (
            precision_recall_curve(
                y_binary[:, class_index],
                probabilities[:, class_index],
            )
        )

        class_pr_auc = auc(
            recall,
            precision,
        )

        axis.plot(
            recall,
            precision,
            linewidth=2,
            label=(
                f"{class_name} "
                f"(AUC={class_pr_auc:.3f})"
            ),
        )

    axis.set_title(
        f"{model_name} - "
        "Precision-Recall Curves (One-vs-Rest)",
        fontweight="bold",
    )
    axis.set_xlabel(
        "Recall",
        fontweight="bold",
    )
    axis.set_ylabel(
        "Precision",
        fontweight="bold",
    )
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1.05)
    axis.legend(
        loc="lower left",
        fontsize="small",
        ncol=2,
    )
    axis.grid(
        True,
        linestyle="--",
        alpha=0.6,
    )

    figure.tight_layout()
    save_both(
        figure,
        figure_dir / "precision_recall_curves_ovr",
    )
    plt.show()
    plt.close(figure)

    print(
        "\n"
        + "=" * 100
        + f"\n{model_name}\n"
        + "=" * 100
    )

    print(
        f"Accuracy:          "
        f"{metrics['accuracy'] * 100:.2f}%"
    )
    print(
        f"Balanced Accuracy: "
        f"{metrics['balanced_accuracy'] * 100:.2f}%"
    )
    print(
        f"Macro Precision:   "
        f"{metrics['macro_precision']:.4f}"
    )
    print(
        f"Macro Recall:      "
        f"{metrics['macro_recall']:.4f}"
    )
    print(
        f"Macro F1:          "
        f"{metrics['macro_f1']:.4f}"
    )
    print(
        f"Weighted F1:       "
        f"{metrics['weighted_f1']:.4f}"
    )
    print(
        f"Test Log Loss:     "
        f"{metrics['test_log_loss']:.6f}"
    )

    if metrics["macro_roc_auc_ovr"] is not None:
        print(
            f"Macro ROC-AUC:     "
            f"{metrics['macro_roc_auc_ovr']:.4f}"
        )

    if (
        metrics["macro_average_precision_ovr"]
        is not None
    ):
        print(
            "Macro Average Precision:      "
            f"{metrics['macro_average_precision_ovr']:.4f}"
        )

    print("\n" + report_text)

    return metrics, probabilities


def seed_group(model_name):
    match = re.match(
        r"^(?P<group>.+?)_seed_(?P<seed>\d+)$",
        model_name,
    )

    if not match:
        return None, None

    return (
        match.group("group"),
        int(match.group("seed")),
    )


def save_seed_summaries(run_dir, rows):
    if not rows:
        return

    dataframe = pd.DataFrame(rows)

    parsed = dataframe[
        "model_name"
    ].apply(seed_group)

    dataframe["seed_group"] = [
        item[0]
        for item in parsed
    ]

    dataframe["seed"] = [
        item[1]
        for item in parsed
    ]

    dataframe = dataframe[
        dataframe["seed_group"].notna()
    ].copy()

    if dataframe.empty:
        return

    summary_rows = []

    for group_name, group_dataframe in dataframe.groupby(
        "seed_group"
    ):
        group_dataframe = group_dataframe.sort_values(
            "seed"
        )

        output_dir = (
            run_dir
            / "Test_Evaluation"
            / (
                "Seed_Group_"
                + safe_name(group_name)
            )
        )

        figure_dir = output_dir / "Figures"

        output_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        figure_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        group_dataframe.to_csv(
            output_dir / "per_seed_test_metrics.csv",
            index=False,
        )

        summary_rows.append(
            {
                "seed_group": group_name,
                "n_seeds": int(
                    len(group_dataframe)
                ),
                "accuracy_mean": float(
                    group_dataframe[
                        "accuracy"
                    ].mean()
                ),
                "accuracy_sd": (
                    float(
                        group_dataframe[
                            "accuracy"
                        ].std(ddof=1)
                    )
                    if len(group_dataframe) > 1
                    else 0.0
                ),
                "macro_f1_mean": float(
                    group_dataframe[
                        "macro_f1"
                    ].mean()
                ),
                "macro_f1_sd": (
                    float(
                        group_dataframe[
                            "macro_f1"
                        ].std(ddof=1)
                    )
                    if len(group_dataframe) > 1
                    else 0.0
                ),
                "test_log_loss_mean": float(
                    group_dataframe[
                        "test_log_loss"
                    ].mean()
                ),
                "test_log_loss_sd": (
                    float(
                        group_dataframe[
                            "test_log_loss"
                        ].std(ddof=1)
                    )
                    if len(group_dataframe) > 1
                    else 0.0
                ),
            }
        )

        figure, axis = plt.subplots(
            figsize=(9, 5)
        )

        axis.bar(
            group_dataframe[
                "seed"
            ].astype(str),
            group_dataframe["accuracy"] * 100,
        )

        mean_accuracy = (
            group_dataframe["accuracy"].mean()
            * 100
        )

        axis.axhline(
            mean_accuracy,
            linestyle="--",
            linewidth=2,
            label=(
                f"Mean = "
                f"{mean_accuracy:.2f}%"
            ),
        )

        axis.set_title(
            f"{group_name} - "
            "Test Accuracy Across Seeds",
            fontweight="bold",
        )
        axis.set_xlabel(
            "Random Seed",
            fontweight="bold",
        )
        axis.set_ylabel(
            "Test Accuracy (%)",
            fontweight="bold",
        )
        axis.set_ylim(0, 100)
        axis.grid(
            axis="y",
            linestyle="--",
            alpha=0.6,
        )
        axis.legend()

        figure.tight_layout()

        save_both(
            figure,
            figure_dir
            / "test_accuracy_across_seeds",
        )

        plt.show()
        plt.close(figure)

        figure, axis = plt.subplots(
            figsize=(9, 5)
        )

        axis.bar(
            group_dataframe[
                "seed"
            ].astype(str),
            group_dataframe["test_log_loss"],
        )

        mean_loss = group_dataframe[
            "test_log_loss"
        ].mean()

        axis.axhline(
            mean_loss,
            linestyle="--",
            linewidth=2,
            label=f"Mean = {mean_loss:.4f}",
        )

        axis.set_title(
            f"{group_name} - "
            "Test Log Loss Across Seeds",
            fontweight="bold",
        )
        axis.set_xlabel(
            "Random Seed",
            fontweight="bold",
        )
        axis.set_ylabel(
            "Categorical Cross-Entropy / Log Loss",
            fontweight="bold",
        )
        axis.grid(
            axis="y",
            linestyle="--",
            alpha=0.6,
        )
        axis.legend()

        figure.tight_layout()

        save_both(
            figure,
            figure_dir
            / "test_log_loss_across_seeds",
        )

        plt.show()
        plt.close(figure)

    pd.DataFrame(
        summary_rows
    ).to_csv(
        run_dir
        / "Test_Evaluation"
        / "seed_group_summary.csv",
        index=False,
    )


def evaluate_run(run_dir):
    print(
        "\n"
        + "#" * 110
        + "\nEVALUATING RUN:\n"
        + str(run_dir)
        + "\n"
        + "#" * 110
    )

    bundle = load_test_bundle(run_dir)

    X_test_3d = bundle["X_keras"]
    X_test_raw = bundle["X_rf"]
    y_test_1d = bundle["y"]
    classes = bundle["classes"]

    keras_paths, rf_paths = discover_models(
        run_dir
    )

    rows = []

    if keras_paths and X_test_3d is None:
        print(
            "[SKIPPED] Retained Keras models were found, "
            "but X_test_3d.npy is absent."
        )

    elif X_test_3d is not None:
        for model_path in keras_paths:
            try:
                probabilities = keras_probabilities(
                    model_path,
                    X_test_3d,
                )

                record, _ = evaluate_probabilities(
                    run_dir,
                    model_path.stem,
                    model_path,
                    "keras",
                    y_test_1d,
                    probabilities,
                    classes,
                )

                rows.append(record)

            except Exception as exc:
                print(
                    f"[SKIPPED] {model_path}: {exc}"
                )

    if rf_paths and X_test_raw is None:
        print(
            "[SKIPPED] Tuned Random Forest was found, "
            "but X_test_raw.npy is absent."
        )

    elif X_test_raw is not None:
        for model_path in rf_paths:
            try:
                probabilities = rf_probabilities(
                    model_path,
                    X_test_raw,
                    len(classes),
                )

                record, _ = evaluate_probabilities(
                    run_dir,
                    model_path.stem,
                    model_path,
                    "tuned_random_forest",
                    y_test_1d,
                    probabilities,
                    classes,
                )

                rows.append(record)

            except Exception as exc:
                print(
                    f"[SKIPPED] {model_path}: {exc}"
                )

    if not rows:
        print(
            "No models were evaluated for this run."
        )
        return []

    result_root = (
        run_dir
        / "Test_Evaluation"
    )

    figure_root = (
        result_root
        / "Figures"
    )

    result_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    figure_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    run_dataframe = (
        pd.DataFrame(rows)
        .sort_values(
            "accuracy",
            ascending=False,
        )
    )

    run_dataframe.to_csv(
        result_root
        / "test_evaluation_summary.csv",
        index=False,
    )

    save_seed_summaries(
        run_dir,
        rows,
    )

    figure, axis = plt.subplots(
        figsize=(
            max(
                10,
                0.55 * len(run_dataframe),
            ),
            6,
        )
    )

    axis.bar(
        run_dataframe["model_name"],
        run_dataframe["accuracy"] * 100,
    )

    axis.set_title(
        "Held-Out Test Accuracy by Saved Model",
        fontweight="bold",
    )
    axis.set_xlabel(
        "Saved Model",
        fontweight="bold",
    )
    axis.set_ylabel(
        "Accuracy (%)",
        fontweight="bold",
    )
    axis.set_ylim(0, 100)
    axis.tick_params(
        axis="x",
        rotation=60,
    )
    axis.grid(
        axis="y",
        linestyle="--",
        alpha=0.6,
    )

    figure.tight_layout()

    save_both(
        figure,
        figure_root / "all_models_test_accuracy",
    )

    plt.show()
    plt.close(figure)

    figure, axis = plt.subplots(
        figsize=(
            max(
                10,
                0.55 * len(run_dataframe),
            ),
            6,
        )
    )

    axis.bar(
        run_dataframe["model_name"],
        run_dataframe["macro_f1"],
    )

    axis.set_title(
        "Held-Out Test Macro-F1 by Saved Model",
        fontweight="bold",
    )
    axis.set_xlabel(
        "Saved Model",
        fontweight="bold",
    )
    axis.set_ylabel(
        "Macro-F1",
        fontweight="bold",
    )
    axis.set_ylim(0, 1)
    axis.tick_params(
        axis="x",
        rotation=60,
    )
    axis.grid(
        axis="y",
        linestyle="--",
        alpha=0.6,
    )

    figure.tight_layout()

    save_both(
        figure,
        figure_root / "all_models_macro_f1",
    )

    plt.show()
    plt.close(figure)

    saved_files = sorted(
        path
        for path in result_root.rglob("*")
        if path.is_file()
    )

    print(
        "\n"
        + "=" * 100
        + "\nSAVED TEST-EVALUATION OUTPUTS\n"
        + "=" * 100
    )

    for path in saved_files:
        print(
            path.relative_to(run_dir)
        )

    print(
        "=" * 100
        + f"\nTotal saved evaluation files: "
        f"{len(saved_files)}"
    )

    return rows


if not MODELS_ROOT.exists():
    raise FileNotFoundError(
        "Models root does not exist:\n"
        f"{MODELS_ROOT}"
    )

compatible_runs = find_runs()
selected_runs = choose_runs(
    compatible_runs
)

all_rows = []

for selected_run_dir in selected_runs:
    run_rows = evaluate_run(
        selected_run_dir
    )

    for row in run_rows:
        row = dict(row)
        row["run_directory"] = str(
            selected_run_dir
        )
        all_rows.append(row)

if not all_rows:
    print("\nNo models were evaluated.")

else:
    global_directory = (
        MODELS_ROOT
        / "Test_Evaluation_Summaries"
    )

    global_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    timestamp = datetime.now(
        ZoneInfo("Africa/Johannesburg")
    ).strftime(
        "%Y-%m-%d_%H-%M-%S"
    )

    global_summary_path = (
        global_directory
        / (
            "test_evaluation_summary_"
            f"{timestamp}.csv"
        )
    )

    global_dataframe = pd.DataFrame(
        all_rows
    )

    global_dataframe.to_csv(
        global_summary_path,
        index=False,
    )

    print(
        "\n"
        + "=" * 110
        + "\nUNIVERSAL TEST EVALUATION COMPLETE\n"
        + "=" * 110
    )

    print(
        f"Evaluated records: "
        f"{len(global_dataframe)}"
    )

    print(
        f"Global summary: "
        f"{global_summary_path}"
    )

    display_columns = [
        "run_directory",
        "model_name",
        "accuracy",
        "balanced_accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_f1",
        "macro_roc_auc_ovr",
        "macro_average_precision_ovr",
        "test_log_loss",
    ]

    display(
        global_dataframe[
            [
                column
                for column in display_columns
                if column
                in global_dataframe.columns
            ]
        ]
    )

### Selected-Model Predictive Comparison Figure

This figure compares the held-out predictive performance of the six selected models: the Attention CNN–BiLSTM, Feature-Attention MLP, CNN, LSTM, BiLSTM and tuned Random Forest.

The five neural results correspond to the representative models selected using restored validation performance. The Random Forest was selected using training-only cross-validated macro-F1. Held-out test performance was not used to select any model.

Three complementary metrics are shown:

- **Accuracy** is the proportion of all held-out observations classified correctly.
- **Balanced accuracy** is the unweighted mean of the recall obtained for each device class. It reduces the influence of differences in class support.
- **Macro-F1** is the unweighted mean of the ten class-specific F1 scores. It gives every device class equal importance and therefore penalises poor minority-class performance more strongly than ordinary accuracy.

The code reads only the canonical model-specific evaluation file stored at `Test_Evaluation/<model_identifier>/metrics.csv` inside each selected run. Aggregate run summaries and global summary copies are deliberately ignored. Consequently, the same evaluation is not presented several times merely because its results were copied into multiple CSV files.

Before combining the results, the cell verifies that:

1. every selected evaluation contains 376 held-out observations;
2. the complete held-out label sequence is identical across all six models;
3. the device-class order is identical across all six models;
4. the five neural runs contain exactly the same processed held-out tensor; and
5. the saved input tensor for every model contains 376 rows.

The Random Forest uses the archived raw 297-predictor test matrix because its fitted pipeline performs preprocessing internally. The neural models use the archived 244-predictor processed tensor. Their input arrays therefore cannot be compared directly, but their held-out labels and class order must still match exactly.

When the cell is run, it asks for one run directory for each model. Pressing Enter selects the newest compatible run. If several runs are listed, select the authoritative run used in the thesis.

The cell saves:

- `selected_model_predictive_comparison.png`;
- `selected_model_predictive_comparison.pdf`;
- `selected_model_predictive_comparison_data.csv`; and
- `selected_model_predictive_comparison_provenance.json`

under `Honours_Project/Thesis_Figures`.

The generated CSV records the plotted numerical values and source run directories. The provenance JSON records the canonical metrics source, model-selection basis, input and label file hashes, selected run paths and completed integrity checks.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

from google.colab import drive
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")


PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
MODELS_ROOT = PROJECT_ROOT / "Models"
OUTPUT_DIR = PROJECT_ROOT / "Thesis_Figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PNG_PATH = OUTPUT_DIR / "selected_model_predictive_comparison.png"
PDF_PATH = OUTPUT_DIR / "selected_model_predictive_comparison.pdf"
DATA_PATH = OUTPUT_DIR / "selected_model_predictive_comparison_data.csv"
PROVENANCE_PATH = (
    OUTPUT_DIR / "selected_model_predictive_comparison_provenance.json"
)

EXPECTED_TEST_ROWS = 376
REQUIRED_METRICS = ["accuracy", "balanced_accuracy", "macro_f1"]

MODEL_SPECS = {
    "Attention CNN-BiLSTM": {
        "run_root": MODELS_ROOT / "Tuned" / "CNN_BiLSTM",
        "identifier": "best_validation_cnn_bilstm",
        "input_file": "X_test_3d.npy",
        "selection_basis": "restored validation accuracy",
    },
    "Feature-Attention MLP": {
        "run_root": MODELS_ROOT / "Tuned" / "Feature_Attention_MLP",
        "identifier": "best_validation_mlp",
        "input_file": "X_test_3d.npy",
        "selection_basis": "restored validation accuracy",
    },
    "CNN": {
        "run_root": MODELS_ROOT / "Tuned" / "CNN_Tuned",
        "identifier": "best_validation_cnn",
        "input_file": "X_test_3d.npy",
        "selection_basis": "restored validation accuracy",
    },
    "LSTM": {
        "run_root": MODELS_ROOT / "Tuned" / "LSTM_Tuned",
        "identifier": "best_validation_lstm",
        "input_file": "X_test_3d.npy",
        "selection_basis": "restored validation accuracy",
    },
    "BiLSTM": {
        "run_root": MODELS_ROOT / "Tuned" / "BiLSTM_Tuned",
        "identifier": "best_validation_bilstm",
        "input_file": "X_test_3d.npy",
        "selection_basis": "restored validation accuracy",
    },
    "Tuned Random Forest": {
        "run_root": MODELS_ROOT / "Baselines" / "Tuned_RF",
        "identifier": "tuned_random_forest_pipeline",
        "input_file": "X_test_raw.npy",
        "selection_basis": "training-only cross-validated macro-F1",
    },
}

MODEL_ORDER = list(MODEL_SPECS)


def normalise_fraction(value):
    value = float(value)

    if value > 1.0:
        value /= 100.0

    return value


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def run_sort_key(run_dir):
    try:
        return datetime.strptime(
            run_dir.name[:19],
            "%Y-%m-%d_%H-%M-%S",
        ).timestamp()
    except ValueError:
        return run_dir.stat().st_mtime


def read_canonical_metrics(metrics_path, expected_identifier):
    frame = pd.read_csv(metrics_path)

    required_columns = {
        "model_name",
        "test_rows",
        *REQUIRED_METRICS,
    }

    missing_columns = sorted(
        required_columns.difference(frame.columns)
    )

    if missing_columns:
        raise RuntimeError(
            f"Missing columns in {metrics_path}: "
            f"{missing_columns}"
        )

    rows = frame.loc[
        frame["model_name"].astype(str).str.strip()
        == expected_identifier
    ].copy()

    if len(rows) != 1:
        raise RuntimeError(
            f"Expected exactly one row for "
            f"{expected_identifier} in {metrics_path}; "
            f"found {len(rows)}."
        )

    row = rows.iloc[0].copy()

    if int(row["test_rows"]) != EXPECTED_TEST_ROWS:
        raise RuntimeError(
            f"Expected {EXPECTED_TEST_ROWS} held-out rows "
            f"in {metrics_path}; "
            f"found {int(row['test_rows'])}."
        )

    for metric in REQUIRED_METRICS:
        row[metric] = normalise_fraction(row[metric])

        if not 0.0 <= float(row[metric]) <= 1.0:
            raise RuntimeError(
                f"Invalid {metric} value in "
                f"{metrics_path}: {row[metric]}"
            )

    return row


def discover_run_candidates(display_name, specification):
    run_root = specification["run_root"]
    identifier = specification["identifier"]
    input_file = specification["input_file"]

    if not run_root.exists():
        raise FileNotFoundError(
            f"The run root for {display_name} "
            f"does not exist:\n{run_root}"
        )

    candidates = []

    for run_dir in run_root.iterdir():
        if (
            not run_dir.is_dir()
            or run_dir.name.startswith("_TEMP_")
        ):
            continue

        metrics_path = (
            run_dir
            / "Test_Evaluation"
            / identifier
            / "metrics.csv"
        )

        y_path = run_dir / "y_test_1d.npy"
        classes_path = run_dir / "device_classes.npy"
        input_path = run_dir / input_file

        required_paths = [
            metrics_path,
            y_path,
            classes_path,
            input_path,
        ]

        if not all(path.exists() for path in required_paths):
            continue

        try:
            metrics_row = read_canonical_metrics(
                metrics_path,
                identifier,
            )

            y_true = np.asarray(
                np.load(y_path),
                dtype=np.int64,
            ).reshape(-1)

            device_classes = np.asarray(
                np.load(
                    classes_path,
                    allow_pickle=True,
                )
            ).astype(str)

            test_input = np.load(
                input_path,
                mmap_mode="r",
            )

            if len(y_true) != EXPECTED_TEST_ROWS:
                raise RuntimeError(
                    f"Unexpected label count: "
                    f"{len(y_true)}"
                )

            if test_input.shape[0] != EXPECTED_TEST_ROWS:
                raise RuntimeError(
                    f"Unexpected test-input row count: "
                    f"{test_input.shape[0]}"
                )

            if len(device_classes) != 10:
                raise RuntimeError(
                    f"Unexpected class count: "
                    f"{len(device_classes)}"
                )

            del test_input

        except Exception as exc:
            print(
                f"[SKIPPED] {run_dir}: {exc}"
            )
            continue

        candidates.append(
            {
                "display_name": display_name,
                "run_dir": run_dir,
                "metrics_path": metrics_path,
                "y_path": y_path,
                "classes_path": classes_path,
                "input_path": input_path,
                "metrics_row": metrics_row,
                "sort_key": run_sort_key(run_dir),
            }
        )

    return sorted(
        candidates,
        key=lambda candidate: candidate["sort_key"],
        reverse=True,
    )


def select_run(display_name, specification):
    candidates = discover_run_candidates(
        display_name,
        specification,
    )

    if not candidates:
        raise FileNotFoundError(
            f"No canonical held-out evaluation was found "
            f"for {display_name}.\n"
            "Run the Universal Held-Out Test Evaluation "
            "for the selected saved model first."
        )

    print("\n" + "=" * 100)
    print(f"SELECT MODEL RUN: {display_name}")
    print("=" * 100)

    for number, candidate in enumerate(
        candidates,
        start=1,
    ):
        row = candidate["metrics_row"]

        print(
            f"[{number}] "
            f"accuracy="
            f"{float(row['accuracy']) * 100:.3f}% | "
            f"balanced accuracy="
            f"{float(row['balanced_accuracy']) * 100:.3f}% | "
            f"macro-F1="
            f"{float(row['macro_f1']):.4f}\n"
            f"    {candidate['run_dir']}"
        )

    while True:
        response = input(
            f"Select 1-{len(candidates)} "
            "(Enter = newest run): "
        ).strip()

        if response == "":
            return candidates[0]

        if (
            response.isdigit()
            and 1 <= int(response) <= len(candidates)
        ):
            return candidates[int(response) - 1]

        print("Invalid selection.")


selected_runs = {
    display_name: select_run(
        display_name,
        specification,
    )
    for display_name, specification
    in MODEL_SPECS.items()
}


reference_y = None
reference_classes = None
reference_neural_input = None
reference_neural_model = None

for display_name in MODEL_ORDER:
    selected = selected_runs[display_name]

    y_true = np.asarray(
        np.load(selected["y_path"]),
        dtype=np.int64,
    ).reshape(-1)

    device_classes = np.asarray(
        np.load(
            selected["classes_path"],
            allow_pickle=True,
        )
    ).astype(str)

    if reference_y is None:
        reference_y = y_true
        reference_classes = device_classes

    else:
        if not np.array_equal(
            reference_y,
            y_true,
        ):
            raise RuntimeError(
                f"Held-out labels for {display_name} "
                "do not exactly match the previously "
                "selected model runs. Choose runs created "
                "from the same corrected split."
            )

        if not np.array_equal(
            reference_classes,
            device_classes,
        ):
            raise RuntimeError(
                f"Class order for {display_name} "
                "does not exactly match the previously "
                "selected model runs."
            )

    if selected["input_path"].name == "X_test_3d.npy":
        neural_input = np.asarray(
            np.load(selected["input_path"])
        )

        if reference_neural_input is None:
            reference_neural_input = neural_input
            reference_neural_model = display_name

        elif not np.array_equal(
            reference_neural_input,
            neural_input,
        ):
            raise RuntimeError(
                f"The processed neural held-out tensor "
                f"for {display_name} does not match "
                f"{reference_neural_model}. Choose runs "
                "created from the same corrected "
                "preprocessing pipeline."
            )


result_rows = []

for display_name in MODEL_ORDER:
    specification = MODEL_SPECS[display_name]
    selected = selected_runs[display_name]
    row = selected["metrics_row"]

    result_rows.append(
        {
            "display_model": display_name,
            "model_name": specification["identifier"],
            "selection_basis": (
                specification["selection_basis"]
            ),
            "test_rows": int(row["test_rows"]),
            "accuracy": float(row["accuracy"]),
            "balanced_accuracy": float(
                row["balanced_accuracy"]
            ),
            "macro_f1": float(row["macro_f1"]),
            "accuracy_percent": (
                float(row["accuracy"]) * 100.0
            ),
            "balanced_accuracy_percent": (
                float(row["balanced_accuracy"]) * 100.0
            ),
            "macro_f1_percent": (
                float(row["macro_f1"]) * 100.0
            ),
            "run_dir": str(selected["run_dir"]),
            "canonical_metrics_csv": str(
                selected["metrics_path"]
            ),
        }
    )


results = pd.DataFrame(result_rows)

if results["display_model"].tolist() != MODEL_ORDER:
    raise RuntimeError(
        "The result order does not match MODEL_ORDER."
    )


metric_specs = [
    (
        "accuracy_percent",
        "Accuracy",
        "#276FBF",
    ),
    (
        "balanced_accuracy_percent",
        "Balanced accuracy",
        "#F28E2B",
    ),
    (
        "macro_f1_percent",
        "Macro-F1",
        "#59A14F",
    ),
]

plot_labels = [
    "Attention CNN–BiLSTM",
    "Feature-Attention MLP",
    "CNN",
    "LSTM",
    "BiLSTM",
    "Tuned Random Forest",
]

y_positions = np.arange(len(results))
bar_height = 0.22

figure, axis = plt.subplots(
    figsize=(12.5, 8.4)
)

for metric_index, (
    column,
    label,
    colour,
) in enumerate(metric_specs):

    values = results[column].to_numpy(
        dtype=float
    )

    offsets = (
        y_positions
        + (metric_index - 1) * bar_height
    )

    bars = axis.barh(
        offsets,
        values,
        height=bar_height,
        label=label,
        color=colour,
        edgecolor="black",
        linewidth=0.55,
    )

    axis.bar_label(
        bars,
        labels=[
            f"{value:.2f}"
            for value in values
        ],
        padding=4,
        fontsize=8.5,
    )


axis.set_xlabel(
    "Held-out score (%)",
    fontsize=11,
)

axis.set_ylabel(
    "Model",
    fontsize=11,
)

axis.set_yticks(y_positions)
axis.set_yticklabels(
    plot_labels,
    fontsize=10,
)

axis.invert_yaxis()
axis.set_xlim(0, 100)
axis.set_xticks(
    np.arange(0, 101, 10)
)

axis.grid(
    axis="x",
    linestyle="--",
    linewidth=0.7,
    alpha=0.35,
)

axis.set_axisbelow(True)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)

handles, legend_labels = (
    axis.get_legend_handles_labels()
)

figure.suptitle(
    "Selected-Model Held-Out Predictive Comparison",
    fontsize=14,
    fontweight="bold",
    y=0.975,
)

figure.legend(
    handles,
    legend_labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.925),
    ncol=3,
    frameon=False,
    fontsize=9.5,
    columnspacing=2.0,
    handletextpad=0.6,
)

figure.subplots_adjust(
    left=0.25,
    right=0.97,
    top=0.84,
    bottom=0.10,
)

figure.savefig(
    PNG_PATH,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

figure.savefig(
    PDF_PATH,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(figure)


results.to_csv(
    DATA_PATH,
    index=False,
)

provenance = {
    "generated_at_utc": (
        datetime.now(timezone.utc).isoformat()
    ),
    "expected_test_rows": EXPECTED_TEST_ROWS,
    "selection_rule": (
        "One validation-selected artifact per neural "
        "architecture and the training-cross-validation-"
        "selected Random Forest. Held-out metrics were "
        "used only for final comparison, not for model "
        "selection."
    ),
    "source_policy": (
        "Only each run's canonical model-specific "
        "Test_Evaluation/<model_identifier>/metrics.csv "
        "was read. Aggregate and global summary CSV "
        "copies were deliberately ignored."
    ),
    "integrity_checks": {
        "held_out_row_count": EXPECTED_TEST_ROWS,
        "held_out_labels_identical_across_all_models": True,
        "class_order_identical_across_all_models": True,
        "neural_test_tensor_identical_across_five_neural_models": True,
        "common_class_order": reference_classes.tolist(),
    },
    "models": {},
}

for display_name in MODEL_ORDER:
    specification = MODEL_SPECS[display_name]
    selected = selected_runs[display_name]
    row = selected["metrics_row"]

    provenance["models"][display_name] = {
        "model_identifier": (
            specification["identifier"]
        ),
        "selection_basis": (
            specification["selection_basis"]
        ),
        "run_dir": str(
            selected["run_dir"]
        ),
        "canonical_metrics_csv": str(
            selected["metrics_path"]
        ),
        "test_input_file": str(
            selected["input_path"]
        ),
        "test_input_sha256": sha256_file(
            selected["input_path"]
        ),
        "held_out_labels_file": str(
            selected["y_path"]
        ),
        "held_out_labels_sha256": sha256_file(
            selected["y_path"]
        ),
        "device_classes_file": str(
            selected["classes_path"]
        ),
        "device_classes_sha256": sha256_file(
            selected["classes_path"]
        ),
        "accuracy": float(
            row["accuracy"]
        ),
        "balanced_accuracy": float(
            row["balanced_accuracy"]
        ),
        "macro_f1": float(
            row["macro_f1"]
        ),
    }

PROVENANCE_PATH.write_text(
    json.dumps(
        provenance,
        indent=2,
    ),
    encoding="utf-8",
)


print("\n" + "=" * 100)
print(
    "SELECTED-MODEL PREDICTIVE "
    "COMPARISON COMPLETE"
)
print("=" * 100)

print("Verified:")
print(
    f"  - {EXPECTED_TEST_ROWS} held-out "
    "observations for every model"
)
print(
    "  - identical held-out label sequence "
    "across all six models"
)
print(
    "  - identical class order across "
    "all six models"
)
print(
    "  - identical processed test tensor "
    "across all five neural models"
)

print("\nSelected results:")

print(
    results[
        [
            "display_model",
            "accuracy_percent",
            "balanced_accuracy_percent",
            "macro_f1_percent",
        ]
    ].to_string(index=False)
)

print(f"\nSaved PNG: {PNG_PATH}")
print(f"Saved PDF: {PDF_PATH}")
print(f"Saved data: {DATA_PATH}")
print(f"Saved provenance: {PROVENANCE_PATH}")

### Full Test-Set Inference Demonstration

This section provides an interactive inference demonstration using previously trained and archived project models. It is intended to show how the retained models classify individual held-out network-flow observations in practice without retraining or modifying the models.

The demonstration supports both:

- saved Keras neural models; and
- the tuned Random Forest pipeline.

#### Saved Model Discovery

The code searches recursively under:

`Honours_Project/Models/`

for the retained validation-selected neural models:

- `best_validation_cnn_bilstm.keras`
- `best_validation_mlp.keras`
- `best_validation_cnn.keras`
- `best_validation_lstm.keras`
- `best_validation_bilstm.keras`

and for the retained tuned Random Forest:

`tuned_random_forest_pipeline.joblib`

A model is listed only when its associated held-out test bundle is available in the same run directory. Neural models require:

```text
X_test_3d.npy
y_test_1d.npy
device_classes.npy
```

while the tuned Random Forest requires:

```text
X_test_raw.npy
y_test_1d.npy
device_classes.npy
```

Temporary run directories and previously generated test-evaluation outputs are excluded from model discovery.

#### Model Selection

All compatible saved models are displayed as a numbered list.

The user selects one model by entering its corresponding number. The selected model and its associated held-out test artifacts are then loaded directly from the archived run directory.

The demonstration therefore allows the same interface to be used with the CNN-BiLSTM, Feature-Attention MLP, CNN, LSTM, BiLSTM or tuned Random Forest, provided the required saved artifacts are present.

#### Input Preparation

The code automatically adapts the held-out input representation to the selected model.

For Keras models:

- three-dimensional sequence/convolutional models receive `(samples, features, 1)` input;
- two-dimensional feed-forward models receive the corresponding flattened feature matrix.

For the tuned Random Forest:

- the archived raw tabular feature matrix is used;
- the expected feature count is verified;
- and, where available, the fitted feature names are restored through a `pandas.DataFrame`.

The custom `SelfAttention` class is registered so that previously saved CNN-BiLSTM models can be deserialised correctly. The historical class name is retained for compatibility with saved Keras artifacts, while the implemented operation is the project's learned attention-pooling mechanism.

#### Deterministic Demonstration Order

The held-out test observations are shuffled using the fixed random state:

`42`

This changes only the order in which the observations are presented during the demonstration.

Every held-out test observation is still processed exactly once, and the stored labels and feature values are not altered.

The shuffled order is used so that the live demonstration does not simply follow the original dataset ordering while remaining completely reproducible.

#### Live Inference Display

Inference is performed one network flow at a time.

For every processed flow, the live dashboard displays:

- the current flow number;
- the predicted device class;
- the actual device class;
- prediction confidence;
- whether the prediction was correct or incorrect;
- total correct predictions;
- total incorrect predictions; and
- current running accuracy.

A live running-accuracy graph is displayed beneath the dashboard.

The dashboard itself updates for every network flow, while the graph image is regenerated periodically rather than after every prediction. This prevents repeated graph rendering from causing excessive notebook scrolling or visual instability during the demonstration.

#### Running Accuracy

After each prediction, cumulative accuracy is calculated as:

`correct predictions / flows processed`

The running-accuracy history is retained throughout the demonstration and is used to produce both:

- the live monitoring graph; and
- the final full test-set running-accuracy plot.

At the completion of inference, the code reports:

- total held-out flows processed;
- number of correct classifications;
- number of incorrect classifications; and
- final running accuracy.

The final running accuracy should correspond to the accuracy obtained by the same saved model on the same held-out test set.

Authoritative predictive metrics and diagnostic figures remain the responsibility of the **Universal Held-Out Test Evaluation** section; this cell is primarily an inference demonstration.

#### Prediction Table

A prediction record is created for every processed flow.

Each record contains:

- flow number;
- actual device;
- predicted device;
- prediction confidence;
- prediction status; and
- running accuracy after that flow.

After inference completes, these records are rendered as a scrollable HTML table.

Correct and incorrect predictions are visually distinguished, and the table remains readable in both light and dark notebook interfaces through explicitly defined styling.

The current configuration displays up to:

`376 prediction rows`

from the deterministic shuffled demonstration order.

#### Final Running-Accuracy Plot

After the live demonstration finishes, a final static graph is produced showing running accuracy across the complete held-out inference sequence.

A horizontal reference line identifies the final achieved accuracy.

The graph provides a visual indication of how cumulative predictive performance stabilises as additional held-out observations are processed.

#### Methodological Role

This section does not:

- train a model;
- tune hyperparameters;
- select between model seeds;
- alter model weights;
- modify the held-out test data; or
- replace the formal test-evaluation workflow.

It demonstrates inference using already trained and selected artifacts.

The **Held-Out Test Evaluation** section remains the authoritative source for saved performance metrics, confusion matrices, classification reports, ROC curves, precision-recall curves and model-comparison results.

In [ ]:
from pathlib import Path
import io
import base64
import html
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.layers import Layer
from tensorflow.keras.models import load_model

from sklearn.utils import shuffle
from IPython.display import display, HTML
from google.colab import drive

MODELS_ROOT = Path('/content/drive/MyDrive/Honours_Project/Models')
SHUFFLE_RANDOM_STATE = 42
PLOT_UPDATE_INTERVAL = 25
PREDICTION_TABLE_ROWS = 376

drive.mount('/content/drive')


@tf.keras.utils.register_keras_serializable()
class SelfAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name='att_weight',
            shape=(input_shape[-1], 1),
            initializer='glorot_uniform',
            trainable=True,
        )
        self.b = self.add_weight(
            name='att_bias',
            shape=(input_shape[1], 1),
            initializer='zeros',
            trainable=True,
        )
        super().build(input_shape)

    def call(self, inputs):
        scores = K.tanh(K.dot(inputs, self.W) + self.b)
        weights = K.softmax(scores, axis=1)
        return K.sum(inputs * weights, axis=1)


def get_required_files(model_path, model_type):
    run_dir = model_path.parent
    y_path = run_dir / "y_test_1d.npy"
    classes_path = run_dir / "device_classes.npy"

    if model_type == "keras":
        x_path = run_dir / "X_test_3d.npy"
        input_kind = "prepared_neural"
    elif model_type == "tuned_random_forest":
        x_path = run_dir / "X_test_raw.npy"
        input_kind = "raw_tabular"
    else:
        raise ValueError(f"Unsupported model type: {model_type}")

    if not (x_path.exists() and y_path.exists() and classes_path.exists()):
        return None

    return {
        "model": model_path,
        "type": model_type,
        "input_kind": input_kind,
        "x": x_path,
        "y": y_path,
        "classes": classes_path,
        "run_dir": run_dir,
    }


def discover_models():
    candidates = []
    seen = set()
    keras_patterns = [
        "best_validation_cnn_bilstm.keras",
        "best_validation_mlp.keras",
        "best_validation_cnn.keras",
        "best_validation_lstm.keras",
        "best_validation_bilstm.keras",
    ]

    for pattern in keras_patterns:
        for path in sorted(MODELS_ROOT.rglob(pattern)):
            if any(part.startswith("_TEMP_") for part in path.parts):
                continue
            if path in seen:
                continue
            files = get_required_files(path, "keras")
            if files is not None:
                candidates.append(files)
                seen.add(path)

    for path in sorted(MODELS_ROOT.rglob("tuned_random_forest_pipeline.joblib")):
        if any(part.startswith("_TEMP_") for part in path.parts):
            continue
        if "Test_Evaluation" in path.parts:
            continue
        if path in seen:
            continue
        files = get_required_files(path, "tuned_random_forest")
        if files is not None:
            candidates.append(files)
            seen.add(path)

    return sorted(candidates, key=lambda item: str(item["model"]))


def select_model(candidates):
    if not candidates:
        raise RuntimeError(f'No compatible saved models found under:\n{MODELS_ROOT}')

    print('\n' + '=' * 120)
    print('AVAILABLE SAVED MODELS')
    print('=' * 120)

    for number, candidate in enumerate(candidates, 1):
        try:
            relative_path = candidate['model'].relative_to(MODELS_ROOT)
        except ValueError:
            relative_path = candidate['model']
        type_label = "Keras" if candidate["type"] == "keras" else "Tuned Random Forest"
        print(f"[{number:03d}] [{type_label}] {relative_path}")

    print('=' * 120)
    selection = input('\nSelect model number: ').strip()

    try:
        index = int(selection) - 1
    except ValueError as exc:
        raise ValueError('Please enter a valid model number.') from exc

    if not 0 <= index < len(candidates):
        raise ValueError('Invalid model number.')

    return candidates[index]


def adapt_keras_input(X, model):
    input_shape = model.input_shape

    if isinstance(input_shape, list):
        if len(input_shape) != 1:
            raise RuntimeError('Only single-input models are supported.')
        input_shape = input_shape[0]

    if len(input_shape) == 3:
        if X.ndim == 3:
            adapted = X
        elif X.ndim == 2:
            adapted = X[..., np.newaxis]
        else:
            raise RuntimeError(f'Cannot adapt {X.shape} to {input_shape}')
    elif len(input_shape) == 2:
        if X.ndim == 3 and X.shape[-1] == 1:
            adapted = X[..., 0]
        elif X.ndim == 2:
            adapted = X
        else:
            adapted = X.reshape(len(X), -1)
    else:
        raise RuntimeError(f'Unsupported Keras input shape: {input_shape}')

    adapted = np.asarray(adapted, dtype=np.float32)
    if input_shape[1] is not None and adapted.shape[1] != int(input_shape[1]):
        raise RuntimeError(f'Feature-count mismatch. Model expects {input_shape}; got {adapted.shape}')

    return adapted


def prepare_rf_input(X, model):
    X_array = np.asarray(X, dtype=np.float32)
    if X_array.ndim != 2:
        raise RuntimeError(f"Tuned Random Forest requires a 2-D raw-feature matrix; received {X_array.shape}.")

    expected_features = getattr(model, "n_features_in_", None)
    if expected_features is not None and X_array.shape[1] != int(expected_features):
        raise RuntimeError(
            f"Tuned Random Forest feature-count mismatch. Model expects {expected_features}; "
            f"held-out bundle contains {X_array.shape[1]}."
        )

    feature_names = getattr(model, "feature_names_in_", None)
    if feature_names is not None:
        return pd.DataFrame(X_array, columns=[str(value) for value in feature_names])

    return X_array


def rf_class_order(model):
    model_classes = getattr(model, "classes_", None)
    if model_classes is None and hasattr(model, "named_steps") and "rf" in model.named_steps:
        model_classes = model.named_steps["rf"].classes_

    if model_classes is None:
        raise RuntimeError("Could not determine tuned Random Forest class order.")

    return np.asarray(model_classes)


def render_accuracy_plot_base64(history_x, history_accuracy, stream_limit):
    figure, axis = plt.subplots(figsize=(9.2, 3.2), dpi=100)

    if len(history_x) > 0:
        axis.plot(history_x, history_accuracy, linewidth=2)

    axis.set_title('Live Inference Accuracy Monitor', fontweight='bold')
    axis.set_xlabel('Network Flows Processed')
    axis.set_ylabel('Running Accuracy (%)')
    axis.set_xlim(0, stream_limit)
    axis.set_ylim(0, 100)
    axis.grid(True, linestyle='--', alpha=0.6)
    figure.tight_layout()

    buffer = io.BytesIO()
    figure.savefig(buffer, format='png', dpi=100, facecolor='white')
    plt.close(figure)
    buffer.seek(0)
    encoded = base64.b64encode(buffer.read()).decode('utf-8')
    buffer.close()
    return encoded


def build_live_display(dashboard_text, plot_base64):
    return HTML(f"""
        <div style="width: 950px; height: 690px; box-sizing: border-box; overflow: hidden; padding: 0; margin: 0;">
            <pre style="width: 930px; height: 315px; box-sizing: border-box; overflow: hidden; margin: 0 0 8px 0; padding: 8px; font-family: monospace; font-size: 14px; line-height: 1.25; white-space: pre;">{html.escape(dashboard_text)}</pre>
            <div style="width: 930px; height: 350px; box-sizing: border-box; overflow: hidden; text-align: left;">
                <img src="data:image/png;base64,{plot_base64}" style="width: 920px; height: 340px; object-fit: contain; display: block;" />
            </div>
        </div>
    """)


candidates = discover_models()
selected = select_model(candidates)
model_path = selected['model']

X_test = np.asarray(np.load(selected['x']), dtype=np.float32)
y_test = np.asarray(np.load(selected['y']), dtype=np.int64).reshape(-1)
device_classes = np.asarray(np.load(selected['classes'], allow_pickle=True))

if len(X_test) != len(y_test):
    raise RuntimeError('X_test and y_test contain different numbers of observations.')
if len(y_test) == 0:
    raise RuntimeError('The saved test set is empty.')
if np.min(y_test) < 0 or np.max(y_test) >= len(device_classes):
    raise RuntimeError('Saved class indices are inconsistent with device_classes.npy.')

print('\n' + '=' * 120)
print('SELECTED INFERENCE MODEL')
print('=' * 120)
print(f'Model:\n{model_path}')
print(f'\nModel type: {selected["type"]}')
print(f'Saved test tensor: {X_test.shape}')
print(f'Number of held-out test flows: {len(X_test)}')
print(f'Number of device classes: {len(device_classes)}')

if selected["type"] == "keras":
    tf.keras.backend.clear_session()
    model = load_model(
        model_path,
        custom_objects={"SelfAttention": SelfAttention},
        compile=False,
    )
    model_input = adapt_keras_input(X_test, model)
else:
    model = joblib.load(model_path)
    model_input = prepare_rf_input(X_test, model)

print(f"Model input tensor: {model_input.shape}")

indices = np.arange(len(y_test))
stream_indices = shuffle(indices, random_state=SHUFFLE_RANDOM_STATE)

if isinstance(model_input, pd.DataFrame):
    X_stream = model_input.iloc[stream_indices].reset_index(drop=True)
else:
    X_stream = model_input[stream_indices]

y_stream = y_test[stream_indices]
stream_limit = len(y_stream)

print('\n' + '=' * 120)
print('FULL TEST-SET INFERENCE DEMONSTRATION')
print('=' * 120)
print(f'Flows to process: {stream_limit}')
print('Every held-out test observation will be processed exactly once.')
print('Only the demonstration order has been shuffled.')

total_processed = 0
correct_predictions = 0
incorrect_predictions = 0
history_x = []
history_accuracy = []
prediction_records = []

live_plot_base64 = render_accuracy_plot_base64([], [], stream_limit)

initial_dashboard_text = (
    '============================================================\n'
    '        SMART HOME APPLIANCE INFERENCE DEMONSTRATION\n'
    '============================================================\n'
    f' Network Flow          : 0 / {stream_limit}\n'
    '------------------------------------------------------------\n'
    ' Predicted Device      : Waiting...\n'
    ' Actual Device         : Waiting...\n'
    ' Prediction Confidence : --\n'
    ' Status                : --\n'
    '============================================================\n'
    ' PERFORMANCE\n'
    '------------------------------------------------------------\n'
    ' Correct Predictions   : 0\n'
    ' Incorrect Predictions : 0\n'
    ' Running Accuracy      : --\n'
    '============================================================'
)

live_display_handle = display(
    build_live_display(initial_dashboard_text, live_plot_base64),
    display_id=True,
)

for i in range(stream_limit):
    if isinstance(X_stream, pd.DataFrame):
        single_flow = X_stream.iloc[i:i + 1]
    else:
        single_flow = X_stream[i:i + 1]
    true_label_idx = int(y_stream[i])
    actual_device = str(device_classes[true_label_idx])

    if selected["type"] == "keras":
        probability = np.asarray(model.predict(single_flow, verbose=0))[0]
        pred_label_idx = int(np.argmax(probability))
    else:
        pred_label_idx = int(model.predict(single_flow)[0])
        raw_probability = np.asarray(model.predict_proba(single_flow))[0]
        probability = np.zeros(len(device_classes), dtype=np.float64)

        for column, encoded_class in enumerate(rf_class_order(model)):
            probability[int(encoded_class)] = raw_probability[column]

    predicted_device = str(device_classes[pred_label_idx])
    confidence = float(probability[pred_label_idx])

    total_processed += 1
    if pred_label_idx == true_label_idx:
        correct_predictions += 1
        status = 'CORRECT'
    else:
        incorrect_predictions += 1
        status = 'INCORRECT'

    running_accuracy = (correct_predictions / total_processed) * 100
    history_x.append(total_processed)
    history_accuracy.append(running_accuracy)

    prediction_records.append({
        'Flow': total_processed,
        'Actual Device': actual_device,
        'Predicted Device': predicted_device,
        'Confidence (%)': round(confidence * 100, 2),
        'Status': status,
        'Running Accuracy (%)': round(running_accuracy, 2),
    })

    dashboard_text = (
        '============================================================\n'
        '        SMART HOME APPLIANCE INFERENCE DEMONSTRATION\n'
        '============================================================\n'
        f' Network Flow          : {total_processed} / {stream_limit}\n'
        '------------------------------------------------------------\n'
        f' Predicted Device      : {predicted_device}\n'
        f' Actual Device         : {actual_device}\n'
        f' Prediction Confidence : {confidence * 100:.2f}%\n'
        f' Status                : {status}\n'
        '============================================================\n'
        ' PERFORMANCE\n'
        '------------------------------------------------------------\n'
        f' Correct Predictions   : {correct_predictions}\n'
        f' Incorrect Predictions : {incorrect_predictions}\n'
        f' Running Accuracy      : {running_accuracy:.2f}%\n'
        '============================================================'
    )

    if total_processed % PLOT_UPDATE_INTERVAL == 0 or total_processed == stream_limit:
        live_plot_base64 = render_accuracy_plot_base64(history_x, history_accuracy, stream_limit)

    live_display_handle.update(build_live_display(dashboard_text, live_plot_base64))

final_accuracy = correct_predictions / total_processed

print('\n' + '=' * 100)
print('FULL TEST-SET INFERENCE DEMONSTRATION COMPLETE')
print('=' * 100)
print(f'Total test flows processed : {total_processed}')
print(f'Correct classifications    : {correct_predictions}')
print(f'Incorrect classifications  : {incorrect_predictions}')
print(f'Final running accuracy     : {final_accuracy * 100:.2f}%')
print('=' * 100)
print('Authoritative predictive metrics and diagnostic plots are produced by the Universal Test Evaluation block.')

prediction_df = pd.DataFrame(prediction_records)
sample_rows = min(PREDICTION_TABLE_ROWS, len(prediction_df))
sample_prediction_df = prediction_df.head(sample_rows).copy()

rows_html = ''
for _, row in sample_prediction_df.iterrows():
    if row['Status'] == 'CORRECT':
        status_html = '<span class="status-correct">CORRECT</span>'
    else:
        status_html = '<span class="status-incorrect">INCORRECT</span>'

    rows_html += f"""
        <tr>
            <td>{int(row['Flow'])}</td>
            <td>{html.escape(str(row['Actual Device']))}</td>
            <td>{html.escape(str(row['Predicted Device']))}</td>
            <td>{float(row['Confidence (%)']):.2f}%</td>
            <td>{status_html}</td>
            <td>{float(row['Running Accuracy (%)']):.2f}%</td>
        </tr>
    """

prediction_table_html = f"""
<style>
    .prediction-table-container {{
        max-height: 560px;
        overflow-y: auto;
        overflow-x: auto;
        border: 1px solid #b8bec7;
        border-radius: 8px;
        margin-top: 12px;
        margin-bottom: 20px;
        background-color: #ffffff !important;
    }}
    table.prediction-table {{
        width: 100% !important;
        min-width: 900px !important;
        border-collapse: collapse !important;
        border-spacing: 0 !important;
        background-color: #ffffff !important;
        color: #1f2328 !important;
        font-family: Arial, Helvetica, sans-serif !important;
        font-size: 14px !important;
    }}
    table.prediction-table thead th {{
        position: sticky !important;
        top: 0 !important;
        z-index: 5 !important;
        background-color: #e9eef5 !important;
        color: #111827 !important;
        font-weight: 700 !important;
        padding: 11px 12px !important;
        border-bottom: 2px solid #9ca3af !important;
        text-align: center !important;
        white-space: nowrap !important;
    }}
    table.prediction-table tbody td {{
        background-color: #ffffff !important;
        color: #1f2328 !important;
        padding: 10px 12px !important;
        border-bottom: 1px solid #d7dce2 !important;
        text-align: center !important;
        white-space: nowrap !important;
    }}
    table.prediction-table tbody tr:nth-child(even) td {{
        background-color: #f4f6f8 !important;
        color: #1f2328 !important;
    }}
    table.prediction-table tbody tr:hover td {{
        background-color: #e8f0fe !important;
        color: #111827 !important;
    }}
    .status-correct {{
        display: inline-block;
        min-width: 76px;
        padding: 3px 8px;
        border-radius: 4px;
        background-color: #e7f6ec !important;
        color: #146c2e !important;
        font-weight: 700 !important;
    }}
    .status-incorrect {{
        display: inline-block;
        min-width: 76px;
        padding: 3px 8px;
        border-radius: 4px;
        background-color: #fdeaea !important;
        color: #b42318 !important;
        font-weight: 700 !important;
    }}
    .prediction-table-title {{
        margin-top: 22px;
        margin-bottom: 4px;
    }}
    .prediction-table-description {{
        margin-top: 0;
        margin-bottom: 8px;
    }}
</style>

<h3 class="prediction-table-title">Sample Held-Out Predictions</h3>
<p class="prediction-table-description">
    Showing the first {sample_rows} predictions from the deterministic shuffled demonstration order.
    All {total_processed} held-out test observations were processed during inference.
</p>
<div class="prediction-table-container">
    <table class="prediction-table">
        <thead>
            <tr>
                <th>Flow</th>
                <th>Actual Device</th>
                <th>Predicted Device</th>
                <th>Confidence</th>
                <th>Status</th>
                <th>Running Accuracy</th>
            </tr>
        </thead>
        <tbody>
            {rows_html}
        </tbody>
    </table>
</div>
"""

display(HTML(prediction_table_html))

plt.figure(figsize=(10, 4))
plt.plot(history_x, history_accuracy, linewidth=2, label='Running Accuracy')
plt.axhline(
    y=final_accuracy * 100,
    linestyle='--',
    label=f'Final Running Accuracy = {final_accuracy * 100:.2f}%',
)
plt.title('Full Test-Set Inference Demonstration', fontweight='bold')
plt.xlabel('Network Flows Processed')
plt.ylabel('Running Accuracy (%)')
plt.xlim(0, stream_limit)
plt.ylim(0, 100)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

### LiteRT Model Conversion and Validation

This section converts compatible saved neural-network models into validated **static-batch-one FP32 LiteRT/TFLite deployment artifacts** for subsequent CPU edge-inference profiling.

The conversion workflow operates on previously trained Keras models stored under:

`Honours_Project/Models/`

It does not retrain the models or modify their learned parameters. Instead, it prepares deployment-compatible inference graphs, converts them to LiteRT/TFLite format and verifies that the converted models preserve the behaviour of the corresponding saved Keras models.

#### Compatible Saved Models

The converter recursively searches the project model directory for recognised `.keras` artifacts.

A candidate must:

- correspond to one of the supported neural architectures;
- use either a `best_validation_...` filename or a retained `_seed_...` filename;
- have the required frozen held-out test artifacts in the same run directory; and
- not be located inside a temporary run directory.

The supported architectures are:

- CNN;
- LSTM;
- BiLSTM;
- Feature-Attention MLP; and
- Attention CNN-BiLSTM.

Each compatible run must contain:

```text
X_test_3d.npy
y_test_1d.npy
device_classes.npy
```

The held-out bundle is validated against the expected project dimensions:

- 376 test observations;
- 244 retained predictors;
- one feature channel for sequence-compatible input; and
- 10 device classes.

#### Model Selection

All compatible saved models are displayed with their architecture group, conversion status and selection status.

Validation-selected models whose filenames begin with `best_validation_` are marked as the recommended/default candidates. Seed-specific models remain available for manual conversion.

The following selection commands are supported:

- `new` — convert every validation-selected model for which either the `.tflite` artifact or companion conversion-validation JSON is absent. This discovery check is based on file existence; use `all` or an explicit model selection when an existing conversion must be regenerated.
- `all` — reconvert every validation-selected model and overwrite its deployment artifacts;
- `latest` — convert the validation-selected models from the newest discovered run within each model group;
- explicit model numbers such as `1,4,7-9`; and
- `path` — manually provide the full Google Drive path to a compatible `.keras` model.

This allows the same conversion workflow to be used for both the retained project models and newly generated experimental runs.

#### Static Batch-One Deployment Representation

Every deployment artifact uses a fixed batch size of:

`1`

The resulting input shape therefore represents the project's intended one-flow-at-a-time edge-inference workload.

For sequence-compatible models, the static input shape is:

`(1, 244, 1)`

For compatible two-dimensional models such as the Feature-Attention MLP, the corresponding static batch-one two-dimensional representation is used.

The conversion retains FP32 input and output tensors.

#### CPU-Compatible Recurrent Models

The LSTM, BiLSTM and CNN-BiLSTM architectures require an additional deployment preparation step.

Their recurrent components are rebuilt using CPU-compatible Keras LSTM implementations with:

`use_cudnn=False`

This avoids conversion paths that depend on GPU-specific cuDNN recurrent operations.

For these architectures:

- the original network structure is reconstructed;
- the static batch-one input shape is applied;
- CPU-compatible recurrent layers are substituted;
- the original learned weights are copied into the deployment model; and
- every weight tensor is checked for matching count and shape before transfer.

The CNN and Feature-Attention MLP do not require recurrent-layer reconstruction and can use their loaded Keras model directly for deployment conversion.

#### CNN-BiLSTM Attention Layer

The custom `SelfAttention` class is registered so that the saved CNN-BiLSTM model can be loaded correctly.

The historical class name is retained for Keras serialization compatibility. The implemented operation is the project's learned attention-pooling mechanism: a scalar weight is learned for each sequence position, the weights are normalised across the sequence, and the weighted representation is summed before classification.

#### Original Keras vs Deployment Keras Validation

Before LiteRT/TFLite conversion, the original saved Keras model and the CPU/static deployment Keras model are evaluated using all 376 held-out test observations in batch-one mode.

The converter verifies that:

- all probability outputs are finite;
- the probability matrices agree within the configured numerical tolerance;
- predicted classes agree for 100% of held-out observations; and
- the deployment representation does not change the model's classification behaviour.

The numerical comparison uses:

`absolute tolerance = 1e-5`

`relative tolerance = 1e-5`

If the deployment Keras representation fails these checks, conversion is stopped.

#### Graph Freezing and LiteRT/TFLite Conversion

The validated deployment Keras model is wrapped in a static TensorFlow inference function and converted to a concrete function.

Its variables are then frozen into constants.

The frozen graph is checked to ensure that no `ReadVariableOp` nodes remain before LiteRT/TFLite conversion proceeds.

Conversion uses:

- built-in LiteRT/TFLite operators;
- no custom operators;
- FP32 precision;
- no quantisation;
- static batch size 1; and
- frozen inference variables.

The generated deployment model follows the naming convention:

```text
<source_model>_edge_static_batch1_fp32_frozen.tflite
```

#### LiteRT/TFLite Validation

The generated LiteRT/TFLite artifact is loaded into a TensorFlow Lite interpreter and validated before it is accepted.

The converter verifies:

- exactly one model input;
- exactly one model output;
- the expected static input shape;
- output shape `(1, 10)`;
- FP32 input;
- FP32 output; and
- finite outputs for every held-out observation.

Inference is then performed over all 376 held-out observations one flow at a time.

The resulting LiteRT/TFLite probabilities are compared with the corresponding static Keras probabilities using the same numerical tolerance.

A conversion is accepted only if:

- probability outputs are numerically equivalent within tolerance;
- predicted classes agree for 100% of the held-out test cases;
- held-out accuracy is unchanged; and
- held-out macro-F1 is unchanged.

This ensures that the deployment transformation does not alter the predictive behaviour of the trained model.

#### Conversion Validation Record

Each successful conversion creates a companion validation file:

```text
<source_model>_conversion_validation.json
```

The validation record preserves:

- validation status;
- model architecture/group;
- original Keras model path;
- generated LiteRT/TFLite path;
- held-out input, label and class paths;
- TensorFlow version;
- SHA-256 hash of the source model;
- SHA-256 hash of the LiteRT/TFLite artifact;
- SHA-256 hashes of the held-out companion files;
- number of validation observations;
- static input and output shapes;
- input and output data types;
- static batch size;
- parameter count;
- LiteRT/TFLite file size;
- numerical comparison tolerances;
- maximum probability differences;
- Keras-to-Keras class agreement;
- Keras-to-LiteRT/TFLite class agreement;
- original, deployment and LiteRT/TFLite accuracy;
- original, deployment and LiteRT/TFLite macro-F1; and
- confirmation that no graph variable-read operations remain.

#### Deployment Artifacts

For each successfully converted source model, the run directory receives:

```text
<source_model>_edge_static_batch1_fp32_frozen.tflite
<source_model>_conversion_validation.json
```

These validated deployment artifacts are the inputs used by the subsequent controlled edge-inference profiling workflow.

#### Methodological Role

This section performs **deployment conversion and equivalence validation only**.

It does not:

- retrain models;
- select hyperparameters;
- select neural seeds;
- alter the held-out test set;
- benchmark latency;
- profile process memory; or
- replace the formal predictive evaluation workflow.

Predictive metrics are formally reported by the **Held-Out Test Evaluation** section, while latency and memory measurements are produced separately by the controlled edge-inference profiling sections.

In [ ]:
import gc
import hashlib
import json
from pathlib import Path
import numpy as np
from google.colab import drive

MODELS_ROOT = Path("/content/drive/MyDrive/Honours_Project/Models")
EXPECTED_TEST_SHAPE = (376, 244, 1)
EXPECTED_CLASS_COUNT = 10
PROBABILITY_ATOL = 1e-5
PROBABILITY_RTOL = 1e-5

drive.mount("/content/drive")

import tensorflow as tf
import tensorflow.keras.backend as K
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.layers import (
    BatchNormalization,
    Bidirectional,
    Conv1D,
    Dense,
    Dropout,
    Input,
    LSTM,
    Layer,
    MaxPooling1D,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2


@tf.keras.utils.register_keras_serializable()
class SelfAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="att_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True,
        )
        self.b = self.add_weight(
            name="att_bias",
            shape=(input_shape[1], 1),
            initializer="zeros",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, inputs):
        scores = K.tanh(K.dot(inputs, self.W) + self.b)
        weights = K.softmax(scores, axis=1)
        return K.sum(inputs * weights, axis=1)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def model_kind(path):
    name = path.name.lower()
    if "cnn_bilstm" in name:
        return "CNN-BiLSTM"
    if "bilstm" in name:
        return "BiLSTM"
    if "lstm" in name:
        return "LSTM"
    if "mlp" in name:
        return "MLP"
    if "cnn" in name:
        return "CNN"
    return None


def model_group(path, kind):
    path_text = str(path)

    if "KFold_Models" in path_text:
        return f"{kind}-KFold"

    if "108_Configurations" in path_text:
        return f"{kind}-50-108"

    if "Grid_Search_30_Epochs" in path_text:
        return f"{kind}-30"

    if "Grid_Search_50_Epochs" in path_text:
        return f"{kind}-50"

    return kind


def default_candidate(path, kind):
    name = path.name.lower()
    return name.startswith("best_validation_")


def deployment_paths(source_model):
    stem = source_model.stem
    tflite_path = source_model.parent / f"{stem}_edge_static_batch1_fp32_frozen.tflite"
    validation_path = source_model.parent / f"{stem}_conversion_validation.json"
    return tflite_path, validation_path


def corrected_companion_files(source_model):
    run_dir = source_model.parent
    x_path = run_dir / "X_test_3d.npy"
    y_path = run_dir / "y_test_1d.npy"
    classes_path = run_dir / "device_classes.npy"
    if not all(path.exists() for path in [x_path, y_path, classes_path]):
        return None
    try:
        x = np.load(x_path, mmap_mode="r")
        y = np.load(y_path, mmap_mode="r")
        classes = np.load(classes_path, allow_pickle=True)
        valid = (
            tuple(x.shape) == EXPECTED_TEST_SHAPE
            and tuple(y.shape) == (EXPECTED_TEST_SHAPE[0],)
            and len(classes) == EXPECTED_CLASS_COUNT
        )
        del x, y, classes
        return (x_path, y_path, classes_path) if valid else None
    except Exception:
        return None


def discover_candidates():
    candidates = []
    for path in sorted(MODELS_ROOT.rglob("*.keras")):
        if any(part.startswith("_TEMP_") for part in path.parts):
            continue

        kind = model_kind(path)
        if kind is None:
            continue
        name = path.name.lower()
        recognised = (
            name.startswith("best_validation_")
            or "_seed_" in name
        )
        if not recognised:
            continue
        companions = corrected_companion_files(path)
        if companions is None:
            continue
        tflite_path, validation_path = deployment_paths(path)
        candidates.append({
            "source": path,
            "kind": kind,
            "group": model_group(path, kind),
            "default": default_candidate(path, kind),
            "converted": tflite_path.exists() and validation_path.exists(),
            "x": companions[0],
            "y": companions[1],
            "classes": companions[2],
            "modified": path.stat().st_mtime,
        })
    return candidates


def show_candidates(candidates):
    print("\nCorrected saved-model candidates")
    print("=" * 120)
    for index, candidate in enumerate(candidates, start=1):
        status = "converted" if candidate["converted"] else "new"
        default = "recommended" if candidate["default"] else "seed model"
        print(
            f"[{index:03d}] {candidate['group']:<18} {status:<10} {default:<14} "
            f"{candidate['source']}"
        )
    print("=" * 120)


def parse_number_selection(text, candidate_count):
    selected = []
    for part in text.split(","):
        part = part.strip()
        if not part:
            continue
        if "-" in part:
            start_text, end_text = part.split("-", 1)
            start = int(start_text)
            end = int(end_text)
            selected.extend(range(start, end + 1))
        else:
            selected.append(int(part))
    selected = list(dict.fromkeys(selected))
    if not selected or any(index < 1 or index > candidate_count for index in selected):
        raise ValueError("The model selection contains an invalid candidate number.")
    return [index - 1 for index in selected]


def select_candidates(candidates):
    if not candidates:
        raise RuntimeError(
            "No corrected saved models were found. A candidate must have sibling "
            "X_test_3d.npy, y_test_1d.npy and device_classes.npy files with shapes "
            f"{EXPECTED_TEST_SHAPE}, ({EXPECTED_TEST_SHAPE[0]},) and 10 classes."
        )
    show_candidates(candidates)
    print("\nSelection commands:")
    print("  new          convert every default candidate without a completed conversion")
    print("  all          convert every default candidate, overwriting its deployment files")
    print("  latest       convert every default candidate in the newest run of each model group")
    print("  1,4,7-9      convert the listed candidate numbers")
    print("  path         paste one full Google Drive .keras path")
    choice = input("\nSelect models to convert [new]: ").strip() or "new"
    choice_lower = choice.lower()

    if choice_lower == "new":
        selected = [c for c in candidates if c["default"] and not c["converted"]]
    elif choice_lower == "all":
        selected = [c for c in candidates if c["default"]]
    elif choice_lower == "latest":
        selected = []
        for group in sorted({candidate["group"] for candidate in candidates}):
            group_candidates = [
                candidate for candidate in candidates
                if candidate["group"] == group and candidate["default"]
            ]
            if not group_candidates:
                continue
            latest_run_dir = max(
                group_candidates,
                key=lambda candidate: candidate["modified"],
            )["source"].parent
            selected.extend(
                candidate for candidate in group_candidates
                if candidate["source"].parent == latest_run_dir
            )
    elif choice_lower == "path":
        manual_path = Path(input("Full Google Drive .keras path: ").strip())
        if not manual_path.exists():
            raise FileNotFoundError(manual_path)
        kind = model_kind(manual_path)
        companions = corrected_companion_files(manual_path)
        if kind is None or companions is None:
            raise RuntimeError("The supplied model is not a recognised corrected COMP700 artifact.")
        tflite_path, validation_path = deployment_paths(manual_path)
        selected = [{
            "source": manual_path,
            "kind": kind,
            "group": model_group(manual_path, kind),
            "default": True,
            "converted": tflite_path.exists() and validation_path.exists(),
            "x": companions[0],
            "y": companions[1],
            "classes": companions[2],
            "modified": manual_path.stat().st_mtime,
        }]
    else:
        indices = parse_number_selection(choice, len(candidates))
        selected = [candidates[index] for index in indices]

    if not selected:
        raise RuntimeError("The selection contains no models requiring conversion.")
    return selected


def layer_config(layer, excluded=()):
    config = layer.get_config().copy()
    for key in ["name", "trainable", "dtype", *excluded]:
        config.pop(key, None)
    return config


def find_layers(model, layer_type):
    return [layer for layer in model.layers if isinstance(layer, layer_type)]


def rebuild_lstm(original_model, static_input_shape):
    original_lstm = find_layers(original_model, LSTM)[0]
    dense_layers = find_layers(original_model, Dense)
    dropout_layers = find_layers(original_model, Dropout)
    inputs = Input(batch_shape=static_input_shape, name="input_layer")
    config = layer_config(original_lstm, excluded=("batch_input_shape",))
    config["use_cudnn"] = False
    x = LSTM(**config, name="lstm_cpu")(inputs)
    for index, dense in enumerate(dense_layers[:-1]):
        x = Dense(**layer_config(dense), name=f"dense_hidden_{index}")(x)
        if index < len(dropout_layers):
            x = Dropout(dropout_layers[index].rate, name=f"dropout_{index}")(x)
    outputs = Dense(**layer_config(dense_layers[-1]), name="output")(x)
    return Model(inputs, outputs, name="lstm_edge_static")


def rebuild_bilstm(original_model, static_input_shape):
    original_bidir = find_layers(original_model, Bidirectional)[0]
    dense_layers = find_layers(original_model, Dense)
    dropout_layers = find_layers(original_model, Dropout)
    forward_config = layer_config(original_bidir.forward_layer, excluded=("batch_input_shape",))
    backward_config = layer_config(original_bidir.backward_layer, excluded=("batch_input_shape",))
    forward_config["use_cudnn"] = False
    backward_config["use_cudnn"] = False
    inputs = Input(batch_shape=static_input_shape, name="input_layer")
    forward = LSTM(**forward_config, name="forward_lstm_cpu")
    backward = LSTM(**backward_config, name="backward_lstm_cpu")
    x = Bidirectional(
        forward,
        backward_layer=backward,
        merge_mode=original_bidir.merge_mode,
        name="bidirectional_lstm_cpu",
    )(inputs)
    for index, dense in enumerate(dense_layers[:-1]):
        x = Dense(**layer_config(dense), name=f"dense_hidden_{index}")(x)
        if index < len(dropout_layers):
            x = Dropout(dropout_layers[index].rate, name=f"dropout_{index}")(x)
    outputs = Dense(**layer_config(dense_layers[-1]), name="output")(x)
    return Model(inputs, outputs, name="bilstm_edge_static")


def rebuild_cnn_bilstm(original_model, static_input_shape):
    conv = find_layers(original_model, Conv1D)[0]
    pool = find_layers(original_model, MaxPooling1D)[0]
    normalizations = find_layers(original_model, BatchNormalization)
    bidir = find_layers(original_model, Bidirectional)[0]
    dropout = find_layers(original_model, Dropout)[0]
    output_dense = find_layers(original_model, Dense)[-1]

    inputs = Input(batch_shape=static_input_shape, name="input_layer")
    x = Conv1D(**layer_config(conv), name="conv1d")(inputs)
    x = BatchNormalization(**layer_config(normalizations[0]), name="batch_normalization")(x)
    x = MaxPooling1D(**layer_config(pool), name="max_pooling1d")(x)

    forward_config = layer_config(bidir.forward_layer, excluded=("batch_input_shape",))
    backward_config = layer_config(bidir.backward_layer, excluded=("batch_input_shape",))
    forward_config["use_cudnn"] = False
    backward_config["use_cudnn"] = False
    forward = LSTM(**forward_config, name="forward_lstm_cpu")
    backward = LSTM(**backward_config, name="backward_lstm_cpu")
    x = Bidirectional(
        forward,
        backward_layer=backward,
        merge_mode=bidir.merge_mode,
        name="bidirectional_lstm_cpu",
    )(x)
    x = BatchNormalization(**layer_config(normalizations[1]), name="batch_normalization_1")(x)
    x = Dropout(dropout.rate, name="dropout")(x)
    x = SelfAttention(name="self_attention")(x)
    outputs = Dense(**layer_config(output_dense), name="output")(x)
    return Model(inputs, outputs, name="attention_cnn_bilstm_edge_static")


def prepare_sample(x, index, input_rank, dtype=np.float32):
    if input_rank == 2:
        return np.asarray(x[index:index + 1, :, 0], dtype=dtype)
    return np.asarray(x[index:index + 1], dtype=dtype)


def keras_predictions_batch_one(model, x, input_rank):
    outputs = []
    for index in range(len(x)):
        sample = prepare_sample(x, index, input_rank)
        output = model(sample, training=False)
        if isinstance(output, dict):
            output = next(iter(output.values()))
        outputs.append(np.asarray(output, dtype=np.float32))
    return np.concatenate(outputs, axis=0)


def macro_f1(y_true, y_pred):
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))


def convert_candidate(candidate):
    source_path = candidate["source"]
    tflite_path, validation_path = deployment_paths(source_path)
    x = np.load(candidate["x"], mmap_mode="r")
    y = np.asarray(np.load(candidate["y"]), dtype=np.int64)
    classes = np.load(candidate["classes"], allow_pickle=True)

    tf.keras.backend.clear_session()
    gc.collect()
    original_model = load_model(
        source_path,
        custom_objects={"SelfAttention": SelfAttention},
        compile=False,
    )
    input_rank = len(original_model.input_shape)
    if input_rank not in {2, 3}:
        raise RuntimeError(f"Unsupported model input shape: {original_model.input_shape}")
    static_input_shape = tuple(
        1 if dimension is None else int(dimension)
        for dimension in original_model.input_shape
    )
    expected_static_shape = (
        (1, x.shape[1]) if input_rank == 2 else (1, x.shape[1], x.shape[2])
    )
    if static_input_shape != expected_static_shape:
        raise RuntimeError(
            f"Model/test input mismatch for {source_path}. "
            f"Model={static_input_shape}, test={expected_static_shape}"
        )
    if int(original_model.output_shape[-1]) != len(classes):
        raise RuntimeError("Model output class count does not match device_classes.npy.")

    if candidate["kind"] == "LSTM":
        deployment_model = rebuild_lstm(original_model, static_input_shape)
    elif candidate["kind"] == "BiLSTM":
        deployment_model = rebuild_bilstm(original_model, static_input_shape)
    elif candidate["kind"] == "CNN-BiLSTM":
        deployment_model = rebuild_cnn_bilstm(original_model, static_input_shape)
    else:
        deployment_model = original_model

    if deployment_model is not original_model:
        original_weights = original_model.get_weights()
        deployment_weights = deployment_model.get_weights()
        if len(original_weights) != len(deployment_weights):
            raise RuntimeError("Weight tensor count changed during CPU-compatible rebuilding.")
        for index, (original_weight, deployment_weight) in enumerate(
            zip(original_weights, deployment_weights)
        ):
            if original_weight.shape != deployment_weight.shape:
                raise RuntimeError(
                    f"Weight shape mismatch at index {index}: "
                    f"{original_weight.shape} != {deployment_weight.shape}"
                )
        deployment_model.set_weights(original_weights)

    print("\n" + "=" * 110)
    print(f"Converting {candidate['group']}: {source_path}")
    print(f"Static input shape: {static_input_shape}")

    original_probabilities = keras_predictions_batch_one(original_model, x, input_rank)
    deployment_probabilities = keras_predictions_batch_one(deployment_model, x, input_rank)
    if not np.all(np.isfinite(original_probabilities)) or not np.all(np.isfinite(deployment_probabilities)):
        raise RuntimeError("Keras produced a non-finite probability.")
    keras_allclose = bool(np.allclose(
        original_probabilities,
        deployment_probabilities,
        rtol=PROBABILITY_RTOL,
        atol=PROBABILITY_ATOL,
    ))
    keras_max_difference = float(np.max(np.abs(
        original_probabilities - deployment_probabilities
    )))
    original_classes = np.argmax(original_probabilities, axis=1)
    deployment_classes = np.argmax(deployment_probabilities, axis=1)
    keras_class_agreement = float(np.mean(original_classes == deployment_classes))
    if not keras_allclose or keras_class_agreement != 1.0:
        raise RuntimeError(
            "Original and CPU/static Keras models are not equivalent: "
            f"allclose={keras_allclose}, class agreement={keras_class_agreement:.6f}, "
            f"max difference={keras_max_difference:.12g}"
        )

    @tf.function(input_signature=[
        tf.TensorSpec(shape=static_input_shape, dtype=tf.float32, name="input")
    ])
    def inference_function(inputs):
        return deployment_model(inputs, training=False)

    concrete_function = inference_function.get_concrete_function()
    frozen_function = convert_variables_to_constants_v2(concrete_function)
    frozen_graph = frozen_function.graph.as_graph_def()
    read_variable_ops = [node for node in frozen_graph.node if node.op == "ReadVariableOp"]
    if read_variable_ops:
        raise RuntimeError("Variable freezing left ReadVariableOp nodes in the graph.")

    converter = tf.lite.TFLiteConverter.from_concrete_functions(
        [frozen_function], deployment_model
    )
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
    converter.allow_custom_ops = False
    converter.optimizations = []
    if hasattr(converter, "_experimental_lower_tensor_list_ops"):
        converter._experimental_lower_tensor_list_ops = True
    tflite_bytes = converter.convert()
    with open(tflite_path, "wb") as file:
        file.write(tflite_bytes)

    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    if len(input_details) != 1 or len(output_details) != 1:
        raise RuntimeError("The deployment model must have one input and one output.")
    actual_input_shape = tuple(int(value) for value in input_details[0]["shape"])
    actual_output_shape = tuple(int(value) for value in output_details[0]["shape"])
    if actual_input_shape != static_input_shape:
        raise RuntimeError(f"TFLite input shape mismatch: {actual_input_shape}")
    if actual_output_shape != (1, len(classes)):
        raise RuntimeError(f"TFLite output shape mismatch: {actual_output_shape}")
    if input_details[0]["dtype"] != np.float32 or output_details[0]["dtype"] != np.float32:
        raise RuntimeError("The FP32 deployment artifact does not use float32 I/O.")

    input_index = int(input_details[0]["index"])
    output_index = int(output_details[0]["index"])
    tflite_outputs = []
    for index in range(len(x)):
        sample = prepare_sample(x, index, input_rank, input_details[0]["dtype"])
        interpreter.set_tensor(input_index, sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_index)
        if not np.all(np.isfinite(output)):
            raise RuntimeError(f"TFLite produced a non-finite output at sample {index}.")
        tflite_outputs.append(np.asarray(output, dtype=np.float32))
    tflite_probabilities = np.concatenate(tflite_outputs, axis=0)
    tflite_classes = np.argmax(tflite_probabilities, axis=1)
    tflite_allclose = bool(np.allclose(
        deployment_probabilities,
        tflite_probabilities,
        rtol=PROBABILITY_RTOL,
        atol=PROBABILITY_ATOL,
    ))
    tflite_max_difference = float(np.max(np.abs(
        deployment_probabilities - tflite_probabilities
    )))
    tflite_class_agreement = float(np.mean(deployment_classes == tflite_classes))
    if not tflite_allclose or tflite_class_agreement != 1.0:
        raise RuntimeError(
            "Static Keras and TFLite models are not equivalent: "
            f"allclose={tflite_allclose}, class agreement={tflite_class_agreement:.6f}, "
            f"max difference={tflite_max_difference:.12g}"
        )

    original_accuracy = float(accuracy_score(y, original_classes))
    deployment_accuracy = float(accuracy_score(y, deployment_classes))
    tflite_accuracy = float(accuracy_score(y, tflite_classes))
    original_f1 = macro_f1(y, original_classes)
    deployment_f1 = macro_f1(y, deployment_classes)
    tflite_f1 = macro_f1(y, tflite_classes)
    if not (
        original_accuracy == deployment_accuracy == tflite_accuracy
        and original_f1 == deployment_f1 == tflite_f1
    ):
        raise RuntimeError("Conversion changed accuracy or macro-F1.")

    record = {
        "status": "validated",
        "validated": True,
        "model_group": candidate["group"],
        "model_kind": candidate["kind"],
        "source_model_path": str(source_path),
        "tflite_path": str(tflite_path),
        "test_inputs_path": str(candidate["x"]),
        "test_labels_path": str(candidate["y"]),
        "device_classes_path": str(candidate["classes"]),
        "tensorflow_version": tf.__version__,
        "source_model_sha256": sha256_file(source_path),
        "tflite_sha256": sha256_file(tflite_path),
        "companion_sha256": {
            "test_inputs": sha256_file(candidate["x"]),
            "test_labels": sha256_file(candidate["y"]),
            "device_classes": sha256_file(candidate["classes"]),
        },
        "test_samples": int(len(x)),
        "input_shape": list(actual_input_shape),
        "output_shape": list(actual_output_shape),
        "input_dtype": str(input_details[0]["dtype"]),
        "output_dtype": str(output_details[0]["dtype"]),
        "static_batch_size": 1,
        "parameters": int(deployment_model.count_params()),
        "tflite_size_bytes": int(tflite_path.stat().st_size),
        "probability_atol": PROBABILITY_ATOL,
        "probability_rtol": PROBABILITY_RTOL,
        "keras_allclose": keras_allclose,
        "keras_max_probability_difference": keras_max_difference,
        "keras_class_agreement": keras_class_agreement,
        "tflite_allclose": tflite_allclose,
        "tflite_max_probability_difference": tflite_max_difference,
        "tflite_class_agreement": tflite_class_agreement,
        "original_accuracy": original_accuracy,
        "deployment_accuracy": deployment_accuracy,
        "tflite_accuracy": tflite_accuracy,
        "original_macro_f1": original_f1,
        "deployment_macro_f1": deployment_f1,
        "tflite_macro_f1": tflite_f1,
        "read_variable_ops_remaining": len(read_variable_ops),
    }
    with open(validation_path, "w", encoding="utf-8") as file:
        json.dump(record, file, indent=2)

    print(f"Validated all {len(x)} test observations.")
    print(f"TFLite accuracy: {tflite_accuracy * 100:.2f}%")
    print(f"TFLite macro-F1: {tflite_f1:.4f}")
    print(f"Maximum probability difference: {tflite_max_difference:.12g}")
    print(f"Saved TFLite: {tflite_path}")
    print(f"Saved validation: {validation_path}")

    del interpreter, original_model, deployment_model
    tf.keras.backend.clear_session()
    gc.collect()
    return record


candidates = discover_candidates()
selected_candidates = select_candidates(candidates)
conversion_records = []
for selected_candidate in selected_candidates:
    conversion_records.append(convert_candidate(selected_candidate))

print("\n" + "=" * 110)
print(f"Completed {len(conversion_records)} corrected conversion(s).")
for record in conversion_records:
    print(f"{record['model_group']}: {record['tflite_path']}")

### Controlled Edge-Inference Profiling

This section performs controlled CPU profiling of previously converted and validated LiteRT/TFLite deployment models. It measures **prepared-input batch-one inference latency, process-memory overhead, model size and retained predictive performance** using the validated deployment artifacts generated by the preceding LiteRT conversion section.

The profiler does not retrain models or reconvert Keras models. It operates only on LiteRT artifacts that have already passed conversion-equivalence validation.

#### Validated Artifact Discovery

The profiler recursively scans:

`Honours_Project/Models/`

for deployment artifacts following the naming convention:

`*_edge_static_batch1_fp32_frozen.tflite`

A LiteRT artifact is accepted only when its corresponding:

`*_conversion_validation.json`

file exists and is marked as successfully validated.

Before an artifact is made available for profiling, the code verifies:

- the SHA-256 hash of the LiteRT/TFLite file;
- the SHA-256 hashes of the associated held-out input, label and class files;
- the expected held-out input shape `(376, 244, 1)`;
- the expected label shape `(376,)`; and
- the expected 10-class mapping.

Artifacts that fail any integrity check are skipped rather than profiled.

#### Artifact Selection

All validated LiteRT artifacts are displayed with their model group, source-model name and current profiling status.

The following selection commands are supported:

- `new` — profile all validated artifacts that do not yet have an edge-profile result;
- `latest` — profile validated artifacts from the newest run of each model group;
- `all` — profile every validated artifact discovered;
- model numbers such as `1,3,5-8`; and
- `path` — profile one explicitly supplied `.tflite` artifact.

This allows the same profiling workflow to be used for retained project models and for newly generated reproducibility runs.

#### Controlled Profiling Configuration

The profiler uses the following fixed experimental configuration:

```text
LiteRT inference threads:       1
Fresh processes per model:      3
Warm-up inferences per phase:   50
Latency repetitions:            10
Held-out flows per repetition:  376
Memory sampling interval:       0.002 seconds
Static batch size:              1
```

Each fresh worker process therefore performs:

`10 × 376 = 3,760`

timed inferences.

Across the three fresh-process repetitions, each profiled artifact contributes:

`3 × 3,760 = 11,280`

timed batch-one inference measurements.

#### Fresh-Process Isolation

Each profiling repetition executes in a separate Python process.

The worker process disables GPU use and constrains common numerical libraries to the configured single-thread execution mode.

The profiler sets:

- LiteRT inference threads to one;
- OpenMP threads to one;
- OpenBLAS threads to one;
- MKL threads to one; and
- NumExpr threads to one.

This reduces uncontrolled variation caused by differing thread counts between models.

#### CPU Affinity

Where supported by the operating system, each profiling worker is pinned to one eligible logical CPU.

CPU affinity is required by the current controlled-profiling configuration. The worker verifies that its final affinity contains exactly one logical processor before the corresponding measurements are accepted.

The selected logical CPU and available processor information are recorded in the profiling metadata.

#### LiteRT Runtime

Profiling uses the standalone:

`ai-edge-litert`

runtime rather than the full TensorFlow inference stack.

If required, the profiling dependencies are installed automatically.

Each worker records:

- LiteRT runtime version;
- Python version;
- operating-system platform;
- processor architecture;
- processor description;
- logical CPU count;
- physical CPU count; and
- applied CPU affinity.

#### Memory Profiling

Memory use is measured using both:

- **RSS — Resident Set Size**; and
- **USS — Unique Set Size**.

USS is particularly useful for identifying memory uniquely attributable to the profiling process rather than memory pages shared with other processes.

Memory is sampled at several execution stages:

1. process baseline;
2. after importing the LiteRT runtime;
3. after loading and allocating the LiteRT interpreter;
4. after loading the inference inputs;
5. during warm-up and full-test inference; and
6. at peak observed inference memory.

The profiler derives several memory-overhead quantities, including:

- LiteRT import overhead;
- model-allocation overhead;
- input-data overhead;
- peak inference overhead;
- total peak RSS overhead; and
- total peak USS overhead.

The principal cross-platform memory metric used later in the project is **total peak USS overhead above the worker-process baseline**.

#### Memory and Latency Measurement Separation

Memory sampling and latency measurement are deliberately separated.

The memory-monitoring thread is stopped before the timed latency experiment begins. This prevents the memory-sampling thread from directly interfering with the latency measurements.

A separate latency warm-up phase is performed after memory profiling and before timing begins.

#### Prepared-Input Latency Boundary

The timed inference boundary is:

`set_tensor() → invoke() → get_tensor()`

The input sample is prepared **before** the timer begins.

Therefore the measured latency represents **prepared-input LiteRT inference** and excludes:

- packet capture;
- network I/O;
- flow reconstruction;
- feature extraction;
- preprocessing outside the saved deployment input;
- Python sample slicing/preparation; and
- other end-to-end gateway operations.

The measurements must therefore be interpreted as controlled model-inference latency rather than complete smart-home gateway latency.

#### Latency Statistics

Each fresh worker process records the raw latency of every timed inference.

For each process, the profiler calculates:

- mean latency;
- standard deviation;
- median latency;
- P95 latency;
- P99 latency;
- minimum latency; and
- maximum latency.

The raw measurements are also preserved as NumPy arrays.

After all three fresh worker processes complete, the profiler calculates the mean and sample standard deviation of the process-level metrics.

This produces a reproducible summary while preserving the individual process measurements.

#### Predictive Integrity Check

The worker also processes the complete held-out test set and calculates:

- accuracy; and
- macro-F1.

These values are not used for model selection.

Instead, they act as deployment-integrity checks. The measured LiteRT accuracy and macro-F1 must match the values recorded during the preceding conversion-validation stage.

The worker also verifies the SHA-256 hash of the LiteRT model before accepting the profiling result.

If predictive performance or artifact identity differs from the validated conversion record, profiling is stopped for that artifact.

#### Per-Model Profiling Artifacts

Each profiled model receives an `Edge_Profiles` directory in its saved run folder.

The main outputs are:

```text
Edge_Profiles/
├── <model>_edge_profile.json
├── <model>_edge_profile_processes.csv
├── <model>_latency_process_1.npy
├── <model>_latency_process_2.npy
└── <model>_latency_process_3.npy
```

The JSON profile records:

- model and deployment hashes;
- conversion-validation provenance;
- parameter count;
- LiteRT artifact size;
- validated accuracy and macro-F1;
- CPU configuration;
- CPU affinity;
- process-repetition count;
- warm-up configuration;
- latency measurement scope;
- runtime and environment information;
- per-process measurements; and
- aggregated latency and memory statistics.

The process CSV preserves the individual fresh-process profiling results.

#### Cross-Model Summary

After all selected artifacts have been profiled, the code writes a timestamped cross-model summary to:

```text
Honours_Project/
└── Models/
    └── Edge_Profiling_Summaries/
        └── edge_profile_summary_<timestamp>.csv
```

The summary contains deployment and profiling information including:

- model group;
- source artifact;
- model hashes;
- LiteRT version;
- platform and processor information;
- parameter count;
- model size;
- accuracy;
- macro-F1;
- mean, median, P95 and P99 latency;
- process-level latency variability;
- total peak RSS overhead; and
- total peak USS overhead.

#### Measured Edge-Deployment Dashboard

When enabled, the profiler also generates a measured-results dashboard for the artifacts profiled during the current execution.

The dashboard contains four panels showing:

1. median and P95 LiteRT latency;
2. peak RSS and USS memory overhead;
3. validated FP32 LiteRT artifact size; and
4. retained accuracy and macro-F1.

The dashboard uses measured profiling results only. It does not use FLOP-based estimates, synthetic target-device projections or latency extrapolation.

The accompanying numerical table and figures are saved under:

`Honours_Project/Models/Edge_Profiling_Summaries/`

using the same profiling timestamp.

#### Methodological Role

This section performs **controlled deployment profiling**.

It does not:

- retrain models;
- select model seeds;
- tune hyperparameters;
- modify learned weights;
- perform LiteRT conversion;
- estimate unmeasured target-device performance; or
- measure complete end-to-end network-processing latency.

Its outputs provide controlled prepared-input LiteRT inference measurements for analysing the computational trade-offs between the retained neural architectures.

In [ ]:
# LITERT EDGE PROFILER

import csv
import hashlib
import importlib.util
import json
import os
import platform
import re
import subprocess
import sys
import tempfile
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import drive
from IPython.display import display

drive.mount("/content/drive")

MODELS_ROOT = Path("/content/drive/MyDrive/Honours_Project/Models")

CPU_THREADS = 1
PROCESS_REPETITIONS = 3
WARMUP_INFERENCES = 50
LATENCY_REPETITIONS = 10
MEMORY_SAMPLE_INTERVAL_SECONDS = 0.002

EXPECTED_TEST_SHAPE = (376, 244, 1)
EXPECTED_CLASS_COUNT = 10

REQUIRE_CPU_AFFINITY = True
CREATE_MEASURED_DASHBOARD = True

EDGE_SUFFIX = "_edge_static_batch1_fp32_frozen"
TFLITE_SUFFIX = f"{EDGE_SUFFIX}.tflite"


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def source_stem_from_tflite(path):
    if not path.name.endswith(TFLITE_SUFFIX):
        raise ValueError(f"Unexpected converter filename: {path.name}")
    return path.name[:-len(TFLITE_SUFFIX)]


def validation_path_for(tflite_path):
    return tflite_path.with_name(
        f"{source_stem_from_tflite(tflite_path)}_conversion_validation.json"
    )


def profile_json_path_for(tflite_path):
    return (
        tflite_path.parent
        / "Edge_Profiles"
        / f"{source_stem_from_tflite(tflite_path)}_edge_profile.json"
    )


def extract_run_timestamp(run_dir):
    match = re.match(r"^(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})", run_dir.name)
    if match:
        return datetime.strptime(match.group(1), "%Y-%m-%d_%H-%M-%S")
    return datetime.fromtimestamp(run_dir.stat().st_mtime)


def read_validation_record(tflite_path):
    validation_path = validation_path_for(tflite_path)
    if not validation_path.exists():
        raise FileNotFoundError(
            f"Conversion-validation record is missing:\n{validation_path}"
        )

    with open(validation_path, "r", encoding="utf-8") as handle:
        record = json.load(handle)

    if record.get("status") != "validated":
        raise RuntimeError(f"Artifact is not marked validated: {validation_path}")

    actual_tflite_hash = sha256_file(tflite_path)
    if actual_tflite_hash != record.get("tflite_sha256"):
        raise RuntimeError(f"TFLite hash mismatch: {tflite_path}")

    x_path = Path(record["test_inputs_path"])
    y_path = Path(record["test_labels_path"])
    classes_path = Path(record["device_classes_path"])

    for companion_path in (x_path, y_path, classes_path):
        if not companion_path.exists():
            raise FileNotFoundError(
                f"Required companion file is missing: {companion_path}"
            )

    expected_hashes = record.get("companion_sha256", {})
    companion_mapping = {
        "test_inputs": x_path,
        "test_labels": y_path,
        "device_classes": classes_path,
    }

    for name, companion_path in companion_mapping.items():
        actual_hash = sha256_file(companion_path)
        expected_hash = expected_hashes.get(name)
        if actual_hash != expected_hash:
            raise RuntimeError(
                f"Companion hash mismatch for {name}: {companion_path}"
            )

    x_memmap = np.load(x_path, mmap_mode="r")
    y_memmap = np.load(y_path, mmap_mode="r")
    classes = np.load(classes_path, allow_pickle=True)

    x_shape = tuple(x_memmap.shape)
    y_shape = tuple(y_memmap.shape)
    classes_shape = tuple(classes.shape)

    del x_memmap, y_memmap, classes

    if x_shape != EXPECTED_TEST_SHAPE:
        raise RuntimeError(
            f"Expected test shape {EXPECTED_TEST_SHAPE}; found {x_shape}"
        )

    if y_shape != (EXPECTED_TEST_SHAPE[0],):
        raise RuntimeError(
            f"Expected label shape {(EXPECTED_TEST_SHAPE[0],)}; found {y_shape}"
        )

    if classes_shape != (EXPECTED_CLASS_COUNT,):
        raise RuntimeError(
            f"Expected {EXPECTED_CLASS_COUNT} classes; found {classes_shape}"
        )

    return record


def scan_validated_artifacts():
    artifacts = []
    for path in MODELS_ROOT.rglob(f"*{TFLITE_SUFFIX}"):
        if any(part.startswith("_TEMP_") for part in path.parts):
            continue

        try:
            record = read_validation_record(path)
            run_dir = path.parent
            artifacts.append({
                "path": path,
                "record": record,
                "model_group": record["model_group"],
                "source_stem": source_stem_from_tflite(path),
                "run_dir": run_dir,
                "run_timestamp": extract_run_timestamp(run_dir),
                "profiled": profile_json_path_for(path).exists(),
            })
        except Exception as exc:
            print(f"Skipping {path}: {exc}")

    return sorted(
        artifacts,
        key=lambda item: (
            item["run_timestamp"],
            item["model_group"],
            item["source_stem"],
        ),
        reverse=True,
    )


def latest_run_artifacts(artifacts):
    selected = []
    groups = sorted({item["model_group"] for item in artifacts})

    for group in groups:
        group_items = [
            item for item in artifacts if item["model_group"] == group
        ]
        if not group_items:
            continue

        run_directories = {item["run_dir"] for item in group_items}
        latest_run_dir = max(run_directories, key=extract_run_timestamp)
        selected.extend(
            item for item in group_items if item["run_dir"] == latest_run_dir
        )

    return sorted(
        selected,
        key=lambda item: (item["model_group"], item["source_stem"]),
    )


def parse_number_selection(text, total):
    indexes = set()
    for token in text.replace(" ", "").split(","):
        if not token:
            continue
        if "-" in token:
            start, end = token.split("-", 1)
            indexes.update(range(int(start), int(end) + 1))
        else:
            indexes.add(int(token))

    invalid = [index for index in indexes if index < 1 or index > total]
    if invalid:
        raise ValueError(f"Invalid selection numbers: {invalid}")

    return sorted(indexes)


def choose_artifacts(artifacts):
    print("\nValidated LiteRT artifacts discovered")
    print("=" * 120)
    for index, item in enumerate(artifacts, 1):
        status = "PROFILED" if item["profiled"] else "NEW"
        print(
            f"[{index:>3}] {status:<8} {item['model_group']:<22} {item['source_stem']}\n"
            f"      {item['path']}"
        )
    print("=" * 120)

    print("\nSelection commands")
    print("  new       = all validated artifacts without an edge profile")
    print("  latest    = every validated artifact in the newest run of each model group")
    print("  all       = every validated artifact shown above")
    print("  1,3,5-8   = specific numbered artifacts")
    print("  path      = enter one explicit Google Drive .tflite path")

    selection = input("\nSelect artifacts to profile [new]: ").strip().lower() or "new"

    if selection == "new":
        return [item for item in artifacts if not item["profiled"]]

    if selection == "latest":
        return latest_run_artifacts(artifacts)

    if selection == "all":
        return artifacts

    if selection == "path":
        path = Path(input("Paste the Google Drive .tflite path: ").strip())
        if not path.exists():
            raise FileNotFoundError(path)

        record = read_validation_record(path)
        run_dir = path.parent
        return [{
            "path": path,
            "record": record,
            "model_group": record["model_group"],
            "source_stem": source_stem_from_tflite(path),
            "run_dir": run_dir,
            "run_timestamp": extract_run_timestamp(run_dir),
            "profiled": profile_json_path_for(path).exists(),
        }]

    indexes = parse_number_selection(selection, len(artifacts))
    return [artifacts[index - 1] for index in indexes]


WORKER_CODE = r'''import hashlib
import importlib.metadata
import json
import os
import platform
import sys
import threading
import time
from pathlib import Path
import numpy as np
import psutil

CPU_THREADS = int(sys.argv[5])

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)

cpu_affinity_supported = False
cpu_affinity_applied = False
eligible_cpus_before_pinning = []
cpu_affinity_after = []
pinned_cpu = None

try:
    if hasattr(os, "sched_getaffinity") and hasattr(os, "sched_setaffinity"):
        eligible_cpus_before_pinning = sorted(int(cpu) for cpu in os.sched_getaffinity(0))
        if eligible_cpus_before_pinning:
            pinned_cpu = int(eligible_cpus_before_pinning[0])
            os.sched_setaffinity(0, {pinned_cpu})
            cpu_affinity_after = sorted(int(cpu) for cpu in os.sched_getaffinity(0))
            cpu_affinity_supported = True
            cpu_affinity_applied = (cpu_affinity_after == [pinned_cpu])
except Exception:
    pass

TFLITE_PATH = Path(sys.argv[1])
X_PATH = Path(sys.argv[2])
Y_PATH = Path(sys.argv[3])
RAW_LATENCY_PATH = Path(sys.argv[4])
WARMUP_INFERENCES = int(sys.argv[6])
LATENCY_REPETITIONS = int(sys.argv[7])
MEMORY_SAMPLE_INTERVAL_SECONDS = float(sys.argv[8])

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def cpu_model_name():
    try:
        text = Path("/proc/cpuinfo").read_text(encoding="utf-8", errors="ignore")
        for key in ("model name", "Hardware", "Processor"):
            match = next(
                (line.split(":", 1)[1].strip() for line in text.splitlines() if line.startswith(key) and ":" in line),
                None
            )
            if match:
                return match
    except Exception:
        pass
    return platform.processor() or "unknown"

def macro_f1_score(y_true, y_pred, class_count):
    scores = []
    for label in range(class_count):
        true_positive = np.sum((y_true == label) & (y_pred == label))
        false_positive = np.sum((y_true != label) & (y_pred == label))
        false_negative = np.sum((y_true == label) & (y_pred != label))
        denominator = 2 * true_positive + false_positive + false_negative
        scores.append(0.0 if denominator == 0 else (2 * true_positive / denominator))
    return float(np.mean(scores))

process = psutil.Process(os.getpid())

def rss_mb():
    return float(process.memory_info().rss / (1024 * 1024))

def uss_mb():
    try:
        return float(process.memory_full_info().uss / (1024 * 1024))
    except Exception:
        return None

baseline_rss = rss_mb()
baseline_uss = uss_mb()

from ai_edge_litert.interpreter import Interpreter

litert_imported_rss = rss_mb()
litert_imported_uss = uss_mb()

interpreter = Interpreter(model_path=str(TFLITE_PATH), num_threads=CPU_THREADS)
interpreter.allocate_tensors()

model_loaded_rss = rss_mb()
model_loaded_uss = uss_mb()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

if len(input_details) != 1 or len(output_details) != 1:
    raise RuntimeError("The LiteRT model must have exactly one input and one output.")

input_detail = input_details[0]
output_detail = output_details[0]

input_index = int(input_detail["index"])
output_index = int(output_detail["index"])
input_shape = tuple(int(value) for value in input_detail["shape"])
output_shape = tuple(int(value) for value in output_detail["shape"])
input_dtype = input_detail["dtype"]
output_dtype = output_detail["dtype"]

X = np.load(X_PATH, mmap_mode="r")
y = np.asarray(np.load(Y_PATH, mmap_mode="r"), dtype=np.int64)

if tuple(X.shape) != (376, 244, 1):
    raise RuntimeError(f"Unexpected corrected test shape: {X.shape}")

if tuple(y.shape) != (376,):
    raise RuntimeError(f"Unexpected corrected label shape: {y.shape}")

if len(input_shape) == 2:
    expected_input_shape = (1, X.shape[1])
elif len(input_shape) == 3:
    expected_input_shape = (1, X.shape[1], X.shape[2])
else:
    raise RuntimeError(f"Unsupported LiteRT input rank: {len(input_shape)}")

if input_shape != expected_input_shape:
    raise RuntimeError(f"Static input mismatch. Runtime={input_shape}; expected={expected_input_shape}")

if output_shape[-1] != 10:
    raise RuntimeError(f"Unexpected model output shape: {output_shape}")

inference_ready_rss = rss_mb()
inference_ready_uss = uss_mb()

def sample_at(index):
    if len(input_shape) == 2:
        return np.asarray(X[index:index + 1, :, 0], dtype=input_dtype)
    return np.asarray(X[index:index + 1], dtype=input_dtype)

def infer_prepared(sample):
    interpreter.set_tensor(input_index, sample)
    interpreter.invoke()
    return interpreter.get_tensor(output_index)

def infer(index):
    sample = sample_at(index)
    return infer_prepared(sample)

memory_samples = []
memory_monitor_running = True

def monitor_memory():
    while memory_monitor_running:
        memory_samples.append((time.perf_counter_ns(), rss_mb(), uss_mb()))
        time.sleep(MEMORY_SAMPLE_INTERVAL_SECONDS)

memory_samples.append((time.perf_counter_ns(), inference_ready_rss, inference_ready_uss))

monitor = threading.Thread(target=monitor_memory, daemon=True)
monitor.start()

for index in range(WARMUP_INFERENCES):
    output = infer(index % len(X))
    if not np.all(np.isfinite(output)):
        raise RuntimeError("LiteRT produced a non-finite output during memory warm-up.")

memory_predictions = []
for index in range(len(X)):
    output = infer(index)
    if not np.all(np.isfinite(output)):
        raise RuntimeError(f"LiteRT produced a non-finite output at sample {index}.")
    prediction = int(np.argmax(output, axis=1)[0])
    memory_predictions.append(prediction)

memory_monitor_running = False
monitor.join()

memory_samples.append((time.perf_counter_ns(), rss_mb(), uss_mb()))
memory_predictions = np.asarray(memory_predictions, dtype=np.int64)

accuracy = float(np.mean(memory_predictions == y))
macro_f1 = macro_f1_score(y, memory_predictions, output_shape[-1])

peak_inference_rss = max(sample[1] for sample in memory_samples)
uss_values = [sample[2] for sample in memory_samples if sample[2] is not None]
peak_inference_uss = max(uss_values) if uss_values else None

total_peak_rss = max(
    baseline_rss,
    litert_imported_rss,
    model_loaded_rss,
    inference_ready_rss,
    peak_inference_rss,
)
rss_total_peak_overhead = total_peak_rss - baseline_rss

if (
    baseline_uss is not None
    and litert_imported_uss is not None
    and model_loaded_uss is not None
    and inference_ready_uss is not None
    and peak_inference_uss is not None
):
    total_peak_uss = max(
        baseline_uss,
        litert_imported_uss,
        model_loaded_uss,
        inference_ready_uss,
        peak_inference_uss,
    )
    uss_total_peak_overhead = total_peak_uss - baseline_uss
else:
    total_peak_uss = None
    uss_total_peak_overhead = None

for index in range(WARMUP_INFERENCES):
    output = infer(index % len(X))
    if not np.all(np.isfinite(output)):
        raise RuntimeError("LiteRT produced a non-finite output during latency warm-up.")

latencies_ms = np.empty(LATENCY_REPETITIONS * len(X), dtype=np.float64)
position = 0

for repetition in range(LATENCY_REPETITIONS):
    for index in range(len(X)):
        sample = sample_at(index)
        start_ns = time.perf_counter_ns()
        output = infer_prepared(sample)
        end_ns = time.perf_counter_ns()
        if not np.all(np.isfinite(output)):
            raise RuntimeError(
                f"LiteRT produced a non-finite output during timed inference at sample {index}."
            )
        latencies_ms[position] = (end_ns - start_ns) / 1_000_000.0
        position += 1

RAW_LATENCY_PATH.parent.mkdir(parents=True, exist_ok=True)
np.save(RAW_LATENCY_PATH, latencies_ms)

result = {
    "runtime": "LiteRT",
    "runtime_package": "ai-edge-litert",
    "runtime_version": importlib.metadata.version("ai-edge-litert"),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": cpu_model_name(),
    "logical_cpu_count": psutil.cpu_count(logical=True),
    "physical_cpu_count": psutil.cpu_count(logical=False),
    "cpu_threads": CPU_THREADS,
    "cpu_affinity_supported": cpu_affinity_supported,
    "cpu_affinity_applied": cpu_affinity_applied,
    "eligible_cpus_before_pinning": eligible_cpus_before_pinning,
    "eligible_cpu_count_before_pinning": len(eligible_cpus_before_pinning),
    "pinned_cpu": pinned_cpu,
    "cpu_affinity_after": cpu_affinity_after,
    "tflite_path": str(TFLITE_PATH),
    "tflite_sha256": sha256_file(TFLITE_PATH),
    "test_inputs_path": str(X_PATH),
    "test_inputs_sha256": sha256_file(X_PATH),
    "test_labels_path": str(Y_PATH),
    "test_labels_sha256": sha256_file(Y_PATH),
    "input_shape": list(input_shape),
    "output_shape": list(output_shape),
    "input_dtype": str(np.dtype(input_dtype)),
    "output_dtype": str(np.dtype(output_dtype)),
    "static_batch_size": int(input_shape[0]),
    "model_size_bytes": int(TFLITE_PATH.stat().st_size),
    "test_flows": int(len(X)),
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "latency_scope": "prepared input -> set_tensor + invoke + get_tensor",
    "sample_preparation_included_in_latency": False,
    "memory_monitor_active_during_latency": False,
    "warmup_inferences_per_phase": WARMUP_INFERENCES,
    "latency_repetitions": LATENCY_REPETITIONS,
    "timed_inferences": int(latencies_ms.size),
    "raw_latency_path": str(RAW_LATENCY_PATH),
    "raw_latency_sha256": sha256_file(RAW_LATENCY_PATH),
    "latency_mean_ms": float(np.mean(latencies_ms)),
    "latency_std_ms": float(np.std(latencies_ms, ddof=1)),
    "latency_median_ms": float(np.median(latencies_ms)),
    "latency_p95_ms": float(np.percentile(latencies_ms, 95)),
    "latency_p99_ms": float(np.percentile(latencies_ms, 99)),
    "latency_min_ms": float(np.min(latencies_ms)),
    "latency_max_ms": float(np.max(latencies_ms)),
    "memory_units": "MiB",
    "memory_sample_interval_seconds": MEMORY_SAMPLE_INTERVAL_SECONDS,
    "memory_sample_count": len(memory_samples),
    "baseline_rss_mb": baseline_rss,
    "litert_imported_rss_mb": litert_imported_rss,
    "model_loaded_rss_mb": model_loaded_rss,
    "inference_ready_rss_mb": inference_ready_rss,
    "peak_inference_rss_mb": peak_inference_rss,
    "total_peak_rss_mb": total_peak_rss,
    "litert_import_overhead_rss_mb": litert_imported_rss - baseline_rss,
    "model_allocation_overhead_rss_mb": model_loaded_rss - litert_imported_rss,
    "input_data_overhead_rss_mb": inference_ready_rss - model_loaded_rss,
    "peak_inference_overhead_rss_mb": peak_inference_rss - inference_ready_rss,
    "total_peak_overhead_rss_mb": rss_total_peak_overhead,
    "baseline_uss_mb": baseline_uss,
    "litert_imported_uss_mb": litert_imported_uss,
    "model_loaded_uss_mb": model_loaded_uss,
    "inference_ready_uss_mb": inference_ready_uss,
    "peak_inference_uss_mb": peak_inference_uss,
    "total_peak_uss_mb": total_peak_uss,
    "litert_import_overhead_uss_mb": (
        None if (baseline_uss is None or litert_imported_uss is None)
        else litert_imported_uss - baseline_uss
    ),
    "model_allocation_overhead_uss_mb": (
        None if (litert_imported_uss is None or model_loaded_uss is None)
        else model_loaded_uss - litert_imported_uss
    ),
    "input_data_overhead_uss_mb": (
        None if (model_loaded_uss is None or inference_ready_uss is None)
        else inference_ready_uss - model_loaded_uss
    ),
    "peak_inference_overhead_uss_mb": (
        None if (inference_ready_uss is None or peak_inference_uss is None)
        else peak_inference_uss - inference_ready_uss
    ),
    "total_peak_overhead_uss_mb": uss_total_peak_overhead,
}

print(json.dumps(result, allow_nan=False))
'''


def ensure_litert():
    missing_packages = []
    if importlib.util.find_spec("ai_edge_litert") is None:
        missing_packages.append("ai-edge-litert")
    if importlib.util.find_spec("psutil") is None:
        missing_packages.append("psutil")

    if missing_packages:
        print("Installing missing profiling dependencies: " + ", ".join(missing_packages))
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            *missing_packages,
        ])


def aggregate_process_results(results):
    latency_fields = [
        "latency_mean_ms",
        "latency_std_ms",
        "latency_median_ms",
        "latency_p95_ms",
        "latency_p99_ms",
        "latency_min_ms",
        "latency_max_ms",
    ]
    memory_fields = [
        "baseline_rss_mb",
        "litert_imported_rss_mb",
        "model_loaded_rss_mb",
        "inference_ready_rss_mb",
        "peak_inference_rss_mb",
        "total_peak_rss_mb",
        "litert_import_overhead_rss_mb",
        "model_allocation_overhead_rss_mb",
        "input_data_overhead_rss_mb",
        "peak_inference_overhead_rss_mb",
        "total_peak_overhead_rss_mb",
        "baseline_uss_mb",
        "litert_imported_uss_mb",
        "model_loaded_uss_mb",
        "inference_ready_uss_mb",
        "peak_inference_uss_mb",
        "total_peak_uss_mb",
        "litert_import_overhead_uss_mb",
        "model_allocation_overhead_uss_mb",
        "input_data_overhead_uss_mb",
        "peak_inference_overhead_uss_mb",
        "total_peak_overhead_uss_mb",
    ]

    aggregate = {}
    for field in latency_fields + memory_fields:
        values = [result[field] for result in results if result.get(field) is not None]
        if values:
            aggregate[f"{field}_process_mean"] = float(np.mean(values))
            aggregate[f"{field}_process_sd"] = (
                float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
            )

    return aggregate


def profile_artifact(item):
    tflite_path = item["path"]
    record = read_validation_record(tflite_path)
    output_dir = tflite_path.parent / "Edge_Profiles"
    output_dir.mkdir(parents=True, exist_ok=True)
    source_stem = item["source_stem"]

    print("\n" + "=" * 120)
    print(f"PROFILING: {source_stem}")
    print(tflite_path)
    print(f"CPU threads: {CPU_THREADS}")
    print(f"Fresh-process repetitions: {PROCESS_REPETITIONS}")
    print(f"Warm-ups per phase: {WARMUP_INFERENCES}")
    print(
        f"Timed inferences per process: {LATENCY_REPETITIONS} x "
        f"{EXPECTED_TEST_SHAPE[0]} = {LATENCY_REPETITIONS * EXPECTED_TEST_SHAPE[0]}"
    )

    process_results = []
    for process_index in range(1, PROCESS_REPETITIONS + 1):
        raw_path = output_dir / f"{source_stem}_latency_process_{process_index}.npy"

        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as handle:
            handle.write(WORKER_CODE)
            worker_path = Path(handle.name)

        try:
            completed = subprocess.run(
                [
                    sys.executable,
                    str(worker_path),
                    str(tflite_path),
                    record["test_inputs_path"],
                    record["test_labels_path"],
                    str(raw_path),
                    str(CPU_THREADS),
                    str(WARMUP_INFERENCES),
                    str(LATENCY_REPETITIONS),
                    str(MEMORY_SAMPLE_INTERVAL_SECONDS),
                ],
                capture_output=True,
                text=True,
            )
        finally:
            worker_path.unlink(missing_ok=True)

        if completed.returncode != 0:
            print(completed.stdout)
            print(completed.stderr)
            raise RuntimeError(
                f"Profiler worker {process_index} failed for {tflite_path}"
            )

        output_lines = [line for line in completed.stdout.splitlines() if line.strip()]
        if not output_lines:
            raise RuntimeError(f"Profiler worker {process_index} returned no result.")

        result = json.loads(output_lines[-1])
        result["process_repetition"] = process_index

        if result["runtime"] != "LiteRT":
            raise RuntimeError(
                "The benchmark did not use the standalone LiteRT runtime."
            )

        if REQUIRE_CPU_AFFINITY:
            if not result.get("cpu_affinity_applied", False):
                raise RuntimeError(
                    "CPU affinity could not be applied. Controlled profiling requires "
                    "the worker to be pinned to one logical CPU."
                )

            affinity_after = result.get("cpu_affinity_after", [])
            if len(affinity_after) != 1:
                raise RuntimeError(
                    "Profiler worker was not restricted to exactly one logical CPU. "
                    f"Observed affinity: {affinity_after}"
                )

        if result["tflite_sha256"] != record["tflite_sha256"]:
            raise RuntimeError("Worker profiled a TFLite file with the wrong hash.")

        if not np.isclose(result["accuracy"], record["tflite_accuracy"], atol=1e-12):
            raise RuntimeError("Worker accuracy differs from conversion validation.")

        if not np.isclose(result["macro_f1"], record["tflite_macro_f1"], atol=1e-12):
            raise RuntimeError("Worker Macro-F1 differs from conversion validation.")

        process_results.append(result)
        print(
            f"Process {process_index}: CPU={result['pinned_cpu']}, "
            f"median={result['latency_median_ms']:.6f} ms, "
            f"P95={result['latency_p95_ms']:.6f} ms, "
            f"P99={result['latency_p99_ms']:.6f} ms, "
            f"peak RSS overhead={result['total_peak_overhead_rss_mb']:.3f} MiB"
        )

    profile = {
        "status": "profiled",
        "profiled_at": datetime.now(ZoneInfo("Africa/Johannesburg")).isoformat(),
        "model_group": item["model_group"],
        "source_stem": source_stem,
        "source_model_path": record["source_model_path"],
        "source_model_sha256": record["source_model_sha256"],
        "tflite_path": str(tflite_path),
        "tflite_sha256": record["tflite_sha256"],
        "conversion_validation_path": str(validation_path_for(tflite_path)),
        "parameters": int(record["parameters"]),
        "tflite_size_bytes": int(record["tflite_size_bytes"]),
        "tflite_accuracy": float(record["tflite_accuracy"]),
        "tflite_macro_f1": float(record["tflite_macro_f1"]),
        "tflite_class_agreement": float(record["tflite_class_agreement"]),
        "read_variable_ops_remaining": int(record["read_variable_ops_remaining"]),
        "cpu_only": True,
        "cpu_threads": int(process_results[0]["cpu_threads"]),
        "cpu_affinity_required": REQUIRE_CPU_AFFINITY,
        "cpu_affinity_applied": bool(process_results[0]["cpu_affinity_applied"]),
        "pinned_cpu": process_results[0]["pinned_cpu"],
        "static_batch_size": int(process_results[0]["static_batch_size"]),
        "process_repetitions": PROCESS_REPETITIONS,
        "warmup_inferences_per_phase": WARMUP_INFERENCES,
        "latency_repetitions": LATENCY_REPETITIONS,
        "latency_scope": process_results[0]["latency_scope"],
        "sample_preparation_included_in_latency": process_results[0]["sample_preparation_included_in_latency"],
        "memory_monitor_active_during_latency": process_results[0]["memory_monitor_active_during_latency"],
        "memory_sample_interval_seconds": MEMORY_SAMPLE_INTERVAL_SECONDS,
        "memory_units": process_results[0]["memory_units"],
        "runtime": process_results[0]["runtime"],
        "runtime_version": process_results[0]["runtime_version"],
        "python_version": process_results[0]["python_version"],
        "platform": process_results[0]["platform"],
        "machine": process_results[0]["machine"],
        "processor": process_results[0]["processor"],
        "logical_cpu_count": process_results[0]["logical_cpu_count"],
        "physical_cpu_count": process_results[0]["physical_cpu_count"],
        "eligible_cpu_count_before_pinning": process_results[0]["eligible_cpu_count_before_pinning"],
        "process_results": process_results,
        "aggregate": aggregate_process_results(process_results),
    }

    json_path = profile_json_path_for(tflite_path)
    with open(json_path, "w", encoding="utf-8") as handle:
        json.dump(profile, handle, indent=2, allow_nan=False)

    csv_path = output_dir / f"{source_stem}_edge_profile_processes.csv"
    fieldnames = sorted({key for result in process_results for key in result})
    with open(csv_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(process_results)

    print(f"Saved profile: {json_path}")
    print(f"Saved process table: {csv_path}")
    return profile


def profile_to_summary_row(profile):
    aggregate = profile["aggregate"]
    return {
        "model_group": profile["model_group"],
        "source_stem": profile["source_stem"],
        "source_model_path": profile["source_model_path"],
        "source_model_sha256": profile["source_model_sha256"],
        "tflite_path": profile["tflite_path"],
        "tflite_sha256": profile["tflite_sha256"],
        "runtime": profile["runtime"],
        "runtime_version": profile["runtime_version"],
        "platform": profile["platform"],
        "machine": profile["machine"],
        "processor": profile["processor"],
        "cpu_threads": profile["cpu_threads"],
        "pinned_cpu": profile["pinned_cpu"],
        "process_repetitions": profile["process_repetitions"],
        "parameters": profile["parameters"],
        "tflite_size_bytes": profile["tflite_size_bytes"],
        "tflite_size_kib": profile["tflite_size_bytes"] / 1024.0,
        "accuracy": profile["tflite_accuracy"],
        "accuracy_percent": profile["tflite_accuracy"] * 100.0,
        "macro_f1": profile["tflite_macro_f1"],
        "median_latency_ms": aggregate.get("latency_median_ms_process_mean"),
        "median_latency_process_sd_ms": aggregate.get("latency_median_ms_process_sd"),
        "p95_latency_ms": aggregate.get("latency_p95_ms_process_mean"),
        "p95_latency_process_sd_ms": aggregate.get("latency_p95_ms_process_sd"),
        "p99_latency_ms": aggregate.get("latency_p99_ms_process_mean"),
        "p99_latency_process_sd_ms": aggregate.get("latency_p99_ms_process_sd"),
        "mean_latency_ms": aggregate.get("latency_mean_ms_process_mean"),
        "mean_latency_process_sd_ms": aggregate.get("latency_mean_ms_process_sd"),
        "total_peak_rss_overhead_mb": aggregate.get("total_peak_overhead_rss_mb_process_mean"),
        "total_peak_rss_overhead_process_sd_mb": aggregate.get("total_peak_overhead_rss_mb_process_sd"),
        "total_peak_uss_overhead_mb": aggregate.get("total_peak_overhead_uss_mb_process_mean"),
        "total_peak_uss_overhead_process_sd_mb": aggregate.get("total_peak_overhead_uss_mb_process_sd"),
        "model_allocation_rss_overhead_mb": aggregate.get("model_allocation_overhead_rss_mb_process_mean"),
        "input_data_rss_overhead_mb": aggregate.get("input_data_overhead_rss_mb_process_mean"),
        "peak_inference_rss_overhead_mb": aggregate.get("peak_inference_overhead_rss_mb_process_mean"),
        "memory_units": profile["memory_units"],
    }


def write_global_summary(profiles):
    timestamp = datetime.now(ZoneInfo("Africa/Johannesburg")).strftime("%Y-%m-%d_%H-%M-%S")
    summary_dir = MODELS_ROOT / "Edge_Profiling_Summaries"
    summary_dir.mkdir(parents=True, exist_ok=True)
    summary_path = summary_dir / f"edge_profile_summary_{timestamp}.csv"

    rows = [profile_to_summary_row(profile) for profile in profiles]
    fieldnames = sorted({key for row in rows for key in row})

    with open(summary_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nSaved cross-model summary: {summary_path}")
    return summary_path, rows, timestamp


def create_measured_edge_dashboard(profiles, timestamp):
    if not profiles:
        return None, None, None

    summary_dir = MODELS_ROOT / "Edge_Profiling_Summaries"
    summary_dir.mkdir(parents=True, exist_ok=True)

    rows = [profile_to_summary_row(profile) for profile in profiles]
    frame = pd.DataFrame(rows)

    group_counts = frame["model_group"].value_counts().to_dict()
    labels = [
        str(row["model_group"])
        if group_counts.get(row["model_group"], 0) == 1
        else f"{row['model_group']}\n{row['source_stem']}"
        for _, row in frame.iterrows()
    ]
    frame["display_label"] = labels

    display_columns = [
        "model_group",
        "source_stem",
        "parameters",
        "tflite_size_kib",
        "accuracy_percent",
        "macro_f1",
        "median_latency_ms",
        "p95_latency_ms",
        "p99_latency_ms",
        "total_peak_rss_overhead_mb",
        "total_peak_uss_overhead_mb",
    ]

    display_frame = frame[display_columns].copy().rename(
        columns={
            "model_group": "Model Group",
            "source_stem": "Artifact",
            "parameters": "Parameters",
            "tflite_size_kib": "TFLite Size (KiB)",
            "accuracy_percent": "Accuracy (%)",
            "macro_f1": "Macro-F1",
            "median_latency_ms": "Median Latency (ms)",
            "p95_latency_ms": "P95 Latency (ms)",
            "p99_latency_ms": "P99 Latency (ms)",
            "total_peak_rss_overhead_mb": "Peak RSS Overhead (MiB)",
            "total_peak_uss_overhead_mb": "Peak USS Overhead (MiB)",
        }
    )

    print("\n" + "=" * 120)
    print("MEASURED EDGE DEPLOYMENT SUMMARY")
    print("=" * 120)
    print(
        "All latency and memory values below are measured LiteRT results. "
        "No TFLOP-based or target-device latency extrapolation is used."
    )

    display(
        display_frame.round({
            "TFLite Size (KiB)": 1,
            "Accuracy (%)": 3,
            "Macro-F1": 4,
            "Median Latency (ms)": 4,
            "P95 Latency (ms)": 4,
            "P99 Latency (ms)": 4,
            "Peak RSS Overhead (MiB)": 3,
            "Peak USS Overhead (MiB)": 3,
        })
    )

    table_path = summary_dir / f"edge_profile_dashboard_table_{timestamp}.csv"
    display_frame.to_csv(table_path, index=False)

    labels = frame["display_label"].tolist()
    x = np.arange(len(frame))
    width = 0.36

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    axes[0, 0].bar(x - width / 2, frame["median_latency_ms"], width, label="Median")
    axes[0, 0].bar(x + width / 2, frame["p95_latency_ms"], width, label="P95")
    axes[0, 0].set_title("Measured Batch-One LiteRT Latency")
    axes[0, 0].set_ylabel("Latency (ms)")
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(labels, rotation=35, ha="right")
    axes[0, 0].legend()
    axes[0, 0].grid(axis="y", linestyle="--", alpha=0.35)

    axes[0, 1].bar(x - width / 2, frame["total_peak_rss_overhead_mb"], width, label="RSS")
    uss_values = frame["total_peak_uss_overhead_mb"]
    if not uss_values.isna().all():
        axes[0, 1].bar(x + width / 2, uss_values, width, label="USS")
    axes[0, 1].set_title("Measured Peak Process-Memory Overhead")
    axes[0, 1].set_ylabel("Memory overhead (MiB)")
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(labels, rotation=35, ha="right")
    axes[0, 1].legend()
    axes[0, 1].grid(axis="y", linestyle="--", alpha=0.35)

    axes[1, 0].bar(x, frame["tflite_size_kib"])
    axes[1, 0].set_title("Validated FP32 LiteRT Artifact Size")
    axes[1, 0].set_ylabel("Size (KiB)")
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(labels, rotation=35, ha="right")
    axes[1, 0].grid(axis="y", linestyle="--", alpha=0.35)

    axes[1, 1].bar(x - width / 2, frame["accuracy_percent"], width, label="Accuracy")
    axes[1, 1].bar(x + width / 2, frame["macro_f1"] * 100.0, width, label="Macro-F1 × 100")
    axes[1, 1].set_title("Validated Test Performance of Profiled Artifact")
    axes[1, 1].set_ylabel("Percent")
    axes[1, 1].set_ylim(0, 100)
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(labels, rotation=35, ha="right")
    axes[1, 1].legend()
    axes[1, 1].grid(axis="y", linestyle="--", alpha=0.35)

    fig.suptitle(
        "Universal LiteRT Edge-Profiling Dashboard — Measured Results Only",
        fontsize=15,
        fontweight="bold",
    )
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    figure_png_path = summary_dir / f"edge_profile_dashboard_{timestamp}.png"
    figure_pdf_path = summary_dir / f"edge_profile_dashboard_{timestamp}.pdf"

    fig.savefig(figure_png_path, dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(figure_pdf_path, bbox_inches="tight", facecolor="white")

    plt.show()
    plt.close(fig)

    print(f"Saved dashboard table: {table_path}")
    print(f"Saved measured dashboard PNG: {figure_png_path}")
    print(f"Saved measured dashboard PDF: {figure_pdf_path}")

    return table_path, figure_png_path, figure_pdf_path


if not MODELS_ROOT.exists():
    raise FileNotFoundError(f"Models root does not exist:\n{MODELS_ROOT}")

ensure_litert()
validated_artifacts = scan_validated_artifacts()

if not validated_artifacts:
    raise RuntimeError(
        "No converter-validated LiteRT artifacts were found.\n\n"
        "Run the Universal Converter first."
    )

selected_artifacts = choose_artifacts(validated_artifacts)

if not selected_artifacts:
    print("No artifacts matched the selection. Nothing was profiled.")
else:
    completed_profiles = []
    for item in selected_artifacts:
        completed_profiles.append(profile_artifact(item))

    summary_path, summary_rows, summary_timestamp = write_global_summary(completed_profiles)

    if CREATE_MEASURED_DASHBOARD:
        create_measured_edge_dashboard(completed_profiles, summary_timestamp)

    print("\n" + "=" * 120)
    print("ALL SELECTED LITERT ARTIFACTS WERE PROFILED SUCCESSFULLY")
    print("=" * 120)
    print(f"Artifacts profiled: {len(completed_profiles)}")
    print(f"Fresh process repetitions per artifact: {PROCESS_REPETITIONS}")
    print(f"Timed inferences per process: {LATENCY_REPETITIONS * EXPECTED_TEST_SHAPE[0]}")
    print(
        f"Timed inferences per artifact: "
        f"{PROCESS_REPETITIONS * LATENCY_REPETITIONS * EXPECTED_TEST_SHAPE[0]}"
    )
    print(f"CPU threads: {CPU_THREADS}")
    print(f"CPU affinity required: {REQUIRE_CPU_AFFINITY}")
    print("\nLatency scope:")
    print("  Prepared tensor -> set_tensor() -> invoke() -> get_tensor()")
    print("  Python sample preparation is excluded from timing.")
    print("  Memory sampling is stopped before latency timing.")
    print(f"\nCross-model summary:\n{summary_path}")

### ARM64 Profiling Bundle Assembly

This section packages user-selected, converter-validated neural LiteRT models for controlled profiling on an external ARM64 Linux machine. It does not train, convert, evaluate or profile models inside Colab.

Before running the bundle builder, verify that the current ARM profiler script exists at:

```text
/content/drive/MyDrive/Honours_Project/ARM_Profiling/
    comp700_arm_selected_models_v7.py
```

The selected neural models must already have been converted by the **LiteRT Model Conversion and Validation** cell. The builder lists compatible conversion-validation records found under:

```text
Honours_Project/Models/
```

Select model numbers using inputs such as:

```text
1
1,3,5
2-5
all
```

All selected models must use byte-identical held-out inputs, labels and class ordering. The builder verifies the corresponding SHA-256 hashes before constructing the bundle.

Review the displayed selection and type:

```text
BUILD
```

when prompted.

The bundle builder copies:

- the selected `.tflite` models;
- their conversion-validation records;
- the common held-out test arrays;
- the ARM profiling program;
- a manifest;
- dependency requirements; and
- operating instructions.

Before creating the ZIP, the builder runs the profiler's `inspect` mode inside the current Colab environment. The ZIP is not created if this integrity check fails.

The generated bundle is saved under:

```text
Honours_Project/
└── ARM_Profiling_Bundles/
    └── <bundle-build timestamp>/
        └── COMP700_ARM_SELECTED_MODELS_BUNDLE_V7.zip
```

Transfer the ZIP to the ARM64 Linux machine, extract it and enter the extracted directory.

Run:

```bash
python3 comp700_arm_selected_models_v7.py profile --auto-install --require-arm64
```

The `--auto-install` route creates a private profiling environment, installs the required dependencies, restarts inside that environment, validates the bundle and then performs profiling.

If Internet access is unavailable, install the packages in `requirements.txt` into the Python environment manually and run:

```bash
python3 comp700_arm_selected_models_v7.py profile --require-arm64
```

The controlled ARM64 procedure uses:

```text
FP32 LiteRT
static batch size 1
one inference thread
one pinned logical CPU
three fresh worker processes per model
warm-up excluded from timing
prepared-input timing boundary:
set_tensor → invoke → get_tensor
```

The profiler creates:

```text
results/
├── arm_edge_profile_summary.csv
├── arm_edge_profile_all.json
├── environment.json
└── raw/
```

Keep this complete directory unchanged.

Read the UTC `captured_at` timestamp from `results/environment.json`, convert it to the folder form `YYYY-MM-DD_HH-MM-SS`, and archive the complete result directory under:

```text
Honours_Project/
└── Models/
    └── Edge_Profiling_Summaries/
        └── ARM64/
            └── YYYY-MM-DD_HH-MM-SS/
                └── results/
```

Absolute latency and memory measurements depend on the ARM64 hardware and operating environment. Reproducibility therefore means preserving the validated models, benchmark protocol, environment record, raw measurements and aggregate results—not expecting identical timings on different machines.

In [ ]:
# BUILD USER-SELECTED ARM64 PROFILING BUNDLE
#
# PURPOSE
# -------
# Build an ARM64 profiling bundle from converter-validated LiteRT models created
# by the COMP700 notebook.
#
# The user chooses EXACTLY which validated model artifacts are included.
#
# IMPORTANT
# ---------
# - Run model training first.
# - Run "LiteRT Model Conversion and Validation" for every model you may want
#   to profile.
# - This cell does not train, convert, evaluate, or profile models.
# - It packages existing validated LiteRT artifacts and their common held-out
#   test bundle.
#
# ONE-TIME PROJECT FILE
# ---------------------
# Store the selected-model ARM profiler at:
#
# /content/drive/MyDrive/Honours_Project/ARM_Profiling/
#     comp700_arm_selected_models_v7.py

from __future__ import annotations

from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import re
import shutil
import subprocess
import sys

from google.colab import drive
import numpy as np

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
MODELS_ROOT = PROJECT_ROOT / "Models"
PROFILER_SCRIPT_PATH = PROJECT_ROOT / "ARM_Profiling" / "comp700_arm_selected_models_v7.py"
BUNDLE_OUTPUT_ROOT = PROJECT_ROOT / "ARM_Profiling_Bundles"
REQUIRE_CONFIRMATION = True

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_json(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path: Path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, allow_nan=False)

def extract_run_timestamp(path: Path):
    pattern = re.compile(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})")
    for part in reversed(path.parts):
        match = pattern.fullmatch(part)
        if match is None:
            continue
        try:
            return datetime.strptime(match.group(1), "%Y-%m-%d_%H-%M-%S").replace(tzinfo=timezone.utc)
        except ValueError:
            pass
    return datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc)

def parse_string_constant(script_text: str, name: str):
    match = re.search(rf'^{re.escape(name)}\s*=\s*"([^"]+)"', script_text, flags=re.MULTILINE)
    if match is None:
        raise RuntimeError(f"Could not read {name} from the ARM profiler script.")
    return match.group(1)

def parse_int_constant(script_text: str, name: str):
    match = re.search(rf"^{re.escape(name)}\s*=\s*(\d+)", script_text, flags=re.MULTILINE)
    if match is None:
        raise RuntimeError(f"Could not read {name} from the ARM profiler script.")
    return int(match.group(1))

def parse_float_constant(script_text: str, name: str):
    match = re.search(rf"^{re.escape(name)}\s*=\s*([0-9.]+)", script_text, flags=re.MULTILINE)
    if match is None:
        raise RuntimeError(f"Could not read {name} from the ARM profiler script.")
    return float(match.group(1))

def parse_tuple_constant(script_text: str, name: str):
    match = re.search(rf"^{re.escape(name)}\s*=\s*\(([^)]*)\)", script_text, flags=re.MULTILINE)
    if match is None:
        raise RuntimeError(f"Could not read {name} from the ARM profiler script.")
    return tuple(int(value.strip()) for value in match.group(1).split(",") if value.strip())

def resolve_record_file(
    validation_path: Path,
    record: dict,
    record_field: str,
    fallback_name: str | None = None,
    glob_pattern: str | None = None,
    expected_sha256: str | None = None,
):
    candidates = []
    stored_value = record.get(record_field)
    if stored_value:
        candidates.append(Path(stored_value))
    if fallback_name:
        candidates.append(validation_path.parent / fallback_name)
    for candidate in candidates:
        if not candidate.exists():
            continue
        if expected_sha256 and sha256_file(candidate) != expected_sha256:
            continue
        return candidate
    if glob_pattern:
        for candidate in sorted(validation_path.parent.glob(glob_pattern)):
            if expected_sha256 and sha256_file(candidate) != expected_sha256:
                continue
            return candidate
    return None

def slugify(value: str) -> str:
    value = re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()
    return value or "model"

def parse_selection(text: str, item_count: int):
    text = text.strip().lower()
    if text == "all":
        return list(range(item_count))
    selected = set()
    for token in text.split(","):
        token = token.strip()
        if not token:
            continue
        if "-" in token:
            pieces = token.split("-", 1)
            if len(pieces) != 2 or not pieces[0].isdigit() or not pieces[1].isdigit():
                raise ValueError(f"Invalid range: {token}")
            start, end = int(pieces[0]), int(pieces[1])
            if start > end:
                raise ValueError(f"Invalid descending range: {token}")
            for value in range(start, end + 1):
                selected.add(value - 1)
        else:
            if not token.isdigit():
                raise ValueError(f"Invalid selection token: {token}")
            selected.add(int(token) - 1)
    if not selected:
        raise ValueError("No models were selected.")
    invalid = [index for index in selected if index < 0 or index >= item_count]
    if invalid:
        raise ValueError("Selection contains an out-of-range item.")
    return sorted(selected)

if not PROFILER_SCRIPT_PATH.exists():
    raise FileNotFoundError(
        f"The selected-model ARM profiler is missing.\n\nExpected:\n{PROFILER_SCRIPT_PATH}\n\n"
        "Upload comp700_arm_selected_models_v7.py there first."
    )

profiler_text = PROFILER_SCRIPT_PATH.read_text(encoding="utf-8")
BUNDLE_FORMAT = parse_string_constant(profiler_text, "BUNDLE_FORMAT")
SCRIPT_VERSION = parse_string_constant(profiler_text, "SCRIPT_VERSION")
EXPECTED_STORED_TEST_SHAPE = parse_tuple_constant(profiler_text, "EXPECTED_STORED_TEST_SHAPE")
EXPECTED_TEST_ROWS = parse_int_constant(profiler_text, "EXPECTED_TEST_ROWS")
EXPECTED_FEATURES = parse_int_constant(profiler_text, "EXPECTED_FEATURES")
EXPECTED_CLASS_COUNT = parse_int_constant(profiler_text, "EXPECTED_CLASS_COUNT")
DEFAULT_PROCESS_REPETITIONS = parse_int_constant(profiler_text, "DEFAULT_PROCESS_REPETITIONS")
DEFAULT_LATENCY_REPETITIONS = parse_int_constant(profiler_text, "DEFAULT_LATENCY_REPETITIONS")
DEFAULT_WARMUP_INFERENCES = parse_int_constant(profiler_text, "DEFAULT_WARMUP_INFERENCES")
DEFAULT_CPU_THREADS = parse_int_constant(profiler_text, "DEFAULT_CPU_THREADS")
DEFAULT_MEMORY_SAMPLE_INTERVAL = parse_float_constant(profiler_text, "DEFAULT_MEMORY_SAMPLE_INTERVAL")

if not MODELS_ROOT.exists():
    raise FileNotFoundError(f"Models root does not exist:\n{MODELS_ROOT}")

validation_paths = sorted(MODELS_ROOT.rglob("*_conversion_validation.json"))
if not validation_paths:
    raise RuntimeError(
        "No conversion-validation JSON files were found.\n\n"
        "Run the notebook's 'LiteRT Model Conversion and Validation' section for the models you want to profile."
    )

candidates, rejected = [], []

for validation_path in validation_paths:
    try:
        record = read_json(validation_path)
        if not bool(record.get("validated", False)):
            continue
        if str(record.get("status", "")).lower() != "validated":
            continue
        if int(record.get("read_variable_ops_remaining", -1)) != 0:
            continue
        if not np.isclose(float(record.get("tflite_class_agreement", 0.0)), 1.0, atol=0.0, rtol=0.0):
            continue
        input_shape = tuple(int(value) for value in record.get("input_shape", []))
        if input_shape not in {(1, EXPECTED_FEATURES), (1, EXPECTED_FEATURES, 1)}:
            continue
        output_shape = tuple(int(value) for value in record.get("output_shape", []))
        if output_shape != (1, EXPECTED_CLASS_COUNT):
            continue
        tflite_sha = str(record.get("tflite_sha256", ""))
        if not tflite_sha:
            continue
        tflite_path = resolve_record_file(
            validation_path=validation_path,
            record=record,
            record_field="tflite_path",
            glob_pattern="*_edge_static_batch1_fp32_frozen.tflite",
            expected_sha256=tflite_sha,
        )
        if tflite_path is None:
            continue
        companion_sha = record.get("companion_sha256", {}) or {}
        x_sha = companion_sha.get("test_inputs")
        y_sha = companion_sha.get("test_labels")
        classes_sha = companion_sha.get("device_classes")

        x_path = resolve_record_file(
            validation_path=validation_path,
            record=record,
            record_field="test_inputs_path",
            fallback_name="X_test_3d.npy",
            expected_sha256=x_sha,
        )
        y_path = resolve_record_file(
            validation_path=validation_path,
            record=record,
            record_field="test_labels_path",
            fallback_name="y_test_1d.npy",
            expected_sha256=y_sha,
        )
        classes_path = resolve_record_file(
            validation_path=validation_path,
            record=record,
            record_field="device_classes_path",
            fallback_name="device_classes.npy",
            expected_sha256=classes_sha,
        )

        if x_path is None or y_path is None or classes_path is None:
            continue

        X_test = np.load(x_path, mmap_mode="r")
        y_test = np.load(y_path, mmap_mode="r")
        classes = np.load(classes_path, allow_pickle=True)
        try:
            if tuple(X_test.shape) != EXPECTED_STORED_TEST_SHAPE:
                continue
            if tuple(y_test.shape) != (EXPECTED_TEST_ROWS,):
                continue
            if tuple(classes.shape) != (EXPECTED_CLASS_COUNT,):
                continue
        finally:
            del X_test
            del y_test
            del classes

        actual_x_sha = sha256_file(x_path)
        actual_y_sha = sha256_file(y_path)
        actual_classes_sha = sha256_file(classes_path)

        if x_sha and actual_x_sha != x_sha:
            continue
        if y_sha and actual_y_sha != y_sha:
            continue
        if classes_sha and actual_classes_sha != classes_sha:
            continue

        source_model_path = Path(str(record.get("source_model_path", validation_path.stem)))
        source_stem = source_model_path.stem if source_model_path.stem else validation_path.stem
        model_group = str(record.get("model_group", record.get("model_kind", "Model"))).strip()
        model_kind = str(record.get("model_kind", model_group)).strip()

        candidates.append({
            "validation_path": validation_path,
            "record": record,
            "model_group": model_group,
            "model_kind": model_kind,
            "source_stem": source_stem,
            "tflite_path": tflite_path,
            "x_path": x_path,
            "y_path": y_path,
            "classes_path": classes_path,
            "x_sha256": actual_x_sha,
            "y_sha256": actual_y_sha,
            "classes_sha256": actual_classes_sha,
            "tflite_sha256": sha256_file(tflite_path),
            "run_timestamp": extract_run_timestamp(validation_path.parent),
        })
    except Exception as exc:
        rejected.append({"path": str(validation_path), "reason": str(exc)})

if not candidates:
    raise RuntimeError("No usable converter-validated LiteRT artifacts were found.")

candidates = sorted(
    candidates,
    key=lambda item: (item["run_timestamp"], item["model_group"], item["source_stem"]),
    reverse=True,
)

print("=" * 120)
print("AVAILABLE CONVERTER-VALIDATED LiteRT MODELS")
print("=" * 120)

for index, item in enumerate(candidates, start=1):
    record = item["record"]
    print(f"[{index:>3}] {item['model_group']:<22} {item['source_stem']}")
    print(f"      Run       : {item['validation_path'].parent}")
    print(f"      Accuracy  : {float(record['tflite_accuracy']) * 100.0:.3f}%")
    print(f"      Macro-F1  : {float(record['tflite_macro_f1']):.4f}")
    print(f"      LiteRT    : {item['tflite_path'].name}")

print("\nSelection examples:\n  all       -> include every validated artifact shown above\n  1         -> include one model\n  1,3,5     -> include specific models\n  2-5       -> include a range\n  1,3-5,8   -> combine individual numbers and ranges")

selection_text = input("\nModels to include in the ARM bundle: ")
try:
    selected_indices = parse_selection(selection_text, len(candidates))
except ValueError as exc:
    raise RuntimeError(f"Invalid model selection: {exc}")

selected = [candidates[index] for index in selected_indices]

x_hashes = {item["x_sha256"] for item in selected}
y_hashes = {item["y_sha256"] for item in selected}
classes_hashes = {item["classes_sha256"] for item in selected}

if len(x_hashes) != 1:
    raise RuntimeError("The selected models do not use the same X_test_3d.npy.\n\nChoose models produced from the same held-out split.")
if len(y_hashes) != 1:
    raise RuntimeError("The selected models do not use the same y_test_1d.npy.\n\nChoose models produced from the same held-out split.")
if len(classes_hashes) != 1:
    raise RuntimeError("The selected models do not use the same device_classes.npy.")

print("\n" + "=" * 120)
print("MODELS SELECTED FOR ARM64 PROFILING BUNDLE")
print("=" * 120)

for position, item in enumerate(selected, start=1):
    record = item["record"]
    print(f"[{position}] {item['model_group']} — {item['source_stem']}")
    print(f"    {item['tflite_path']}")
    print(f"    Accuracy: {float(record['tflite_accuracy']) * 100.0:.3f}% | Macro-F1: {float(record['tflite_macro_f1']):.4f}")

print(f"\nSelected model count: {len(selected)}")

if REQUIRE_CONFIRMATION:
    confirmation = input("\nType BUILD to create this bundle: ").strip()
    if confirmation != "BUILD":
        raise RuntimeError("Bundle creation cancelled.")

build_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_UTC")
output_run_dir = BUNDLE_OUTPUT_ROOT / build_timestamp
output_run_dir.mkdir(parents=True, exist_ok=False)

working_root = Path("/content") / f"_COMP700_ARM_SELECTED_BUILD_{build_timestamp}"
bundle_dir = working_root / BUNDLE_FORMAT

if working_root.exists():
    shutil.rmtree(working_root)

(bundle_dir / "models").mkdir(parents=True, exist_ok=True)
(bundle_dir / "validation").mkdir(parents=True, exist_ok=True)
(bundle_dir / "data").mkdir(parents=True, exist_ok=True)

bundle_profiler_path = bundle_dir / "comp700_arm_selected_models_v7.py"
shutil.copy2(PROFILER_SCRIPT_PATH, bundle_profiler_path)
(bundle_dir / "requirements.txt").write_text("numpy\npsutil\nai-edge-litert\n", encoding="utf-8")

first_item = selected[0]
bundle_x_path = bundle_dir / "data" / "X_test_3d.npy"
bundle_y_path = bundle_dir / "data" / "y_test_1d.npy"
bundle_classes_path = bundle_dir / "data" / "device_classes.npy"

shutil.copy2(first_item["x_path"], bundle_x_path)
shutil.copy2(first_item["y_path"], bundle_y_path)
shutil.copy2(first_item["classes_path"], bundle_classes_path)

manifest_models = []
used_keys = set()
group_counts = {}

for item in selected:
    group_counts[item["model_group"]] = group_counts.get(item["model_group"], 0) + 1

for item in selected:
    record = item["record"]
    key_base = slugify(item["source_stem"])
    key = key_base
    if key in used_keys:
        key = f"{key_base}_{item['tflite_sha256'][:8]}"
    used_keys.add(key)

    display_name = (
        item["model_group"]
        if group_counts[item["model_group"]] == 1
        else f"{item['model_group']} — {item['source_stem']}"
    )

    target_model_path = bundle_dir / "models" / f"{key}.tflite"
    target_validation_path = bundle_dir / "validation" / f"{key}_conversion_validation.json"

    shutil.copy2(item["tflite_path"], target_model_path)
    shutil.copy2(item["validation_path"], target_validation_path)

    try:
        source_run = str(item["validation_path"].parent.relative_to(MODELS_ROOT))
    except ValueError:
        source_run = str(item["validation_path"].parent)

    manifest_models.append({
        "key": key,
        "display_name": display_name,
        "model_group": item["model_group"],
        "source_run": source_run,
        "model_file": str(Path("models") / f"{key}.tflite"),
        "validation_file": str(Path("validation") / f"{key}_conversion_validation.json"),
        "model_sha256": sha256_file(target_model_path),
        "validation_sha256": sha256_file(target_validation_path),
        "input_shape": [int(value) for value in record["input_shape"]],
        "output_shape": [int(value) for value in record["output_shape"]],
        "parameters": int(record["parameters"]),
        "tflite_size_bytes": int(target_model_path.stat().st_size),
        "expected_accuracy": float(record["tflite_accuracy"]),
        "expected_macro_f1": float(record["tflite_macro_f1"]),
        "source_model_sha256": str(record["source_model_sha256"]),
        "tflite_class_agreement": float(record["tflite_class_agreement"]),
        "tflite_max_probability_difference": float(record["tflite_max_probability_difference"]),
        "read_variable_ops_remaining": int(record["read_variable_ops_remaining"]),
    })

manifest = {
    "format": BUNDLE_FORMAT,
    "pipeline_script_version": SCRIPT_VERSION,
    "description": "User-selected converter-validated neural LiteRT artifacts assembled from COMP700 notebook outputs for controlled ARM64 profiling.",
    "selected_model_count": int(len(manifest_models)),
    "data": {
        "x_test_file": "data/X_test_3d.npy",
        "y_test_file": "data/y_test_1d.npy",
        "device_classes_file": "data/device_classes.npy",
        "x_test_sha256": sha256_file(bundle_x_path),
        "y_test_sha256": sha256_file(bundle_y_path),
        "device_classes_sha256": sha256_file(bundle_classes_path),
        "stored_x_test_shape": list(EXPECTED_STORED_TEST_SHAPE),
        "expected_class_count": int(EXPECTED_CLASS_COUNT),
    },
    "benchmark_defaults": {
        "process_repetitions": int(DEFAULT_PROCESS_REPETITIONS),
        "latency_repetitions": int(DEFAULT_LATENCY_REPETITIONS),
        "warmup_inferences_per_phase": int(DEFAULT_WARMUP_INFERENCES),
        "cpu_threads": int(DEFAULT_CPU_THREADS),
        "cpu_affinity": "one permitted logical CPU",
        "memory_sample_interval_seconds": float(DEFAULT_MEMORY_SAMPLE_INTERVAL),
        "batch_size": 1,
        "precision": "FP32",
        "latency_scope": "prepared model input -> set_tensor + invoke + get_tensor",
        "sample_preparation_included_in_latency": False,
        "memory_monitor_active_during_latency": False,
    },
    "models": manifest_models,
}

write_json(bundle_dir / "manifest.json", manifest)

start_here_text = f"""COMP700 ARM64 SELECTED-MODEL PROFILING
=======================================

This bundle contains {len(manifest_models)} user-selected converter-validated
LiteRT model artifact(s).

On the ARM64 Linux server:

1. Unzip the bundle and enter its directory.

2. Run the controlled ARM64 profile:

   python3 comp700_arm_selected_models_v7.py profile --auto-install --require-arm64

The profiler creates this structure unchanged:

results/
├── arm_edge_profile_summary.csv
├── arm_edge_profile_all.json
├── environment.json
└── raw/

After profiling, keep the complete results/ directory unchanged.

For Google Drive archiving, open results/environment.json, read captured_at,
create a folder named YYYY-MM-DD_HH-MM-SS from that timestamp, and upload
the complete results/ directory under:

Honours_Project/Models/Edge_Profiling_Summaries/ARM64/
    YYYY-MM-DD_HH-MM-SS/results/
"""

(bundle_dir / "START_HERE.txt").write_text(start_here_text, encoding="utf-8")

inspection = subprocess.run(
    [sys.executable, str(bundle_profiler_path), "inspect", "--bundle-root", str(bundle_dir)],
    capture_output=True,
    text=True,
)

print("\n" + "=" * 120)
print("GENERATED BUNDLE INTEGRITY CHECK")
print("=" * 120)
print(inspection.stdout)

if inspection.returncode != 0:
    print(inspection.stderr)
    raise RuntimeError("The generated bundle failed its own inspect command. The ZIP was not created.")

zip_base = output_run_dir / BUNDLE_FORMAT
zip_path_string = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(working_root),
    base_dir=BUNDLE_FORMAT,
)
zip_path = Path(zip_path_string)

if len(list((bundle_dir / "models").glob("*.tflite"))) != len(manifest_models):
    raise RuntimeError("Generated bundle model count does not match the selection.")

if len(list((bundle_dir / "validation").glob("*_conversion_validation.json"))) != len(manifest_models):
    raise RuntimeError("Generated validation-record count does not match the selection.")

shutil.rmtree(working_root, ignore_errors=True)

print("\n" + "=" * 120)
print("USER-SELECTED ARM64 PROFILING BUNDLE CREATED SUCCESSFULLY")
print("=" * 120)
print(f"\nModels included: {len(manifest_models)}")
for model in manifest_models:
    print(f"  - {model['display_name']}")
print(f"\nSaved ZIP:\n{zip_path}")
print("\nNext step:\nCopy the ZIP to an ARM64 Linux server and follow the ARM64 Profiling Instructions markdown cell in the notebook.")

### Cross-Platform LiteRT Edge-Profiling Summary

This cell combines one user-selected x86_64 profiling summary with one user-selected ARM64 profiling summary.

It searches:

```text
Honours_Project/Models/Edge_Profiling_Summaries/
```

for x86_64 summaries named:

```text
edge_profile_summary_<timestamp>.csv
```

It searches:

```text
Honours_Project/Models/Edge_Profiling_Summaries/ARM64/
```

for archived ARM64 results at:

```text
<timestamp>/results/arm_edge_profile_summary.csv
```

The newest compatible artifact is the default, but the user must verify that the selected summaries correspond to the intended model generation.

Both summaries must contain exactly one result for each of:

- CNN;
- Feature-Attention MLP;
- LSTM;
- BiLSTM; and
- CNN-BiLSTM.

The cell validates the platform architecture, model set, process-repetition count, inference-thread count and finite latency and memory measurements before combining the sources.

The resulting two-panel figure compares:

1. median prepared-input batch-one latency; and
2. total peak USS overhead above the worker baseline.

The latency boundary is:

```text
set_tensor → invoke → get_tensor
```

It excludes packet capture, flow reconstruction, feature extraction and preprocessing before the prepared model input.

Outputs are saved as:

```text
Honours_Project/Models/Edge_Profiling_Summaries/
├── Cross_Platform_Edge_Profile.png
├── Cross_Platform_Edge_Profile.pdf
├── Cross_Platform_Edge_Profile_Data.csv
└── Cross_Platform_Edge_Profile_Provenance.json
```

The provenance file records the exact selected x86_64 summary, ARM64 summary and ARM environment metadata.

In [ ]:
import json
import re
from pathlib import Path

from google.colab import drive
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
MODELS_ROOT = PROJECT_ROOT / "Models"
EDGE_SUMMARY_ROOT = MODELS_ROOT / "Edge_Profiling_Summaries"

if not EDGE_SUMMARY_ROOT.exists():
    raise FileNotFoundError(
        f"Edge-profiling summary directory does not exist:\n"
        f"{EDGE_SUMMARY_ROOT}"
    )


def select_artifact(label, candidates):
    candidates = sorted(
        {
            Path(path).resolve()
            for path in candidates
            if Path(path).is_file()
        },
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError(
            f"No compatible {label} artifact was found."
        )

    print("\n" + "=" * 100)
    print(f"SELECT {label.upper()}")
    print("=" * 100)

    for number, path in enumerate(candidates, start=1):
        print(f"[{number}] {path}")

    while True:
        response = input(
            f"Select 1-{len(candidates)} "
            f"(Enter = newest): "
        ).strip()

        if response == "":
            return candidates[0]

        if response.isdigit():
            selection = int(response)

            if 1 <= selection <= len(candidates):
                return candidates[selection - 1]

        print("Invalid selection.")


X86_SUMMARY_PATH = select_artifact(
    "x86_64 edge-profile summary",
    EDGE_SUMMARY_ROOT.glob(
        "edge_profile_summary_*.csv"
    ),
)

ARM_SUMMARY_PATH = select_artifact(
    "ARM64 edge-profile summary",
    (
        EDGE_SUMMARY_ROOT
        / "ARM64"
    ).glob(
        "*/results/arm_edge_profile_summary.csv"
    ),
)

ARM_RESULTS_DIR = ARM_SUMMARY_PATH.parent
ARM_RUN_DIR = ARM_RESULTS_DIR.parent
ARM_ENVIRONMENT_PATH = (
    ARM_RESULTS_DIR
    / "environment.json"
)

PNG_PATH = (
    EDGE_SUMMARY_ROOT
    / "Cross_Platform_Edge_Profile.png"
)

PDF_PATH = (
    EDGE_SUMMARY_ROOT
    / "Cross_Platform_Edge_Profile.pdf"
)

FINAL_DATA_PATH = (
    EDGE_SUMMARY_ROOT
    / "Cross_Platform_Edge_Profile_Data.csv"
)

PROVENANCE_PATH = (
    EDGE_SUMMARY_ROOT
    / "Cross_Platform_Edge_Profile_Provenance.json"
)

MODEL_ORDER = ["CNN", "Feature-Attention MLP", "LSTM", "BiLSTM", "CNN-BiLSTM"]
MODEL_DISPLAY_LABELS = ["CNN", "Feature-Attention\nMLP", "LSTM", "BiLSTM", "CNN-BiLSTM"]

def canonical_model(row):
    possible_fields = ["model_key", "display_name", "model_group", "source_stem"]
    text = " ".join(str(row.get(field, "")) for field in possible_fields).lower()
    normalised = text.replace("_", "-").replace(" ", "-")

    if (
        "cnn-bilstm" in normalised
        or "cnn-bilstm-50" in normalised
        or "attention-cnn-bilstm" in normalised
        or "cnn-bilstm-final" in normalised
        or ("cnn" in normalised and "bilstm" in normalised)
    ):
        return "CNN-BiLSTM"

    if (
        "feature-attention" in normalised
        or "mlp-50" in normalised
        or re.search(r"(^|-)mlp($|-)", normalised)
    ):
        return "Feature-Attention MLP"

    if "bilstm" in normalised or "bidirectional-lstm" in normalised:
        return "BiLSTM"

    if "lstm" in normalised:
        return "LSTM"

    if "cnn" in normalised:
        return "CNN"

    return None

def canonical_machine(text):
    text = str(text).lower()
    arm_tokens = ["aarch64", "arm64", "armv8", "neoverse"]
    x86_tokens = ["x86_64", "amd64", "x86-64"]

    if any(token in text for token in arm_tokens):
        return "ARM64"
    if any(token in text for token in x86_tokens):
        return "x86_64"
    return None

required_authoritative_paths = [
    X86_SUMMARY_PATH,
    ARM_SUMMARY_PATH,
    ARM_ENVIRONMENT_PATH,
]

missing_authoritative_paths = [
    path for path in required_authoritative_paths if not path.exists()
]

if missing_authoritative_paths:
    raise FileNotFoundError(
        "One or more authoritative edge-profiling artifacts are missing:\n\n"
        + "\n".join(f"  - {path}" for path in missing_authoritative_paths)
        + "\n\nThese paths are deliberately fixed because they correspond to "
        "the profiling results reported in the thesis."
    )

print("=" * 100)
print("AUTHORITATIVE CROSS-PLATFORM PROFILING ARTIFACTS")
print("=" * 100)
print(f"\nx86_64 summary:\n{X86_SUMMARY_PATH}")
print(f"\nARM64 summary:\n{ARM_SUMMARY_PATH}")
print(f"\nARM64 environment:\n{ARM_ENVIRONMENT_PATH}")

arm_raw = pd.read_csv(ARM_SUMMARY_PATH)
required_arm_columns = {
    "model_group",
    "model_key",
    "display_name",
    "median_latency_ms",
    "total_peak_uss_overhead_mib",
    "process_repetitions",
    "cpu_threads",
}

missing_arm_columns = required_arm_columns - set(arm_raw.columns)
if missing_arm_columns:
    raise RuntimeError(
        "The authoritative ARM64 summary does not contain the expected columns:\n"
        + "\n".join(f"  - {column}" for column in sorted(missing_arm_columns))
    )

with open(ARM_ENVIRONMENT_PATH, "r", encoding="utf-8") as handle:
    arm_environment = json.load(handle)

arm_machine_text = " ".join([
    str(arm_environment.get("machine", "")),
    str(arm_environment.get("platform", "")),
    str(arm_environment.get("processor", "")),
    str(arm_environment.get("lscpu", "")),
])

if canonical_machine(arm_machine_text) != "ARM64":
    raise RuntimeError(
        "The authoritative ARM environment metadata does not identify an ARM64/aarch64 system."
    )

arm_raw = arm_raw.copy()
arm_raw["Canonical_Model"] = arm_raw.apply(canonical_model, axis=1)

unrecognised_arm = arm_raw[arm_raw["Canonical_Model"].isna()]
if not unrecognised_arm.empty:
    print("\nUnrecognised ARM64 rows:")
    display(unrecognised_arm)
    raise RuntimeError(
        "One or more authoritative ARM64 profiling rows could not be mapped to a retained model."
    )

arm_counts = arm_raw["Canonical_Model"].value_counts()
for model_name in MODEL_ORDER:
    count = int(arm_counts.get(model_name, 0))
    if count != 1:
        raise RuntimeError(
            f"The authoritative ARM64 run must contain exactly one result for {model_name}; found {count}."
        )

if len(arm_raw) != len(MODEL_ORDER):
    raise RuntimeError(
        f"The authoritative ARM64 summary contains an unexpected number of rows. Expected {len(MODEL_ORDER)}; found {len(arm_raw)}."
    )

arm_frame = arm_raw.set_index("Canonical_Model").loc[MODEL_ORDER].reset_index()
arm_standard = pd.DataFrame({
    "Platform": "ARM64",
    "Model": arm_frame["Canonical_Model"],
    "Median_Latency_ms": pd.to_numeric(arm_frame["median_latency_ms"], errors="raise"),
    "Total_Peak_USS_Overhead_MiB": pd.to_numeric(arm_frame["total_peak_uss_overhead_mib"], errors="raise"),
    "Process_Repetitions": pd.to_numeric(arm_frame["process_repetitions"], errors="raise"),
    "CPU_Threads": pd.to_numeric(arm_frame["cpu_threads"], errors="raise"),
    "Source_Model_Group": arm_frame["model_group"].astype(str),
    "Source_Artifact": arm_frame["model_key"].astype(str),
    "Source_Summary_CSV": str(ARM_SUMMARY_PATH),
})

print("\n" + "=" * 100)
print("AUTHORITATIVE ARM64 PROFILING RESULT VERIFIED")
print("=" * 100)
display(arm_standard)

x86_raw = pd.read_csv(X86_SUMMARY_PATH)
required_x86_columns = {
    "model_group",
    "source_stem",
    "machine",
    "platform",
    "median_latency_ms",
    "total_peak_uss_overhead_mb",
    "process_repetitions",
    "cpu_threads",
}

missing_x86_columns = required_x86_columns - set(x86_raw.columns)
if missing_x86_columns:
    raise RuntimeError(
        "The authoritative x86_64 summary does not contain the expected columns:\n"
        + "\n".join(f"  - {column}" for column in sorted(missing_x86_columns))
    )

x86_raw = x86_raw.copy()
x86_raw["Canonical_Model"] = x86_raw.apply(canonical_model, axis=1)
x86_raw["Canonical_Machine"] = x86_raw.apply(
    lambda row: canonical_machine(
        " ".join([
            str(row.get("machine", "")),
            str(row.get("platform", "")),
            str(row.get("processor", "")),
        ])
    ),
    axis=1,
)

unrecognised_x86_models = x86_raw[x86_raw["Canonical_Model"].isna()]
if not unrecognised_x86_models.empty:
    print("\nUnrecognised x86_64 model rows:")
    display(unrecognised_x86_models)
    raise RuntimeError(
        "One or more authoritative x86_64 profiling rows could not be mapped to a retained model."
    )

non_x86_rows = x86_raw[x86_raw["Canonical_Machine"] != "x86_64"]
if not non_x86_rows.empty:
    print("\nRows not identified as x86_64:")
    display(non_x86_rows)
    raise RuntimeError(
        "One or more rows in the authoritative x86_64 summary do not identify an x86_64 platform."
    )

x86_counts = x86_raw["Canonical_Model"].value_counts()
for model_name in MODEL_ORDER:
    count = int(x86_counts.get(model_name, 0))
    if count != 1:
        raise RuntimeError(
            f"The authoritative x86_64 run must contain exactly one result for {model_name}; found {count}."
        )

if len(x86_raw) != len(MODEL_ORDER):
    raise RuntimeError(
        f"The authoritative x86_64 summary contains an unexpected number of rows. Expected {len(MODEL_ORDER)}; found {len(x86_raw)}."
    )

x86_frame = x86_raw.set_index("Canonical_Model").loc[MODEL_ORDER].reset_index()
x86_standard = pd.DataFrame({
    "Platform": "x86_64",
    "Model": x86_frame["Canonical_Model"],
    "Median_Latency_ms": pd.to_numeric(x86_frame["median_latency_ms"], errors="raise"),
    "Total_Peak_USS_Overhead_MiB": pd.to_numeric(x86_frame["total_peak_uss_overhead_mb"], errors="raise"),
    "Process_Repetitions": pd.to_numeric(x86_frame["process_repetitions"], errors="raise"),
    "CPU_Threads": pd.to_numeric(x86_frame["cpu_threads"], errors="raise"),
    "Source_Model_Group": x86_frame["model_group"].astype(str),
    "Source_Artifact": x86_frame["source_stem"].astype(str),
    "Source_Summary_CSV": str(X86_SUMMARY_PATH),
})

print("\n" + "=" * 100)
print("AUTHORITATIVE x86_64 PROFILING RESULT VERIFIED")
print("=" * 100)
display(x86_standard)

combined = pd.concat([x86_standard, arm_standard], ignore_index=True)
expected_pairs = {
    (platform, model)
    for platform in ["x86_64", "ARM64"]
    for model in MODEL_ORDER
}
observed_pairs = {
    (row["Platform"], row["Model"])
    for _, row in combined.iterrows()
}

if observed_pairs != expected_pairs:
    missing_pairs = sorted(expected_pairs - observed_pairs)
    unexpected_pairs = sorted(observed_pairs - expected_pairs)
    raise RuntimeError(
        f"Cross-platform result integrity check failed.\n\nMissing pairs: {missing_pairs}\nUnexpected pairs: {unexpected_pairs}"
    )

for column in ["Median_Latency_ms", "Total_Peak_USS_Overhead_MiB"]:
    values = combined[column].to_numpy(dtype=np.float64)
    if not np.all(np.isfinite(values)):
        raise RuntimeError(f"{column} contains NaN or Inf.")

if (combined["Median_Latency_ms"] <= 0).any():
    raise RuntimeError("Median latency must be positive.")

if (combined["Total_Peak_USS_Overhead_MiB"] < 0).any():
    raise RuntimeError("USS overhead cannot be negative.")

process_repetition_values = sorted(
    combined["Process_Repetitions"].dropna().astype(int).unique().tolist()
)
cpu_thread_values = sorted(
    combined["CPU_Threads"].dropna().astype(int).unique().tolist()
)

if len(process_repetition_values) != 1:
    raise RuntimeError(
        f"x86_64 and ARM64 results do not use the same number of fresh-process repetitions:\n{process_repetition_values}"
    )

if len(cpu_thread_values) != 1:
    raise RuntimeError(
        f"x86_64 and ARM64 results do not use the same LiteRT thread count:\n{cpu_thread_values}"
    )

PROCESS_REPETITIONS = process_repetition_values[0]
CPU_THREADS = cpu_thread_values[0]

model_order_map = {model: index for index, model in enumerate(MODEL_ORDER)}
platform_order_map = {"x86_64": 0, "ARM64": 1}

combined["_model_order"] = combined["Model"].map(model_order_map)
combined["_platform_order"] = combined["Platform"].map(platform_order_map)
combined = combined.sort_values(["_platform_order", "_model_order"]).reset_index(drop=True)

figure_data = combined.drop(columns=["_model_order", "_platform_order"])
figure_data.to_csv(FINAL_DATA_PATH, index=False)

print("\n" + "=" * 100)
print("MEASUREMENTS USED FOR FINAL FIGURE")
print("=" * 100)
display(figure_data[["Platform", "Model", "Median_Latency_ms", "Total_Peak_USS_Overhead_MiB"]])

def metric_vector(platform, metric):
    frame = (
        combined[combined["Platform"] == platform]
        .set_index("Model")
        .loc[MODEL_ORDER]
    )
    return frame[metric].to_numpy(dtype=np.float64)

x86_median_latency = metric_vector("x86_64", "Median_Latency_ms")
arm_median_latency = metric_vector("ARM64", "Median_Latency_ms")
x86_peak_uss = metric_vector("x86_64", "Total_Peak_USS_Overhead_MiB")
arm_peak_uss = metric_vector("ARM64", "Total_Peak_USS_Overhead_MiB")

x = np.arange(len(MODEL_ORDER))
bar_width = 0.36
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

ax = axes[0]
bars_x86_latency = ax.bar(x - bar_width / 2, x86_median_latency, width=bar_width, label="x86_64")
bars_arm_latency = ax.bar(x + bar_width / 2, arm_median_latency, width=bar_width, label="ARM64")
ax.set_yscale("log")
ax.set_title("(a) Median Batch-One Inference Latency", fontweight="bold")
ax.set_ylabel("Median latency (ms, log scale)")
ax.set_xticks(x)
ax.set_xticklabels(MODEL_DISPLAY_LABELS, rotation=20, ha="right")
ax.grid(axis="y", linestyle="--", alpha=0.4, which="both")
ax.legend(frameon=False)

for bars in [bars_x86_latency, bars_arm_latency]:
    for bar in bars:
        value = bar.get_height()
        ax.annotate(
            f"{value:.4f}",
            xy=(bar.get_x() + bar.get_width() / 2, value),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90,
        )

ax = axes[1]
bars_x86_uss = ax.bar(x - bar_width / 2, x86_peak_uss, width=bar_width, label="x86_64")
bars_arm_uss = ax.bar(x + bar_width / 2, arm_peak_uss, width=bar_width, label="ARM64")
ax.set_title("(b) Total Peak USS Memory Overhead", fontweight="bold")
ax.set_ylabel("Peak USS overhead (MiB)")
ax.set_xticks(x)
ax.set_xticklabels(MODEL_DISPLAY_LABELS, rotation=20, ha="right")
maximum_uss = float(max(np.max(x86_peak_uss), np.max(arm_peak_uss)))
ax.set_ylim(0, maximum_uss * 1.16)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(frameon=False)

for bars in [bars_x86_uss, bars_arm_uss]:
    for bar in bars:
        value = bar.get_height()
        ax.annotate(
            f"{value:.3f}",
            xy=(bar.get_x() + bar.get_width() / 2, value),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90,
        )

thread_text = (
    f"{CPU_THREADS} inference thread"
    if CPU_THREADS == 1
    else f"{CPU_THREADS} inference threads"
)
figure_note = (
    f"Prepared-input FP32 LiteRT inference; batch size 1; CPU only; {thread_text}. "
    f"Values are means across {PROCESS_REPETITIONS} fresh-process repetitions."
)
fig.text(0.5, 0.01, figure_note, ha="center", fontsize=9)
fig.tight_layout(rect=[0.02, 0.07, 0.99, 1.00])

fig.savefig(PNG_PATH, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(PDF_PATH, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)

provenance = {
    "selection_policy": "fixed_authoritative_thesis_artifacts",
    "authoritative_runs": {
        "x86_64": {
            "summary_csv": str(X86_SUMMARY_PATH),
        },
        "ARM64": {
            "run_directory": str(ARM_RUN_DIR),
            "results_directory": str(ARM_RESULTS_DIR),
            "summary_csv": str(ARM_SUMMARY_PATH),
            "environment_json": str(ARM_ENVIRONMENT_PATH),
        },
    },
    "arm_environment": {
        "captured_at": arm_environment.get("captured_at"),
        "hostname": arm_environment.get("hostname"),
        "platform": arm_environment.get("platform"),
        "machine": arm_environment.get("machine"),
        "processor": arm_environment.get("processor"),
        "python_version": arm_environment.get("python_version"),
        "ai_edge_litert_version": arm_environment.get("ai_edge_litert_version"),
        "selected_profile_cpu": arm_environment.get("selected_profile_cpu"),
    },
    "models": MODEL_ORDER,
    "process_repetitions": int(PROCESS_REPETITIONS),
    "cpu_threads": int(CPU_THREADS),
    "figure_png": str(PNG_PATH),
    "figure_pdf": str(PDF_PATH),
    "figure_data_csv": str(FINAL_DATA_PATH),
}

with open(PROVENANCE_PATH, "w", encoding="utf-8") as handle:
    json.dump(provenance, handle, indent=2)

print("\n" + "=" * 100)
print("CROSS-PLATFORM EDGE-PROFILING FIGURE COMPLETE")
print("=" * 100)
print("\nSelection policy:\nFixed authoritative thesis artifacts")
print(f"\nAuthoritative x86_64 source:\n{X86_SUMMARY_PATH}")
print(f"\nAuthoritative ARM64 run:\n{ARM_RUN_DIR}")
print(f"\nAuthoritative ARM64 source:\n{ARM_SUMMARY_PATH}")
print(f"\nARM64 environment:\n{ARM_ENVIRONMENT_PATH}")
print(f"\nSaved figure data:\n{FINAL_DATA_PATH}")
print(f"\nSaved provenance record:\n{PROVENANCE_PATH}")
print(f"\nSaved PNG:\n{PNG_PATH}")
print(f"\nSaved PDF:\n{PDF_PATH}")

### Predictive-Deployment Trade-off Figure

This cell combines predictive performance with deployment measurements for the five validation-selected neural models:

- CNN;
- Feature-Attention MLP;
- LSTM;
- BiLSTM; and
- CNN-BiLSTM.

The user selects:

1. a cross-platform profiling dataset;
2. one held-out evaluation result for each model; and
3. one corresponding LiteRT profile for each model.

The selected cross-platform dataset must contain exactly ten rows: five models on x86_64 and the same five models on ARM64.

The figure contains separate x86_64 and ARM64 panels. Each point represents one model:

- horizontal position: median prepared-input LiteRT latency on a logarithmic scale;
- vertical position: held-out accuracy of the selected validation-selected model; and
- marker area: FP32 LiteRT file size.

The Random Forest is excluded because this figure compares LiteRT neural deployment artifacts.

Outputs are written to:

```text
Honours_Project/Thesis_Figures/
├── predictive_deployment_tradeoff.png
├── predictive_deployment_tradeoff.pdf
├── predictive_deployment_tradeoff_data.csv
└── predictive_deployment_tradeoff_provenance.json
```

Repeated held-out summary rows referring to the same underlying model artifact are collapsed during discovery, with canonical model-specific `metrics.csv` files preferred where available.

The evaluation, LiteRT-profile and cross-platform sources are nevertheless selected independently. The cell verifies the expected models, platforms, row counts and numerical validity, but it does not prove that all three selected sources came from the same training generation. Inspect the displayed run names and paths before accepting each selection. The provenance JSON preserves every selected source so that the final figure can be audited.

In [ ]:
from pathlib import Path
import json

from google.colab import drive
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
MODELS_ROOT = PROJECT_ROOT / "Models"
OUTPUT_DIR = PROJECT_ROOT / "Thesis_Figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_REQUESTS = {
    "CNN": "best_validation_cnn",
    "Feature-Attention MLP": "best_validation_mlp",
    "LSTM": "best_validation_lstm",
    "BiLSTM": "best_validation_bilstm",
    "CNN--BiLSTM": "best_validation_cnn_bilstm",
}


def valid_text(value):
    if value is None or pd.isna(value):
        return ""
    value = str(value).strip()
    return "" if value.lower() == "nan" else value


def choose(label, candidates, description):
    if not candidates:
        raise FileNotFoundError(
            f"No {description} candidate was found for {label}."
        )

    print("\n" + "=" * 100)
    print(f"SELECT {description.upper()}: {label}")
    print("=" * 100)

    for number, candidate in enumerate(candidates, start=1):
        print(
            f"[{number}] {candidate['summary']}\n"
            f"    {candidate['path']}"
        )

    while True:
        response = input(
            f"Select 1-{len(candidates)} (Enter = newest): "
        ).strip()

        if response == "":
            return candidates[0]

        if response.isdigit() and 1 <= int(response) <= len(candidates):
            return candidates[int(response) - 1]

        print("Invalid selection.")


def evaluation_candidates(model_identifier):
    records_by_artifact = {}

    for csv_path in MODELS_ROOT.rglob("*.csv"):
        try:
            frame = pd.read_csv(csv_path)
        except Exception:
            continue

        model_column = next(
            (
                column
                for column in ["model_name", "model"]
                if column in frame.columns
            ),
            None,
        )

        required_columns = {
            "accuracy",
        }

        if model_column is None or not required_columns.issubset(frame.columns):
            continue

        matching_rows = frame.loc[
            frame[model_column].astype(str).str.strip() == model_identifier
        ]

        for _, row in matching_rows.iterrows():
            if (
                "test_rows" in row.index
                and pd.notna(row["test_rows"])
                and int(row["test_rows"]) != 376
            ):
                continue

            accuracy = float(row["accuracy"])
            accuracy_percent = accuracy * 100 if accuracy <= 1 else accuracy

            model_source = valid_text(row.get("model_source"))
            run_directory = valid_text(row.get("run_directory"))
            evaluated_at = valid_text(row.get("evaluated_at"))

            if model_source:
                artifact_identity = f"model_source::{model_source}"
                source_path = Path(model_source)
                model_filename = source_path.name

                if run_directory:
                    run_name = Path(run_directory).name
                else:
                    parent_names = list(source_path.parts)
                    run_name = next(
                        (
                            name
                            for name in reversed(parent_names)
                            if name[:4].isdigit() and "_" in name
                        ),
                        source_path.parent.name,
                    )

            elif run_directory:
                artifact_identity = (
                    f"run_directory::{run_directory}::{model_identifier}"
                )
                run_name = Path(run_directory).name
                model_filename = model_identifier

            else:
                artifact_identity = (
                    f"fallback::{model_identifier}::{evaluated_at}::"
                    f"{accuracy_percent:.12f}"
                )
                run_name = "unknown run"
                model_filename = model_identifier

            try:
                evaluated_timestamp = pd.to_datetime(
                    evaluated_at,
                    utc=True,
                ).timestamp()
            except Exception:
                evaluated_timestamp = csv_path.stat().st_mtime

            if csv_path.name == "metrics.csv":
                source_priority = 3
            elif csv_path.name == "test_evaluation_summary.csv":
                source_priority = 2
            else:
                source_priority = 1

            record = {
                "path": str(csv_path),
                "modified": csv_path.stat().st_mtime,
                "sort_time": evaluated_timestamp,
                "accuracy_percent": accuracy_percent,
                "model_source": model_source,
                "run_directory": run_directory,
                "evaluated_at": evaluated_at,
                "source_priority": source_priority,
                "summary": (
                    f"run={run_name} | "
                    f"held-out accuracy={accuracy_percent:.3f}% | "
                    f"model={model_filename}"
                ),
            }

            existing = records_by_artifact.get(artifact_identity)

            if existing is None:
                records_by_artifact[artifact_identity] = record
                continue

            existing_preference = (
                existing["source_priority"],
                existing["modified"],
            )

            new_preference = (
                record["source_priority"],
                record["modified"],
            )

            if new_preference > existing_preference:
                records_by_artifact[artifact_identity] = record

    return sorted(
        records_by_artifact.values(),
        key=lambda record: (
            record["sort_time"],
            record["modified"],
        ),
        reverse=True,
    )


def profile_candidates(model_identifier):
    records = []
    seen_paths = set()

    for profile_path in MODELS_ROOT.rglob(
        f"{model_identifier}_edge_profile.json"
    ):
        resolved_path = str(profile_path.resolve())

        if resolved_path in seen_paths:
            continue

        try:
            profile = json.loads(
                profile_path.read_text(encoding="utf-8")
            )
            size_bytes = int(profile["tflite_size_bytes"])
        except Exception:
            continue

        seen_paths.add(resolved_path)

        run_name = next(
            (
                part
                for part in reversed(profile_path.parts)
                if part[:4].isdigit() and "_" in part
            ),
            profile_path.parent.parent.name,
        )

        records.append(
            {
                "path": str(profile_path),
                "modified": profile_path.stat().st_mtime,
                "size_kib": size_bytes / 1024,
                "summary": (
                    f"run={run_name} | "
                    f"LiteRT size={size_bytes / 1024:.1f} KiB"
                ),
            }
        )

    return sorted(
        records,
        key=lambda record: record["modified"],
        reverse=True,
    )


def cross_platform_candidates():
    records = []
    seen_paths = set()

    for csv_path in MODELS_ROOT.rglob(
        "Cross_Platform_Edge_Profile_Data.csv"
    ):
        resolved_path = str(csv_path.resolve())

        if resolved_path in seen_paths:
            continue

        try:
            frame = pd.read_csv(csv_path)
        except Exception:
            continue

        required_columns = {
            "Platform",
            "Model",
            "Median_Latency_ms",
        }

        if not required_columns.issubset(frame.columns):
            continue

        source_summaries = []

        if "Source_Summary_CSV" in frame.columns:
            source_summaries = sorted(
                frame["Source_Summary_CSV"]
                .dropna()
                .astype(str)
                .unique()
                .tolist()
            )

        source_description = ""

        if source_summaries:
            source_description = (
                " | sources="
                + ", ".join(
                    Path(source).name
                    for source in source_summaries
                )
            )

        seen_paths.add(resolved_path)

        records.append(
            {
                "path": str(csv_path),
                "modified": csv_path.stat().st_mtime,
                "summary": (
                    f"{len(frame)} model-platform rows"
                    f"{source_description}"
                ),
            }
        )

    return sorted(
        records,
        key=lambda record: record["modified"],
        reverse=True,
    )


edge_source = choose(
    "Cross-platform profiling data",
    cross_platform_candidates(),
    "cross-platform summary",
)

selections = {}

for display_name, model_identifier in MODEL_REQUESTS.items():
    evaluation = choose(
        display_name,
        evaluation_candidates(model_identifier),
        "held-out evaluation",
    )

    profile = choose(
        display_name,
        profile_candidates(model_identifier),
        "LiteRT profile",
    )

    selections[display_name] = {
        "model_identifier": model_identifier,
        "accuracy_percent": evaluation["accuracy_percent"],
        "evaluation_csv": evaluation["path"],
        "model_source": evaluation["model_source"],
        "run_directory": evaluation["run_directory"],
        "evaluated_at": evaluation["evaluated_at"],
        "size_kib": profile["size_kib"],
        "profile_json": profile["path"],
    }

edge_frame = pd.read_csv(edge_source["path"])

edge_frame["Model"] = edge_frame["Model"].replace(
    {
        "CNN-BiLSTM": "CNN--BiLSTM",
    }
)

expected_models = set(MODEL_REQUESTS)
expected_platforms = {
    "x86_64",
    "ARM64",
}

actual_models = set(edge_frame["Model"])
actual_platforms = set(edge_frame["Platform"])

if actual_models != expected_models:
    raise RuntimeError(
        "Unexpected model set in the cross-platform summary.\n"
        f"Expected: {sorted(expected_models)}\n"
        f"Found: {sorted(actual_models)}"
    )

if actual_platforms != expected_platforms:
    raise RuntimeError(
        "Unexpected platform set in the cross-platform summary.\n"
        f"Expected: {sorted(expected_platforms)}\n"
        f"Found: {sorted(actual_platforms)}"
    )

if len(edge_frame) != 10:
    raise RuntimeError(
        "The cross-platform summary must contain exactly one row for "
        "each of the five models on each of the two platforms.\n"
        f"Expected 10 rows; found {len(edge_frame)}."
    )

duplicate_pairs = edge_frame.duplicated(
    subset=["Platform", "Model"],
    keep=False,
)

if duplicate_pairs.any():
    duplicates = edge_frame.loc[
        duplicate_pairs,
        ["Platform", "Model"],
    ]

    raise RuntimeError(
        "Duplicate model-platform rows were found:\n"
        f"{duplicates.to_string(index=False)}"
    )

edge_frame["Median_Latency_ms"] = pd.to_numeric(
    edge_frame["Median_Latency_ms"],
    errors="raise",
)

if not np.all(
    np.isfinite(
        edge_frame["Median_Latency_ms"].to_numpy(dtype=float)
    )
):
    raise RuntimeError(
        "The cross-platform latency column contains NaN or infinity."
    )

if (edge_frame["Median_Latency_ms"] <= 0).any():
    raise RuntimeError(
        "All median latency measurements must be positive."
    )

edge_frame["Selected_Accuracy_Percent"] = edge_frame["Model"].map(
    {
        model: values["accuracy_percent"]
        for model, values in selections.items()
    }
)

edge_frame["TFLite_Size_KiB"] = edge_frame["Model"].map(
    {
        model: values["size_kib"]
        for model, values in selections.items()
    }
)

if edge_frame[
    [
        "Selected_Accuracy_Percent",
        "TFLite_Size_KiB",
    ]
].isna().any().any():
    raise RuntimeError(
        "At least one selected model could not be mapped to its "
        "accuracy or LiteRT size."
    )

model_colours = {
    "CNN": "#4E79A7",
    "Feature-Attention MLP": "#F28E2B",
    "LSTM": "#59A14F",
    "BiLSTM": "#E15759",
    "CNN--BiLSTM": "#B07AA1",
}

annotation_layout = {
    "CNN": {
        "xytext": (8, -10),
        "horizontalalignment": "left",
        "verticalalignment": "top",
    },
    "Feature-Attention MLP": {
        "xytext": (0, 11),
        "horizontalalignment": "left",
        "verticalalignment": "bottom",
    },
    "LSTM": {
        "xytext": (-8, 10),
        "horizontalalignment": "right",
        "verticalalignment": "bottom",
    },
    "BiLSTM": {
        "xytext": (8, -10),
        "horizontalalignment": "left",
        "verticalalignment": "top",
    },
    "CNN--BiLSTM": {
        "xytext": (-8, 8),
        "horizontalalignment": "right",
        "verticalalignment": "bottom",
    },
}

minimum_size = edge_frame["TFLite_Size_KiB"].min()
maximum_size = edge_frame["TFLite_Size_KiB"].max()


def marker_area(size_kib):
    if maximum_size == minimum_size:
        return 180

    return (
        90
        + 310
        * (size_kib - minimum_size)
        / (maximum_size - minimum_size)
    )


figure, axes = plt.subplots(
    1,
    2,
    figsize=(13.2, 6.8),
    sharey=True,
)

for axis, platform in zip(
    axes,
    ["x86_64", "ARM64"],
):
    platform_frame = (
        edge_frame.loc[
            edge_frame["Platform"] == platform
        ]
        .copy()
        .sort_values("Median_Latency_ms")
    )

    if len(platform_frame) != 5:
        raise RuntimeError(
            f"Expected five rows for {platform}; "
            f"found {len(platform_frame)}."
        )

    for row in platform_frame.itertuples(index=False):
        axis.scatter(
            row.Median_Latency_ms,
            row.Selected_Accuracy_Percent,
            s=marker_area(row.TFLite_Size_KiB),
            color=model_colours[row.Model],
            edgecolor="black",
            linewidth=0.7,
            alpha=0.85,
            zorder=3,
        )

        label_settings = annotation_layout[row.Model]

        axis.annotate(
            row.Model,
            (
                row.Median_Latency_ms,
                row.Selected_Accuracy_Percent,
            ),
            xytext=label_settings["xytext"],
            textcoords="offset points",
            horizontalalignment=(
                label_settings["horizontalalignment"]
            ),
            verticalalignment=(
                label_settings["verticalalignment"]
            ),
            fontsize=8,
            annotation_clip=False,
            zorder=4,
        )

    axis.set_xscale("log")
    axis.set_xlabel(
        "Median prepared-input latency (ms, log scale)"
    )
    axis.set_title(
        platform,
        fontweight="bold",
    )
    axis.grid(
        True,
        which="both",
        linestyle="--",
        alpha=0.3,
        zorder=0,
    )

axes[0].set_ylabel(
    "Validation-selected model held-out accuracy (%)"
)

all_accuracies = edge_frame[
    "Selected_Accuracy_Percent"
].to_numpy(dtype=float)

axes[0].set_ylim(
    max(
        0,
        np.floor(all_accuracies.min()) - 3,
    ),
    min(
        100,
        np.ceil(all_accuracies.max()) + 3,
    ),
)

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        markerfacecolor=colour,
        markeredgecolor="black",
        label=model,
    )
    for model, colour in model_colours.items()
]

figure.suptitle(
    "Predictive Accuracy and LiteRT Deployment Trade-off",
    fontweight="bold",
    y=0.965,
)

figure.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=5,
    frameon=False,
    bbox_to_anchor=(0.5, 0.070),
)

figure.text(
    0.5,
    0.025,
    "Marker area is proportional to FP32 LiteRT file size.",
    ha="center",
    va="center",
    fontsize=9,
)

figure.subplots_adjust(
    left=0.075,
    right=0.985,
    top=0.855,
    bottom=0.225,
    wspace=0.14,
)

png_path = (
    OUTPUT_DIR
    / "predictive_deployment_tradeoff.png"
)

pdf_path = (
    OUTPUT_DIR
    / "predictive_deployment_tradeoff.pdf"
)

data_path = (
    OUTPUT_DIR
    / "predictive_deployment_tradeoff_data.csv"
)

provenance_path = (
    OUTPUT_DIR
    / "predictive_deployment_tradeoff_provenance.json"
)

figure.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

figure.savefig(
    pdf_path,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(figure)

edge_frame.to_csv(
    data_path,
    index=False,
)

cross_platform_sources = []

if "Source_Summary_CSV" in edge_frame.columns:
    cross_platform_sources = sorted(
        edge_frame["Source_Summary_CSV"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

provenance = {
    "cross_platform_source": edge_source["path"],
    "underlying_cross_platform_summaries": (
        cross_platform_sources
    ),
    "models": selections,
    "output_files": {
        "png": str(png_path),
        "pdf": str(pdf_path),
        "data_csv": str(data_path),
    },
}

provenance_path.write_text(
    json.dumps(
        provenance,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 100)
print("PREDICTIVE DEPLOYMENT TRADE-OFF FIGURE COMPLETE")
print("=" * 100)
print(f"Saved PNG:        {png_path}")
print(f"Saved PDF:        {pdf_path}")
print(f"Saved data:       {data_path}")
print(f"Saved provenance: {provenance_path}")

### CNN-BiLSTM Five-Seed Performance Visualisation

This cell generates the held-out five-seed performance figure from saved evaluation artifacts.

It searches:

```text
Honours_Project/Models/Tuned/CNN_BiLSTM/
```

for compatible CNN-BiLSTM runs and:

```text
Honours_Project/Models/Baselines/Tuned_RF/
```

for evaluated tuned Random Forest runs. The user selects the intended run for each model rather than relying on a hard-coded timestamp.

The CNN-BiLSTM run must contain:

- `validation_seed_results.csv`;
- held-out evaluations for seeds `42, 7, 123, 2024, 99`; and
- an unambiguous validation-selected seed.

The figure reports:

- each seed model's held-out accuracy;
- the five-seed mean;
- one sample standard deviation;
- minimum and maximum seed accuracy;
- the seed selected using validation evidence; and
- the independently tuned Random Forest held-out accuracy as a reference.

The five held-out accuracies describe run-to-run variability. They are not used to change the validation-selected CNN-BiLSTM model.

Outputs are saved to:

```text
Honours_Project/CNN_BiLSTM_Five_Seed_Performance/
├── Figure_CNN_BiLSTM_Five_Seed_Performance.png
├── Figure_CNN_BiLSTM_Five_Seed_Performance.pdf
└── CNN_BiLSTM_Five_Seed_Performance_Summary.csv
```

The summary CSV records the selected source directories and metrics files in addition to the plotted values.

In [ ]:
# ==============================================================================
# CNN-BILSTM FIVE-SEED HELD-OUT PERFORMANCE FIGURE
# ==============================================================================

import re
from pathlib import Path

from google.colab import drive
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ==============================================================================
# 1. GOOGLE DRIVE AND PROJECT PATHS
# ==============================================================================

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Honours_Project"
)

MODELS_ROOT = PROJECT_ROOT / "Models"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "CNN_BiLSTM_Five_Seed_Performance"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_PNG = (
    OUTPUT_DIR
    / "Figure_CNN_BiLSTM_Five_Seed_Performance.png"
)

OUTPUT_PDF = (
    OUTPUT_DIR
    / "Figure_CNN_BiLSTM_Five_Seed_Performance.pdf"
)

SUMMARY_OUTPUT = (
    OUTPUT_DIR
    / "CNN_BiLSTM_Five_Seed_Performance_Summary.csv"
)

EXPECTED_SEEDS = {
    42,
    7,
    123,
    2024,
    99,
}

SEED_ORDER = [
    42,
    7,
    123,
    2024,
    99,
]


# ==============================================================================
# 2. SAVED-RUN SELECTION
# ==============================================================================

def select_run_directory(label, candidates):
    candidates = sorted(
        {
            Path(candidate).resolve()
            for candidate in candidates
            if Path(candidate).is_dir()
        },
        key=lambda candidate: candidate.name,
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError(
            f"No compatible {label} run directory was found."
        )

    print("\n" + "=" * 100)
    print(f"SELECT {label.upper()} RUN")
    print("=" * 100)

    for number, candidate in enumerate(
        candidates,
        start=1,
    ):
        print(f"[{number}] {candidate}")

    while True:
        response = input(
            f"Select 1-{len(candidates)} "
            f"(Enter = newest): "
        ).strip()

        if response == "":
            selected = candidates[0]
            print(f"\nSelected newest run:\n{selected}")
            return selected

        if response.isdigit():
            selection = int(response)

            if 1 <= selection <= len(candidates):
                selected = candidates[
                    selection - 1
                ]

                print(
                    f"\nSelected run:\n{selected}"
                )

                return selected

        print("Invalid selection.")


cnn_run_candidates = [
    validation_path.parent
    for validation_path in (
        MODELS_ROOT
        / "Tuned"
        / "CNN_BiLSTM"
    ).glob(
        "*/validation_seed_results.csv"
    )
]

rf_run_candidates = [
    metrics_path.parents[2]
    for metrics_path in (
        MODELS_ROOT
        / "Baselines"
        / "Tuned_RF"
    ).glob(
        "*/Test_Evaluation/"
        "tuned_random_forest_pipeline/"
        "metrics.csv"
    )
]

CNN_RUN_DIR = select_run_directory(
    "CNN-BiLSTM",
    cnn_run_candidates,
)

RF_RUN_DIR = select_run_directory(
    "tuned Random Forest",
    rf_run_candidates,
)


# ==============================================================================
# 3. CNN-BILSTM VALIDATION-SELECTION RECORD
# ==============================================================================

VALIDATION_RESULTS_PATH = (
    CNN_RUN_DIR
    / "validation_seed_results.csv"
)

if not VALIDATION_RESULTS_PATH.exists():
    raise FileNotFoundError(
        "Could not locate the CNN-BiLSTM "
        "validation-selection record:\n"
        f"{VALIDATION_RESULTS_PATH}"
    )

validation_dataframe = pd.read_csv(
    VALIDATION_RESULTS_PATH
)

required_validation_columns = {
    "model",
    "seed",
    "selected_champion",
}

missing_validation_columns = (
    required_validation_columns
    - set(validation_dataframe.columns)
)

if missing_validation_columns:
    raise RuntimeError(
        "validation_seed_results.csv does not "
        "contain the expected columns.\n"
        f"Missing: {sorted(missing_validation_columns)}\n"
        f"Found:   {list(validation_dataframe.columns)}"
    )

cnn_validation_dataframe = validation_dataframe[
    validation_dataframe["model"]
    .astype(str)
    .str.contains(
        "CNN-BiLSTM",
        case=False,
        regex=False,
    )
].copy()

if cnn_validation_dataframe.empty:
    raise RuntimeError(
        "No CNN-BiLSTM rows were found in:\n"
        f"{VALIDATION_RESULTS_PATH}"
    )

champion_mask = (
    cnn_validation_dataframe[
        "selected_champion"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(
        [
            "true",
            "1",
            "yes",
        ]
    )
)

champion_rows = cnn_validation_dataframe[
    champion_mask
]

if len(champion_rows) != 1:
    raise RuntimeError(
        "Expected exactly one validation-selected "
        "CNN-BiLSTM seed.\n"
        f"Found: {len(champion_rows)}"
    )

VALIDATION_SELECTED_SEED = int(
    champion_rows.iloc[0]["seed"]
)

if VALIDATION_SELECTED_SEED not in EXPECTED_SEEDS:
    raise RuntimeError(
        "The validation-selected seed is not one of "
        "the expected five seeds.\n"
        f"Selected: {VALIDATION_SELECTED_SEED}\n"
        f"Expected: {sorted(EXPECTED_SEEDS)}"
    )


# ==============================================================================
# 4. HELD-OUT TEST RESULT DISCOVERY
# ==============================================================================

SEED_MODEL_PATTERN = re.compile(
    r"^cnn_bilstm(?:_final)?_seed_(\d+)$",
    re.IGNORECASE,
)


def find_cnn_bilstm_seed_test_results(
    run_directory,
):
    run_directory = Path(run_directory)

    test_evaluation_directory = (
        run_directory
        / "Test_Evaluation"
    )

    metrics_files = sorted(
        test_evaluation_directory.glob(
            "cnn_bilstm_seed_*/metrics.csv"
        )
    )

    if not metrics_files:
        raise FileNotFoundError(
            "No canonical CNN-BiLSTM seed metrics "
            "files were found under:\n"
            f"{test_evaluation_directory}\n\n"
            "Run the universal held-out evaluator "
            "for all five CNN-BiLSTM seed models first."
        )

    discovered_rows = []

    for metrics_path in metrics_files:
        try:
            metrics_dataframe = pd.read_csv(
                metrics_path
            )
        except Exception as exception:
            print(
                "Skipping unreadable metrics file:\n"
                f"{metrics_path}\n"
                f"Reason: {exception}"
            )
            continue

        required_columns = {
            "model_name",
            "accuracy",
            "test_rows",
        }

        missing_columns = (
            required_columns
            - set(metrics_dataframe.columns)
        )

        if missing_columns:
            continue

        for _, row in metrics_dataframe.iterrows():
            model_name = str(
                row["model_name"]
            ).strip()

            match = SEED_MODEL_PATTERN.fullmatch(
                model_name
            )

            if match is None:
                continue

            seed = int(
                match.group(1)
            )

            if seed not in EXPECTED_SEEDS:
                continue

            try:
                accuracy = float(
                    row["accuracy"]
                )

                test_rows = int(
                    row["test_rows"]
                )
            except Exception:
                continue

            if test_rows != 376:
                continue

            if not np.isfinite(accuracy):
                continue

            if not 0.0 <= accuracy <= 1.0:
                continue

            evaluated_at = ""

            if (
                "evaluated_at"
                in metrics_dataframe.columns
            ):
                evaluated_at = str(
                    row["evaluated_at"]
                )

            discovered_rows.append(
                {
                    "seed": seed,
                    "accuracy_percent": (
                        accuracy * 100.0
                    ),
                    "evaluated_at": evaluated_at,
                    "source_csv": str(
                        metrics_path
                    ),
                }
            )

    if not discovered_rows:
        raise RuntimeError(
            "No valid CNN-BiLSTM held-out seed "
            "results were found under:\n"
            f"{test_evaluation_directory}\n\n"
            "Expected model names such as:\n"
            "cnn_bilstm_seed_42"
        )

    results_dataframe = pd.DataFrame(
        discovered_rows
    )

    duplicate_seed_mask = (
        results_dataframe["seed"]
        .duplicated(keep=False)
    )

    if duplicate_seed_mask.any():
        duplicate_rows = (
            results_dataframe[
                duplicate_seed_mask
            ]
            .sort_values(
                [
                    "seed",
                    "source_csv",
                ]
            )
        )

        print(
            "\nDuplicate canonical seed rows:"
        )

        print(
            duplicate_rows.to_string(
                index=False
            )
        )

        duplicate_seeds = sorted(
            duplicate_rows["seed"]
            .astype(int)
            .unique()
            .tolist()
        )

        raise RuntimeError(
            "More than one canonical metrics.csv "
            "was found for these seeds:\n"
            f"{duplicate_seeds}"
        )

    found_seeds = set(
        results_dataframe["seed"]
        .astype(int)
    )

    missing_seeds = (
        EXPECTED_SEEDS
        - found_seeds
    )

    unexpected_seeds = (
        found_seeds
        - EXPECTED_SEEDS
    )

    if found_seeds != EXPECTED_SEEDS:
        raise RuntimeError(
            "Five-seed integrity check failed.\n"
            f"Expected:   {sorted(EXPECTED_SEEDS)}\n"
            f"Found:      {sorted(found_seeds)}\n"
            f"Missing:    {sorted(missing_seeds)}\n"
            f"Unexpected: {sorted(unexpected_seeds)}"
        )

    return (
        results_dataframe
        .sort_values("seed")
        .reset_index(drop=True)
    )


def find_tuned_rf_test_accuracy(
    run_directory,
):
    run_directory = Path(run_directory)

    metrics_path = (
        run_directory
        / "Test_Evaluation"
        / "tuned_random_forest_pipeline"
        / "metrics.csv"
    )

    if not metrics_path.exists():
        raise FileNotFoundError(
            "The canonical tuned Random Forest "
            "metrics file was not found:\n"
            f"{metrics_path}"
        )

    metrics_dataframe = pd.read_csv(
        metrics_path
    )

    required_columns = {
        "model_name",
        "accuracy",
        "test_rows",
    }

    missing_columns = (
        required_columns
        - set(metrics_dataframe.columns)
    )

    if missing_columns:
        raise RuntimeError(
            "The tuned Random Forest metrics file "
            "does not contain the expected columns.\n"
            f"Missing: {sorted(missing_columns)}\n"
            f"Found:   {list(metrics_dataframe.columns)}"
        )

    if len(metrics_dataframe) != 1:
        raise RuntimeError(
            "Expected exactly one row in the tuned "
            "Random Forest metrics file.\n"
            f"Found: {len(metrics_dataframe)}\n"
            f"File: {metrics_path}"
        )

    row = metrics_dataframe.iloc[0]

    model_name = str(
        row["model_name"]
    ).strip()

    if model_name != "tuned_random_forest_pipeline":
        raise RuntimeError(
            "The canonical Random Forest metrics "
            "file contains the wrong model name.\n"
            f"Found: {model_name}"
        )

    try:
        test_rows = int(
            row["test_rows"]
        )

        accuracy = float(
            row["accuracy"]
        )
    except Exception as exception:
        raise RuntimeError(
            "The tuned Random Forest metrics contain "
            "invalid numerical values."
        ) from exception

    if test_rows != 376:
        raise RuntimeError(
            "The tuned Random Forest evaluation "
            "does not use the expected 376-row "
            "held-out test set.\n"
            f"Found: {test_rows}"
        )

    if not np.isfinite(accuracy):
        raise RuntimeError(
            "The tuned Random Forest accuracy is "
            "NaN or infinite."
        )

    if not 0.0 <= accuracy <= 1.0:
        raise RuntimeError(
            "The tuned Random Forest accuracy is "
            "outside the expected [0, 1] range.\n"
            f"Found: {accuracy}"
        )

    return (
        metrics_path,
        accuracy * 100.0,
    )


# ==============================================================================
# 5. LOAD AND VERIFY THE AUTHORITATIVE RESULTS
# ==============================================================================

seed_dataframe = (
    find_cnn_bilstm_seed_test_results(
        CNN_RUN_DIR
    )
)

(
    rf_source_csv,
    random_forest_accuracy,
) = find_tuned_rf_test_accuracy(
    RF_RUN_DIR
)

seed_dataframe = (
    seed_dataframe
    .set_index("seed")
    .loc[SEED_ORDER]
    .reset_index()
)

values = seed_dataframe[
    "accuracy_percent"
].to_numpy(
    dtype=np.float64
)

seeds = seed_dataframe[
    "seed"
].to_numpy(
    dtype=int
)

if len(values) != 5:
    raise RuntimeError(
        "Expected exactly five CNN-BiLSTM "
        f"held-out accuracies; found {len(values)}."
    )

if not np.all(np.isfinite(values)):
    raise RuntimeError(
        "One or more CNN-BiLSTM held-out "
        "accuracies are NaN or infinite."
    )

if np.any(values < 0.0) or np.any(values > 100.0):
    raise RuntimeError(
        "One or more CNN-BiLSTM held-out "
        "accuracies are outside [0, 100]."
    )


# ==============================================================================
# 6. DESCRIPTIVE STATISTICS
# ==============================================================================

number_of_seeds = len(values)

mean_accuracy = float(
    np.mean(values)
)

sample_sd = float(
    np.std(
        values,
        ddof=1,
    )
)

minimum_accuracy = float(
    np.min(values)
)

maximum_accuracy = float(
    np.max(values)
)

selected_seed_row = seed_dataframe[
    seed_dataframe["seed"]
    == VALIDATION_SELECTED_SEED
]

if len(selected_seed_row) != 1:
    raise RuntimeError(
        "Could not locate the validation-selected "
        "seed inside the five held-out results."
    )

selected_seed_test_accuracy = float(
    selected_seed_row.iloc[0][
        "accuracy_percent"
    ]
)


# ==============================================================================
# 7. PRINT VERIFIED NUMERICAL RESULTS
# ==============================================================================

print("\n" + "=" * 100)
print("FINAL RETAINED CNN-BILSTM FIVE-SEED RESULTS")
print("=" * 100)

print(
    "\nAuthoritative CNN-BiLSTM run:\n"
    f"{CNN_RUN_DIR}"
)

print(
    "\nAuthoritative tuned Random Forest run:\n"
    f"{RF_RUN_DIR}"
)

print(
    "\nValidation-selected CNN-BiLSTM seed:\n"
    f"{VALIDATION_SELECTED_SEED}"
)

print("\nHeld-out CNN-BiLSTM test accuracies:")

for seed, accuracy in zip(
    seeds,
    values,
):
    if seed == VALIDATION_SELECTED_SEED:
        marker = "  <-- validation-selected"
    else:
        marker = ""

    print(
        f"  Seed {seed:4d}: "
        f"{accuracy:.3f}%"
        f"{marker}"
    )

print("\nDescriptive five-seed summary:")

print(
    f"  Number of seeds: {number_of_seeds}"
)

print(
    f"  Mean accuracy : {mean_accuracy:.3f}%"
)

print(
    "  Sample SD     : "
    f"{sample_sd:.3f} percentage points"
)

print(
    f"  Minimum       : {minimum_accuracy:.3f}%"
)

print(
    f"  Maximum       : {maximum_accuracy:.3f}%"
)

print(
    "\nValidation-selected seed "
    "held-out accuracy:"
)

print(
    f"  Seed {VALIDATION_SELECTED_SEED}: "
    f"{selected_seed_test_accuracy:.3f}%"
)

print("\nTuned Random Forest reference:")

print(
    "  Held-out accuracy: "
    f"{random_forest_accuracy:.3f}%"
)

print(
    f"  Source CSV: {rf_source_csv}"
)


# ==============================================================================
# 8. GENERATE THE FIVE-SEED FIGURE
# ==============================================================================

figure, axis = plt.subplots(
    figsize=(10.5, 6.2),
    constrained_layout=True,
)

x_positions = np.arange(
    len(seeds),
    dtype=np.float64,
)

axis.scatter(
    x_positions,
    values,
    s=90,
    color="#276FBF",
    edgecolor="black",
    linewidth=0.6,
    zorder=4,
    label="CNN-BiLSTM seed model",
)

selected_position = int(
    np.where(
        seeds
        == VALIDATION_SELECTED_SEED
    )[0][0]
)

axis.scatter(
    [selected_position],
    [selected_seed_test_accuracy],
    s=190,
    marker="*",
    color="#F28E2B",
    edgecolor="black",
    linewidth=0.9,
    zorder=5,
    label=(
        "Validation-selected seed "
        f"({VALIDATION_SELECTED_SEED})"
    ),
)

axis.axhline(
    mean_accuracy,
    color="#59A14F",
    linestyle="-",
    linewidth=1.7,
    label=(
        "CNN-BiLSTM mean "
        f"({mean_accuracy:.2f}%)"
    ),
)

axis.axhspan(
    mean_accuracy - sample_sd,
    mean_accuracy + sample_sd,
    color="#59A14F",
    alpha=0.12,
    label=(
        "CNN-BiLSTM mean ± SD "
        f"({sample_sd:.2f} pp)"
    ),
)

axis.axhline(
    random_forest_accuracy,
    color="#E15759",
    linestyle="--",
    linewidth=1.7,
    label=(
        "Tuned Random Forest "
        f"({random_forest_accuracy:.2f}%)"
    ),
)

for x_position, accuracy in zip(
    x_positions,
    values,
):
    axis.annotate(
        f"{accuracy:.2f}%",
        (
            x_position,
            accuracy,
        ),
        xytext=(0, 9),
        textcoords="offset points",
        horizontalalignment="center",
        verticalalignment="bottom",
        fontsize=9,
    )

all_y_values = np.concatenate(
    [
        values,
        np.asarray(
            [
                mean_accuracy - sample_sd,
                mean_accuracy + sample_sd,
                random_forest_accuracy,
            ],
            dtype=np.float64,
        ),
    ]
)

minimum_y = float(
    np.min(all_y_values)
)

maximum_y = float(
    np.max(all_y_values)
)

axis.set_ylim(
    minimum_y - 0.8,
    maximum_y + 1.0,
)

axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    [
        str(seed)
        for seed in seeds
    ]
)

axis.set_xlabel(
    "CNN-BiLSTM training seed"
)

axis.set_ylabel(
    "Held-out test accuracy (%)"
)

axis.set_title(
    "CNN-BiLSTM Held-Out Test Accuracy "
    "Across Five Training Seeds",
    fontweight="bold",
)

axis.grid(
    axis="y",
    linestyle=":",
    alpha=0.45,
)

axis.spines[
    [
        "top",
        "right",
    ]
].set_visible(False)

axis.legend(
    loc="best",
    frameon=False,
)

figure.savefig(
    OUTPUT_PNG,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

figure.savefig(
    OUTPUT_PDF,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(figure)


# ==============================================================================
# 9. SAVE THE NUMERICAL RESULTS AND PROVENANCE
# ==============================================================================

summary_dataframe = (
    seed_dataframe.copy()
)

summary_dataframe[
    "validation_selected"
] = (
    summary_dataframe["seed"]
    == VALIDATION_SELECTED_SEED
)

summary_dataframe[
    "five_seed_mean_accuracy_percent"
] = mean_accuracy

summary_dataframe[
    "five_seed_sample_sd_percentage_points"
] = sample_sd

summary_dataframe[
    "five_seed_minimum_accuracy_percent"
] = minimum_accuracy

summary_dataframe[
    "five_seed_maximum_accuracy_percent"
] = maximum_accuracy

summary_dataframe[
    "tuned_rf_reference_accuracy_percent"
] = random_forest_accuracy

summary_dataframe[
    "cnn_bilstm_run_directory"
] = str(CNN_RUN_DIR)

summary_dataframe[
    "rf_run_directory"
] = str(RF_RUN_DIR)

summary_dataframe[
    "validation_results_csv"
] = str(VALIDATION_RESULTS_PATH)

summary_dataframe[
    "rf_metrics_csv"
] = str(rf_source_csv)

summary_dataframe.to_csv(
    SUMMARY_OUTPUT,
    index=False,
)


# ==============================================================================
# 10. COMPLETION SUMMARY
# ==============================================================================

print("\n" + "=" * 100)
print("FIVE-SEED FIGURE COMPLETE")
print("=" * 100)

print(
    "\nCNN-BiLSTM five-seed mean:\n"
    f"{mean_accuracy:.3f}%"
)

print(
    "\nCNN-BiLSTM sample SD:\n"
    f"{sample_sd:.3f} percentage points"
)

print(
    "\nValidation-selected seed:\n"
    f"{VALIDATION_SELECTED_SEED}"
)

print(
    "\nValidation-selected held-out accuracy:\n"
    f"{selected_seed_test_accuracy:.3f}%"
)

print(
    "\nTuned Random Forest held-out accuracy:\n"
    f"{random_forest_accuracy:.3f}%"
)

print(
    "\nOutput directory:\n"
    f"{OUTPUT_DIR}"
)

print(
    "\nSaved PNG:\n"
    f"{OUTPUT_PNG}"
)

print(
    "\nSaved PDF:\n"
    f"{OUTPUT_PDF}"
)

print(
    "\nSaved numerical summary:\n"
    f"{SUMMARY_OUTPUT}"
)

## Explainable AI (XAI) Analysis


### SHAP Common Setup and Integrity Checks

This cell establishes the shared, validated state used by the SHAP analyses.

It dynamically discovers complete CNN-BiLSTM runs under:

```text
Honours_Project/Models/Tuned/CNN_BiLSTM/
```

and asks the user to select the intended run. No historical timestamp or `Layers_1-3_Grid_Search_50_Epochs` path is used.

The selected run must contain the validation-selected model, five-seed validation record, fixed held-out arrays, feature names, preprocessing objects, split indices and zero-variance record.

The cell:

- reconstructs the 1,877-row deduplicated corpus;
- verifies the stored 1,200/301/376 split indices;
- checks the 53 removed and 244 retained predictors;
- verifies class ordering and saved array shapes;
- loads the validation-selected CNN-BiLSTM using the registered `SelfAttention` class;
- confirms that the copied champion model matches the selected seed model;
- reproduces held-out predictions;
- reconstructs the 1,200 real pre-SMOTE training observations;
- fits K-means with 20 clusters to the 1,200 real, pre-SMOTE, training-processed observations using `random_state=42` and `n_init=20`, then selects the real observation nearest each cluster centre to form the 20-case SHAP background;
- selects a class-balanced 100-observation subset for the stability experiment;
- calculates weights that restore the complete held-out class distribution during stability aggregation; and
- builds the semantic feature-family mapping.

The SHAP configuration is:

```text
Stability repetitions:          3
Stability seeds:                1701, 2701, 3701
Stability cases:                100
Kernel SHAP samples:            4096
Full held-out cases:            376
Kernel checkpoint size:         10
Partition maximum evaluations:  4096
Partition batch size:           256
Partition checkpoint size:      10
```

Outputs are created beneath the selected run:

```text
<selected CNN-BiLSTM run>/
└── SHAP_Final/
    ├── Common/
    ├── Stability_3x4096/
    │   └── Figures/
    └── Full376/
        ├── Kernel_Checkpoints/
        ├── Partition_Checkpoints/
        └── Figures/
```

This cell creates the runtime objects needed by the stability and Partition SHAP cells. It does not train or select another model.

In [ ]:
!pip install -q shap

import gc
import hashlib
import json
import os
import random
import shutil
from pathlib import Path

os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import tensorflow as tf
import tensorflow.keras.backend as K
from google.colab import drive
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Layer
from tensorflow.keras.models import load_model

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

RANDOM_STATE = 42

EXPECTED_UNIQUE_ROWS = 1877
EXPECTED_TRAIN_ROWS = 1200
EXPECTED_VALIDATION_ROWS = 301
EXPECTED_TEST_ROWS = 376
EXPECTED_RAW_FEATURES = 297
EXPECTED_ZERO_VARIANCE = 53
EXPECTED_FEATURES = 244
EXPECTED_CLASSES = 10

BACKGROUND_SIZE = 20
STABILITY_SAMPLES_PER_CLASS = 10
STABILITY_NSAMPLES = 4096
STABILITY_SEEDS = [1701, 2701, 3701]

KERNEL_NSAMPLES = 4096
KERNEL_BASE_SEED = 4701
KERNEL_CHUNK_SIZE = 10

PARTITION_MAX_EVALS = 4096
PARTITION_BATCH_SIZE = 256
PARTITION_CHUNK_SIZE = 10

FORCE_RECOMPUTE = False

PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
CNN_BILSTM_ROOT = PROJECT_ROOT / "Models" / "Tuned" / "CNN_BiLSTM"


def select_cnn_bilstm_run():
    required_names = [
        "best_validation_cnn_bilstm.keras",
        "validation_seed_results.csv",
        "X_test_3d.npy",
        "y_test_1d.npy",
        "device_classes.npy",
        "feature_names.npy",
        "split_indices.npz",
        "imputer.joblib",
        "scaler.joblib",
        "one_hot_encoder.joblib",
        "zero_variance_columns.txt",
    ]

    candidates = []

    if CNN_BILSTM_ROOT.exists():
        for candidate in CNN_BILSTM_ROOT.iterdir():
            if candidate.is_dir() and all(
                (candidate / name).exists() for name in required_names
            ):
                candidates.append(candidate.resolve())

    candidates = sorted(
        candidates,
        key=lambda path: path.name,
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError(
            "No complete tuned CNN-BiLSTM run was found under:\n"
            f"{CNN_BILSTM_ROOT}"
        )

    print("\n" + "=" * 100)
    print("SELECT CNN-BILSTM RUN FOR SHAP")
    print("=" * 100)

    for number, candidate in enumerate(candidates, start=1):
        print(f"[{number}] {candidate}")

    while True:
        response = input(
            f"Select 1-{len(candidates)} (Enter = newest): "
        ).strip()

        if response == "":
            return candidates[0]

        if response.isdigit() and 1 <= int(response) <= len(candidates):
            return candidates[int(response) - 1]

        print("Invalid selection.")


RUN_DIR = select_cnn_bilstm_run()

TRAIN_DATA_PATH = PROJECT_ROOT / "iot_device_train.csv"
TEST_DATA_PATH = PROJECT_ROOT / "iot_device_test.csv"

MODEL_PATH = RUN_DIR / "best_validation_cnn_bilstm.keras"
VALIDATION_RESULTS_PATH = RUN_DIR / "validation_seed_results.csv"
X_TEST_PATH = RUN_DIR / "X_test_3d.npy"
Y_TEST_PATH = RUN_DIR / "y_test_1d.npy"
CLASSES_PATH = RUN_DIR / "device_classes.npy"
FEATURE_NAMES_PATH = RUN_DIR / "feature_names.npy"
IMPUTER_PATH = RUN_DIR / "imputer.joblib"
SCALER_PATH = RUN_DIR / "scaler.joblib"
ENCODER_PATH = RUN_DIR / "one_hot_encoder.joblib"
ZERO_VARIANCE_PATH = RUN_DIR / "zero_variance_columns.txt"
SPLIT_INDICES_PATH = RUN_DIR / "split_indices.npz"

SHAP_ROOT = RUN_DIR / "SHAP_Final"
COMMON_DIR = SHAP_ROOT / "Common"
STABILITY_DIR = SHAP_ROOT / "Stability_3x4096"
STABILITY_FIGURES_DIR = STABILITY_DIR / "Figures"
FULL376_DIR = SHAP_ROOT / "Full376"
FULL376_FIGURES_DIR = FULL376_DIR / "Figures"
KERNEL_CHECKPOINT_DIR = FULL376_DIR / "Kernel_Checkpoints"
PARTITION_CHECKPOINT_DIR = FULL376_DIR / "Partition_Checkpoints"

for directory in [
    SHAP_ROOT,
    COMMON_DIR,
    STABILITY_DIR,
    STABILITY_FIGURES_DIR,
    FULL376_DIR,
    FULL376_FIGURES_DIR,
    KERNEL_CHECKPOINT_DIR,
    PARTITION_CHECKPOINT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


@tf.keras.utils.register_keras_serializable(package="COMP700")
class SelfAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="att_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True,
        )
        self.b = self.add_weight(
            name="att_bias",
            shape=(input_shape[1], 1),
            initializer="zeros",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, inputs):
        scores = K.tanh(K.dot(inputs, self.W) + self.b)
        weights = K.softmax(scores, axis=1)
        return K.sum(inputs * weights, axis=1)


SELF_ATTENTION_CUSTOM_OBJECTS = {
    "SelfAttention": SelfAttention,
    "Custom>SelfAttention": SelfAttention,
    "COMP700>SelfAttention": SelfAttention,
}


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)

    return digest.hexdigest()


def save_figure(fig, base_path):
    base_path = Path(base_path)

    fig.savefig(
        base_path.with_suffix(".png"),
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        base_path.with_suffix(".pdf"),
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )


def safe_spearman(a, b):
    value = spearmanr(np.asarray(a), np.asarray(b)).statistic
    return float(value) if np.isfinite(value) else float("nan")


def top_overlap(a, b, k):
    top_a = set(np.argsort(np.asarray(a))[::-1][:k].tolist())
    top_b = set(np.argsort(np.asarray(b))[::-1][:k].tolist())
    return len(top_a.intersection(top_b)) / float(k)


def normalise_multiclass_shap(values, n_samples):
    if isinstance(values, shap.Explanation):
        array = np.asarray(values.values)
    elif isinstance(values, list):
        array = np.stack(
            [np.asarray(item) for item in values],
            axis=-1,
        )
    else:
        array = np.asarray(values)

    target = (
        n_samples,
        EXPECTED_FEATURES,
        EXPECTED_CLASSES,
    )

    if array.shape == target:
        return array.astype(np.float64)

    if array.shape == (
        EXPECTED_CLASSES,
        n_samples,
        EXPECTED_FEATURES,
    ):
        return np.transpose(
            array,
            (1, 2, 0),
        ).astype(np.float64)

    if array.shape == (
        n_samples,
        EXPECTED_CLASSES,
        EXPECTED_FEATURES,
    ):
        return np.transpose(
            array,
            (0, 2, 1),
        ).astype(np.float64)

    raise RuntimeError(
        f"Unexpected SHAP shape {array.shape}; "
        f"expected compatibility with {target}."
    )


def assign_feature_family(feature_name):
    name = str(feature_name).strip().lower()

    if name.startswith("ttl"):
        return "TTL statistics"

    if (
        name.startswith("suffix_")
        or name.startswith("domain_")
        or name.startswith("subdomain_")
        or name in {
            "ssl_dom_server_name_alexarank",
            "http_dom_host_alexarank",
        }
    ):
        return "Domain / suffix metadata"

    if "_ratio" in name or name.endswith("ratio"):
        return "Directional ratios"

    if (
        name == "duration"
        or "inter_arrivel" in name
        or name.startswith("http_time_")
        or name.startswith("ssl_handshake_duration_")
    ):
        return "Timing / inter-arrival"

    if (
        "packet_size" in name
        or name in {
            "bytes",
            "bytes_a",
            "bytes_b",
        }
        or "_bytes_" in name
        or name.startswith("bytes_")
    ):
        return "Packet size / byte volume"

    if (
        name.startswith("tcp_analysis_")
        or name in {
            "reset",
            "reset_a",
            "reset_b",
            "push",
            "push_a",
            "push_b",
            "ack",
            "ack_a",
            "ack_b",
        }
    ):
        return "TCP control / state"

    if (
        name.startswith("b_port_is_")
        or name in {
            "b_is_user_port",
            "b_is_system_port",
            "is_numeric_url_with_port_http",
        }
    ):
        return "Port metadata"

    if name.startswith("ssl_") or name == "is_ssl":
        return "SSL / TLS metadata"

    if (
        name.startswith("http_")
        or name in {
            "is_http",
            "is_cloud_http",
            "is_numeric_url_http",
        }
    ):
        return "HTTP metadata"

    if name in {
        "packets",
        "packets_a",
        "packets_b",
    }:
        return "Packet / transaction counts"

    if name.startswith("ds_field_"):
        return "IP-header metadata"

    return "Other / unclassified"


def aggregate_family_importance(decision_shap, sample_weights=None):
    absolute_values = np.abs(
        np.asarray(decision_shap, dtype=np.float64)
    )

    if (
        absolute_values.ndim != 2
        or absolute_values.shape[1] != EXPECTED_FEATURES
    ):
        raise RuntimeError(
            f"Expected decision SHAP shape "
            f"(n, {EXPECTED_FEATURES}); "
            f"found {absolute_values.shape}."
        )

    rows = []

    for family in family_names:
        indices = family_to_indices[family]
        per_sample_total = np.sum(
            absolute_values[:, indices],
            axis=1,
        )

        if sample_weights is None:
            total = float(np.mean(per_sample_total))
        else:
            total = float(
                np.average(
                    per_sample_total,
                    weights=sample_weights,
                )
            )

        rows.append(
            {
                "feature_family": family,
                "number_of_features": int(len(indices)),
                "total_mean_abs_shap": total,
                "mean_abs_shap_per_feature": (
                    total / float(len(indices))
                ),
            }
        )

    result = pd.DataFrame(rows)

    grand_total = float(
        result["total_mean_abs_shap"].sum()
    )

    result["share_of_total_abs_shap"] = (
        result["total_mean_abs_shap"] / grand_total
    )

    result = (
        result
        .sort_values(
            "share_of_total_abs_shap",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    result.insert(
        0,
        "rank",
        np.arange(1, len(result) + 1),
    )

    return result


required_paths = [
    RUN_DIR,
    TRAIN_DATA_PATH,
    TEST_DATA_PATH,
    MODEL_PATH,
    VALIDATION_RESULTS_PATH,
    X_TEST_PATH,
    Y_TEST_PATH,
    CLASSES_PATH,
    FEATURE_NAMES_PATH,
    IMPUTER_PATH,
    SCALER_PATH,
    ENCODER_PATH,
    ZERO_VARIANCE_PATH,
    SPLIT_INDICES_PATH,
]

missing = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required project artifacts:\n"
        + "\n".join(missing)
    )

X_test_3d_saved = np.asarray(
    np.load(X_TEST_PATH),
    dtype=np.float32,
)
y_test_saved = np.asarray(
    np.load(Y_TEST_PATH),
    dtype=np.int64,
).reshape(-1)
device_classes = np.asarray(
    np.load(CLASSES_PATH, allow_pickle=True)
)
feature_names = [
    str(value)
    for value in np.load(
        FEATURE_NAMES_PATH,
        allow_pickle=True,
    )
]

imputer = joblib.load(IMPUTER_PATH)
scaler = joblib.load(SCALER_PATH)
one_hot_encoder = joblib.load(ENCODER_PATH)

with open(
    ZERO_VARIANCE_PATH,
    "r",
    encoding="utf-8",
) as handle:
    saved_zero_variance_columns = [
        line.strip()
        for line in handle
        if line.strip()
    ]

if X_test_3d_saved.shape != (
    EXPECTED_TEST_ROWS,
    EXPECTED_FEATURES,
    1,
):
    raise RuntimeError(
        f"Unexpected saved test tensor: "
        f"{X_test_3d_saved.shape}"
    )

if y_test_saved.shape != (EXPECTED_TEST_ROWS,):
    raise RuntimeError(
        f"Unexpected saved labels: "
        f"{y_test_saved.shape}"
    )

if len(device_classes) != EXPECTED_CLASSES:
    raise RuntimeError(
        f"Expected {EXPECTED_CLASSES} classes; "
        f"found {len(device_classes)}."
    )

if len(feature_names) != EXPECTED_FEATURES:
    raise RuntimeError(
        f"Expected {EXPECTED_FEATURES} features; "
        f"found {len(feature_names)}."
    )

validation_df = pd.read_csv(
    VALIDATION_RESULTS_PATH
)

cnn_rows = validation_df[
    validation_df["model"]
    .astype(str)
    .str.contains(
        "CNN-BiLSTM",
        case=False,
        regex=False,
    )
].copy()

selected_mask = (
    cnn_rows["selected_champion"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "yes"])
)

selected_rows = cnn_rows[selected_mask]

if len(selected_rows) != 1:
    raise RuntimeError(
        "Expected exactly one selected CNN-BiLSTM seed; "
        f"found {len(selected_rows)}."
    )

CHAMPION_SEED = int(
    selected_rows.iloc[0]["seed"]
)

seed_model_candidates = [
    RUN_DIR / f"cnn_bilstm_seed_{CHAMPION_SEED}.keras",
    RUN_DIR / f"cnn_bilstm_final_seed_{CHAMPION_SEED}.keras",
]

SEED_MODEL_PATH = next(
    (
        path
        for path in seed_model_candidates
        if path.exists()
    ),
    None,
)

if SEED_MODEL_PATH is None:
    raise FileNotFoundError(
        "Could not locate selected CNN-BiLSTM "
        f"seed model for seed {CHAMPION_SEED}."
    )

tf.keras.backend.clear_session()
gc.collect()

model = load_model(
    MODEL_PATH,
    custom_objects=SELF_ATTENTION_CUSTOM_OBJECTS,
    compile=False,
)

seed_model = load_model(
    SEED_MODEL_PATH,
    custom_objects=SELF_ATTENTION_CUSTOM_OBJECTS,
    compile=False,
)

champion_probs_saved = np.asarray(
    model.predict(
        X_test_3d_saved,
        verbose=0,
    ),
    dtype=np.float64,
)

seed_probs_saved = np.asarray(
    seed_model.predict(
        X_test_3d_saved,
        verbose=0,
    ),
    dtype=np.float64,
)

champion_seed_max_difference = float(
    np.max(
        np.abs(
            champion_probs_saved
            - seed_probs_saved
        )
    )
)

if not np.allclose(
    champion_probs_saved,
    seed_probs_saved,
    atol=1e-6,
    rtol=1e-6,
):
    raise RuntimeError(
        "best_validation_cnn_bilstm.keras "
        "does not match the selected seed artifact."
    )

del seed_model, seed_probs_saved
gc.collect()

train_source_df = pd.read_csv(
    TRAIN_DATA_PATH
)
test_source_df = pd.read_csv(
    TEST_DATA_PATH
)

combined_df = (
    pd.concat(
        [
            train_source_df,
            test_source_df,
        ],
        ignore_index=True,
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

if len(combined_df) != EXPECTED_UNIQUE_ROWS:
    raise RuntimeError(
        f"Expected {EXPECTED_UNIQUE_ROWS} unique rows; "
        f"found {len(combined_df)}."
    )

X_full = (
    combined_df
    .drop(columns=["device_category"])
    .replace([np.inf, -np.inf], np.nan)
)
y_full = combined_df[
    "device_category"
].astype(str)

if X_full.shape[1] != EXPECTED_RAW_FEATURES:
    raise RuntimeError(
        f"Expected {EXPECTED_RAW_FEATURES} raw predictors; "
        f"found {X_full.shape[1]}."
    )

all_indices = np.arange(
    len(combined_df),
    dtype=np.int64,
)

(
    X_development_raw,
    X_test_raw,
    y_development_raw,
    y_test_raw,
    index_development,
    index_test,
) = train_test_split(
    X_full,
    y_full,
    all_indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_full,
)

(
    X_train_raw,
    X_validation_raw,
    y_train_raw,
    y_validation_raw,
    index_train,
    index_validation,
) = train_test_split(
    X_development_raw,
    y_development_raw,
    index_development,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_development_raw,
)

actual_split = (
    len(X_train_raw),
    len(X_validation_raw),
    len(X_test_raw),
)

if actual_split != (
    EXPECTED_TRAIN_ROWS,
    EXPECTED_VALIDATION_ROWS,
    EXPECTED_TEST_ROWS,
):
    raise RuntimeError(
        f"Unexpected reconstructed split: "
        f"{actual_split}"
    )

with np.load(
    SPLIT_INDICES_PATH,
    allow_pickle=False,
) as saved_split:
    saved_index_train = np.asarray(
        saved_split["train"],
        dtype=np.int64,
    )
    saved_index_validation = np.asarray(
        saved_split["validation"],
        dtype=np.int64,
    )
    saved_index_test = np.asarray(
        saved_split["test"],
        dtype=np.int64,
    )

if not np.array_equal(
    np.asarray(index_train, dtype=np.int64),
    saved_index_train,
):
    raise RuntimeError(
        "Reconstructed training indices differ "
        "from split_indices.npz."
    )

if not np.array_equal(
    np.asarray(index_validation, dtype=np.int64),
    saved_index_validation,
):
    raise RuntimeError(
        "Reconstructed validation indices differ "
        "from split_indices.npz."
    )

if not np.array_equal(
    np.asarray(index_test, dtype=np.int64),
    saved_index_test,
):
    raise RuntimeError(
        "Reconstructed test indices differ "
        "from split_indices.npz."
    )

training_variances = X_train_raw.var()

zero_variance_columns = (
    training_variances[
        training_variances == 0
    ]
    .index
    .tolist()
)

if len(zero_variance_columns) != EXPECTED_ZERO_VARIANCE:
    raise RuntimeError(
        f"Expected {EXPECTED_ZERO_VARIANCE} "
        "zero-variance predictors; "
        f"found {len(zero_variance_columns)}."
    )

if zero_variance_columns != saved_zero_variance_columns:
    raise RuntimeError(
        "Saved zero-variance schema "
        "differs from reconstruction."
    )

zero_set = set(zero_variance_columns)

reconstructed_features = [
    column
    for column in X_train_raw.columns
    if column not in zero_set
]

if reconstructed_features != feature_names:
    raise RuntimeError(
        "Retained feature order differs "
        "from the saved experiment."
    )

X_train_real_scaled = scaler.transform(
    imputer.transform(
        X_train_raw[feature_names]
    )
).astype(np.float32)

X_test_reconstructed_scaled = scaler.transform(
    imputer.transform(
        X_test_raw[feature_names]
    )
).astype(np.float32)

X_test_reconstructed_3d = (
    X_test_reconstructed_scaled[
        ...,
        np.newaxis,
    ]
)

if not np.allclose(
    X_test_reconstructed_3d,
    X_test_3d_saved,
    atol=1e-6,
    rtol=1e-6,
):
    raise RuntimeError(
        "Reconstructed held-out tensor "
        "differs from X_test_3d.npy."
    )

encoded_test_labels = one_hot_encoder.transform(
    y_test_raw
    .to_numpy()
    .reshape(-1, 1)
)

if hasattr(encoded_test_labels, "toarray"):
    encoded_test_labels = (
        encoded_test_labels.toarray()
    )

y_test_reconstructed = np.argmax(
    np.asarray(encoded_test_labels),
    axis=1,
).astype(np.int64)

if not np.array_equal(
    y_test_reconstructed,
    y_test_saved,
):
    raise RuntimeError(
        "Reconstructed held-out labels "
        "differ from y_test_1d.npy."
    )

X_explain = (
    X_test_3d_saved[:, :, 0]
    .astype(np.float32)
)
y_explain = y_test_saved.copy()
explain_probabilities = champion_probs_saved.copy()

predicted_explain_classes = np.argmax(
    explain_probabilities,
    axis=1,
)

prediction_confidence = np.max(
    explain_probabilities,
    axis=1,
)

test_accuracy = float(
    accuracy_score(
        y_explain,
        predicted_explain_classes,
    )
)

test_macro_f1 = float(
    f1_score(
        y_explain,
        predicted_explain_classes,
        average="macro",
        zero_division=0,
    )
)

background_kmeans = KMeans(
    n_clusters=BACKGROUND_SIZE,
    random_state=RANDOM_STATE,
    n_init=20,
)

background_labels = (
    background_kmeans.fit_predict(
        X_train_real_scaled
    )
)

background_indices = []

for cluster_index in range(BACKGROUND_SIZE):
    members = np.where(
        background_labels == cluster_index
    )[0]

    if len(members) == 0:
        raise RuntimeError(
            f"Background cluster "
            f"{cluster_index} is empty."
        )

    center = (
        background_kmeans
        .cluster_centers_[cluster_index]
    )

    distances = np.sum(
        (
            X_train_real_scaled[members]
            - center
        ) ** 2,
        axis=1,
    )

    background_indices.append(
        int(
            members[
                np.argmin(distances)
            ]
        )
    )

background_indices = np.asarray(
    background_indices,
    dtype=np.int64,
)

X_background = (
    X_train_real_scaled[
        background_indices
    ]
    .astype(np.float32)
)

background_df = pd.DataFrame(
    {
        "background_position": np.arange(
            BACKGROUND_SIZE
        ),
        "training_relative_position": (
            background_indices
        ),
        "combined_dataset_index": (
            X_train_raw
            .index
            .to_numpy()[
                background_indices
            ]
        ),
        "device_category": (
            y_train_raw
            .to_numpy()[
                background_indices
            ]
        ),
    }
)

background_df.to_csv(
    COMMON_DIR / "Background_Observations.csv",
    index=False,
)

rng = np.random.default_rng(
    RANDOM_STATE
)

stability_test_indices = []

for class_index in range(EXPECTED_CLASSES):
    class_indices = np.where(
        y_explain == class_index
    )[0]

    selected = rng.choice(
        class_indices,
        size=STABILITY_SAMPLES_PER_CLASS,
        replace=False,
    )

    stability_test_indices.extend(
        selected.tolist()
    )

stability_test_indices = np.asarray(
    stability_test_indices,
    dtype=np.int64,
)

X_stability = X_explain[
    stability_test_indices
].astype(np.float32)

y_stability = y_explain[
    stability_test_indices
]

stability_probabilities = (
    explain_probabilities[
        stability_test_indices
    ]
)

stability_predicted_classes = np.argmax(
    stability_probabilities,
    axis=1,
)

stability_confidence = np.max(
    stability_probabilities,
    axis=1,
)

full_test_class_counts = np.bincount(
    y_explain,
    minlength=EXPECTED_CLASSES,
)

stability_class_counts = np.bincount(
    y_stability,
    minlength=EXPECTED_CLASSES,
)

stability_weights = np.asarray(
    [
        (
            full_test_class_counts[cls]
            / len(y_explain)
        )
        / stability_class_counts[cls]
        for cls in y_stability
    ],
    dtype=np.float64,
)

if not np.isclose(
    stability_weights.sum(),
    1.0,
    atol=1e-12,
):
    raise RuntimeError(
        "Stability-sample weights "
        "do not sum to one."
    )

stability_observations_df = pd.DataFrame(
    {
        "shap_sample_position": np.arange(
            len(stability_test_indices)
        ),
        "test_relative_index": (
            stability_test_indices
        ),
        "combined_dataset_index": (
            X_test_raw
            .index
            .to_numpy()[
                stability_test_indices
            ]
        ),
        "actual_class_index": y_stability,
        "actual_class": [
            str(device_classes[value])
            for value in y_stability
        ],
        "predicted_class_index": (
            stability_predicted_classes
        ),
        "predicted_class": [
            str(device_classes[value])
            for value in stability_predicted_classes
        ],
        "correct": (
            y_stability
            == stability_predicted_classes
        ),
        "prediction_confidence": (
            stability_confidence
        ),
        "test_distribution_weight": (
            stability_weights
        ),
    }
)

stability_observations_df.to_csv(
    COMMON_DIR
    / "Stability_Explained_Test_Observations.csv",
    index=False,
)

feature_family_df = pd.DataFrame(
    {
        "feature_index": np.arange(
            EXPECTED_FEATURES
        ),
        "feature": feature_names,
        "feature_family": [
            assign_feature_family(feature)
            for feature in feature_names
        ],
    }
)

unclassified = feature_family_df[
    feature_family_df["feature_family"]
    == "Other / unclassified"
]

if not unclassified.empty:
    display(unclassified)
    raise RuntimeError(
        "The final feature-family mapping "
        "contains unclassified predictors."
    )

family_counts_df = (
    feature_family_df["feature_family"]
    .value_counts()
    .rename_axis("feature_family")
    .reset_index(name="number_of_features")
)

family_names = sorted(
    feature_family_df[
        "feature_family"
    ]
    .unique()
    .tolist()
)

family_to_indices = {
    family: (
        feature_family_df.loc[
            feature_family_df[
                "feature_family"
            ] == family,
            "feature_index",
        ]
        .to_numpy(dtype=int)
    )
    for family in family_names
}

feature_family_df.to_csv(
    COMMON_DIR / "Feature_Family_Mapping.csv",
    index=False,
)

family_counts_df.to_csv(
    COMMON_DIR / "Feature_Family_Counts.csv",
    index=False,
)


def predict_wrapper(X_2d):
    X_2d = np.asarray(
        X_2d,
        dtype=np.float32,
    )

    if X_2d.ndim == 1:
        X_2d = X_2d.reshape(1, -1)

    if X_2d.shape[1] != EXPECTED_FEATURES:
        raise RuntimeError(
            f"Expected {EXPECTED_FEATURES} features; "
            f"received {X_2d.shape[1]}."
        )

    return np.asarray(
        model.predict(
            X_2d[..., np.newaxis],
            verbose=0,
        ),
        dtype=np.float64,
    )


if not np.allclose(
    predict_wrapper(X_explain[:5]),
    explain_probabilities[:5],
    atol=1e-6,
    rtol=1e-6,
):
    raise RuntimeError(
        "Prediction-wrapper integrity check failed."
    )

common_metadata = {
    "run_directory": str(RUN_DIR),
    "selected_model": str(MODEL_PATH),
    "selected_model_sha256": sha256_file(
        MODEL_PATH
    ),
    "split_indices_sha256": sha256_file(
        SPLIT_INDICES_PATH
    ),
    "x_test_sha256": sha256_file(
        X_TEST_PATH
    ),
    "y_test_sha256": sha256_file(
        Y_TEST_PATH
    ),
    "device_classes_sha256": sha256_file(
        CLASSES_PATH
    ),
    "feature_names_sha256": sha256_file(
        FEATURE_NAMES_PATH
    ),
    "selected_seed": CHAMPION_SEED,
    "unique_observations": int(
        len(combined_df)
    ),
    "split": {
        "train": EXPECTED_TRAIN_ROWS,
        "validation": EXPECTED_VALIDATION_ROWS,
        "test": EXPECTED_TEST_ROWS,
    },
    "features": EXPECTED_FEATURES,
    "classes": EXPECTED_CLASSES,
    "heldout_accuracy": test_accuracy,
    "heldout_macro_f1": test_macro_f1,
    "background_size": BACKGROUND_SIZE,
    "background_source": (
        "20 real pre-SMOTE training representatives"
    ),
    "stability_subset_size": int(
        len(stability_test_indices)
    ),
    "stability_seeds": STABILITY_SEEDS,
    "stability_nsamples": STABILITY_NSAMPLES,
    "full_kernel_nsamples": KERNEL_NSAMPLES,
    "partition_max_evals": PARTITION_MAX_EVALS,
    "feature_family_count": int(
        len(family_names)
    ),
    "unclassified_feature_count": 0,
}

with open(
    COMMON_DIR / "SHAP_Common_Metadata.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        common_metadata,
        handle,
        indent=2,
    )

print("=" * 100)
print("FINAL SHAP COMMON SETUP COMPLETE")
print("=" * 100)
print(f"Selected seed            : {CHAMPION_SEED}")
print(
    f"Model/source max diff    : "
    f"{champion_seed_max_difference:.12g}"
)
print(f"Unique observations      : {len(combined_df)}")
print(f"Split                    : {actual_split}")
print(f"Predictors               : {len(feature_names)}")
print(f"Held-out cases           : {len(X_explain)}")
print(f"Held-out accuracy        : {test_accuracy * 100:.3f}%")
print(f"Held-out macro-F1        : {test_macro_f1:.4f}")
print(
    f"Shared background        : "
    f"{len(X_background)} real training cases"
)
print(f"Stability subset         : {len(X_stability)}")
print(f"Feature families         : {len(family_names)}")
print("\nFeature-family counts:")
display(family_counts_df)
print(f"\nSHAP output root:\n{SHAP_ROOT}")
print("\nRun Block 2 next.")

### Kernel SHAP Monte-Carlo Stability Analysis

This section evaluates the **run-to-run stability of Kernel SHAP explanations** for the retained CNN-BiLSTM model before the full held-out test set is explained.

It uses the common SHAP setup established in the preceding block, including:

- the fixed validation-selected CNN-BiLSTM;
- the same 20 real pre-SMOTE training observations as the SHAP background;
- the fixed 100-case class-balanced held-out stability subset;
- the test-distribution correction weights;
- the 244 retained predictors;
- the 10 device classes; and
- the common semantic feature-family mapping.

The purpose of this block is to determine whether the feature-importance conclusions produced by Kernel SHAP remain sufficiently consistent across repeated stochastic runs at the selected computational budget.

#### Kernel SHAP Configuration

A single `KernelExplainer` is constructed using the saved prediction wrapper and the common 20-case background:

```text
Background size:          20 real training observations
Stability subset size:   100 held-out observations
Samples per class:       10
Kernel SHAP nsamples:    4096
Number of repetitions:   3
Seeds:                   1701, 2701, 3701
L1 regularisation:       0.0
```

The same:

- trained model;
- background observations;
- explained observations;
- preprocessing state; and
- SHAP computational budget

are retained across all three repetitions.

Only the random seed changes between repetitions.

#### Expected Values

The Kernel SHAP expected values are extracted once from the shared explainer.

Because the classifier has 10 output classes, the expected-value vector must contain exactly:

`10 values`

The expected values are saved to:

`SHAP_Expected_Values.npy`

The block also preserves copies of the background observations, explained test observations and feature-family mapping inside the stability-results directory so that the stability experiment remains self-describing.

Three 4096-Sample SHAP Repetitions

Kernel SHAP is executed three times on the same 100 held-out observations.

Before each repetition, the following random generators are assigned the corresponding fixed seed:

- NumPy;
- Python `random`; and
- TensorFlow.

Each SHAP result is normalised into the common multiclass tensor shape:

`(100 observations, 244 features, 10 classes)`

Execution stops if a repetition does not produce the expected dimensions.

The complete multiclass SHAP tensor from each repetition is preserved as:

```text
SHAP_Values_4096_Rep_1_Seed_1701.npy
SHAP_Values_4096_Rep_2_Seed_2701.npy
SHAP_Values_4096_Rep_3_Seed_3701.npy
```

#### Predicted-Decision SHAP Values

Although the complete multiclass SHAP tensor is retained, global feature importance is calculated for the class actually predicted for each held-out observation.

For every sample, the block extracts the SHAP vector corresponding to that observation's predicted class.

This produces a predicted-decision explanation matrix with dimensions:

`100 × 244`

Absolute SHAP values are then averaged using the previously calculated test-distribution weights.

These weights correct for the deliberately class-balanced 100-case stability sample so that aggregate importance better reflects the class proportions of the complete held-out test set.

#### Per-Repetition Feature Importance

For every repetition, the block creates a ranked table containing:

- feature rank;
- feature index;
- feature name;
- semantic feature family; and
- weighted mean absolute SHAP value.

The three per-run feature-importance files follow the naming pattern:

```text
Feature_Importance_4096_Rep_1_Seed_1701.csv
Feature_Importance_4096_Rep_2_Seed_2701.csv
Feature_Importance_4096_Rep_3_Seed_3701.csv
```

These files are the repetition-specific feature-importance results for the final stability experiment.

#### Feature-Level Stability Metrics

Each pair of SHAP repetitions is compared using several complementary agreement measures.

For all 244 feature-importance values, the block calculates:

- Spearman rank correlation;
- cosine similarity; and
- relative L1 difference.

It also compares the overlap between the highest-ranked features using:

- top-10 overlap;
- top-20 overlap; and
- top-50 overlap.

With three SHAP repetitions, this produces the three pairwise comparisons:

```text
Repetition 1 vs Repetition 2
Repetition 1 vs Repetition 3
Repetition 2 vs Repetition 3
```

The results are saved to:

`SHAP_4096_Feature_Stability.csv`

These statistics quantify whether the overall ranking and the most influential predictors remain consistent across independent seeded SHAP approximations.

#### Consensus SHAP Importance

The three complete SHAP tensors are averaged to create a consensus explanation tensor.

Predicted-decision SHAP values are then extracted from this averaged tensor, and class-distribution-weighted absolute importance is recalculated.

For every feature, the consensus analysis records:

- mean weighted absolute SHAP importance;
- standard deviation of weighted importance across the three runs;
- number of repetitions in which the feature appeared in the top 10;
- number of repetitions in which the feature appeared in the top 20;
- mean rank;
- minimum rank; and
- maximum rank.

The complete consensus table is saved as:

`SHAP_4096_Feature_Consensus.csv`

Two stricter consensus subsets are also produced:

- `SHAP_4096_Strong_Top10_Consensus.csv` — features appearing in the top 10 in all three repetitions;
- `SHAP_4096_Strong_Top20_Consensus.csv` — features appearing in the top 20 in all three repetitions.

#### Averaged Global Feature Importance

A final ranked feature-level table is generated from the averaged SHAP explanation.

It contains:

- rank;
- feature index;
- feature name;
- feature family;
- weighted mean absolute SHAP; and
- run-to-run standard deviation.

The table is saved as:

`SHAP_4096_Averaged_Global_Importance.csv`

This provides the principal feature-level summary of the three-run stability experiment.

#### Feature-Family Stability

Predicted-decision SHAP values from each repetition are also aggregated according to the common semantic feature-family mapping established in the SHAP setup block.

For every repetition and feature family, the analysis calculates:

- number of predictors in the family;
- total weighted mean absolute SHAP;
- share of total absolute SHAP; and
- mean absolute SHAP per feature.

The repetition-level results are saved to:

`SHAP_4096_Family_By_Run.csv`

A family-level consensus table then reports:

- mean share of total absolute SHAP;
- standard deviation of family share across runs;
- mean total weighted absolute SHAP; and
- mean absolute SHAP per feature.

This is saved as:

`SHAP_4096_Family_Consensus.csv`

#### Feature-Family Agreement

The semantic-family rankings are also compared pairwise between the three repetitions.

The family-level stability analysis calculates:

- Spearman rank correlation;
- top-3 family overlap; and
- top-5 family overlap.

These results are written to:

`SHAP_4096_Family_Stability.csv`

This provides a higher-level stability assessment that complements the individual-feature comparison.

#### Local-Fidelity Check

For each explained observation, the predicted-class probability is reconstructed from:

`Kernel SHAP expected value + sum of predicted-class SHAP values`

The reconstructed value is compared with the CNN-BiLSTM's actual predicted-class probability.

The block records the:

- mean absolute reconstruction error; and
- maximum absolute reconstruction error

for each of the three repetitions and for the averaged SHAP explanation.

The results are saved to:

`SHAP_4096_Local_Fidelity.csv`

This checks that the generated explanations retain the expected local additive relationship with the model output.

#### Stability-Sample Class Summary

The code also records the predictive behaviour of the fixed 100-case explanation sample by actual device class.

For each class, it reports:

- number of explained observations;
- number classified correctly; and
- sample accuracy.

The results are saved to:

`SHAP_4096_Explanation_Sample_By_Class.csv`

This provides context for interpreting SHAP results derived from the deliberately class-balanced explanation subset.

#### Stability Summary

A machine-readable summary is written to:

`SHAP_4096_Analysis_Summary.json`

It records the main characteristics and stability results of the experiment, including:

- experimental design;
- SHAP background definition;
- explanation-sample size;
- explanation-sample accuracy;
- minimum and maximum feature-level Spearman agreement;
- minimum and maximum top-k feature overlap;
- minimum and maximum cosine similarity;
- number of strong top-10 consensus features;
- number of strong top-20 consensus features;
- family-level rank and overlap stability; and
- averaged local-fidelity errors.

#### Stability Figures

Three principal figures are generated in the stability `Figures` directory.

**Global feature consensus**

Displays the 20 most important predictors according to the averaged weighted absolute Kernel SHAP values.

**Feature-family importance**

Displays the mean share of total absolute SHAP assigned to each semantic feature family across the three repetitions.

**Run-to-run stability**

Displays the pairwise agreement between the three repetitions using:

- all-feature Spearman correlation;
- top-10 overlap; and
- top-20 overlap.

Each figure is saved in both PNG and PDF format.

#### Output Structure

The stability experiment is stored under:

```text
SHAP_Final/
└── Stability_3x4096/
    ├── SHAP_Expected_Values.npy
    ├── Background_Observations.csv
    ├── Explained_Test_Observations.csv
    ├── Feature_Family_Mapping.csv
    ├── SHAP_Values_4096_Rep_1_Seed_1701.npy
    ├── SHAP_Values_4096_Rep_2_Seed_2701.npy
    ├── SHAP_Values_4096_Rep_3_Seed_3701.npy
    ├── Feature_Importance_4096_Rep_1_Seed_1701.csv
    ├── Feature_Importance_4096_Rep_2_Seed_2701.csv
    ├── Feature_Importance_4096_Rep_3_Seed_3701.csv
    ├── SHAP_4096_Feature_Stability.csv
    ├── SHAP_4096_Feature_Consensus.csv
    ├── SHAP_4096_Strong_Top10_Consensus.csv
    ├── SHAP_4096_Strong_Top20_Consensus.csv
    ├── SHAP_4096_Averaged_Global_Importance.csv
    ├── SHAP_4096_Family_By_Run.csv
    ├── SHAP_4096_Family_Consensus.csv
    ├── SHAP_4096_Family_Stability.csv
    ├── SHAP_4096_Local_Fidelity.csv
    ├── SHAP_4096_Explanation_Sample_By_Class.csv
    ├── SHAP_4096_Analysis_Summary.json
    └── Figures/
```

#### Methodological Role

This section is a **Monte-Carlo stability audit for Kernel SHAP**.

It does not:

- retrain the CNN-BiLSTM;
- alter the selected model;
- change the SHAP background;
- select features for model training;
- use SHAP results to modify predictive performance;
- explain the complete 376-case held-out set; or
- perform the later correlation-aware Partition SHAP comparison.

Its role is to establish that the principal Kernel SHAP feature and feature-family conclusions are sufficiently stable at the selected `nsamples = 4096` computational budget before the full held-out explanation is produced.

In [ ]:
print("=" * 100)
print("FINAL 3 x 4096 KERNEL SHAP STABILITY AUDIT")
print("=" * 100)

stability_explainer = shap.KernelExplainer(predict_wrapper, X_background)

stability_expected_values = np.asarray(stability_explainer.expected_value, dtype=np.float64).reshape(-1)
if stability_expected_values.shape != (EXPECTED_CLASSES,):
    raise RuntimeError(f"Unexpected Kernel SHAP expected-value shape: {stability_expected_values.shape}")

np.save(STABILITY_DIR / "SHAP_Expected_Values.npy", stability_expected_values)
background_df.to_csv(STABILITY_DIR / "Background_Observations.csv", index=False)
stability_observations_df.to_csv(STABILITY_DIR / "Explained_Test_Observations.csv", index=False)
feature_family_df.to_csv(STABILITY_DIR / "Feature_Family_Mapping.csv", index=False)

runs = []
for rep, seed in enumerate(STABILITY_SEEDS, start=1):
    print("\n" + "-" * 100)
    print(f"Kernel SHAP | nsamples={STABILITY_NSAMPLES} | repetition={rep}/3 | seed={seed}")
    print("-" * 100)

    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)

    raw_values = stability_explainer.shap_values(X_stability, nsamples=STABILITY_NSAMPLES, l1_reg=0.0)
    shap_array = normalise_multiclass_shap(raw_values, n_samples=len(X_stability))

    expected_shape = (len(X_stability), EXPECTED_FEATURES, EXPECTED_CLASSES)
    if shap_array.shape != expected_shape:
        raise RuntimeError(f"Unexpected stability SHAP shape: {shap_array.shape}")

    decision_shap = shap_array[np.arange(len(X_stability)), :, stability_predicted_classes]
    importance = np.average(np.abs(decision_shap), axis=0, weights=stability_weights)
    reconstructed = stability_expected_values[stability_predicted_classes] + decision_shap.sum(axis=1)
    residual = np.abs(reconstructed - stability_confidence)

    np.save(STABILITY_DIR / f"SHAP_Values_4096_Rep_{rep}_Seed_{seed}.npy", shap_array)

    per_run_df = pd.DataFrame({
        "feature_index": np.arange(EXPECTED_FEATURES),
        "feature": feature_names,
        "feature_family": feature_family_df["feature_family"].to_numpy(),
        "weighted_mean_abs_shap": importance,
    }).sort_values("weighted_mean_abs_shap", ascending=False).reset_index(drop=True)
    per_run_df.insert(0, "rank", np.arange(1, len(per_run_df) + 1))
    per_run_df.to_csv(STABILITY_DIR / f"Feature_Importance_4096_Rep_{rep}_Seed_{seed}.csv", index=False)

    runs.append({
        "rep": rep,
        "seed": seed,
        "shap": shap_array,
        "decision_shap": decision_shap,
        "importance": importance,
        "residual": residual,
    })

    del raw_values
    gc.collect()

feature_stability_rows = []
for i in range(len(runs)):
    for j in range(i + 1, len(runs)):
        a = runs[i]["importance"]
        b = runs[j]["importance"]
        cosine_denom = np.linalg.norm(a) * np.linalg.norm(b)
        cosine = float(np.dot(a, b) / cosine_denom)

        feature_stability_rows.append({
            "rep_a": runs[i]["rep"],
            "seed_a": runs[i]["seed"],
            "rep_b": runs[j]["rep"],
            "seed_b": runs[j]["seed"],
            "all_feature_spearman": safe_spearman(a, b),
            "top10_overlap_fraction": top_overlap(a, b, 10),
            "top20_overlap_fraction": top_overlap(a, b, 20),
            "top50_overlap_fraction": top_overlap(a, b, 50),
            "cosine_similarity": cosine,
            "relative_l1_difference": float(np.sum(np.abs(a - b)) / np.sum(np.abs(b))),
        })

feature_stability_df = pd.DataFrame(feature_stability_rows)
feature_stability_df.to_csv(STABILITY_DIR / "SHAP_4096_Feature_Stability.csv", index=False)

averaged_shap = np.mean(np.stack([run["shap"] for run in runs], axis=0), axis=0)
averaged_decision_shap = averaged_shap[np.arange(len(averaged_shap)), :, stability_predicted_classes]
averaged_importance = np.average(np.abs(averaged_decision_shap), axis=0, weights=stability_weights)

run_importances = np.stack([run["importance"] for run in runs], axis=0)
run_to_run_sd = np.std(run_importances, axis=0, ddof=1)

ranks, top10_sets, top20_sets = [], [], []
for importance in run_importances:
    order = np.argsort(importance)[::-1]
    rank = np.empty(len(importance), dtype=int)
    rank[order] = np.arange(1, len(importance) + 1)
    ranks.append(rank)
    top10_sets.append(set(order[:10]))
    top20_sets.append(set(order[:20]))

ranks = np.stack(ranks, axis=0)

consensus_df = pd.DataFrame({
    "feature_index": np.arange(EXPECTED_FEATURES),
    "feature": feature_names,
    "feature_family": feature_family_df["feature_family"].to_numpy(),
    "mean_weighted_abs_shap": averaged_importance,
    "sd_weighted_abs_shap_across_runs": run_to_run_sd,
    "top10_presence_count": [sum(idx in s for s in top10_sets) for idx in range(EXPECTED_FEATURES)],
    "top20_presence_count": [sum(idx in s for s in top20_sets) for idx in range(EXPECTED_FEATURES)],
    "mean_rank": ranks.mean(axis=0),
    "minimum_rank": ranks.min(axis=0),
    "maximum_rank": ranks.max(axis=0),
}).sort_values(["top10_presence_count", "top20_presence_count", "mean_weighted_abs_shap"], ascending=[False, False, False]).reset_index(drop=True)

consensus_df.to_csv(STABILITY_DIR / "SHAP_4096_Feature_Consensus.csv", index=False)
consensus_df[consensus_df["top10_presence_count"] == 3].to_csv(STABILITY_DIR / "SHAP_4096_Strong_Top10_Consensus.csv", index=False)
consensus_df[consensus_df["top20_presence_count"] == 3].to_csv(STABILITY_DIR / "SHAP_4096_Strong_Top20_Consensus.csv", index=False)

global_df = pd.DataFrame({
    "feature_index": np.arange(EXPECTED_FEATURES),
    "feature": feature_names,
    "feature_family": feature_family_df["feature_family"].to_numpy(),
    "weighted_mean_abs_shap": averaged_importance,
    "run_to_run_sd": run_to_run_sd,
}).sort_values("weighted_mean_abs_shap", ascending=False).reset_index(drop=True)
global_df.insert(0, "rank", np.arange(1, len(global_df) + 1))
global_df.to_csv(STABILITY_DIR / "SHAP_4096_Averaged_Global_Importance.csv", index=False)

family_by_run_rows = []
family_share_vectors = []
for run in runs:
    family_df = aggregate_family_importance(run["decision_shap"], sample_weights=stability_weights)
    family_df.insert(0, "seed", run["seed"])
    family_df.insert(0, "rep", run["rep"])
    family_by_run_rows.append(family_df)

    ordered = family_df.set_index("feature_family").loc[family_names, "share_of_total_abs_shap"].to_numpy(dtype=np.float64)
    family_share_vectors.append(ordered)

family_by_run_df = pd.concat(family_by_run_rows, ignore_index=True)
family_by_run_df.to_csv(STABILITY_DIR / "SHAP_4096_Family_By_Run.csv", index=False)

family_share_vectors = np.stack(family_share_vectors, axis=0)

family_consensus_rows = []
for family_index, family in enumerate(family_names):
    shares = family_share_vectors[:, family_index]
    family_run_subset = family_by_run_df[family_by_run_df["feature_family"] == family]
    family_consensus_rows.append({
        "feature_family": family,
        "number_of_features": int(len(family_to_indices[family])),
        "mean_share_of_total_abs_shap": float(np.mean(shares)),
        "sd_share_across_runs": float(np.std(shares, ddof=1)),
        "mean_total_weighted_abs_shap": float(family_run_subset["total_mean_abs_shap"].mean()),
        "mean_abs_shap_per_feature": float(family_run_subset["mean_abs_shap_per_feature"].mean()),
    })

family_consensus_df = pd.DataFrame(family_consensus_rows).sort_values("mean_share_of_total_abs_shap", ascending=False).reset_index(drop=True)
family_consensus_df.insert(0, "rank", np.arange(1, len(family_consensus_df) + 1))
family_consensus_df.to_csv(STABILITY_DIR / "SHAP_4096_Family_Consensus.csv", index=False)

family_stability_rows = []
for i in range(3):
    for j in range(i + 1, 3):
        a = family_share_vectors[i]
        b = family_share_vectors[j]
        family_stability_rows.append({
            "rep_a": runs[i]["rep"],
            "seed_a": runs[i]["seed"],
            "rep_b": runs[j]["rep"],
            "seed_b": runs[j]["seed"],
            "family_spearman": safe_spearman(a, b),
            "top3_family_overlap_fraction": top_overlap(a, b, 3),
            "top5_family_overlap_fraction": top_overlap(a, b, 5),
        })

family_stability_df = pd.DataFrame(family_stability_rows)
family_stability_df.to_csv(STABILITY_DIR / "SHAP_4096_Family_Stability.csv", index=False)

averaged_reconstructed_probability = stability_expected_values[stability_predicted_classes] + averaged_decision_shap.sum(axis=1)
averaged_residual = np.abs(averaged_reconstructed_probability - stability_confidence)

fidelity_rows = [
    {
        "rep": run["rep"],
        "seed": run["seed"],
        "mean_predicted_class_reconstruction_error": float(run["residual"].mean()),
        "max_predicted_class_reconstruction_error": float(run["residual"].max()),
    }
    for run in runs
]
fidelity_rows.append({
    "rep": "AVERAGED",
    "seed": "-",
    "mean_predicted_class_reconstruction_error": float(averaged_residual.mean()),
    "max_predicted_class_reconstruction_error": float(averaged_residual.max()),
})
fidelity_df = pd.DataFrame(fidelity_rows)
fidelity_df.to_csv(STABILITY_DIR / "SHAP_4096_Local_Fidelity.csv", index=False)

class_support_df = (
    stability_observations_df
    .groupby("actual_class", sort=False)["correct"]
    .agg(correct="sum", sample_count="count")
    .reset_index()
)
class_support_df["sample_accuracy"] = class_support_df["correct"] / class_support_df["sample_count"]
class_support_df.to_csv(STABILITY_DIR / "SHAP_4096_Explanation_Sample_By_Class.csv", index=False)

summary = {
    "design": "Three independent Kernel SHAP runs at nsamples=4096 on the same fixed 100-case class-balanced held-out subset.",
    "background": "20 real pre-SMOTE training representatives",
    "explanation_sample_size": int(len(X_stability)),
    "explanation_sample_accuracy": float(np.mean(y_stability == stability_predicted_classes)),
    "feature_stability": {
        "all_feature_spearman_min": float(feature_stability_df["all_feature_spearman"].min()),
        "all_feature_spearman_max": float(feature_stability_df["all_feature_spearman"].max()),
        "top10_overlap_min": float(feature_stability_df["top10_overlap_fraction"].min()),
        "top10_overlap_max": float(feature_stability_df["top10_overlap_fraction"].max()),
        "top20_overlap_min": float(feature_stability_df["top20_overlap_fraction"].min()),
        "top20_overlap_max": float(feature_stability_df["top20_overlap_fraction"].max()),
        "top50_overlap_min": float(feature_stability_df["top50_overlap_fraction"].min()),
        "top50_overlap_max": float(feature_stability_df["top50_overlap_fraction"].max()),
        "cosine_min": float(feature_stability_df["cosine_similarity"].min()),
        "cosine_max": float(feature_stability_df["cosine_similarity"].max()),
        "strong_top10_consensus_count": int((consensus_df["top10_presence_count"] == 3).sum()),
        "strong_top20_consensus_count": int((consensus_df["top20_presence_count"] == 3).sum()),
    },
    "family_stability": {
        "pairwise_spearman_min": float(family_stability_df["family_spearman"].min()),
        "pairwise_spearman_max": float(family_stability_df["family_spearman"].max()),
        "top3_overlap_min": float(family_stability_df["top3_family_overlap_fraction"].min()),
        "top5_overlap_min": float(family_stability_df["top5_family_overlap_fraction"].min()),
    },
    "local_fidelity": {
        "averaged_mean_predicted_class_reconstruction_error": float(averaged_residual.mean()),
        "averaged_max_predicted_class_reconstruction_error": float(averaged_residual.max()),
    },
}

with open(STABILITY_DIR / "SHAP_4096_Analysis_Summary.json", "w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

plot_df = global_df.head(20).sort_values("weighted_mean_abs_shap")
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(plot_df["feature"], plot_df["weighted_mean_abs_shap"])
ax.set_xlabel("Weighted mean |Kernel SHAP value|")
ax.set_ylabel("Feature")
ax.set_title("CNN-BiLSTM Kernel SHAP Stability Consensus (3 x 4096)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_figure(fig, STABILITY_FIGURES_DIR / "SHAP_4096_Global_Consensus")
plt.show()
plt.close(fig)

family_plot_df = family_consensus_df.sort_values("mean_share_of_total_abs_shap")
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.barh(family_plot_df["feature_family"], family_plot_df["mean_share_of_total_abs_shap"] * 100.0)
ax.set_xlabel("Mean share of total |SHAP| (%)")
ax.set_ylabel("Feature family")
ax.set_title("Kernel SHAP Stability Consensus by Feature Family")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_figure(fig, STABILITY_FIGURES_DIR / "SHAP_4096_Family_Importance")
plt.show()
plt.close(fig)

labels = [f"Rep {int(row.rep_a)} vs {int(row.rep_b)}" for row in feature_stability_df.itertuples()]
x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(x, feature_stability_df["all_feature_spearman"], marker="o", label="All-feature Spearman")
ax.plot(x, feature_stability_df["top10_overlap_fraction"], marker="o", label="Top-10 overlap")
ax.plot(x, feature_stability_df["top20_overlap_fraction"], marker="o", label="Top-20 overlap")
ax.set_ylim(0, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Agreement")
ax.set_title("Kernel SHAP Run-to-Run Stability")
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_figure(fig, STABILITY_FIGURES_DIR / "SHAP_4096_Consensus_Stability")
plt.show()
plt.close(fig)

print("\n" + "=" * 100)
print("3 x 4096 KERNEL SHAP STABILITY AUDIT COMPLETE")
print("=" * 100)
print("\nFeature-level stability:")
display(feature_stability_df)
print("\nFamily-level stability:")
display(family_stability_df)
print("\nTop-20 consensus features:")
display(global_df.head(20))
print("\nFamily consensus:")
display(family_consensus_df)
print("\nLocal fidelity:")
display(fidelity_df)
print(f"\nSaved to:\n{STABILITY_DIR}")
print("\nRun Block 3 next.")

### Full Held-Out Kernel SHAP Analysis

This section computes the final Kernel SHAP explanation over all 376 held-out observations for the validation-selected CNN-BiLSTM.

The SHAP Common Setup cell must have been run at least once for the selected model so that the common metadata, background-observation record and feature-family mapping exist in Google Drive. However, this full Kernel SHAP cell does not require the Common Setup cell to have been run in the current Colab session.

In a fresh runtime, the cell mounts Google Drive, discovers compatible CNN-BiLSTM runs, asks the user to select the intended run and independently reloads the saved model, test tensors, feature names, preprocessing objects, split indices and SHAP common artifacts. It reconstructs and verifies the corrected corpus and the 1,200/301/376 split before resuming the explanation.

The cell validates saved metadata and checkpoints so that chunks from different models, datasets, feature orders or SHAP backgrounds cannot be combined silently.

#### Full Held-Out Explanation Set

Kernel SHAP is evaluated on the complete held-out test partition:

`376 observations`

using:

`nsamples = 4096`

for each explained case.

No class balancing or sample weighting is applied in this full-test analysis because the complete held-out test distribution is explained directly.

The shared SHAP background remains:

`20 real pre-SMOTE training representatives`

#### Kernel SHAP Explainer

A `KernelExplainer` is created using:

- the fixed CNN-BiLSTM prediction wrapper; and
- the common 20-observation training background.

The resulting expected-value vector must contain one value for each of the 10 output classes.

The expected values are saved as:

`Kernel_SHAP_Full376_Expected_Values.npy`

#### Checkpointed SHAP Computation

Because full Kernel SHAP over 376 observations is computationally expensive, the held-out set is processed in small checkpoints.

The configured chunk size is:

`10 observations`

Each checkpoint therefore stores a SHAP tensor with the form:

`(chunk observations, 244 features, 10 classes)`

Checkpoint files follow the naming convention:

```text
kernel_4096_<start>_<end>.npy
```

and are written to:

`SHAP_Final/Full376/Kernel_Checkpoints/`

A `Kernel_SHAP_Resume_Metadata.json` file records the selected-model hash, dataset hashes, background hash, SHAP budget and expected tensor dimensions. Existing checkpoints are reused only when this metadata matches the current analysis. Checkpoints and the final tensor are saved atomically through a temporary file before replacing the final `.npy` path. Setting `FORCE_RECOMPUTE=True` deliberately removes the existing Kernel checkpoints and starts again from observation zero; its default value is `False`.

Before recomputing a chunk, the code checks whether a corresponding checkpoint already exists with the expected shape.

Valid completed chunks are reused automatically.

If a checkpoint exists but has an invalid shape, it is deleted and recomputed.

This makes the full Kernel SHAP process resumable after Colab runtime interruptions.

#### Deterministic Chunk Seeds

Each Kernel SHAP chunk receives a deterministic random seed derived from:

`KERNEL_BASE_SEED + chunk_start_index`

The NumPy, Python and TensorFlow random generators are all set before computing each chunk.

This provides reproducible stochastic approximation while allowing the complete 376-case analysis to be split into independently restartable sections.

#### Full Multiclass SHAP Tensor

After all chunks have been completed, the checkpoint tensors are concatenated into the final multiclass SHAP tensor:

`(376, 244, 10)`

The resulting tensor is saved as:

`Kernel_SHAP_Full376_4096.npy`

This artifact contains a SHAP value for every combination of:

- held-out observation;
- retained predictor; and
- output class.

#### Local Additivity and Fidelity Check

The full multiclass model output is reconstructed using:

`expected value + sum of SHAP values`

for every observation and every output class.

The reconstructed probabilities are compared directly with the CNN-BiLSTM prediction probabilities.

For every held-out case, the code records:

- actual class;
- predicted class;
- whether the classification was correct;
- mean output reconstruction residual; and
- maximum output reconstruction residual.

The results are saved to:

`Kernel_SHAP_Full376_Local_Fidelity.csv`

The overall mean and maximum reconstruction residuals are also reported.

These values provide a numerical check that the Kernel SHAP explanations satisfy the expected additive relationship with the model outputs to high numerical precision.

#### Predicted-Decision SHAP Importance

For global interpretation, the SHAP values associated with each observation's **predicted class** are extracted.

This produces a decision-specific explanation matrix with dimensions:

`376 × 244`

The absolute predicted-decision SHAP values are then aggregated across all held-out observations.

For every predictor, the analysis calculates:

- mean absolute Kernel SHAP value; and
- standard deviation of absolute Kernel SHAP values across the 376 held-out cases.

The resulting ranked table is saved as:

`Kernel_SHAP_Full376_Global_Importance.csv`

#### Global Feature Importance

The global feature-importance table contains:

- rank;
- feature index;
- feature name;
- semantic feature family;
- mean absolute Kernel SHAP value; and
- standard deviation across held-out observations.

The ranking therefore reflects how strongly each predictor contributes, on average, to the model's actual predicted decisions over the complete held-out test set.

#### Semantic Feature-Family Importance

The predicted-decision SHAP values are also aggregated using the semantic feature-family mapping defined in the common setup block.

For each family, the analysis calculates:

- number of features;
- total mean absolute SHAP contribution;
- mean absolute SHAP per feature; and
- share of total absolute SHAP.

The resulting table is saved as:

`Kernel_SHAP_Full376_Family_Importance.csv`

This provides a higher-level interpretation of which categories of network-flow information contribute most strongly to the CNN-BiLSTM's held-out predictions.

#### Class-Conditional SHAP Analysis

A separate class-specific analysis is performed for each device category.

For a given true class, the code selects all held-out observations belonging to that class and examines the SHAP values associated with that class's model output.

For every device category, it records:

- class support;
- number of correctly classified observations; and
- top 20 features ranked by mean absolute SHAP for the true-class output.

The results are saved to:

`Kernel_SHAP_Full376_Class_Conditional_Top20.csv`

This allows the global explanation to be supplemented with class-specific evidence about which predictors are most influential for each appliance category.

#### Global Feature-Importance Figure

The 20 highest-ranked global predictors are visualised using their mean absolute predicted-decision Kernel SHAP values.

The figure is saved as:

```text
01_Kernel_SHAP_Full376_Global.png
01_Kernel_SHAP_Full376_Global.pdf
```

This figure provides the principal global feature-level summary of the full-test Kernel SHAP analysis.

#### SHAP Beeswarm Summary

A standard SHAP summary plot is generated from the full predicted-decision SHAP matrix and the corresponding held-out feature values.

The plot displays the 20 most influential predictors and shows both:

- magnitude of SHAP contribution; and
- direction of contribution across individual held-out observations.

The figure is saved as:

```text
02_Kernel_SHAP_Full376_Beeswarm.png
02_Kernel_SHAP_Full376_Beeswarm.pdf
```

#### Feature-Family Figure

The semantic feature families are visualised according to their share of total absolute Kernel SHAP contribution.

The figure is saved as:

```text
03_Kernel_SHAP_Full376_Families.png
03_Kernel_SHAP_Full376_Families.pdf
```

This provides a compact architecture-level view of which types of traffic information dominate the model's decisions.

#### Highest-Confidence Misclassification

If the retained model makes at least one error on the held-out test set, the code identifies the misclassified observation with the highest prediction confidence.

A local Kernel SHAP waterfall explanation is generated for the **predicted class** of that observation.

The figure shows how individual predictors contributed to the incorrect high-confidence decision.

Its title reports both:

- the true device class; and
- the predicted device class.

The figure is saved as:

```text
04_Kernel_SHAP_High_Confidence_Error.png
04_Kernel_SHAP_High_Confidence_Error.pdf
```

This provides a concrete local explanation of a difficult model failure.

#### Kernel SHAP Summary Metadata

A machine-readable summary is written to:

`Kernel_SHAP_Full376_Summary.json`

The summary records:

- SHAP method;
- number of explained cases;
- Kernel SHAP sample count;
- background size;
- background definition;
- whether sample weighting was used;
- held-out accuracy;
- held-out macro-F1;
- mean local-fidelity residual;
- maximum local-fidelity residual; and
- checkpoint chunk size.

#### Output Structure

The principal full-test Kernel SHAP artifacts are stored under:

```text
SHAP_Final/
└── Full376/
    ├── Kernel_SHAP_Full376_Expected_Values.npy
    ├── Kernel_SHAP_Full376_4096.npy
    ├── Kernel_SHAP_Full376_Local_Fidelity.csv
    ├── Kernel_SHAP_Full376_Global_Importance.csv
    ├── Kernel_SHAP_Full376_Family_Importance.csv
    ├── Kernel_SHAP_Full376_Class_Conditional_Top20.csv
    ├── Kernel_SHAP_Full376_Summary.json
    ├── Kernel_SHAP_Resume_Metadata.json
    ├── Kernel_Checkpoints/
    │   └── kernel_4096_<start>_<end>.npy
    └── Figures/
        ├── 01_Kernel_SHAP_Full376_Global.png
        ├── 01_Kernel_SHAP_Full376_Global.pdf
        ├── 02_Kernel_SHAP_Full376_Beeswarm.png
        ├── 02_Kernel_SHAP_Full376_Beeswarm.pdf
        ├── 03_Kernel_SHAP_Full376_Families.png
        ├── 03_Kernel_SHAP_Full376_Families.pdf
        ├── 04_Kernel_SHAP_High_Confidence_Error.png
        └── 04_Kernel_SHAP_High_Confidence_Error.pdf
```

The high-confidence-error figure is generated only when at least one held-out observation is misclassified.

#### Methodological Role

This section performs the **authoritative full held-out Kernel SHAP analysis** for the retained CNN-BiLSTM.

It does not:

- retrain the model;
- select a different neural seed;
- modify the held-out observations;
- use SHAP results for model selection;
- rebalance the full held-out explanation set; or
- replace the later correlation-aware Partition SHAP analysis.

Its purpose is to explain the actual predictions made by the retained model across the complete 376-case held-out test set and to establish the principal Kernel SHAP feature-level, class-conditional and feature-family interpretation used in the final analysis.

In [ ]:
!pip install -q shap

import gc
import hashlib
import json
import os
import random
import shutil
from pathlib import Path

os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import tensorflow as tf
import tensorflow.keras.backend as K
from google.colab import drive
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Layer
from tensorflow.keras.models import load_model

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

RANDOM_STATE = 42

EXPECTED_UNIQUE_ROWS = 1877
EXPECTED_TRAIN_ROWS = 1200
EXPECTED_VALIDATION_ROWS = 301
EXPECTED_TEST_ROWS = 376
EXPECTED_RAW_FEATURES = 297
EXPECTED_FEATURES = 244
EXPECTED_CLASSES = 10

BACKGROUND_SIZE = 20
KERNEL_NSAMPLES = 4096
KERNEL_BASE_SEED = 4701
KERNEL_CHUNK_SIZE = 10

# False resumes from valid saved checkpoints.
# Set True only when you deliberately want to delete every Kernel SHAP
# checkpoint and restart from observation 0.
FORCE_RECOMPUTE = False

PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
CNN_BILSTM_ROOT = PROJECT_ROOT / "Models" / "Tuned" / "CNN_BiLSTM"
TRAIN_DATA_PATH = PROJECT_ROOT / "iot_device_train.csv"
TEST_DATA_PATH = PROJECT_ROOT / "iot_device_test.csv"


def select_cnn_bilstm_shap_run():
    required_relative_paths = [
        Path("best_validation_cnn_bilstm.keras"),
        Path("X_test_3d.npy"),
        Path("y_test_1d.npy"),
        Path("device_classes.npy"),
        Path("feature_names.npy"),
        Path("imputer.joblib"),
        Path("scaler.joblib"),
        Path("split_indices.npz"),
        Path("SHAP_Final/Common/SHAP_Common_Metadata.json"),
        Path("SHAP_Final/Common/Background_Observations.csv"),
        Path("SHAP_Final/Common/Feature_Family_Mapping.csv"),
    ]

    candidates = []

    if CNN_BILSTM_ROOT.exists():
        for candidate in CNN_BILSTM_ROOT.iterdir():
            if candidate.is_dir() and all(
                (candidate / relative_path).exists()
                for relative_path in required_relative_paths
            ):
                candidates.append(candidate.resolve())

    candidates = sorted(
        candidates,
        key=lambda path: path.name,
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError(
            "No CNN-BiLSTM run containing the required SHAP common artifacts "
            f"was found under:\n{CNN_BILSTM_ROOT}\n\n"
            "Run SHAP Block 1 once for the intended model before running this block."
        )

    print("\n" + "=" * 100)
    print("SELECT CNN-BILSTM RUN FOR RESUMABLE KERNEL SHAP")
    print("=" * 100)

    for number, candidate in enumerate(candidates, start=1):
        print(f"[{number}] {candidate}")

    while True:
        response = input(
            f"Select 1-{len(candidates)} (Enter = newest): "
        ).strip()

        if response == "":
            return candidates[0]

        if response.isdigit() and 1 <= int(response) <= len(candidates):
            return candidates[int(response) - 1]

        print("Invalid selection.")


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(chunk_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def sha256_array(array):
    contiguous = np.ascontiguousarray(array)

    digest = hashlib.sha256()
    digest.update(
        str(contiguous.dtype).encode("utf-8")
    )
    digest.update(
        str(contiguous.shape).encode("utf-8")
    )
    digest.update(contiguous.tobytes())

    return digest.hexdigest()


def atomic_save_npy(path, array):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open("wb") as handle:
        np.save(handle, array)
        handle.flush()
        os.fsync(handle.fileno())

    temporary_path.replace(path)


def save_figure(fig, base_path):
    base_path = Path(base_path)

    fig.savefig(
        base_path.with_suffix(".png"),
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

    fig.savefig(
        base_path.with_suffix(".pdf"),
        bbox_inches="tight",
        facecolor="white",
    )


def normalise_multiclass_shap(values, n_samples):
    if isinstance(values, shap.Explanation):
        array = np.asarray(values.values)

    elif isinstance(values, list):
        array = np.stack(
            [
                np.asarray(item)
                for item in values
            ],
            axis=-1,
        )

    else:
        array = np.asarray(values)

    target_shape = (
        n_samples,
        EXPECTED_FEATURES,
        EXPECTED_CLASSES,
    )

    if array.shape == target_shape:
        return array.astype(np.float64)

    if array.shape == (
        EXPECTED_CLASSES,
        n_samples,
        EXPECTED_FEATURES,
    ):
        return np.transpose(
            array,
            (1, 2, 0),
        ).astype(np.float64)

    if array.shape == (
        n_samples,
        EXPECTED_CLASSES,
        EXPECTED_FEATURES,
    ):
        return np.transpose(
            array,
            (0, 2, 1),
        ).astype(np.float64)

    raise RuntimeError(
        f"Unexpected SHAP shape {array.shape}; "
        f"expected compatibility with {target_shape}."
    )


@tf.keras.utils.register_keras_serializable(
    package="COMP700"
)
class SelfAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="att_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True,
        )

        self.b = self.add_weight(
            name="att_bias",
            shape=(input_shape[1], 1),
            initializer="zeros",
            trainable=True,
        )

        super().build(input_shape)

    def call(self, inputs):
        scores = K.tanh(
            K.dot(inputs, self.W)
            + self.b
        )

        weights = K.softmax(
            scores,
            axis=1,
        )

        return K.sum(
            inputs * weights,
            axis=1,
        )


SELF_ATTENTION_CUSTOM_OBJECTS = {
    "SelfAttention": SelfAttention,
    "Custom>SelfAttention": SelfAttention,
    "COMP700>SelfAttention": SelfAttention,
}


RUN_DIR = select_cnn_bilstm_shap_run()

MODEL_PATH = (
    RUN_DIR
    / "best_validation_cnn_bilstm.keras"
)
X_TEST_PATH = RUN_DIR / "X_test_3d.npy"
Y_TEST_PATH = RUN_DIR / "y_test_1d.npy"
CLASSES_PATH = RUN_DIR / "device_classes.npy"
FEATURE_NAMES_PATH = RUN_DIR / "feature_names.npy"
IMPUTER_PATH = RUN_DIR / "imputer.joblib"
SCALER_PATH = RUN_DIR / "scaler.joblib"
SPLIT_INDICES_PATH = RUN_DIR / "split_indices.npz"

SHAP_ROOT = RUN_DIR / "SHAP_Final"
COMMON_DIR = SHAP_ROOT / "Common"
FULL376_DIR = SHAP_ROOT / "Full376"
FULL376_FIGURES_DIR = FULL376_DIR / "Figures"
KERNEL_CHECKPOINT_DIR = (
    FULL376_DIR
    / "Kernel_Checkpoints"
)

COMMON_METADATA_PATH = (
    COMMON_DIR
    / "SHAP_Common_Metadata.json"
)
BACKGROUND_OBSERVATIONS_PATH = (
    COMMON_DIR
    / "Background_Observations.csv"
)
FEATURE_FAMILY_PATH = (
    COMMON_DIR
    / "Feature_Family_Mapping.csv"
)
KERNEL_RESUME_METADATA_PATH = (
    FULL376_DIR
    / "Kernel_SHAP_Resume_Metadata.json"
)

required_paths = [
    TRAIN_DATA_PATH,
    TEST_DATA_PATH,
    MODEL_PATH,
    X_TEST_PATH,
    Y_TEST_PATH,
    CLASSES_PATH,
    FEATURE_NAMES_PATH,
    IMPUTER_PATH,
    SCALER_PATH,
    SPLIT_INDICES_PATH,
    COMMON_METADATA_PATH,
    BACKGROUND_OBSERVATIONS_PATH,
    FEATURE_FAMILY_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "The standalone Kernel SHAP block cannot continue "
        "because these files are missing:\n"
        + "\n".join(
            f"- {path}"
            for path in missing_paths
        )
    )

FULL376_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
FULL376_FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if (
    FORCE_RECOMPUTE
    and KERNEL_CHECKPOINT_DIR.exists()
):
    shutil.rmtree(
        KERNEL_CHECKPOINT_DIR
    )

    print(
        "Deleted existing Kernel SHAP checkpoints "
        "because FORCE_RECOMPUTE=True."
    )

KERNEL_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

with COMMON_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    common_metadata = json.load(handle)

provenance_checks = {
    "selected_model_sha256": MODEL_PATH,
    "x_test_sha256": X_TEST_PATH,
    "y_test_sha256": Y_TEST_PATH,
    "device_classes_sha256": CLASSES_PATH,
    "feature_names_sha256": FEATURE_NAMES_PATH,
}

for (
    metadata_key,
    artifact_path,
) in provenance_checks.items():
    expected_hash = common_metadata.get(
        metadata_key
    )

    if not expected_hash:
        raise RuntimeError(
            "SHAP_Common_Metadata.json does not contain "
            f"{metadata_key}. "
            "Rerun the updated SHAP Block 1 "
            "for this model run."
        )

    actual_hash = sha256_file(
        artifact_path
    )

    if actual_hash != expected_hash:
        raise RuntimeError(
            "SHAP provenance mismatch for "
            f"{artifact_path.name}.\n"
            f"Expected: {expected_hash}\n"
            f"Actual:   {actual_hash}"
        )

X_test_3d = np.asarray(
    np.load(
        X_TEST_PATH,
        allow_pickle=False,
    ),
    dtype=np.float32,
)

y_explain = np.asarray(
    np.load(
        Y_TEST_PATH,
        allow_pickle=False,
    ),
    dtype=np.int64,
).reshape(-1)

device_classes = np.asarray(
    np.load(
        CLASSES_PATH,
        allow_pickle=True,
    )
)

feature_names = [
    str(value)
    for value in np.load(
        FEATURE_NAMES_PATH,
        allow_pickle=True,
    )
]

if X_test_3d.shape != (
    EXPECTED_TEST_ROWS,
    EXPECTED_FEATURES,
    1,
):
    raise RuntimeError(
        f"Unexpected X_test_3d shape: "
        f"{X_test_3d.shape}"
    )

if y_explain.shape != (
    EXPECTED_TEST_ROWS,
):
    raise RuntimeError(
        f"Unexpected y_test_1d shape: "
        f"{y_explain.shape}"
    )

if len(device_classes) != EXPECTED_CLASSES:
    raise RuntimeError(
        f"Expected {EXPECTED_CLASSES} device classes; "
        f"found {len(device_classes)}."
    )

if len(feature_names) != EXPECTED_FEATURES:
    raise RuntimeError(
        f"Expected {EXPECTED_FEATURES} retained features; "
        f"found {len(feature_names)}."
    )

X_explain = (
    X_test_3d[:, :, 0]
    .astype(np.float32)
)

feature_family_df = (
    pd.read_csv(FEATURE_FAMILY_PATH)
    .sort_values("feature_index")
    .reset_index(drop=True)
)

required_family_columns = {
    "feature_index",
    "feature",
    "feature_family",
}

if not required_family_columns.issubset(
    feature_family_df.columns
):
    raise RuntimeError(
        "Feature_Family_Mapping.csv "
        "has an unexpected schema."
    )

if (
    feature_family_df["feature"]
    .astype(str)
    .tolist()
    != feature_names
):
    raise RuntimeError(
        "Feature_Family_Mapping.csv does not match "
        "the saved retained feature order."
    )

family_names = sorted(
    feature_family_df[
        "feature_family"
    ]
    .astype(str)
    .unique()
    .tolist()
)

family_to_indices = {
    family: (
        feature_family_df.loc[
            (
                feature_family_df[
                    "feature_family"
                ].astype(str)
                == family
            ),
            "feature_index",
        ]
        .to_numpy(dtype=int)
    )
    for family in family_names
}


def aggregate_family_importance(
    decision_shap,
):
    absolute_values = np.abs(
        np.asarray(
            decision_shap,
            dtype=np.float64,
        )
    )

    if (
        absolute_values.ndim != 2
        or absolute_values.shape[1]
        != EXPECTED_FEATURES
    ):
        raise RuntimeError(
            "Expected decision SHAP shape "
            f"(n, {EXPECTED_FEATURES}); "
            f"found {absolute_values.shape}."
        )

    rows = []

    for family in family_names:
        indices = family_to_indices[family]

        per_sample_total = np.sum(
            absolute_values[:, indices],
            axis=1,
        )

        total = float(
            np.mean(per_sample_total)
        )

        rows.append(
            {
                "feature_family": family,
                "number_of_features": int(
                    len(indices)
                ),
                "total_mean_abs_shap": total,
                "mean_abs_shap_per_feature": (
                    total
                    / float(len(indices))
                ),
            }
        )

    result = pd.DataFrame(rows)

    grand_total = float(
        result[
            "total_mean_abs_shap"
        ].sum()
    )

    if (
        not np.isfinite(grand_total)
        or grand_total <= 0.0
    ):
        raise RuntimeError(
            "Invalid total SHAP magnitude: "
            f"{grand_total}"
        )

    result[
        "share_of_total_abs_shap"
    ] = (
        result[
            "total_mean_abs_shap"
        ]
        / grand_total
    )

    result = (
        result
        .sort_values(
            "share_of_total_abs_shap",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    result.insert(
        0,
        "rank",
        np.arange(1, len(result) + 1),
    )

    return result


train_source_df = pd.read_csv(
    TRAIN_DATA_PATH
)
test_source_df = pd.read_csv(
    TEST_DATA_PATH
)

combined_df = (
    pd.concat(
        [
            train_source_df,
            test_source_df,
        ],
        ignore_index=True,
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

if len(combined_df) != EXPECTED_UNIQUE_ROWS:
    raise RuntimeError(
        f"Expected {EXPECTED_UNIQUE_ROWS} unique rows; "
        f"found {len(combined_df)}."
    )

X_full = (
    combined_df
    .drop(columns=["device_category"])
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)

y_full = (
    combined_df[
        "device_category"
    ]
    .astype(str)
)

if X_full.shape[1] != EXPECTED_RAW_FEATURES:
    raise RuntimeError(
        f"Expected {EXPECTED_RAW_FEATURES} raw predictors; "
        f"found {X_full.shape[1]}."
    )

all_indices = np.arange(
    len(combined_df),
    dtype=np.int64,
)

(
    X_development_raw,
    X_test_raw,
    y_development_raw,
    y_test_raw,
    index_development,
    index_test,
) = train_test_split(
    X_full,
    y_full,
    all_indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_full,
)

(
    X_train_raw,
    X_validation_raw,
    y_train_raw,
    y_validation_raw,
    index_train,
    index_validation,
) = train_test_split(
    X_development_raw,
    y_development_raw,
    index_development,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_development_raw,
)

actual_split = (
    len(X_train_raw),
    len(X_validation_raw),
    len(X_test_raw),
)

expected_split = (
    EXPECTED_TRAIN_ROWS,
    EXPECTED_VALIDATION_ROWS,
    EXPECTED_TEST_ROWS,
)

if actual_split != expected_split:
    raise RuntimeError(
        f"Unexpected reconstructed split: "
        f"{actual_split}"
    )

with np.load(
    SPLIT_INDICES_PATH,
    allow_pickle=False,
) as saved_split:
    saved_index_train = np.asarray(
        saved_split["train"],
        dtype=np.int64,
    )

    saved_index_validation = np.asarray(
        saved_split["validation"],
        dtype=np.int64,
    )

    saved_index_test = np.asarray(
        saved_split["test"],
        dtype=np.int64,
    )

if not np.array_equal(
    np.asarray(
        index_train,
        dtype=np.int64,
    ),
    saved_index_train,
):
    raise RuntimeError(
        "Reconstructed training indices differ "
        "from split_indices.npz."
    )

if not np.array_equal(
    np.asarray(
        index_validation,
        dtype=np.int64,
    ),
    saved_index_validation,
):
    raise RuntimeError(
        "Reconstructed validation indices differ "
        "from split_indices.npz."
    )

if not np.array_equal(
    np.asarray(
        index_test,
        dtype=np.int64,
    ),
    saved_index_test,
):
    raise RuntimeError(
        "Reconstructed test indices differ "
        "from split_indices.npz."
    )

imputer = joblib.load(
    IMPUTER_PATH
)
scaler = joblib.load(
    SCALER_PATH
)

X_train_real_scaled = scaler.transform(
    imputer.transform(
        X_train_raw[feature_names]
    )
).astype(np.float32)

background_observations_df = pd.read_csv(
    BACKGROUND_OBSERVATIONS_PATH
)

required_background_columns = {
    "background_position",
    "training_relative_position",
    "combined_dataset_index",
}

if not required_background_columns.issubset(
    background_observations_df.columns
):
    raise RuntimeError(
        "Background_Observations.csv "
        "has an unexpected schema."
    )

background_observations_df = (
    background_observations_df
    .sort_values("background_position")
    .reset_index(drop=True)
)

background_indices = (
    background_observations_df[
        "training_relative_position"
    ]
    .to_numpy(dtype=np.int64)
)

if len(background_indices) != BACKGROUND_SIZE:
    raise RuntimeError(
        f"Expected {BACKGROUND_SIZE} saved background "
        f"observations; found {len(background_indices)}."
    )

if (
    len(np.unique(background_indices))
    != BACKGROUND_SIZE
):
    raise RuntimeError(
        "Saved background training positions "
        "are not unique."
    )

if (
    np.any(background_indices < 0)
    or np.any(
        background_indices
        >= EXPECTED_TRAIN_ROWS
    )
):
    raise RuntimeError(
        "A saved background position lies "
        "outside the training partition."
    )

reconstructed_combined_indices = (
    X_train_raw
    .index
    .to_numpy()[
        background_indices
    ]
)

saved_combined_indices = (
    background_observations_df[
        "combined_dataset_index"
    ]
    .to_numpy(dtype=np.int64)
)

if not np.array_equal(
    reconstructed_combined_indices,
    saved_combined_indices,
):
    raise RuntimeError(
        "Saved background observations do not match "
        "the reconstructed training partition."
    )

X_background = (
    X_train_real_scaled[
        background_indices
    ]
    .astype(np.float32)
)

if X_background.shape != (
    BACKGROUND_SIZE,
    EXPECTED_FEATURES,
):
    raise RuntimeError(
        f"Unexpected background shape: "
        f"{X_background.shape}"
    )

tf.keras.backend.clear_session()
gc.collect()

model = load_model(
    MODEL_PATH,
    custom_objects=(
        SELF_ATTENTION_CUSTOM_OBJECTS
    ),
    compile=False,
)


def predict_wrapper(X_2d):
    X_2d = np.asarray(
        X_2d,
        dtype=np.float32,
    )

    if X_2d.ndim == 1:
        X_2d = X_2d.reshape(
            1,
            -1,
        )

    if X_2d.ndim != 2:
        raise RuntimeError(
            "Kernel SHAP wrapper expects 2-D input; "
            f"received {X_2d.shape}."
        )

    if X_2d.shape[1] != EXPECTED_FEATURES:
        raise RuntimeError(
            f"Expected {EXPECTED_FEATURES} features; "
            f"received {X_2d.shape[1]}."
        )

    probabilities = np.asarray(
        model.predict(
            X_2d[..., np.newaxis],
            verbose=0,
        ),
        dtype=np.float64,
    )

    expected_shape = (
        len(X_2d),
        EXPECTED_CLASSES,
    )

    if probabilities.shape != expected_shape:
        raise RuntimeError(
            "Unexpected CNN-BiLSTM probability shape: "
            f"{probabilities.shape}; "
            f"expected {expected_shape}."
        )

    if not np.all(
        np.isfinite(probabilities)
    ):
        raise RuntimeError(
            "CNN-BiLSTM probabilities "
            "contain NaN or Inf."
        )

    return probabilities


explain_probabilities = predict_wrapper(
    X_explain
)

predicted_explain_classes = np.argmax(
    explain_probabilities,
    axis=1,
)

prediction_confidence = np.max(
    explain_probabilities,
    axis=1,
)

test_accuracy = float(
    accuracy_score(
        y_explain,
        predicted_explain_classes,
    )
)

test_macro_f1 = float(
    f1_score(
        y_explain,
        predicted_explain_classes,
        average="macro",
        zero_division=0,
    )
)

metadata_accuracy = common_metadata.get(
    "heldout_accuracy"
)
metadata_macro_f1 = common_metadata.get(
    "heldout_macro_f1"
)

if (
    metadata_accuracy is not None
    and not np.isclose(
        test_accuracy,
        float(metadata_accuracy),
        atol=1e-10,
        rtol=1e-10,
    )
):
    raise RuntimeError(
        "Reloaded model accuracy does not match "
        "SHAP_Common_Metadata.json.\n"
        f"Metadata: {float(metadata_accuracy):.12g}\n"
        f"Reloaded: {test_accuracy:.12g}"
    )

if (
    metadata_macro_f1 is not None
    and not np.isclose(
        test_macro_f1,
        float(metadata_macro_f1),
        atol=1e-10,
        rtol=1e-10,
    )
):
    raise RuntimeError(
        "Reloaded model macro-F1 does not match "
        "SHAP_Common_Metadata.json.\n"
        f"Metadata: {float(metadata_macro_f1):.12g}\n"
        f"Reloaded: {test_macro_f1:.12g}"
    )

probe_probabilities = predict_wrapper(
    X_explain[:5]
)

if not np.allclose(
    probe_probabilities,
    explain_probabilities[:5],
    atol=1e-7,
    rtol=1e-7,
):
    raise RuntimeError(
        "Prediction-wrapper integrity check failed."
    )

current_resume_metadata = {
    "selected_run": str(RUN_DIR),
    "selected_model_sha256": sha256_file(
        MODEL_PATH
    ),
    "x_test_sha256": sha256_file(
        X_TEST_PATH
    ),
    "y_test_sha256": sha256_file(
        Y_TEST_PATH
    ),
    "feature_names_sha256": sha256_file(
        FEATURE_NAMES_PATH
    ),
    "background_array_sha256": sha256_array(
        X_background
    ),
    "background_size": BACKGROUND_SIZE,
    "kernel_nsamples": KERNEL_NSAMPLES,
    "kernel_base_seed": KERNEL_BASE_SEED,
    "kernel_chunk_size": KERNEL_CHUNK_SIZE,
    "expected_test_rows": EXPECTED_TEST_ROWS,
    "expected_features": EXPECTED_FEATURES,
    "expected_classes": EXPECTED_CLASSES,
    "shap_version": str(shap.__version__),
    "tensorflow_version": str(tf.__version__),
    "numpy_version": str(np.__version__),
}

if KERNEL_RESUME_METADATA_PATH.exists():
    with KERNEL_RESUME_METADATA_PATH.open(
        "r",
        encoding="utf-8",
    ) as handle:
        saved_resume_metadata = json.load(
            handle
        )

    invariant_keys = [
        "selected_model_sha256",
        "x_test_sha256",
        "y_test_sha256",
        "feature_names_sha256",
        "background_array_sha256",
        "background_size",
        "kernel_nsamples",
        "kernel_base_seed",
        "kernel_chunk_size",
        "expected_test_rows",
        "expected_features",
        "expected_classes",
    ]

    mismatches = [
        key
        for key in invariant_keys
        if (
            saved_resume_metadata.get(key)
            != current_resume_metadata.get(key)
        )
    ]

    if mismatches:
        raise RuntimeError(
            "The saved Kernel SHAP checkpoints were created "
            "with a different model, dataset, background or "
            "configuration. Refusing to mix experiments.\n"
            "Mismatched fields:\n  - "
            + "\n  - ".join(mismatches)
        )

    version_keys = [
        "shap_version",
        "tensorflow_version",
        "numpy_version",
    ]

    version_changes = [
        key
        for key in version_keys
        if (
            saved_resume_metadata.get(key)
            != current_resume_metadata.get(key)
        )
    ]

    if version_changes:
        print(
            "WARNING: Runtime package versions differ from "
            "those recorded during a previous resume session:\n  - "
            + "\n  - ".join(version_changes)
        )

else:
    existing_checkpoint_count = len(
        list(
            KERNEL_CHECKPOINT_DIR.glob(
                "kernel_4096_*.npy"
            )
        )
    )

    current_resume_metadata[
        "preexisting_checkpoints_when_metadata_created"
    ] = int(existing_checkpoint_count)

    current_resume_metadata[
        "note"
    ] = (
        "Metadata created by the standalone "
        "resume-safe Kernel SHAP block."
    )

    with KERNEL_RESUME_METADATA_PATH.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            current_resume_metadata,
            handle,
            indent=2,
        )

print("=" * 100)
print("STANDALONE KERNEL SHAP SETUP VERIFIED")
print("=" * 100)
print(f"Selected run       : {RUN_DIR}")
print(f"Model              : {MODEL_PATH.name}")
print(f"Held-out accuracy  : {test_accuracy * 100:.3f}%")
print(f"Held-out macro-F1  : {test_macro_f1:.4f}")
print(f"Background shape   : {X_background.shape}")
print(f"Explanation shape  : {X_explain.shape}")
print(f"Checkpoint folder  : {KERNEL_CHECKPOINT_DIR}")

print("\n" + "=" * 100)
print("FULL 376-CASE KERNEL SHAP — RESUME-SAFE")
print("=" * 100)

kernel_explainer = shap.KernelExplainer(
    predict_wrapper,
    X_background,
)

kernel_expected_values = np.asarray(
    kernel_explainer.expected_value,
    dtype=np.float64,
).reshape(-1)

if kernel_expected_values.shape != (
    EXPECTED_CLASSES,
):
    raise RuntimeError(
        "Unexpected Kernel SHAP expected-value shape: "
        f"{kernel_expected_values.shape}"
    )

if not np.all(
    np.isfinite(kernel_expected_values)
):
    raise RuntimeError(
        "Kernel SHAP expected values "
        "contain NaN or Inf."
    )

kernel_expected_path = (
    FULL376_DIR
    / "Kernel_SHAP_Full376_Expected_Values.npy"
)

if kernel_expected_path.exists():
    saved_expected_values = np.asarray(
        np.load(
            kernel_expected_path,
            allow_pickle=False,
        ),
        dtype=np.float64,
    ).reshape(-1)

    if (
        saved_expected_values.shape
        != (EXPECTED_CLASSES,)
        or not np.allclose(
            saved_expected_values,
            kernel_expected_values,
            atol=1e-6,
            rtol=1e-6,
        )
    ):
        raise RuntimeError(
            "The recomputed Kernel SHAP expected values "
            "do not match the saved values. Refusing to "
            "combine checkpoints from inconsistent explainers."
        )

else:
    atomic_save_npy(
        kernel_expected_path,
        kernel_expected_values,
    )


def load_valid_kernel_checkpoint(
    path,
    expected_shape,
):
    path = Path(path)

    if not path.exists():
        return None

    try:
        array = np.asarray(
            np.load(
                path,
                allow_pickle=False,
            ),
            dtype=np.float64,
        )

    except Exception as exc:
        print(
            f"Invalid checkpoint "
            f"{path.name}: {exc}"
        )
        return None

    if array.shape != expected_shape:
        print(
            "Invalid checkpoint shape for "
            f"{path.name}: found {array.shape}, "
            f"expected {expected_shape}"
        )
        return None

    if not np.all(np.isfinite(array)):
        print(
            "Invalid checkpoint values for "
            f"{path.name}: contains NaN or Inf"
        )
        return None

    return array


kernel_chunk_specs = []
completed_before_resume = 0

for start in range(
    0,
    EXPECTED_TEST_ROWS,
    KERNEL_CHUNK_SIZE,
):
    end = min(
        start + KERNEL_CHUNK_SIZE,
        EXPECTED_TEST_ROWS,
    )

    checkpoint_path = (
        KERNEL_CHECKPOINT_DIR
        / (
            "kernel_4096_"
            f"{start:03d}_{end:03d}.npy"
        )
    )

    expected_chunk_shape = (
        end - start,
        EXPECTED_FEATURES,
        EXPECTED_CLASSES,
    )

    kernel_chunk_specs.append(
        (
            start,
            end,
            checkpoint_path,
            expected_chunk_shape,
        )
    )

    existing_chunk = (
        load_valid_kernel_checkpoint(
            checkpoint_path,
            expected_chunk_shape,
        )
    )

    if existing_chunk is not None:
        completed_before_resume += 1
    else:
        checkpoint_path.unlink(
            missing_ok=True
        )

    del existing_chunk

print(
    "Valid checkpoints found: "
    f"{completed_before_resume}/"
    f"{len(kernel_chunk_specs)}"
)

completed_observations = sum(
    end - start
    for (
        start,
        end,
        path,
        shape,
    ) in kernel_chunk_specs
    if path.exists()
)

print(
    "Held-out observations already complete: "
    f"{completed_observations}/"
    f"{EXPECTED_TEST_ROWS}"
)

for (
    start,
    end,
    checkpoint_path,
    expected_chunk_shape,
) in kernel_chunk_specs:
    existing_chunk = (
        load_valid_kernel_checkpoint(
            checkpoint_path,
            expected_chunk_shape,
        )
    )

    if existing_chunk is not None:
        print(
            "Skipping completed Kernel chunk "
            f"{start}:{end}"
        )
        del existing_chunk
        continue

    checkpoint_path.unlink(
        missing_ok=True
    )

    print(
        f"\nKernel SHAP {start}:{end} "
        f"of {EXPECTED_TEST_ROWS}"
    )

    chunk_seed = (
        KERNEL_BASE_SEED
        + start
    )

    np.random.seed(chunk_seed)
    random.seed(chunk_seed)
    tf.random.set_seed(chunk_seed)

    raw_values = (
        kernel_explainer.shap_values(
            X_explain[start:end],
            nsamples=KERNEL_NSAMPLES,
            l1_reg=0.0,
        )
    )

    chunk_values = (
        normalise_multiclass_shap(
            raw_values,
            n_samples=end - start,
        )
    )

    if (
        chunk_values.shape
        != expected_chunk_shape
    ):
        raise RuntimeError(
            "Unexpected Kernel chunk shape: "
            f"{chunk_values.shape}"
        )

    if not np.all(
        np.isfinite(chunk_values)
    ):
        raise RuntimeError(
            f"Kernel chunk {start}:{end} "
            "contains NaN or Inf."
        )

    atomic_save_npy(
        checkpoint_path,
        chunk_values,
    )

    print(
        "Saved completed checkpoint: "
        f"{checkpoint_path.name}"
    )

    del raw_values, chunk_values
    gc.collect()

kernel_chunks = []

for (
    start,
    end,
    checkpoint_path,
    expected_chunk_shape,
) in kernel_chunk_specs:
    chunk = load_valid_kernel_checkpoint(
        checkpoint_path,
        expected_chunk_shape,
    )

    if chunk is None:
        raise RuntimeError(
            "Required Kernel checkpoint remains "
            f"missing or invalid: {start}:{end}"
        )

    kernel_chunks.append(chunk)

kernel_shap = np.concatenate(
    kernel_chunks,
    axis=0,
).astype(np.float64)

del kernel_chunks
gc.collect()

expected_full_shape = (
    EXPECTED_TEST_ROWS,
    EXPECTED_FEATURES,
    EXPECTED_CLASSES,
)

if kernel_shap.shape != expected_full_shape:
    raise RuntimeError(
        "Unexpected full Kernel SHAP shape: "
        f"{kernel_shap.shape}; "
        f"expected {expected_full_shape}."
    )

if not np.all(
    np.isfinite(kernel_shap)
):
    raise RuntimeError(
        "Full Kernel SHAP tensor "
        "contains NaN or Inf."
    )

kernel_full_path = (
    FULL376_DIR
    / "Kernel_SHAP_Full376_4096.npy"
)

atomic_save_npy(
    kernel_full_path,
    kernel_shap,
)

print(
    "\nFull Kernel SHAP tensor:",
    kernel_shap.shape,
)

kernel_reconstructed = (
    kernel_expected_values[
        np.newaxis,
        :
    ]
    + np.sum(
        kernel_shap,
        axis=1,
    )
)

kernel_residuals = np.abs(
    kernel_reconstructed
    - explain_probabilities
)

kernel_mean_residual = float(
    np.mean(kernel_residuals)
)

kernel_max_residual = float(
    np.max(kernel_residuals)
)

kernel_fidelity_df = pd.DataFrame(
    {
        "test_index": np.arange(
            EXPECTED_TEST_ROWS
        ),
        "actual_class": [
            str(device_classes[value])
            for value in y_explain
        ],
        "predicted_class": [
            str(device_classes[value])
            for value
            in predicted_explain_classes
        ],
        "correct": (
            y_explain
            == predicted_explain_classes
        ),
        "mean_output_residual": (
            np.mean(
                kernel_residuals,
                axis=1,
            )
        ),
        "max_output_residual": (
            np.max(
                kernel_residuals,
                axis=1,
            )
        ),
    }
)

kernel_fidelity_df.to_csv(
    FULL376_DIR
    / "Kernel_SHAP_Full376_Local_Fidelity.csv",
    index=False,
)

print("\nKernel SHAP local fidelity:")
print(
    f"  Mean residual : "
    f"{kernel_mean_residual:.12g}"
)
print(
    f"  Max residual  : "
    f"{kernel_max_residual:.12g}"
)

kernel_decision_shap = kernel_shap[
    np.arange(EXPECTED_TEST_ROWS),
    :,
    predicted_explain_classes,
]

kernel_absolute_decision = np.abs(
    kernel_decision_shap
)

kernel_global_importance = np.mean(
    kernel_absolute_decision,
    axis=0,
)

kernel_global_sd = np.std(
    kernel_absolute_decision,
    axis=0,
    ddof=1,
)

kernel_global_df = pd.DataFrame(
    {
        "feature_index": np.arange(
            EXPECTED_FEATURES
        ),
        "feature": feature_names,
        "feature_family": (
            feature_family_df[
                "feature_family"
            ].to_numpy()
        ),
        "mean_abs_kernel_shap": (
            kernel_global_importance
        ),
        "sd_abs_kernel_shap_across_376_cases": (
            kernel_global_sd
        ),
    }
)

kernel_global_df = (
    kernel_global_df
    .sort_values(
        "mean_abs_kernel_shap",
        ascending=False,
    )
    .reset_index(drop=True)
)

kernel_global_df.insert(
    0,
    "rank",
    np.arange(
        1,
        len(kernel_global_df) + 1,
    ),
)

kernel_global_df.to_csv(
    FULL376_DIR
    / "Kernel_SHAP_Full376_Global_Importance.csv",
    index=False,
)

kernel_family_df = (
    aggregate_family_importance(
        kernel_decision_shap
    )
)

kernel_family_df.to_csv(
    FULL376_DIR
    / "Kernel_SHAP_Full376_Family_Importance.csv",
    index=False,
)

kernel_class_rows = []

for (
    class_index,
    class_name,
) in enumerate(device_classes):
    mask = (
        y_explain == class_index
    )

    class_values = kernel_shap[
        mask,
        :,
        class_index,
    ]

    class_mean_abs = np.mean(
        np.abs(class_values),
        axis=0,
    )

    class_predictions = (
        predicted_explain_classes[
            mask
        ]
    )

    class_correct = int(
        np.sum(
            class_predictions
            == class_index
        )
    )

    ranked_indices = np.argsort(
        class_mean_abs
    )[::-1][:20]

    for (
        rank,
        feature_index,
    ) in enumerate(
        ranked_indices,
        start=1,
    ):
        kernel_class_rows.append(
            {
                "device_class": str(
                    class_name
                ),
                "support": int(
                    np.sum(mask)
                ),
                "correctly_classified": (
                    class_correct
                ),
                "rank": rank,
                "feature": (
                    feature_names[
                        feature_index
                    ]
                ),
                "feature_family": (
                    feature_family_df.loc[
                        feature_index,
                        "feature_family",
                    ]
                ),
                "mean_abs_kernel_shap_for_true_class_output": (
                    class_mean_abs[
                        feature_index
                    ]
                ),
            }
        )

kernel_class_df = pd.DataFrame(
    kernel_class_rows
)

kernel_class_df.to_csv(
    FULL376_DIR
    / "Kernel_SHAP_Full376_Class_Conditional_Top20.csv",
    index=False,
)

plot_df = (
    kernel_global_df
    .head(20)
    .sort_values(
        "mean_abs_kernel_shap"
    )
)

fig, ax = plt.subplots(
    figsize=(10, 8)
)

ax.barh(
    plot_df["feature"],
    plot_df[
        "mean_abs_kernel_shap"
    ],
)

ax.set_xlabel(
    "Mean |Kernel SHAP value| "
    "across 376 held-out cases"
)
ax.set_ylabel("Feature")
ax.set_title(
    "CNN-BiLSTM Full-Test "
    "Kernel SHAP Importance"
)
ax.spines[
    ["top", "right"]
].set_visible(False)

fig.tight_layout()

save_figure(
    fig,
    FULL376_FIGURES_DIR
    / "01_Kernel_SHAP_Full376_Global",
)

plt.show()
plt.close(fig)

plt.figure(
    figsize=(11, 8)
)

shap.summary_plot(
    kernel_decision_shap,
    X_explain,
    feature_names=feature_names,
    max_display=20,
    show=False,
)

plt.title(
    "CNN-BiLSTM Full-Test "
    "Kernel SHAP Summary"
)

plt.tight_layout()

plt.savefig(
    FULL376_FIGURES_DIR
    / "02_Kernel_SHAP_Full376_Beeswarm.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.savefig(
    FULL376_FIGURES_DIR
    / "02_Kernel_SHAP_Full376_Beeswarm.pdf",
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close()

family_plot_df = (
    kernel_family_df
    .sort_values(
        "share_of_total_abs_shap"
    )
)

fig, ax = plt.subplots(
    figsize=(10, 6.5)
)

ax.barh(
    family_plot_df[
        "feature_family"
    ],
    (
        family_plot_df[
            "share_of_total_abs_shap"
        ]
        * 100.0
    ),
)

ax.set_xlabel(
    "Share of total |Kernel SHAP| (%)"
)
ax.set_ylabel("Feature family")
ax.set_title(
    "CNN-BiLSTM Full-Test Kernel SHAP "
    "by Feature Family"
)
ax.spines[
    ["top", "right"]
].set_visible(False)

fig.tight_layout()

save_figure(
    fig,
    FULL376_FIGURES_DIR
    / "03_Kernel_SHAP_Full376_Families",
)

plt.show()
plt.close(fig)

wrong_indices = np.where(
    predicted_explain_classes
    != y_explain
)[0]

if len(wrong_indices) > 0:
    error_index = int(
        wrong_indices[
            np.argmax(
                prediction_confidence[
                    wrong_indices
                ]
            )
        ]
    )

    true_class_index = int(
        y_explain[error_index]
    )

    predicted_class_index = int(
        predicted_explain_classes[
            error_index
        ]
    )

    local_explanation = shap.Explanation(
        values=kernel_shap[
            error_index,
            :,
            predicted_class_index,
        ],
        base_values=kernel_expected_values[
            predicted_class_index
        ],
        data=X_explain[
            error_index
        ],
        feature_names=feature_names,
    )

    shap.plots.waterfall(
        local_explanation,
        max_display=15,
        show=False,
    )

    plt.title(
        "Kernel SHAP — Highest-Confidence "
        "Misclassification\n"
        f"True: "
        f"{device_classes[true_class_index]} | "
        f"Predicted: "
        f"{device_classes[predicted_class_index]}"
    )

    plt.tight_layout()

    current_fig = plt.gcf()

    save_figure(
        current_fig,
        FULL376_FIGURES_DIR
        / "04_Kernel_SHAP_High_Confidence_Error",
    )

    plt.show()
    plt.close(current_fig)

kernel_summary = {
    "method": "Kernel SHAP",
    "cases": EXPECTED_TEST_ROWS,
    "nsamples": KERNEL_NSAMPLES,
    "background_size": BACKGROUND_SIZE,
    "background": (
        "20 real pre-SMOTE "
        "training representatives"
    ),
    "sample_weighting": False,
    "heldout_accuracy": test_accuracy,
    "heldout_macro_f1": test_macro_f1,
    "mean_local_fidelity_residual": (
        kernel_mean_residual
    ),
    "maximum_local_fidelity_residual": (
        kernel_max_residual
    ),
    "checkpoint_chunk_size": (
        KERNEL_CHUNK_SIZE
    ),
    "selected_model_sha256": sha256_file(
        MODEL_PATH
    ),
    "background_array_sha256": sha256_array(
        X_background
    ),
}

with open(
    FULL376_DIR
    / "Kernel_SHAP_Full376_Summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        kernel_summary,
        handle,
        indent=2,
    )

print("\n" + "=" * 100)
print("FULL 376-CASE KERNEL SHAP COMPLETE")
print("=" * 100)

print("\nTop-25 features:")
display(
    kernel_global_df.head(25)
)

print("\nFamily importance:")
display(kernel_family_df)

print(
    f"\nSaved to:\n{FULL376_DIR}"
)

print("\nRun Block 4 next.")

### Correlation-Aware Partition SHAP

This section computes the **correlation-aware Partition SHAP explanations for all 376 held-out test observations** and is designed to resume safely after a Colab runtime interruption.

It builds directly on the completed SHAP common-setup and full-test Kernel SHAP stages.

The block does not recompute valid Kernel SHAP results. It reloads the completed Kernel SHAP artifacts from Google Drive, validates them and then computes only the Partition SHAP results that are still missing.

#### Runtime Prerequisites

Before continuing, the block verifies that the variables, functions and objects created by:

`SHAP Block 1 — Common Setup`

are present in the current runtime.

These include the:

- reconstructed dataset information;
- selected CNN-BiLSTM model path;
- held-out feature and label arrays;
- model predictions;
- retained feature names;
- feature-family mapping;
- SHAP configuration;
- custom `SelfAttention` layer;
- SHAP normalisation utilities; and
- figure and integrity helpers.

If these symbols are missing, the block stops with an explicit instruction to rerun **Block 1 only**.

Previously completed Kernel SHAP or Partition SHAP calculations do not need to be rerun simply because the Colab runtime has restarted.

#### Reuse of Completed Kernel SHAP Results

The following completed Kernel SHAP artifacts from the preceding full-test analysis must already exist:

```text
Kernel_SHAP_Full376_4096.npy
Kernel_SHAP_Full376_Expected_Values.npy
Kernel_SHAP_Full376_Global_Importance.csv
Kernel_SHAP_Full376_Family_Importance.csv
Kernel_SHAP_Full376_Summary.json
```

The block reloads these artifacts directly from Google Drive.

The saved Kernel SHAP tensor is required to have shape:

`(376, 244, 10)`

and the Kernel expected-value vector must have shape:

`(10,)`

The code also verifies that the stored SHAP tensor contains only finite values.

The previously recorded Kernel SHAP mean and maximum local-fidelity residuals are reloaded from the saved summary metadata.

This allows the subsequent Partition SHAP analysis to continue without recomputing the expensive 376-case Kernel SHAP experiment.

#### Resume-Safe Partition SHAP Design

Partition SHAP is performed using checkpointed computation.

The principal completed Partition artifacts are:

```text
Partition_SHAP_Full376_4096.npy
Partition_SHAP_Full376_Base_Values.npy
Partition_SHAP_Feature_Clustering.npy
```

Before performing any new computation, the block checks whether the complete Partition SHAP tensor and complete base-value tensor already exist and are valid.

A saved array is accepted only if:

- it can be loaded successfully;
- it has the expected shape; and
- every value is finite.

If both full tensors are valid, they are loaded immediately and no Partition SHAP chunks are recomputed.

#### Partition Checkpoints

If complete Partition SHAP tensors are not available, the block checks the individual saved checkpoint pairs.

Each held-out chunk has two files:

```text
partition_4096_values_<start>_<end>.npy
partition_4096_base_<start>_<end>.npy
```

The values checkpoint must have shape:

`(chunk_size, 244, 10)`

and the corresponding base-value checkpoint must have shape:

`(chunk_size, 10)`

Both files must be valid before a chunk is considered complete.

If one member of a checkpoint pair is missing, corrupted, incorrectly shaped or contains invalid values, that checkpoint pair is treated as incomplete and is recomputed.

Valid checkpoint pairs are retained.

#### Atomic Checkpoint Saving

Partition SHAP checkpoints are written using an atomic-save procedure.

Each NumPy array is first written to a temporary file, flushed to disk and synchronised before replacing the final checkpoint path.

This reduces the risk of leaving a partially written `.npy` file if the Colab runtime disconnects or execution is interrupted during a save operation.

#### Correlation-Aware Feature Hierarchy

Partition SHAP uses a hierarchical grouping of correlated predictors.

The hierarchy is constructed from the:

`1,200 real training observations`

after the saved training preprocessing has been applied.

The hierarchy uses:

- average linkage;
- correlation distance; and
- `random_state = 42`.

For 244 retained predictors, the expected hierarchy shape is:

`(243, 4)`

The hierarchy is saved as:

`Partition_SHAP_Feature_Clustering.npy`

If a valid saved hierarchy already exists, it is reused.

If the saved hierarchy is missing, corrupted or incorrectly shaped, it is rebuilt from the real training data and saved again.

The held-out test data are therefore **not** used to derive the feature-correlation hierarchy.

#### Dedicated Partition SHAP Model Wrapper

When new Partition SHAP chunks must be calculated, the retained CNN-BiLSTM model is reloaded from its authoritative saved `.keras` artifact.

A dedicated Partition SHAP prediction wrapper is created for that model.

The wrapper:

- accepts the two-dimensional representation used by SHAP;
- verifies the expected 244 predictors;
- restores the singleton channel dimension required by the CNN-BiLSTM;
- performs model inference; and
- returns the 10-class probability matrix.

Before the Partition explainer is constructed, the wrapper is checked against the held-out probabilities established by the common setup block.

The wrapper must reproduce those probabilities within a tolerance of:

`1e-6`

This prevents a stale or overwritten global model reference from being used accidentally during the long-running Partition SHAP computation.

#### Partition SHAP Explainer

The correlation-aware masker is constructed using:

- the same 20 real pre-SMOTE training representatives used by the Kernel SHAP analysis; and
- the training-derived correlation hierarchy.

The Partition SHAP explainer is then created using the dedicated CNN-BiLSTM prediction wrapper.

The analysis retains the same:

- 244 predictors;
- 10 device classes;
- feature names; and
- device-class output names

used throughout the final SHAP workflow.

#### Partition SHAP Computational Budget

Each missing chunk is explained using:

```text
Maximum evaluations: 4096
Batch size:          256
```

The chunk size is inherited from the common SHAP configuration.

For each chunk, the explainer returns:

- multiclass Partition SHAP values; and
- corresponding base values.

The SHAP values are normalised to the standard project representation:

`(samples, 244 features, 10 classes)`

The base values are normalised to:

`(samples, 10 classes)`

If the explainer returns a single 10-class base-value vector for a chunk, the vector is repeated across the observations in that chunk.

#### Chunk Validation

Before a newly computed checkpoint is accepted, the block verifies:

- the complete expected SHAP tensor shape;
- the complete expected base-value shape;
- finite SHAP values; and
- finite base values.

Only after these checks pass are the checkpoint files saved.

This allows completed work to be reused safely after future runtime interruptions.

#### Full Partition SHAP Reassembly

After all required chunks are present, every checkpoint pair is validated again.

The checkpoint tensors are then concatenated in held-out test order.

The final expected shapes are:

```text
Partition SHAP values:
(376, 244, 10)

Partition SHAP base values:
(376, 10)
```

The reassembled tensors are again checked for finite values.

They are then saved atomically as:

```text
Partition_SHAP_Full376_4096.npy
Partition_SHAP_Full376_Base_Values.npy
```

These complete tensors can be loaded directly by subsequent analysis and comparison code without rerunning Partition SHAP.

#### Resume Behaviour

The block follows the following recovery sequence:

1. reuse the complete Partition tensors if both are valid;
2. otherwise inspect the individual chunk checkpoints;
3. retain all valid checkpoint pairs;
4. recompute only missing or invalid chunks;
5. rebuild the feature-correlation hierarchy only if necessary;
6. reassemble the full 376-case tensors; and
7. save the completed tensors back to Google Drive.

This design is particularly important for Partition SHAP because the complete full-test explanation is computationally expensive and may span multiple Colab sessions.

#### Methodological Role

This section performs the **resume-safe correlation-aware Partition SHAP computation** for the complete held-out test set.

It does not:

- retrain the CNN-BiLSTM;
- select another model or neural seed;
- recompute completed Kernel SHAP results;
- use the held-out test data to construct the correlation hierarchy;
- discard valid Partition checkpoints after a runtime interruption; or
- modify the trained model or test observations.

Its purpose is to create a complete, validated and restartable Partition SHAP explanation tensor that can subsequently be compared with the completed Kernel SHAP explanations.

In [ ]:
import gc
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import tensorflow as tf
from IPython.display import display
from tensorflow.keras.models import load_model

print("=" * 100)
print("FULL 376-CASE CORRELATION-AWARE PARTITION SHAP — RESUME-SAFE")
print("=" * 100)

required_common_symbols = [
    "RANDOM_STATE",
    "EXPECTED_TEST_ROWS",
    "EXPECTED_FEATURES",
    "EXPECTED_CLASSES",
    "BACKGROUND_SIZE",
    "KERNEL_NSAMPLES",
    "PARTITION_MAX_EVALS",
    "PARTITION_BATCH_SIZE",
    "PARTITION_CHUNK_SIZE",
    "MODEL_PATH",
    "CHAMPION_SEED",
    "SHAP_ROOT",
    "FULL376_DIR",
    "FULL376_FIGURES_DIR",
    "PARTITION_CHECKPOINT_DIR",
    "X_train_real_scaled",
    "X_background",
    "X_explain",
    "y_explain",
    "explain_probabilities",
    "predicted_explain_classes",
    "prediction_confidence",
    "test_accuracy",
    "test_macro_f1",
    "device_classes",
    "feature_names",
    "feature_family_df",
    "family_names",
    "family_to_indices",
    "SelfAttention",
    "SELF_ATTENTION_CUSTOM_OBJECTS",
    "normalise_multiclass_shap",
    "aggregate_family_importance",
    "safe_spearman",
    "top_overlap",
    "save_figure",
    "sha256_file",
]

missing_common_symbols = [
    name
    for name in required_common_symbols
    if name not in globals()
]

if missing_common_symbols:
    raise RuntimeError(
        "Fresh/incomplete runtime detected. "
        "Rerun SHAP Block 1 (Common Setup) once, "
        "then run this Block 4 directly. "
        "Do NOT rerun Blocks 2 or 3.\n\n"
        "Missing Block-1 symbols:\n  - "
        + "\n  - ".join(missing_common_symbols)
    )

FULL376_DIR = Path(FULL376_DIR)
FULL376_FIGURES_DIR = Path(
    FULL376_FIGURES_DIR
)
PARTITION_CHECKPOINT_DIR = Path(
    PARTITION_CHECKPOINT_DIR
)

FULL376_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
FULL376_FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
PARTITION_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def atomic_save_npy(path, array):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        path.name + ".tmp"
    )

    with open(temporary, "wb") as handle:
        np.save(handle, array)
        handle.flush()
        os.fsync(handle.fileno())

    temporary.replace(path)


kernel_tensor_path = (
    FULL376_DIR
    / "Kernel_SHAP_Full376_4096.npy"
)
kernel_expected_path = (
    FULL376_DIR
    / "Kernel_SHAP_Full376_Expected_Values.npy"
)
kernel_global_path = (
    FULL376_DIR
    / "Kernel_SHAP_Full376_Global_Importance.csv"
)
kernel_family_path = (
    FULL376_DIR
    / "Kernel_SHAP_Full376_Family_Importance.csv"
)
kernel_summary_path = (
    FULL376_DIR
    / "Kernel_SHAP_Full376_Summary.json"
)

required_kernel_paths = [
    kernel_tensor_path,
    kernel_expected_path,
    kernel_global_path,
    kernel_family_path,
    kernel_summary_path,
]

missing_kernel = [
    str(path)
    for path in required_kernel_paths
    if not path.exists()
]

if missing_kernel:
    raise FileNotFoundError(
        "The completed Block-3 Kernel SHAP "
        "artifacts are not all present on Google Drive.\n\n"
        "Missing:\n"
        + "\n".join(missing_kernel)
    )

kernel_shap = np.asarray(
    np.load(
        kernel_tensor_path,
        allow_pickle=False,
    ),
    dtype=np.float64,
)

kernel_expected_values = np.asarray(
    np.load(
        kernel_expected_path,
        allow_pickle=False,
    ),
    dtype=np.float64,
).reshape(-1)

expected_kernel_shape = (
    EXPECTED_TEST_ROWS,
    EXPECTED_FEATURES,
    EXPECTED_CLASSES,
)

if kernel_shap.shape != expected_kernel_shape:
    raise RuntimeError(
        f"Unexpected saved Kernel SHAP shape: "
        f"{kernel_shap.shape}; "
        f"expected {expected_kernel_shape}."
    )

if kernel_expected_values.shape != (
    EXPECTED_CLASSES,
):
    raise RuntimeError(
        "Unexpected saved Kernel expected-value "
        f"shape: {kernel_expected_values.shape}."
    )

if not np.all(np.isfinite(kernel_shap)):
    raise RuntimeError(
        "Saved Kernel SHAP tensor "
        "contains NaN or Inf."
    )

kernel_global_df = pd.read_csv(
    kernel_global_path
)
kernel_family_df = pd.read_csv(
    kernel_family_path
)

with open(
    kernel_summary_path,
    "r",
    encoding="utf-8",
) as handle:
    kernel_summary = json.load(handle)

kernel_mean_residual = float(
    kernel_summary[
        "mean_local_fidelity_residual"
    ]
)
kernel_max_residual = float(
    kernel_summary[
        "maximum_local_fidelity_residual"
    ]
)

kernel_decision_shap = kernel_shap[
    np.arange(EXPECTED_TEST_ROWS),
    :,
    predicted_explain_classes,
]

print(
    "\nLoaded completed Kernel SHAP "
    "from Google Drive."
)
print(
    "Kernel tensor shape:",
    kernel_shap.shape,
)
print(
    f"Kernel mean residual: "
    f"{kernel_mean_residual:.12g}"
)
print(
    f"Kernel max residual : "
    f"{kernel_max_residual:.12g}"
)

partition_full_values_path = (
    FULL376_DIR
    / "Partition_SHAP_Full376_4096.npy"
)
partition_full_base_path = (
    FULL376_DIR
    / "Partition_SHAP_Full376_Base_Values.npy"
)
partition_clustering_path = (
    FULL376_DIR
    / "Partition_SHAP_Feature_Clustering.npy"
)

expected_partition_shape = (
    EXPECTED_TEST_ROWS,
    EXPECTED_FEATURES,
    EXPECTED_CLASSES,
)

expected_partition_base_shape = (
    EXPECTED_TEST_ROWS,
    EXPECTED_CLASSES,
)


def load_valid_npy(path, expected_shape):
    path = Path(path)

    if not path.exists():
        return None

    try:
        array = np.asarray(
            np.load(
                path,
                allow_pickle=False,
            ),
            dtype=np.float64,
        )
    except Exception as exc:
        print(
            f"[Invalid saved file] "
            f"{path.name}: {exc}"
        )
        return None

    if array.shape != expected_shape:
        print(
            f"[Invalid shape] {path.name}: "
            f"found {array.shape}, "
            f"expected {expected_shape}"
        )
        return None

    if not np.all(np.isfinite(array)):
        print(
            f"[Invalid values] {path.name}: "
            "contains NaN or Inf"
        )
        return None

    return array


def partition_checkpoint_paths(start, end):
    values_path = (
        PARTITION_CHECKPOINT_DIR
        / (
            "partition_4096_values_"
            f"{start:03d}_{end:03d}.npy"
        )
    )
    base_path = (
        PARTITION_CHECKPOINT_DIR
        / (
            "partition_4096_base_"
            f"{start:03d}_{end:03d}.npy"
        )
    )
    return values_path, base_path


def checkpoint_pair_is_valid(start, end):
    (
        values_path,
        base_path,
    ) = partition_checkpoint_paths(
        start,
        end,
    )

    expected_values_shape = (
        end - start,
        EXPECTED_FEATURES,
        EXPECTED_CLASSES,
    )
    expected_base_shape = (
        end - start,
        EXPECTED_CLASSES,
    )

    values = load_valid_npy(
        values_path,
        expected_values_shape,
    )
    base = load_valid_npy(
        base_path,
        expected_base_shape,
    )

    valid = (
        values is not None
        and base is not None
    )

    del values, base
    gc.collect()

    return valid


partition_shap = load_valid_npy(
    partition_full_values_path,
    expected_partition_shape,
)

partition_base_values = load_valid_npy(
    partition_full_base_path,
    expected_partition_base_shape,
)

full_partition_available = (
    partition_shap is not None
    and partition_base_values is not None
)

if full_partition_available:
    print(
        "\nLoaded COMPLETE Partition SHAP "
        "tensors from Google Drive."
    )
    print(
        "Partition tensor shape:",
        partition_shap.shape,
    )
    print(
        "Partition base-value shape:",
        partition_base_values.shape,
    )

else:
    partition_shap = None
    partition_base_values = None

    print(
        "\nComplete Partition tensors were "
        "not both available/valid."
    )
    print(
        "Checking saved Partition chunk checkpoints..."
    )

    chunk_specs = []
    completed_chunks = 0
    missing_chunks = []

    for start in range(
        0,
        EXPECTED_TEST_ROWS,
        PARTITION_CHUNK_SIZE,
    ):
        end = min(
            start + PARTITION_CHUNK_SIZE,
            EXPECTED_TEST_ROWS,
        )

        (
            values_path,
            base_path,
        ) = partition_checkpoint_paths(
            start,
            end,
        )

        chunk_specs.append(
            (
                start,
                end,
                values_path,
                base_path,
            )
        )

        if checkpoint_pair_is_valid(
            start,
            end,
        ):
            completed_chunks += 1
        else:
            values_path.unlink(
                missing_ok=True
            )
            base_path.unlink(
                missing_ok=True
            )
            missing_chunks.append(
                (start, end)
            )

    print(
        "Valid saved Partition chunks: "
        f"{completed_chunks}/{len(chunk_specs)}"
    )

    if missing_chunks:
        print(
            "Chunks still requiring computation: "
            f"{len(missing_chunks)}"
        )
    else:
        print(
            "All Partition chunks are "
            "already present on Drive."
        )

    if missing_chunks:
        feature_clustering = None

        if partition_clustering_path.exists():
            try:
                candidate_clustering = np.asarray(
                    np.load(
                        partition_clustering_path,
                        allow_pickle=False,
                    ),
                    dtype=np.float64,
                )

                expected_clustering_shape = (
                    EXPECTED_FEATURES - 1,
                    4,
                )

                if (
                    candidate_clustering.shape
                    == expected_clustering_shape
                ):
                    feature_clustering = (
                        candidate_clustering
                    )
                    print(
                        "Loaded saved correlation "
                        "hierarchy from Drive:",
                        feature_clustering.shape,
                    )
                else:
                    print(
                        "Saved hierarchy has an "
                        "unexpected shape; rebuilding. "
                        f"Found {candidate_clustering.shape}; "
                        "expected "
                        f"{expected_clustering_shape}."
                    )

            except Exception as exc:
                print(
                    "Could not load saved correlation "
                    "hierarchy; rebuilding it. "
                    f"Reason: {exc}"
                )

        if feature_clustering is None:
            print(
                "\nBuilding correlation hierarchy "
                "from 1,200 real training observations..."
            )

            feature_clustering = shap.utils.hclust(
                X_train_real_scaled,
                linkage="average",
                metric="correlation",
                random_state=RANDOM_STATE,
            )

            expected_clustering_shape = (
                EXPECTED_FEATURES - 1,
                4,
            )

            if (
                feature_clustering.shape
                != expected_clustering_shape
            ):
                raise RuntimeError(
                    "Unexpected Partition clustering "
                    f"shape: {feature_clustering.shape}"
                )

            atomic_save_npy(
                partition_clustering_path,
                feature_clustering,
            )

            print(
                "Saved correlation hierarchy:",
                feature_clustering.shape,
            )

        tf.keras.backend.clear_session()
        gc.collect()

        partition_shap_model = load_model(
            MODEL_PATH,
            custom_objects=(
                SELF_ATTENTION_CUSTOM_OBJECTS
            ),
            compile=False,
        )

        def partition_predict_wrapper(
            X_2d,
            _model=partition_shap_model,
        ):
            X_2d = np.asarray(
                X_2d,
                dtype=np.float32,
            )

            if X_2d.ndim == 1:
                X_2d = X_2d.reshape(
                    1,
                    -1,
                )

            if X_2d.ndim != 2:
                raise RuntimeError(
                    "Partition SHAP wrapper expects "
                    "2-D input; received "
                    f"{X_2d.shape}."
                )

            if X_2d.shape[1] != EXPECTED_FEATURES:
                raise RuntimeError(
                    f"Expected {EXPECTED_FEATURES} "
                    "features; received "
                    f"{X_2d.shape[1]}."
                )

            probabilities = np.asarray(
                _model.predict(
                    X_2d[..., np.newaxis],
                    verbose=0,
                ),
                dtype=np.float64,
            )

            expected_probability_shape = (
                len(X_2d),
                EXPECTED_CLASSES,
            )

            if (
                probabilities.shape
                != expected_probability_shape
            ):
                raise RuntimeError(
                    "Unexpected CNN-BiLSTM "
                    "probability shape: "
                    f"{probabilities.shape}"
                )

            return probabilities

        resume_wrapper_check = (
            partition_predict_wrapper(
                X_explain[:5]
            )
        )

        if not np.allclose(
            resume_wrapper_check,
            explain_probabilities[:5],
            atol=1e-6,
            rtol=1e-6,
        ):
            raise RuntimeError(
                "Resume-time CNN-BiLSTM wrapper "
                "does not reproduce the Block-1 "
                "held-out probabilities."
            )

        print(
            "Dedicated Partition SHAP "
            "CNN-BiLSTM wrapper verified."
        )
        print(
            "Maximum probability difference:",
            float(
                np.max(
                    np.abs(
                        resume_wrapper_check
                        - explain_probabilities[:5]
                    )
                )
            ),
        )

        partition_masker = shap.maskers.Partition(
            X_background,
            clustering=feature_clustering,
        )

        partition_explainer = shap.Explainer(
            partition_predict_wrapper,
            partition_masker,
            algorithm="partition",
            output_names=[
                str(value)
                for value in device_classes
            ],
            feature_names=feature_names,
        )

        for chunk_number, (
            start,
            end,
        ) in enumerate(
            missing_chunks,
            start=1,
        ):
            (
                values_path,
                base_path,
            ) = partition_checkpoint_paths(
                start,
                end,
            )

            print(
                "\nPartition SHAP resume chunk "
                f"{chunk_number}/"
                f"{len(missing_chunks)}: "
                f"{start}:{end} of "
                f"{EXPECTED_TEST_ROWS}"
            )

            explanation = partition_explainer(
                X_explain[start:end],
                max_evals=PARTITION_MAX_EVALS,
                batch_size=PARTITION_BATCH_SIZE,
                silent=False,
            )

            chunk_values = normalise_multiclass_shap(
                explanation.values,
                n_samples=(end - start),
            )

            raw_base = np.asarray(
                explanation.base_values,
                dtype=np.float64,
            )

            if (
                raw_base.ndim == 1
                and raw_base.shape
                == (EXPECTED_CLASSES,)
            ):
                chunk_base = np.repeat(
                    raw_base[np.newaxis, :],
                    end - start,
                    axis=0,
                )

            elif raw_base.shape == (
                end - start,
                EXPECTED_CLASSES,
            ):
                chunk_base = raw_base

            else:
                raise RuntimeError(
                    "Unexpected Partition "
                    "base-value shape: "
                    f"{raw_base.shape}"
                )

            expected_values_shape = (
                end - start,
                EXPECTED_FEATURES,
                EXPECTED_CLASSES,
            )

            if (
                chunk_values.shape
                != expected_values_shape
            ):
                raise RuntimeError(
                    "Unexpected Partition "
                    f"chunk shape: {chunk_values.shape}"
                )

            if not np.all(
                np.isfinite(chunk_values)
            ):
                raise RuntimeError(
                    f"Partition chunk {start}:{end} "
                    "contains NaN or Inf."
                )

            if not np.all(
                np.isfinite(chunk_base)
            ):
                raise RuntimeError(
                    "Partition base chunk "
                    f"{start}:{end} "
                    "contains NaN or Inf."
                )

            atomic_save_npy(
                values_path,
                chunk_values,
            )
            atomic_save_npy(
                base_path,
                chunk_base,
            )

            print(
                "Saved checkpoint pair for "
                f"{start}:{end}"
            )

            del (
                explanation,
                chunk_values,
                chunk_base,
                raw_base,
            )
            gc.collect()

        del (
            partition_explainer,
            partition_masker,
            partition_shap_model,
        )
        gc.collect()

    partition_value_files = []
    partition_base_files = []

    for (
        start,
        end,
        values_path,
        base_path,
    ) in chunk_specs:
        if not checkpoint_pair_is_valid(
            start,
            end,
        ):
            raise RuntimeError(
                "A required Partition checkpoint "
                "is still missing or invalid "
                f"after resume: {start}:{end}"
            )

        partition_value_files.append(
            values_path
        )
        partition_base_files.append(
            base_path
        )

    partition_shap = np.concatenate(
        [
            np.load(
                path,
                allow_pickle=False,
            )
            for path in partition_value_files
        ],
        axis=0,
    ).astype(np.float64)

    partition_base_values = np.concatenate(
        [
            np.load(
                path,
                allow_pickle=False,
            )
            for path in partition_base_files
        ],
        axis=0,
    ).astype(np.float64)

    if (
        partition_shap.shape
        != expected_partition_shape
    ):
        raise RuntimeError(
            "Unexpected reassembled Partition "
            f"SHAP shape: {partition_shap.shape}"
        )

    if (
        partition_base_values.shape
        != expected_partition_base_shape
    ):
        raise RuntimeError(
            "Unexpected reassembled Partition "
            "base-value shape: "
            f"{partition_base_values.shape}"
        )

    if not np.all(
        np.isfinite(partition_shap)
    ):
        raise RuntimeError(
            "Reassembled Partition SHAP tensor "
            "contains NaN or Inf."
        )

    if not np.all(
        np.isfinite(partition_base_values)
    ):
        raise RuntimeError(
            "Reassembled Partition base values "
            "contain NaN or Inf."
        )

    atomic_save_npy(
        partition_full_values_path,
        partition_shap,
    )
    atomic_save_npy(
        partition_full_base_path,
        partition_base_values,
    )

    print(
        "\nPartition SHAP resume/reassembly complete."
    )
    print(
        "Partition tensor shape:",
        partition_shap.shape,
    )
    print(
        "Partition base-value shape:",
        partition_base_values.shape,
    )

### Full-Test SHAP Post-Processing and Cross-Method Comparison

This saved-artifact cell compares completed full-test Kernel SHAP and correlation-aware Partition SHAP results. It does not recompute either explanation method.

It dynamically searches:

```text
Honours_Project/Models/Tuned/CNN_BiLSTM/
```

for runs containing the complete required SHAP tensors and asks the user to select the intended run.

Required artifacts include:

```text
X_test_3d.npy
y_test_1d.npy
device_classes.npy
feature_names.npy
SHAP_Final/Common/Feature_Family_Mapping.csv
SHAP_Final/Full376/Kernel_SHAP_Full376_4096.npy
SHAP_Final/Full376/Kernel_SHAP_Full376_Expected_Values.npy
SHAP_Final/Full376/Partition_SHAP_Full376_4096.npy
SHAP_Final/Full376/Partition_SHAP_Full376_Base_Values.npy
```

The expected dimensions are:

```text
held-out observations: 376
predictors:             244
classes:                10
SHAP tensor:            (376, 244, 10)
```

The cell validates shapes, feature order and finite values. It loads or creates the cached CNN-BiLSTM probability matrix, then checks the additive reconstruction fidelity of both explanation methods.

For every observation, comparisons use the SHAP values associated with the class predicted by the model. The cell calculates:

- Kernel and Partition global feature importance;
- Kernel and Partition feature-family importance;
- all-feature Spearman rank correlation;
- top-10, top-20 and top-50 overlap;
- shared top-10 predictors;
- family-level Spearman correlation; and
- per-observation local-fidelity residuals.

Outputs are written beneath:

```text
<selected CNN-BiLSTM run>/SHAP_Final/Full376/
```

Principal outputs include:

```text
Kernel_SHAP_Full376_Global_Importance_Recomputed.csv
Partition_SHAP_Full376_Global_Importance.csv
Kernel_SHAP_Full376_Family_Importance_Recomputed.csv
Partition_SHAP_Full376_Family_Importance.csv
SHAP_Full376_Kernel_vs_Partition_Features.csv
SHAP_Full376_Family_Comparison.csv
SHAP_Full376_Shared_Top10.csv
SHAP_Full376_Cross_Method_Summary.json
```

The comparison figures are saved in `Full376/Figures/` as PNG and PDF files.

This is the final SHAP interpretation and agreement-analysis stage. It does not retrain the model, change the selected seed, alter the held-out data or use explanation results for model selection.

In [ ]:
import gc
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import drive
from scipy.stats import spearmanr

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Honours_Project"
)
CNN_BILSTM_ROOT = (
    PROJECT_ROOT
    / "Models"
    / "Tuned"
    / "CNN_BiLSTM"
)


def select_completed_shap_run():
    required_relative_paths = [
        Path("best_validation_cnn_bilstm.keras"),
        Path("X_test_3d.npy"),
        Path("y_test_1d.npy"),
        Path("device_classes.npy"),
        Path("feature_names.npy"),
        Path(
            "SHAP_Final/Common/"
            "SHAP_Common_Metadata.json"
        ),
        Path(
            "SHAP_Final/Common/"
            "Feature_Family_Mapping.csv"
        ),
        Path(
            "SHAP_Final/Full376/"
            "Kernel_SHAP_Full376_4096.npy"
        ),
        Path(
            "SHAP_Final/Full376/"
            "Kernel_SHAP_Full376_Expected_Values.npy"
        ),
        Path(
            "SHAP_Final/Full376/"
            "Partition_SHAP_Full376_4096.npy"
        ),
        Path(
            "SHAP_Final/Full376/"
            "Partition_SHAP_Full376_Base_Values.npy"
        ),
    ]

    candidates = []

    if CNN_BILSTM_ROOT.exists():
        for candidate in CNN_BILSTM_ROOT.iterdir():
            if candidate.is_dir() and all(
                (
                    candidate / relative
                ).exists()
                for relative
                in required_relative_paths
            ):
                candidates.append(
                    candidate.resolve()
                )

    candidates = sorted(
        candidates,
        key=lambda path: path.name,
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError(
            "No tuned CNN-BiLSTM run with "
            "complete Kernel and Partition "
            "SHAP artifacts was found under:\n"
            f"{CNN_BILSTM_ROOT}"
        )

    print("\n" + "=" * 100)
    print(
        "SELECT COMPLETED CNN-BILSTM SHAP RUN"
    )
    print("=" * 100)

    for number, candidate in enumerate(
        candidates,
        start=1,
    ):
        print(
            f"[{number}] {candidate}"
        )

    while True:
        response = input(
            f"Select 1-{len(candidates)} "
            "(Enter = newest): "
        ).strip()

        if response == "":
            return candidates[0]

        if (
            response.isdigit()
            and 1 <= int(response) <= len(candidates)
        ):
            return candidates[
                int(response) - 1
            ]

        print("Invalid selection.")


def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(chunk_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


RUN_DIR = select_completed_shap_run()

SHAP_ROOT = RUN_DIR / "SHAP_Final"
COMMON_DIR = SHAP_ROOT / "Common"
FULL376_DIR = SHAP_ROOT / "Full376"
FULL376_FIGURES_DIR = (
    FULL376_DIR / "Figures"
)
FULL376_FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXPECTED_TEST_ROWS = 376
EXPECTED_FEATURES = 244
EXPECTED_CLASSES = 10

X_TEST_PATH = RUN_DIR / "X_test_3d.npy"
Y_TEST_PATH = RUN_DIR / "y_test_1d.npy"
CLASSES_PATH = RUN_DIR / "device_classes.npy"
FEATURE_NAMES_PATH = (
    RUN_DIR / "feature_names.npy"
)
MODEL_PATH = (
    RUN_DIR
    / "best_validation_cnn_bilstm.keras"
)
FEATURE_MAPPING_PATH = (
    COMMON_DIR
    / "Feature_Family_Mapping.csv"
)
COMMON_METADATA_PATH = (
    COMMON_DIR
    / "SHAP_Common_Metadata.json"
)

kernel_values_path = (
    FULL376_DIR
    / "Kernel_SHAP_Full376_4096.npy"
)
kernel_expected_path = (
    FULL376_DIR
    / "Kernel_SHAP_Full376_Expected_Values.npy"
)
partition_values_path = (
    FULL376_DIR
    / "Partition_SHAP_Full376_4096.npy"
)
partition_base_path = (
    FULL376_DIR
    / "Partition_SHAP_Full376_Base_Values.npy"
)
MODEL_PROBABILITIES_CACHE = (
    FULL376_DIR
    / "CNN_BiLSTM_Full376_Model_Probabilities.npy"
)

required_paths = [
    X_TEST_PATH,
    Y_TEST_PATH,
    CLASSES_PATH,
    FEATURE_NAMES_PATH,
    FEATURE_MAPPING_PATH,
    COMMON_METADATA_PATH,
    kernel_values_path,
    kernel_expected_path,
    partition_values_path,
    partition_base_path,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Cannot run standalone SHAP post-processing "
        "because these saved artifacts are missing:\n"
        + "\n".join(
            f"- {path}"
            for path in missing_paths
        )
        + "\n\nThis cell will not recompute "
        "Kernel or Partition SHAP."
    )

with COMMON_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    common_metadata = json.load(handle)

provenance_checks = {
    "selected_model_sha256": MODEL_PATH,
    "x_test_sha256": X_TEST_PATH,
    "y_test_sha256": Y_TEST_PATH,
    "device_classes_sha256": CLASSES_PATH,
    "feature_names_sha256": FEATURE_NAMES_PATH,
}

for (
    metadata_key,
    artifact_path,
) in provenance_checks.items():
    expected_hash = common_metadata.get(
        metadata_key
    )

    if not expected_hash:
        raise RuntimeError(
            "SHAP_Common_Metadata.json does not "
            f"contain {metadata_key}. "
            "Rerun the updated SHAP Block 1 "
            "for this model run."
        )

    actual_hash = sha256_file(
        artifact_path
    )

    if actual_hash != expected_hash:
        raise RuntimeError(
            "SHAP provenance mismatch for "
            f"{artifact_path.name}.\n"
            f"Expected: {expected_hash}\n"
            f"Actual:   {actual_hash}"
        )

X_test_3d = np.asarray(
    np.load(
        X_TEST_PATH,
        allow_pickle=False,
    ),
    dtype=np.float32,
)

y_true = np.asarray(
    np.load(
        Y_TEST_PATH,
        allow_pickle=False,
    ),
    dtype=np.int64,
).reshape(-1)

device_classes = np.asarray(
    np.load(
        CLASSES_PATH,
        allow_pickle=True,
    )
)

feature_names = [
    str(value)
    for value in np.load(
        FEATURE_NAMES_PATH,
        allow_pickle=True,
    )
]

feature_family_df = (
    pd.read_csv(FEATURE_MAPPING_PATH)
    .sort_values("feature_index")
    .reset_index(drop=True)
)

if X_test_3d.shape != (
    EXPECTED_TEST_ROWS,
    EXPECTED_FEATURES,
    1,
):
    raise RuntimeError(
        f"Unexpected X_test_3d shape: "
        f"{X_test_3d.shape}"
    )

if y_true.shape != (
    EXPECTED_TEST_ROWS,
):
    raise RuntimeError(
        f"Unexpected y_test shape: "
        f"{y_true.shape}"
    )

if len(device_classes) != EXPECTED_CLASSES:
    raise RuntimeError(
        f"Expected {EXPECTED_CLASSES} classes; "
        f"found {len(device_classes)}."
    )

if len(feature_names) != EXPECTED_FEATURES:
    raise RuntimeError(
        f"Expected {EXPECTED_FEATURES} features; "
        f"found {len(feature_names)}."
    )

required_mapping_columns = {
    "feature_index",
    "feature",
    "feature_family",
}

if not required_mapping_columns.issubset(
    feature_family_df.columns
):
    raise RuntimeError(
        "Feature_Family_Mapping.csv does not "
        "contain the expected columns."
    )

if (
    feature_family_df["feature"]
    .astype(str)
    .tolist()
    != [
        str(value)
        for value in feature_names
    ]
):
    raise RuntimeError(
        "Feature_Family_Mapping.csv is not "
        "aligned with feature_names.npy."
    )

kernel_shap = np.asarray(
    np.load(
        kernel_values_path,
        allow_pickle=False,
    ),
    dtype=np.float64,
)

kernel_expected_values = np.asarray(
    np.load(
        kernel_expected_path,
        allow_pickle=False,
    ),
    dtype=np.float64,
).reshape(-1)

partition_shap = np.asarray(
    np.load(
        partition_values_path,
        allow_pickle=False,
    ),
    dtype=np.float64,
)

partition_base_values = np.asarray(
    np.load(
        partition_base_path,
        allow_pickle=False,
    ),
    dtype=np.float64,
)

expected_probability_shape = (
    EXPECTED_TEST_ROWS,
    EXPECTED_CLASSES,
)

model_probabilities = None

if MODEL_PROBABILITIES_CACHE.exists():
    try:
        cached = np.asarray(
            np.load(
                MODEL_PROBABILITIES_CACHE,
                allow_pickle=False,
            ),
            dtype=np.float64,
        )

        if (
            cached.shape
            == expected_probability_shape
            and np.all(np.isfinite(cached))
        ):
            model_probabilities = cached
            print(
                "Loaded cached CNN-BiLSTM "
                "test probabilities."
            )
        else:
            print(
                "Ignoring invalid probability "
                "cache and recreating it."
            )

    except Exception as exc:
        print(
            "Could not load probability cache; "
            f"recreating it: {exc}"
        )

if model_probabilities is None:
    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            "No valid probability cache exists "
            "and the selected model is missing:\n"
            f"{MODEL_PATH}"
        )

    print(
        "Running one ordinary 376-case "
        "CNN-BiLSTM inference pass "
        "(SHAP is NOT being recomputed)."
    )

    import tensorflow as tf
    import tensorflow.keras.backend as K
    from tensorflow.keras.layers import Layer
    from tensorflow.keras.models import load_model

    @tf.keras.utils.register_keras_serializable(
        package="COMP700"
    )
    class SelfAttention(Layer):
        def __init__(self, **kwargs):
            super().__init__(**kwargs)

        def build(self, input_shape):
            self.W = self.add_weight(
                name="att_weight",
                shape=(input_shape[-1], 1),
                initializer="glorot_uniform",
                trainable=True,
            )
            self.b = self.add_weight(
                name="att_bias",
                shape=(input_shape[1], 1),
                initializer="zeros",
                trainable=True,
            )
            super().build(input_shape)

        def call(self, inputs):
            scores = K.tanh(
                K.dot(inputs, self.W)
                + self.b
            )
            weights = K.softmax(
                scores,
                axis=1,
            )
            return K.sum(
                inputs * weights,
                axis=1,
            )

    SELF_ATTENTION_CUSTOM_OBJECTS = {
        "SelfAttention": SelfAttention,
        "Custom>SelfAttention": SelfAttention,
        "COMP700>SelfAttention": SelfAttention,
    }

    tf.keras.backend.clear_session()
    gc.collect()

    selected_model = load_model(
        MODEL_PATH,
        custom_objects=(
            SELF_ATTENTION_CUSTOM_OBJECTS
        ),
        compile=False,
    )

    model_probabilities = np.asarray(
        selected_model.predict(
            X_test_3d,
            verbose=0,
        ),
        dtype=np.float64,
    )

    if (
        model_probabilities.shape
        != expected_probability_shape
    ):
        raise RuntimeError(
            "Unexpected selected-model "
            "probability shape: "
            f"{model_probabilities.shape}"
        )

    if not np.all(
        np.isfinite(model_probabilities)
    ):
        raise RuntimeError(
            "Selected-model probabilities "
            "contain NaN or Inf."
        )

    np.save(
        MODEL_PROBABILITIES_CACHE,
        model_probabilities,
    )

    print(
        "Saved reusable probability cache:\n"
        f"{MODEL_PROBABILITIES_CACHE}"
    )

    del selected_model
    gc.collect()

family_names = sorted(
    feature_family_df[
        "feature_family"
    ]
    .astype(str)
    .unique()
    .tolist()
)

family_to_indices = {
    family: (
        feature_family_df.loc[
            feature_family_df[
                "feature_family"
            ].astype(str) == family,
            "feature_index",
        ]
        .to_numpy(dtype=int)
    )
    for family in family_names
}


def aggregate_family_importance(
    decision_shap,
):
    absolute_values = np.abs(
        np.asarray(
            decision_shap,
            dtype=np.float64,
        )
    )

    if (
        absolute_values.ndim != 2
        or absolute_values.shape[1]
        != EXPECTED_FEATURES
    ):
        raise RuntimeError(
            "Expected decision SHAP shape "
            f"(n, {EXPECTED_FEATURES}); "
            f"found {absolute_values.shape}."
        )

    rows = []

    for family in family_names:
        indices = family_to_indices[family]

        per_sample_total = np.sum(
            absolute_values[:, indices],
            axis=1,
        )

        total = float(
            np.mean(per_sample_total)
        )

        rows.append(
            {
                "feature_family": family,
                "number_of_features": int(
                    len(indices)
                ),
                "total_mean_abs_shap": total,
                "mean_abs_shap_per_feature": (
                    total / float(len(indices))
                ),
            }
        )

    result = pd.DataFrame(rows)

    grand_total = float(
        result[
            "total_mean_abs_shap"
        ].sum()
    )

    if (
        not np.isfinite(grand_total)
        or grand_total <= 0.0
    ):
        raise RuntimeError(
            "Invalid total SHAP magnitude: "
            f"{grand_total}"
        )

    result[
        "share_of_total_abs_shap"
    ] = (
        result["total_mean_abs_shap"]
        / grand_total
    )

    result = (
        result
        .sort_values(
            "share_of_total_abs_shap",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    result.insert(
        0,
        "rank",
        np.arange(1, len(result) + 1),
    )

    return result


print("=" * 100)
print(
    "SAVED SHAP ARTIFACTS LOADED — "
    "STARTING POST-PROCESSING ONLY"
)
print("=" * 100)
print(
    "Kernel tensor    :",
    kernel_shap.shape,
)
print(
    "Partition tensor :",
    partition_shap.shape,
)
print(
    "Output directory :",
    FULL376_DIR,
)

expected_tensor_shape = (
    EXPECTED_TEST_ROWS,
    EXPECTED_FEATURES,
    EXPECTED_CLASSES,
)

if kernel_shap.shape != expected_tensor_shape:
    raise RuntimeError(
        "Unexpected Kernel SHAP shape: "
        f"{kernel_shap.shape}"
    )

if partition_shap.shape != expected_tensor_shape:
    raise RuntimeError(
        "Unexpected Partition SHAP shape: "
        f"{partition_shap.shape}"
    )

if kernel_expected_values.shape != (
    EXPECTED_CLASSES,
):
    raise RuntimeError(
        "Unexpected Kernel expected-value shape: "
        f"{kernel_expected_values.shape}"
    )

if partition_base_values.shape != (
    expected_probability_shape
):
    raise RuntimeError(
        "Unexpected Partition base-value shape: "
        f"{partition_base_values.shape}"
    )

if model_probabilities.shape != (
    expected_probability_shape
):
    raise RuntimeError(
        "Unexpected model-probability shape: "
        f"{model_probabilities.shape}"
    )

for name, array in {
    "Kernel SHAP": kernel_shap,
    "Kernel expected values": (
        kernel_expected_values
    ),
    "Partition SHAP": partition_shap,
    "Partition base values": (
        partition_base_values
    ),
    "model probabilities": model_probabilities,
}.items():
    if not np.all(np.isfinite(array)):
        raise RuntimeError(
            f"{name} contains NaN or Inf."
        )

predicted_classes = np.argmax(
    model_probabilities,
    axis=1,
)

row_indices = np.arange(
    EXPECTED_TEST_ROWS
)

kernel_reconstructed = (
    kernel_expected_values[np.newaxis, :]
    + kernel_shap.sum(axis=1)
)

partition_reconstructed = (
    partition_base_values
    + partition_shap.sum(axis=1)
)

kernel_residuals = np.abs(
    model_probabilities
    - kernel_reconstructed
)

partition_residuals = np.abs(
    model_probabilities
    - partition_reconstructed
)

kernel_decision_shap = kernel_shap[
    row_indices,
    :,
    predicted_classes,
]

partition_decision_shap = partition_shap[
    row_indices,
    :,
    predicted_classes,
]

kernel_importance = np.mean(
    np.abs(kernel_decision_shap),
    axis=0,
)

partition_importance = np.mean(
    np.abs(partition_decision_shap),
    axis=0,
)

feature_mapping = (
    feature_family_df
    .sort_values("feature_index")
    .reset_index(drop=True)
)

if (
    feature_mapping["feature"]
    .astype(str)
    .tolist()
    != [
        str(value)
        for value in feature_names
    ]
):
    raise RuntimeError(
        "Feature-family mapping is not aligned "
        "with the retained feature order."
    )


def descending_ranks(values):
    values = np.asarray(values)

    order = np.argsort(
        -values,
        kind="mergesort",
    )

    ranks = np.empty(
        len(order),
        dtype=int,
    )

    ranks[order] = np.arange(
        1,
        len(order) + 1,
    )

    return ranks


def overlap_fraction(a, b, k):
    top_a = set(
        np.argsort(
            np.asarray(a)
        )[::-1][:k].tolist()
    )
    top_b = set(
        np.argsort(
            np.asarray(b)
        )[::-1][:k].tolist()
    )

    return (
        len(top_a.intersection(top_b))
        / float(k)
    )


def save_png_pdf(figure, stem):
    png_path = (
        FULL376_FIGURES_DIR
        / f"{stem}.png"
    )
    pdf_path = (
        FULL376_FIGURES_DIR
        / f"{stem}.pdf"
    )

    figure.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )
    figure.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white",
    )

    return png_path, pdf_path


kernel_ranks = descending_ranks(
    kernel_importance
)
partition_ranks = descending_ranks(
    partition_importance
)

kernel_global_df = pd.DataFrame(
    {
        "feature_index": np.arange(
            EXPECTED_FEATURES
        ),
        "feature": [
            str(value)
            for value in feature_names
        ],
        "feature_family": (
            feature_mapping[
                "feature_family"
            ].astype(str)
        ),
        "mean_abs_kernel_shap": (
            kernel_importance
        ),
        "sd_abs_kernel_shap_across_376_cases": (
            np.std(
                np.abs(
                    kernel_decision_shap
                ),
                axis=0,
                ddof=1,
            )
        ),
        "rank": kernel_ranks,
    }
).sort_values(
    [
        "rank",
        "feature_index",
    ]
).reset_index(drop=True)

partition_global_df = pd.DataFrame(
    {
        "feature_index": np.arange(
            EXPECTED_FEATURES
        ),
        "feature": [
            str(value)
            for value in feature_names
        ],
        "feature_family": (
            feature_mapping[
                "feature_family"
            ].astype(str)
        ),
        "mean_abs_partition_shap": (
            partition_importance
        ),
        "sd_abs_partition_shap_across_376_cases": (
            np.std(
                np.abs(
                    partition_decision_shap
                ),
                axis=0,
                ddof=1,
            )
        ),
        "rank": partition_ranks,
    }
).sort_values(
    [
        "rank",
        "feature_index",
    ]
).reset_index(drop=True)

kernel_family_df = (
    aggregate_family_importance(
        kernel_decision_shap
    )
)

partition_family_df = (
    aggregate_family_importance(
        partition_decision_shap
    )
)

kernel_fidelity_df = pd.DataFrame(
    {
        "test_index": row_indices,
        "actual_class": [
            str(device_classes[index])
            for index in y_true
        ],
        "predicted_class": [
            str(device_classes[index])
            for index in predicted_classes
        ],
        "mean_output_residual": (
            kernel_residuals.mean(axis=1)
        ),
        "max_output_residual": (
            kernel_residuals.max(axis=1)
        ),
        "predicted_class_residual": (
            kernel_residuals[
                row_indices,
                predicted_classes,
            ]
        ),
    }
)

partition_fidelity_df = pd.DataFrame(
    {
        "test_index": row_indices,
        "actual_class": [
            str(device_classes[index])
            for index in y_true
        ],
        "predicted_class": [
            str(device_classes[index])
            for index in predicted_classes
        ],
        "mean_output_residual": (
            partition_residuals.mean(axis=1)
        ),
        "max_output_residual": (
            partition_residuals.max(axis=1)
        ),
        "predicted_class_residual": (
            partition_residuals[
                row_indices,
                predicted_classes,
            ]
        ),
    }
)

feature_comparison_df = pd.DataFrame(
    {
        "feature_index": np.arange(
            EXPECTED_FEATURES
        ),
        "feature": [
            str(value)
            for value in feature_names
        ],
        "feature_family": (
            feature_mapping[
                "feature_family"
            ].astype(str)
        ),
        "kernel_mean_abs_shap": (
            kernel_importance
        ),
        "partition_mean_abs_shap": (
            partition_importance
        ),
        "kernel_share_of_total": (
            kernel_importance
            / kernel_importance.sum()
        ),
        "partition_share_of_total": (
            partition_importance
            / partition_importance.sum()
        ),
        "kernel_rank": kernel_ranks,
        "partition_rank": partition_ranks,
    }
)

feature_comparison_df[
    "in_kernel_top10"
] = (
    feature_comparison_df[
        "kernel_rank"
    ] <= 10
)

feature_comparison_df[
    "in_partition_top10"
] = (
    feature_comparison_df[
        "partition_rank"
    ] <= 10
)

feature_comparison_df[
    "shared_top10"
] = (
    feature_comparison_df[
        "in_kernel_top10"
    ]
    & feature_comparison_df[
        "in_partition_top10"
    ]
)

family_comparison_df = (
    kernel_family_df.rename(
        columns={
            "rank": "kernel_rank",
            "total_mean_abs_shap": (
                "kernel_total_mean_abs_shap"
            ),
            "mean_abs_shap_per_feature": (
                "kernel_mean_abs_shap_per_feature"
            ),
            "share_of_total_abs_shap": (
                "kernel_share_of_total_abs_shap"
            ),
        }
    )
    .merge(
        partition_family_df.rename(
            columns={
                "rank": "partition_rank",
                "total_mean_abs_shap": (
                    "partition_total_mean_abs_shap"
                ),
                "mean_abs_shap_per_feature": (
                    "partition_mean_abs_shap_per_feature"
                ),
                "share_of_total_abs_shap": (
                    "partition_share_of_total_abs_shap"
                ),
            }
        ),
        on=[
            "feature_family",
            "number_of_features",
        ],
        how="inner",
        validate="one_to_one",
    )
)

feature_spearman = float(
    spearmanr(
        kernel_importance,
        partition_importance,
    ).statistic
)

family_spearman = float(
    spearmanr(
        family_comparison_df[
            "kernel_total_mean_abs_shap"
        ],
        family_comparison_df[
            "partition_total_mean_abs_shap"
        ],
    ).statistic
)

summary = {
    "cases": int(EXPECTED_TEST_ROWS),
    "features": int(EXPECTED_FEATURES),
    "classes": int(EXPECTED_CLASSES),
    "decision_output": "predicted class",
    "kernel_mean_absolute_output_residual": float(
        kernel_residuals.mean()
    ),
    "kernel_maximum_absolute_output_residual": float(
        kernel_residuals.max()
    ),
    "partition_mean_absolute_output_residual": float(
        partition_residuals.mean()
    ),
    "partition_maximum_absolute_output_residual": float(
        partition_residuals.max()
    ),
    "all_feature_spearman": feature_spearman,
    "top10_overlap_fraction": overlap_fraction(
        kernel_importance,
        partition_importance,
        10,
    ),
    "top20_overlap_fraction": overlap_fraction(
        kernel_importance,
        partition_importance,
        20,
    ),
    "top50_overlap_fraction": overlap_fraction(
        kernel_importance,
        partition_importance,
        50,
    ),
    "shared_top10_count": int(
        feature_comparison_df[
            "shared_top10"
        ].sum()
    ),
    "family_spearman": family_spearman,
}

kernel_global_df.to_csv(
    FULL376_DIR
    / "Kernel_SHAP_Full376_Global_Importance_Recomputed.csv",
    index=False,
)

partition_global_df.to_csv(
    FULL376_DIR
    / "Partition_SHAP_Full376_Global_Importance.csv",
    index=False,
)

kernel_family_df.to_csv(
    FULL376_DIR
    / "Kernel_SHAP_Full376_Family_Importance_Recomputed.csv",
    index=False,
)

partition_family_df.to_csv(
    FULL376_DIR
    / "Partition_SHAP_Full376_Family_Importance.csv",
    index=False,
)

kernel_fidelity_df.to_csv(
    FULL376_DIR
    / "Kernel_SHAP_Full376_Local_Fidelity_Recomputed.csv",
    index=False,
)

partition_fidelity_df.to_csv(
    FULL376_DIR
    / "Partition_SHAP_Full376_Local_Fidelity.csv",
    index=False,
)

feature_comparison_df.to_csv(
    FULL376_DIR
    / "SHAP_Full376_Kernel_vs_Partition_Features.csv",
    index=False,
)

family_comparison_df.to_csv(
    FULL376_DIR
    / "SHAP_Full376_Family_Comparison.csv",
    index=False,
)

shared_top10_df = (
    feature_comparison_df.loc[
        feature_comparison_df[
            "shared_top10"
        ],
        [
            "feature",
            "feature_family",
            "kernel_rank",
            "partition_rank",
        ],
    ]
    .sort_values(
        [
            "kernel_rank",
            "partition_rank",
        ]
    )
)

shared_top10_df.to_csv(
    FULL376_DIR
    / "SHAP_Full376_Shared_Top10.csv",
    index=False,
)

with open(
    FULL376_DIR
    / "SHAP_Full376_Cross_Method_Summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
        allow_nan=False,
    )

top20_union = feature_comparison_df.loc[
    (
        feature_comparison_df[
            "kernel_rank"
        ] <= 20
    )
    | (
        feature_comparison_df[
            "partition_rank"
        ] <= 20
    )
].copy()

top20_union[
    "display_order"
] = top20_union[
    [
        "kernel_share_of_total",
        "partition_share_of_total",
    ]
].max(axis=1)

top20_union = top20_union.sort_values(
    [
        "display_order",
        "kernel_rank",
        "partition_rank",
    ],
    ascending=[
        True,
        False,
        False,
    ],
)

y_positions = np.arange(
    len(top20_union)
)
bar_height = 0.38

figure, axis = plt.subplots(
    figsize=(
        13,
        max(
            8,
            0.43 * len(top20_union),
        ),
    )
)

axis.barh(
    y_positions - bar_height / 2,
    top20_union[
        "kernel_share_of_total"
    ],
    bar_height,
    label="Kernel SHAP",
)

axis.barh(
    y_positions + bar_height / 2,
    top20_union[
        "partition_share_of_total"
    ],
    bar_height,
    label="Partition SHAP",
)

axis.set_yticks(y_positions)
axis.set_yticklabels(
    top20_union["feature"]
)
axis.set_xlabel(
    "Normalised global absolute attribution"
)
axis.set_ylabel("Feature")
axis.set_title(
    "Full-Test Kernel SHAP vs Partition SHAP"
)
axis.grid(
    axis="x",
    linestyle="--",
    alpha=0.25,
)
axis.legend(loc="lower right")
figure.tight_layout()

feature_figure_paths = save_png_pdf(
    figure,
    "SHAP_Full376_Kernel_vs_Partition_Features",
)

plt.show()
plt.close(figure)

family_plot_df = (
    family_comparison_df.copy()
)

family_plot_df[
    "display_order"
] = family_plot_df[
    [
        "kernel_share_of_total_abs_shap",
        "partition_share_of_total_abs_shap",
    ]
].max(axis=1)

family_plot_df = (
    family_plot_df.sort_values(
        "display_order",
        ascending=True,
    )
)

y_positions = np.arange(
    len(family_plot_df)
)

figure, axis = plt.subplots(
    figsize=(12, 7)
)

axis.barh(
    y_positions - bar_height / 2,
    (
        family_plot_df[
            "kernel_share_of_total_abs_shap"
        ]
        * 100.0
    ),
    bar_height,
    label="Kernel SHAP",
)

axis.barh(
    y_positions + bar_height / 2,
    (
        family_plot_df[
            "partition_share_of_total_abs_shap"
        ]
        * 100.0
    ),
    bar_height,
    label="Partition SHAP",
)

axis.set_yticks(y_positions)
axis.set_yticklabels(
    family_plot_df[
        "feature_family"
    ]
)
axis.set_xlabel(
    "Share of total absolute "
    "predicted-class attribution (%)"
)
axis.set_ylabel("Feature family")
axis.set_title(
    "Full-Test SHAP Feature-Family Comparison"
)
axis.grid(
    axis="x",
    linestyle="--",
    alpha=0.25,
)
axis.legend(loc="lower right")
figure.tight_layout()

family_figure_paths = save_png_pdf(
    figure,
    "SHAP_Full376_Family_Comparison",
)

plt.show()
plt.close(figure)

print("=" * 100)
print(
    "FULL-TEST SHAP CROSS-METHOD "
    "ANALYSIS COMPLETE"
)
print("=" * 100)
print(
    json.dumps(
        summary,
        indent=2,
    )
)
print("\nShared top-10 predictors:")
print(
    shared_top10_df.to_string(
        index=False
    )
)
print(
    "\nSaved feature figure:",
    feature_figure_paths[0],
)
print(
    "Saved family figure :",
    family_figure_paths[0],
)

## Post-hoc Difficult-Class Error Analysis

The held-out results identified `lights`, `socket` and `water_sensor` as the most difficult device classes. This section investigates why these classes are confused and whether the problem is associated with unstable optimisation, overlapping feature distributions, limited real class coverage, synthetic oversampling or differences in the decision boundaries learned by the selected deep-learning and classical models.

This is a post-hoc diagnostic analysis. It does not participate in preprocessing, hyperparameter tuning, early stopping, seed selection or model selection. The held-out observations are used only to explain the error behaviour of the already selected models. Any subsequent model modification motivated by these results must be developed using training and validation data and must not be selected according to improved performance on the existing held-out test set.

All analysis outputs are written to the selected CNN–BiLSTM run under:

`Post_Hoc_Difficult_Class_Error_Analysis/`

Each numbered analysis cell mounts Google Drive and reloads its required saved artifacts independently. The cells therefore do not need to be executed consecutively or in the same Colab session. The contrastive SHAP cell is the exception only in terms of saved prerequisites: complete Kernel and Partition SHAP tensors must already exist, but they may have been produced in an earlier session.

### Selected CNN–BiLSTM Confusion, Confidence and Case-Level Audit

This analysis examines the validation-selected CNN–BiLSTM at the individual-observation level. It reconstructs predictions from the archived probability matrix and records the predicted class, true-class probability, predicted-class probability, prediction margin, normalised entropy and rank of the correct class.

The analysis distinguishes low-confidence boundary errors from high-confidence errors. Low-confidence errors suggest that an observation lies close to a learned class boundary. High-confidence errors may indicate substantial feature overlap, ambiguous labels, missing discriminatory information or an unrepresentative training distribution.

The reduced confusion matrix concentrates on `lights`, `socket` and `water_sensor`, while retaining an `other` column to reveal whether observations leave this three-class confusion cluster.

In [ ]:
from pathlib import Path
import json

from google.colab import drive
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
RUNS_ROOT = PROJECT_ROOT / "Models" / "Tuned" / "CNN_BiLSTM"
DIFFICULT_CLASSES = ["lights", "socket", "water_sensor"]


def select_cnn_bilstm_run(required_paths):
    candidates = []

    if not RUNS_ROOT.exists():
        raise FileNotFoundError(f"Run root does not exist: {RUNS_ROOT}")

    for run_dir in RUNS_ROOT.iterdir():
        if not run_dir.is_dir():
            continue
        if not all((run_dir / relative_path).exists() for relative_path in required_paths):
            continue

        metadata_path = run_dir / "run_metadata.json"
        metadata = {}
        if metadata_path.exists():
            try:
                metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            except Exception:
                metadata = {}

        selection = metadata.get("selection", {})
        candidates.append(
            {
                "run_dir": run_dir,
                "modified": run_dir.stat().st_mtime,
                "validation_accuracy": selection.get("champion_validation_accuracy"),
                "champion_seed": selection.get("champion_seed"),
            }
        )

    candidates.sort(key=lambda item: item["modified"], reverse=True)

    if not candidates:
        raise FileNotFoundError(
            "No compatible CNN-BiLSTM run was found. The run must contain:\n"
            + "\n".join(f"  - {path}" for path in required_paths)
        )

    print("=" * 100)
    print("SELECT CNN-BILSTM RUN FOR DIFFICULT-CLASS CASE AUDIT")
    print("=" * 100)

    for number, candidate in enumerate(candidates, start=1):
        validation = candidate["validation_accuracy"]
        validation_text = "unknown" if validation is None else f"{validation * 100:.3f}%"
        print(
            f"[{number}] validation accuracy={validation_text} | "
            f"champion seed={candidate['champion_seed']}\n"
            f"    {candidate['run_dir']}"
        )

    while True:
        response = input(f"Select 1-{len(candidates)} (Enter = newest): ").strip()
        if response == "":
            return candidates[0]["run_dir"]
        if response.isdigit() and 1 <= int(response) <= len(candidates):
            return candidates[int(response) - 1]["run_dir"]
        print("Invalid selection.")


RUN_DIR = select_cnn_bilstm_run(
    [
        "y_test_1d.npy",
        "device_classes.npy",
        "feature_names.npy",
        "Test_Evaluation/best_validation_cnn_bilstm/prediction_probabilities.npy",
        "Test_Evaluation/best_validation_cnn_bilstm/confusion_matrix.npy",
    ]
)

EVALUATION_DIR = RUN_DIR / "Test_Evaluation" / "best_validation_cnn_bilstm"
OUTPUT_DIR = (
    RUN_DIR
    / "Post_Hoc_Difficult_Class_Error_Analysis"
    / "01_Selected_Model_Case_Audit"
)
FIGURES_DIR = OUTPUT_DIR / "Figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

y_true = np.load(RUN_DIR / "y_test_1d.npy").astype(int)
device_classes = np.load(RUN_DIR / "device_classes.npy", allow_pickle=True).astype(str)
feature_names = np.load(RUN_DIR / "feature_names.npy", allow_pickle=True).astype(str)
probabilities = np.load(EVALUATION_DIR / "prediction_probabilities.npy").astype(float)
archived_confusion = np.load(EVALUATION_DIR / "confusion_matrix.npy").astype(int)

if probabilities.shape != (len(y_true), len(device_classes)):
    raise RuntimeError(
        f"Unexpected probability shape {probabilities.shape}; "
        f"expected {(len(y_true), len(device_classes))}."
    )

if len(y_true) != 376 or len(device_classes) != 10 or len(feature_names) != 244:
    raise RuntimeError(
        f"Unexpected artifacts: test rows={len(y_true)}, "
        f"classes={len(device_classes)}, features={len(feature_names)}."
    )

missing_classes = sorted(set(DIFFICULT_CLASSES) - set(device_classes))
if missing_classes:
    raise RuntimeError(f"Missing difficult classes: {missing_classes}")

predictions = np.argmax(probabilities, axis=1)
calculated_confusion = pd.crosstab(
    pd.Categorical(device_classes[y_true], categories=device_classes),
    pd.Categorical(device_classes[predictions], categories=device_classes),
    dropna=False,
).to_numpy()

if not np.array_equal(calculated_confusion, archived_confusion):
    raise RuntimeError(
        "Predictions reconstructed from the probability matrix do not reproduce "
        "the archived confusion matrix."
    )

probability_order = np.argsort(probabilities, axis=1)
runner_up_indices = probability_order[:, -2]

predicted_probability = probabilities[np.arange(len(y_true)), predictions]
runner_up_probability = probabilities[np.arange(len(y_true)), runner_up_indices]
true_probability = probabilities[np.arange(len(y_true)), y_true]
prediction_margin = predicted_probability - runner_up_probability
true_class_rank = 1 + np.sum(
    probabilities > true_probability[:, np.newaxis],
    axis=1,
)

normalised_probabilities = np.clip(probabilities, 1e-12, None)
normalised_probabilities = (
    normalised_probabilities
    / normalised_probabilities.sum(axis=1, keepdims=True)
)
normalised_entropy = (
    -np.sum(
        normalised_probabilities * np.log(normalised_probabilities),
        axis=1,
    )
    / np.log(len(device_classes))
)

audit = pd.DataFrame(
    {
        "test_index": np.arange(len(y_true)),
        "true_class_index": y_true,
        "true_class": device_classes[y_true],
        "predicted_class_index": predictions,
        "predicted_class": device_classes[predictions],
        "runner_up_class": device_classes[runner_up_indices],
        "correct": predictions == y_true,
        "predicted_probability": predicted_probability,
        "runner_up_probability": runner_up_probability,
        "true_class_probability": true_probability,
        "prediction_margin": prediction_margin,
        "normalised_entropy": normalised_entropy,
        "true_class_rank": true_class_rank,
    }
)

audit["true_class_is_difficult"] = audit["true_class"].isin(DIFFICULT_CLASSES)
audit["predicted_class_is_difficult"] = audit["predicted_class"].isin(DIFFICULT_CLASSES)
audit["involves_difficult_class"] = (
    audit["true_class_is_difficult"]
    | audit["predicted_class_is_difficult"]
)

audit["error_type"] = np.where(
    audit["correct"],
    "correct",
    audit["true_class"] + " -> " + audit["predicted_class"],
)

difficult_audit = audit.loc[audit["involves_difficult_class"]].copy()
true_difficult_audit = audit.loc[audit["true_class_is_difficult"]].copy()
high_confidence_errors = true_difficult_audit.loc[
    ~true_difficult_audit["correct"]
].sort_values(
    ["predicted_probability", "prediction_margin"],
    ascending=False,
)

summary = (
    true_difficult_audit.groupby(["true_class", "correct"], as_index=False)
    .agg(
        observations=("test_index", "count"),
        mean_predicted_probability=("predicted_probability", "mean"),
        median_predicted_probability=("predicted_probability", "median"),
        mean_true_class_probability=("true_class_probability", "mean"),
        mean_prediction_margin=("prediction_margin", "mean"),
        median_prediction_margin=("prediction_margin", "median"),
        mean_normalised_entropy=("normalised_entropy", "mean"),
        median_true_class_rank=("true_class_rank", "median"),
    )
)

audit.to_csv(OUTPUT_DIR / "all_held_out_case_audit.csv", index=False)
difficult_audit.to_csv(
    OUTPUT_DIR / "cases_involving_difficult_classes.csv",
    index=False,
)
true_difficult_audit.to_csv(
    OUTPUT_DIR / "true_difficult_class_cases.csv",
    index=False,
)
high_confidence_errors.to_csv(
    OUTPUT_DIR / "high_confidence_difficult_class_errors.csv",
    index=False,
)
summary.to_csv(
    OUTPUT_DIR / "difficult_class_confidence_summary.csv",
    index=False,
)

prediction_categories = DIFFICULT_CLASSES + ["other"]
reduced_confusion = pd.DataFrame(
    0,
    index=DIFFICULT_CLASSES,
    columns=prediction_categories,
    dtype=int,
)

for true_class in DIFFICULT_CLASSES:
    subset = audit.loc[audit["true_class"] == true_class]
    for predicted_class, count in subset["predicted_class"].value_counts().items():
        output_class = (
            predicted_class
            if predicted_class in DIFFICULT_CLASSES
            else "other"
        )
        reduced_confusion.loc[true_class, output_class] += int(count)

reduced_percent = reduced_confusion.div(
    reduced_confusion.sum(axis=1),
    axis=0,
) * 100.0

reduced_confusion.to_csv(
    OUTPUT_DIR / "difficult_class_reduced_confusion_counts.csv"
)
reduced_percent.to_csv(
    OUTPUT_DIR / "difficult_class_reduced_confusion_row_percent.csv"
)

annotation = reduced_confusion.astype(str) + "\n(" + reduced_percent.round(1).astype(str) + "%)"

figure, axis = plt.subplots(figsize=(8.5, 5.5))
sns.heatmap(
    reduced_percent,
    annot=annotation,
    fmt="",
    cmap="Blues",
    vmin=0,
    vmax=100,
    linewidths=0.5,
    cbar_kws={"label": "Percentage of true-class observations"},
    ax=axis,
)
axis.set_title("Selected CNN-BiLSTM Difficult-Class Confusion")
axis.set_xlabel("Predicted class")
axis.set_ylabel("True class")
figure.tight_layout()
figure.savefig(
    FIGURES_DIR / "01_difficult_class_reduced_confusion.png",
    dpi=300,
    bbox_inches="tight",
)
figure.savefig(
    FIGURES_DIR / "01_difficult_class_reduced_confusion.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(figure)

figure, axes = plt.subplots(1, 2, figsize=(13, 5.5))

sns.boxplot(
    data=true_difficult_audit,
    x="true_class",
    y="prediction_margin",
    hue="correct",
    order=DIFFICULT_CLASSES,
    ax=axes[0],
)
axes[0].set_title("Prediction Margin")
axes[0].set_xlabel("True class")
axes[0].set_ylabel("Top probability minus second probability")
axes[0].legend(title="Correct")

sns.boxplot(
    data=true_difficult_audit,
    x="true_class",
    y="normalised_entropy",
    hue="correct",
    order=DIFFICULT_CLASSES,
    ax=axes[1],
)
axes[1].set_title("Prediction Entropy")
axes[1].set_xlabel("True class")
axes[1].set_ylabel("Normalised entropy")
axes[1].legend(title="Correct")

figure.suptitle(
    "Selected CNN-BiLSTM Confidence for Difficult Classes",
    fontweight="bold",
)
figure.tight_layout()
figure.savefig(
    FIGURES_DIR / "02_difficult_class_confidence.png",
    dpi=300,
    bbox_inches="tight",
)
figure.savefig(
    FIGURES_DIR / "02_difficult_class_confidence.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(figure)

provenance = {
    "analysis": "Selected CNN-BiLSTM difficult-class case audit",
    "selected_run": str(RUN_DIR),
    "probability_source": str(
        EVALUATION_DIR / "prediction_probabilities.npy"
    ),
    "test_rows": int(len(y_true)),
    "retained_features": int(len(feature_names)),
    "difficult_classes": DIFFICULT_CLASSES,
    "selection_role": "post-hoc diagnostic only",
}

(OUTPUT_DIR / "provenance.json").write_text(
    json.dumps(provenance, indent=2),
    encoding="utf-8",
)

print("\nSelected run:")
print(RUN_DIR)

print("\nDifficult-class confidence summary:")
display(summary)

print("\nHighest-confidence difficult-class errors:")
display(high_confidence_errors.head(20))

print(f"\nSaved outputs to:\n{OUTPUT_DIR}")

### Five-Seed Difficult-Class Stability

A single selected model may conceal instability across independent training seeds. This analysis reconstructs predictions for all five CNN–BiLSTM seeds and calculates per-seed precision, recall and F1 for the difficult classes.

The analysis also identifies observations that are misclassified by several or all seeds. Repeated errors across seeds suggest persistent feature overlap or insufficient discriminatory information. Errors that change substantially between seeds suggest optimisation instability or a poorly determined decision boundary.

In [ ]:
from pathlib import Path
import json
import re

from google.colab import drive
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
RUNS_ROOT = PROJECT_ROOT / "Models" / "Tuned" / "CNN_BiLSTM"
DIFFICULT_CLASSES = ["lights", "socket", "water_sensor"]
EXPECTED_SEEDS = [42, 7, 123, 2024, 99]


def select_cnn_bilstm_run(required_paths):
    candidates = []

    if not RUNS_ROOT.exists():
        raise FileNotFoundError(f"Run root does not exist: {RUNS_ROOT}")

    for run_dir in RUNS_ROOT.iterdir():
        if not run_dir.is_dir():
            continue
        if not all((run_dir / path).exists() for path in required_paths):
            continue

        metadata = {}
        metadata_path = run_dir / "run_metadata.json"
        if metadata_path.exists():
            try:
                metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            except Exception:
                metadata = {}

        selection = metadata.get("selection", {})
        candidates.append(
            {
                "run_dir": run_dir,
                "modified": run_dir.stat().st_mtime,
                "validation_accuracy": selection.get("champion_validation_accuracy"),
                "champion_seed": selection.get("champion_seed"),
            }
        )

    candidates.sort(key=lambda item: item["modified"], reverse=True)

    if not candidates:
        raise FileNotFoundError("No compatible CNN-BiLSTM run was found.")

    print("=" * 100)
    print("SELECT CNN-BILSTM RUN FOR FIVE-SEED DIFFICULT-CLASS ANALYSIS")
    print("=" * 100)

    for number, candidate in enumerate(candidates, start=1):
        validation = candidate["validation_accuracy"]
        validation_text = "unknown" if validation is None else f"{validation * 100:.3f}%"
        print(
            f"[{number}] validation accuracy={validation_text} | "
            f"champion seed={candidate['champion_seed']}\n"
            f"    {candidate['run_dir']}"
        )

    while True:
        response = input(f"Select 1-{len(candidates)} (Enter = newest): ").strip()
        if response == "":
            return candidates[0]["run_dir"]
        if response.isdigit() and 1 <= int(response) <= len(candidates):
            return candidates[int(response) - 1]["run_dir"]
        print("Invalid selection.")


RUN_DIR = select_cnn_bilstm_run(
    [
        "y_test_1d.npy",
        "device_classes.npy",
        "Test_Evaluation",
    ]
)

TEST_EVALUATION_DIR = RUN_DIR / "Test_Evaluation"
OUTPUT_DIR = (
    RUN_DIR
    / "Post_Hoc_Difficult_Class_Error_Analysis"
    / "02_Five_Seed_Stability"
)
FIGURES_DIR = OUTPUT_DIR / "Figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

y_true = np.load(RUN_DIR / "y_test_1d.npy").astype(int)
device_classes = np.load(
    RUN_DIR / "device_classes.npy",
    allow_pickle=True,
).astype(str)

if len(y_true) != 376 or len(device_classes) != 10:
    raise RuntimeError(
        f"Unexpected test artifacts: rows={len(y_true)}, "
        f"classes={len(device_classes)}."
    )

seed_directories = {}
pattern = re.compile(r"cnn_bilstm_seed_(\d+)$")

for directory in TEST_EVALUATION_DIR.iterdir():
    if not directory.is_dir():
        continue
    match = pattern.fullmatch(directory.name)
    if match is None:
        continue
    probability_path = directory / "prediction_probabilities.npy"
    if probability_path.exists():
        seed_directories[int(match.group(1))] = directory

missing_seeds = sorted(set(EXPECTED_SEEDS) - set(seed_directories))
if missing_seeds:
    raise FileNotFoundError(
        f"Missing held-out prediction probabilities for seeds: {missing_seeds}"
    )

seed_order = EXPECTED_SEEDS
class_indices = {
    class_name: int(np.where(device_classes == class_name)[0][0])
    for class_name in DIFFICULT_CLASSES
}

metric_rows = []
predictions_by_seed = {}
aggregate_confusion = np.zeros(
    (len(device_classes), len(device_classes)),
    dtype=int,
)

for seed in seed_order:
    probability_path = (
        seed_directories[seed]
        / "prediction_probabilities.npy"
    )
    probabilities = np.load(probability_path).astype(float)

    if probabilities.shape != (len(y_true), len(device_classes)):
        raise RuntimeError(
            f"Unexpected probability shape for seed {seed}: "
            f"{probabilities.shape}"
        )

    predictions = np.argmax(probabilities, axis=1)
    predictions_by_seed[seed] = predictions

    seed_confusion = confusion_matrix(
        y_true,
        predictions,
        labels=np.arange(len(device_classes)),
    )
    aggregate_confusion += seed_confusion

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        predictions,
        labels=np.arange(len(device_classes)),
        zero_division=0,
    )

    for class_index, class_name in enumerate(device_classes):
        metric_rows.append(
            {
                "seed": seed,
                "class": class_name,
                "precision": precision[class_index],
                "recall": recall[class_index],
                "f1_score": f1[class_index],
                "support": int(support[class_index]),
                "probability_path": str(probability_path),
            }
        )

seed_metrics = pd.DataFrame(metric_rows)
difficult_seed_metrics = seed_metrics.loc[
    seed_metrics["class"].isin(DIFFICULT_CLASSES)
].copy()

stability_summary = (
    difficult_seed_metrics.groupby("class", as_index=False)
    .agg(
        mean_precision=("precision", "mean"),
        sd_precision=("precision", "std"),
        mean_recall=("recall", "mean"),
        sd_recall=("recall", "std"),
        mean_f1=("f1_score", "mean"),
        sd_f1=("f1_score", "std"),
        minimum_f1=("f1_score", "min"),
        maximum_f1=("f1_score", "max"),
    )
)

prediction_matrix = np.vstack(
    [predictions_by_seed[seed] for seed in seed_order]
)

persistent_rows = []

for test_index, true_index in enumerate(y_true):
    true_class = device_classes[true_index]
    if true_class not in DIFFICULT_CLASSES:
        continue

    seed_predictions = prediction_matrix[:, test_index]
    row = {
        "test_index": test_index,
        "true_class": true_class,
        "correct_seed_count": int(np.sum(seed_predictions == true_index)),
        "error_seed_count": int(np.sum(seed_predictions != true_index)),
    }

    for seed_position, seed in enumerate(seed_order):
        row[f"prediction_seed_{seed}"] = device_classes[
            seed_predictions[seed_position]
        ]

    row["unique_predictions"] = " | ".join(
        sorted(set(device_classes[seed_predictions]))
    )
    persistent_rows.append(row)

persistent_errors = pd.DataFrame(persistent_rows).sort_values(
    ["error_seed_count", "true_class", "test_index"],
    ascending=[False, True, True],
)

seed_metrics.to_csv(
    OUTPUT_DIR / "all_seed_per_class_metrics.csv",
    index=False,
)
difficult_seed_metrics.to_csv(
    OUTPUT_DIR / "difficult_class_seed_metrics.csv",
    index=False,
)
stability_summary.to_csv(
    OUTPUT_DIR / "difficult_class_stability_summary.csv",
    index=False,
)
persistent_errors.to_csv(
    OUTPUT_DIR / "difficult_class_repeated_case_results.csv",
    index=False,
)
np.save(
    OUTPUT_DIR / "aggregate_five_seed_confusion_matrix.npy",
    aggregate_confusion,
)

figure, axis = plt.subplots(figsize=(9.5, 5.8))

seed_positions = np.arange(len(seed_order))
seed_labels = [str(seed) for seed in seed_order]

class_colours = {
    "lights": "#4E79A7",
    "socket": "#E15759",
    "water_sensor": "#59A14F",
}

class_markers = {
    "lights": "o",
    "socket": "s",
    "water_sensor": "^",
}

class_offsets = {
    "lights": -0.18,
    "socket": 0.00,
    "water_sensor": 0.18,
}

summary_by_class = stability_summary.set_index("class")

for class_name in DIFFICULT_CLASSES:
    class_rows = (
        difficult_seed_metrics.loc[
            difficult_seed_metrics["class"] == class_name,
            ["seed", "f1_score"],
        ]
        .set_index("seed")
        .reindex(seed_order)
    )

    if class_rows["f1_score"].isna().any():
        missing = [
            seed
            for seed, value in class_rows["f1_score"].items()
            if pd.isna(value)
        ]
        raise RuntimeError(
            f"Missing {class_name} F1 values for seeds: {missing}"
        )

    mean_f1 = float(summary_by_class.loc[class_name, "mean_f1"])
    sd_f1 = float(summary_by_class.loc[class_name, "sd_f1"])

    axis.scatter(
        seed_positions + class_offsets[class_name],
        class_rows["f1_score"].to_numpy(dtype=float),
        s=80,
        marker=class_markers[class_name],
        color=class_colours[class_name],
        edgecolor="black",
        linewidth=0.7,
        label=(
            f"{class_name} "
            f"(mean={mean_f1:.3f}, SD={sd_f1:.3f})"
        ),
        zorder=3,
    )

axis.set_title(
    "CNN-BiLSTM Difficult-Class F1 Across Five Training Seeds",
    fontweight="bold",
)
axis.set_xlabel("Training seed")
axis.set_ylabel("Held-out F1")
axis.set_xticks(seed_positions)
axis.set_xticklabels(seed_labels)
axis.set_xlim(-0.55, len(seed_order) - 0.45)
axis.set_ylim(0, 1)
axis.grid(axis="y", linestyle="--", alpha=0.35)
axis.legend(
    title="Class and five-seed summary",
    loc="best",
    frameon=True,
)

figure.tight_layout()

figure.savefig(
    FIGURES_DIR / "01_difficult_class_f1_across_seeds.png",
    dpi=300,
    bbox_inches="tight",
)
figure.savefig(
    FIGURES_DIR / "01_difficult_class_f1_across_seeds.pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

prediction_categories = DIFFICULT_CLASSES + ["other"]
reduced_counts = pd.DataFrame(
    0,
    index=DIFFICULT_CLASSES,
    columns=prediction_categories,
    dtype=int,
)

difficult_indices = {
    name: int(np.where(device_classes == name)[0][0])
    for name in DIFFICULT_CLASSES
}

for true_class in DIFFICULT_CLASSES:
    true_index = difficult_indices[true_class]

    for predicted_class in DIFFICULT_CLASSES:
        predicted_index = difficult_indices[predicted_class]
        reduced_counts.loc[true_class, predicted_class] = int(
            aggregate_confusion[true_index, predicted_index]
        )

    other_indices = [
        index
        for index, class_name in enumerate(device_classes)
        if class_name not in DIFFICULT_CLASSES
    ]
    reduced_counts.loc[true_class, "other"] = int(
        aggregate_confusion[true_index, other_indices].sum()
    )

reduced_percent = reduced_counts.div(
    reduced_counts.sum(axis=1),
    axis=0,
) * 100.0

reduced_counts.to_csv(
    OUTPUT_DIR / "aggregate_difficult_class_confusion_counts.csv"
)
reduced_percent.to_csv(
    OUTPUT_DIR / "aggregate_difficult_class_confusion_row_percent.csv"
)

annotation = reduced_counts.astype(str) + "\n(" + reduced_percent.round(1).astype(str) + "%)"

figure, axis = plt.subplots(figsize=(8.5, 5.5))
sns.heatmap(
    reduced_percent,
    annot=annotation,
    fmt="",
    cmap="Blues",
    vmin=0,
    vmax=100,
    linewidths=0.5,
    cbar_kws={"label": "Percentage across five seed predictions"},
    ax=axis,
)
axis.set_title("Aggregate Five-Seed Difficult-Class Confusion")
axis.set_xlabel("Predicted class")
axis.set_ylabel("True class")
figure.tight_layout()
figure.savefig(
    FIGURES_DIR / "02_aggregate_five_seed_difficult_confusion.png",
    dpi=300,
    bbox_inches="tight",
)
figure.savefig(
    FIGURES_DIR / "02_aggregate_five_seed_difficult_confusion.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(figure)

provenance = {
    "analysis": "CNN-BiLSTM five-seed difficult-class stability",
    "selected_run": str(RUN_DIR),
    "seeds": seed_order,
    "test_rows": int(len(y_true)),
    "difficult_classes": DIFFICULT_CLASSES,
    "selection_role": "post-hoc diagnostic only",
}

(OUTPUT_DIR / "provenance.json").write_text(
    json.dumps(provenance, indent=2),
    encoding="utf-8",
)

print("\nFive-seed difficult-class stability:")
display(stability_summary)

print("\nCases misclassified by all five seeds:")
display(
    persistent_errors.loc[
        persistent_errors["error_seed_count"] == len(seed_order)
    ].head(30)
)

print(f"\nSaved outputs to:\n{OUTPUT_DIR}")

### Difficult-Class Comparison Across the Selected Models

The CNN, LSTM, BiLSTM, Feature-Attention MLP and Attention CNN–BiLSTM learn different neural representations of the same corrected data. The tuned Random Forest is included as the principal classical comparator. Comparing their class-level behaviour helps distinguish error patterns shared across model families from errors associated with a particular learned representation.

Random Forest remains a comparator rather than part of the proposed deep-learning architecture, and no ensemble is constructed. All six models are compared using their archived validation-selected artifacts and the same 376 held-out labels. This is a post-hoc diagnostic comparison and does not alter model selection.

In [ ]:
from pathlib import Path
import json

from google.colab import drive
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
MODELS_ROOT = PROJECT_ROOT / "Models"
DIFFICULT_CLASSES = ["lights", "socket", "water_sensor"]

MODEL_SPECS = {
    "Attention CNN-BiLSTM": {
        "run_root": MODELS_ROOT / "Tuned" / "CNN_BiLSTM",
        "identifier": "best_validation_cnn_bilstm",
        "model_family": "deep learning",
    },
    "Feature-Attention MLP": {
        "run_root": MODELS_ROOT / "Tuned" / "Feature_Attention_MLP",
        "identifier": "best_validation_mlp",
        "model_family": "deep learning",
    },
    "CNN": {
        "run_root": MODELS_ROOT / "Tuned" / "CNN_Tuned",
        "identifier": "best_validation_cnn",
        "model_family": "deep learning",
    },
    "LSTM": {
        "run_root": MODELS_ROOT / "Tuned" / "LSTM_Tuned",
        "identifier": "best_validation_lstm",
        "model_family": "deep learning",
    },
    "BiLSTM": {
        "run_root": MODELS_ROOT / "Tuned" / "BiLSTM_Tuned",
        "identifier": "best_validation_bilstm",
        "model_family": "deep learning",
    },
    "Tuned Random Forest": {
        "run_root": MODELS_ROOT / "Baselines" / "Tuned_RF",
        "identifier": "tuned_random_forest_pipeline",
        "model_family": "classical machine learning",
    },
}


def select_model_run(display_name, specification):
    run_root = specification["run_root"]
    identifier = specification["identifier"]
    candidates = []

    if not run_root.exists():
        raise FileNotFoundError(f"Model run root does not exist: {run_root}")

    for run_dir in run_root.iterdir():
        if not run_dir.is_dir():
            continue

        evaluation_dir = run_dir / "Test_Evaluation" / identifier
        required = [
            run_dir / "device_classes.npy",
            run_dir / "y_test_1d.npy",
            evaluation_dir / "per_class_metrics.csv",
            evaluation_dir / "metrics.csv",
            evaluation_dir / "confusion_matrix.npy",
            evaluation_dir / "prediction_probabilities.npy",
        ]

        if not all(path.exists() for path in required):
            continue

        metadata = {}
        metadata_path = run_dir / "run_metadata.json"
        if metadata_path.exists():
            try:
                metadata = json.loads(
                    metadata_path.read_text(encoding="utf-8")
                )
            except Exception:
                metadata = {}

        if display_name == "Tuned Random Forest":
            selection_value = (
                metadata.get("search", {}).get("best_cv_macro_f1")
            )
            selection_label = "best CV macro-F1"
            seed_text = "random_state=42"
        else:
            selection = metadata.get("selection", {})
            selection_value = selection.get(
                "champion_validation_accuracy"
            )
            selection_label = "validation accuracy"
            seed_text = (
                f"champion seed={selection.get('champion_seed')}"
            )

        candidates.append(
            {
                "run_dir": run_dir,
                "evaluation_dir": evaluation_dir,
                "modified": run_dir.stat().st_mtime,
                "selection_value": selection_value,
                "selection_label": selection_label,
                "seed_text": seed_text,
            }
        )

    candidates.sort(
        key=lambda item: item["modified"],
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError(
            f"No compatible run was found for {display_name}."
        )

    print("\n" + "=" * 100)
    print(f"SELECT RUN: {display_name}")
    print("=" * 100)

    for number, candidate in enumerate(candidates, start=1):
        value = candidate["selection_value"]
        value_text = (
            "unknown"
            if value is None
            else f"{value * 100:.3f}%"
        )
        print(
            f"[{number}] "
            f"{candidate['selection_label']}={value_text} | "
            f"{candidate['seed_text']}\n"
            f"    {candidate['run_dir']}"
        )

    while True:
        response = input(
            f"Select 1-{len(candidates)} (Enter = newest): "
        ).strip()

        if response == "":
            return candidates[0]

        if response.isdigit() and 1 <= int(response) <= len(candidates):
            return candidates[int(response) - 1]

        print("Invalid selection.")


selected_runs = {
    display_name: select_model_run(
        display_name,
        specification,
    )
    for display_name, specification in MODEL_SPECS.items()
}

PRIMARY_RUN = selected_runs[
    "Attention CNN-BiLSTM"
]["run_dir"]

OUTPUT_DIR = (
    PRIMARY_RUN
    / "Post_Hoc_Difficult_Class_Error_Analysis"
    / "03_All_Model_Comparison"
)
FIGURES_DIR = OUTPUT_DIR / "Figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

metric_frames = []
overall_rows = []
confusion_rows = []

reference_classes = None
reference_y_true = None

for display_name, selected in selected_runs.items():
    run_dir = selected["run_dir"]
    evaluation_dir = selected["evaluation_dir"]

    device_classes = np.load(
        run_dir / "device_classes.npy",
        allow_pickle=True,
    ).astype(str)

    y_true = np.load(
        run_dir / "y_test_1d.npy"
    ).astype(int)

    if reference_classes is None:
        reference_classes = device_classes
        reference_y_true = y_true
    else:
        if not np.array_equal(
            reference_classes,
            device_classes,
        ):
            raise RuntimeError(
                f"Class order for {display_name} does not match "
                "the other models."
            )

        if not np.array_equal(
            reference_y_true,
            y_true,
        ):
            raise RuntimeError(
                f"Held-out labels for {display_name} do not match "
                "the other models."
            )

    if len(y_true) != 376:
        raise RuntimeError(
            f"Unexpected held-out row count for {display_name}: "
            f"{len(y_true)}"
        )

    metrics = pd.read_csv(
        evaluation_dir / "per_class_metrics.csv"
    )

    required_columns = {
        "class",
        "precision",
        "recall",
        "f1_score",
        "support",
    }

    if not required_columns.issubset(metrics.columns):
        raise RuntimeError(
            f"Unexpected per-class metric columns for "
            f"{display_name}: {metrics.columns.tolist()}"
        )

    metrics.insert(0, "model", display_name)
    metrics.insert(
        1,
        "model_family",
        MODEL_SPECS[display_name]["model_family"],
    )
    metrics["run_dir"] = str(run_dir)
    metric_frames.append(metrics)

    overall = pd.read_csv(
        evaluation_dir / "metrics.csv"
    )

    if len(overall) != 1:
        raise RuntimeError(
            f"Expected one overall metrics row for "
            f"{display_name}; found {len(overall)}."
        )

    required_overall = {
        "accuracy",
        "balanced_accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
    }

    if not required_overall.issubset(overall.columns):
        raise RuntimeError(
            f"Unexpected overall metric columns for "
            f"{display_name}: {overall.columns.tolist()}"
        )

    overall_row = overall.iloc[0].to_dict()
    overall_row.update(
        {
            "model": display_name,
            "model_family": MODEL_SPECS[
                display_name
            ]["model_family"],
            "run_dir": str(run_dir),
        }
    )
    overall_rows.append(overall_row)

    confusion = np.load(
        evaluation_dir / "confusion_matrix.npy"
    ).astype(int)

    probabilities = np.load(
        evaluation_dir / "prediction_probabilities.npy"
    ).astype(float)

    expected_confusion_shape = (
        len(device_classes),
        len(device_classes),
    )

    if confusion.shape != expected_confusion_shape:
        raise RuntimeError(
            f"Unexpected confusion-matrix shape for "
            f"{display_name}: {confusion.shape}"
        )

    expected_probability_shape = (
        len(y_true),
        len(device_classes),
    )

    if probabilities.shape != expected_probability_shape:
        raise RuntimeError(
            f"Unexpected probability shape for "
            f"{display_name}: {probabilities.shape}"
        )

    predictions = np.argmax(
        probabilities,
        axis=1,
    )

    calculated_confusion = np.zeros_like(confusion)
    np.add.at(
        calculated_confusion,
        (y_true, predictions),
        1,
    )

    if not np.array_equal(
        calculated_confusion,
        confusion,
    ):
        raise RuntimeError(
            f"Archived probabilities do not reproduce the "
            f"confusion matrix for {display_name}."
        )

    for true_class in DIFFICULT_CLASSES:
        true_index = int(
            np.where(
                device_classes == true_class
            )[0][0]
        )

        row = {
            "model": display_name,
            "model_family": MODEL_SPECS[
                display_name
            ]["model_family"],
            "true_class": true_class,
            "support": int(
                confusion[true_index].sum()
            ),
        }

        for predicted_class in DIFFICULT_CLASSES:
            predicted_index = int(
                np.where(
                    device_classes == predicted_class
                )[0][0]
            )

            row[f"predicted_{predicted_class}"] = int(
                confusion[
                    true_index,
                    predicted_index,
                ]
            )

        other_indices = [
            index
            for index, class_name in enumerate(device_classes)
            if class_name not in DIFFICULT_CLASSES
        ]

        row["predicted_other"] = int(
            confusion[
                true_index,
                other_indices,
            ].sum()
        )

        confusion_rows.append(row)

all_metrics = pd.concat(
    metric_frames,
    ignore_index=True,
)

overall_metrics = pd.DataFrame(
    overall_rows
)

difficult_metrics = all_metrics.loc[
    all_metrics["class"].isin(
        DIFFICULT_CLASSES
    )
].copy()

difficult_confusions = pd.DataFrame(
    confusion_rows
)

all_metrics.to_csv(
    OUTPUT_DIR / "all_model_all_class_metrics.csv",
    index=False,
)

overall_metrics.to_csv(
    OUTPUT_DIR / "all_model_overall_metrics.csv",
    index=False,
)

difficult_metrics.to_csv(
    OUTPUT_DIR / "all_model_difficult_class_metrics.csv",
    index=False,
)

difficult_confusions.to_csv(
    OUTPUT_DIR / "all_model_difficult_class_confusions.csv",
    index=False,
)

model_order = list(MODEL_SPECS)

f1_matrix = (
    difficult_metrics.pivot(
        index="class",
        columns="model",
        values="f1_score",
    )
    .reindex(
        index=DIFFICULT_CLASSES,
        columns=model_order,
    )
)

figure, axis = plt.subplots(
    figsize=(13.5, 4.8)
)

sns.heatmap(
    f1_matrix,
    annot=True,
    fmt=".4f",
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"label": "Held-out F1"},
    ax=axis,
)

axis.set_title(
    "Difficult-Class F1 Across the Selected Models"
)
axis.set_xlabel("Model")
axis.set_ylabel("True class")
axis.tick_params(
    axis="x",
    rotation=25,
)

figure.tight_layout()

figure.savefig(
    FIGURES_DIR
    / "01_all_model_difficult_class_f1.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    FIGURES_DIR
    / "01_all_model_difficult_class_f1.pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

long_confusions = difficult_confusions.melt(
    id_vars=[
        "model",
        "model_family",
        "true_class",
        "support",
    ],
    value_vars=[
        "predicted_lights",
        "predicted_socket",
        "predicted_water_sensor",
        "predicted_other",
    ],
    var_name="prediction_group",
    value_name="observations",
)

long_confusions["prediction_group"] = (
    long_confusions["prediction_group"]
    .str.replace(
        "predicted_",
        "",
        regex=False,
    )
)

figure, axes = plt.subplots(
    1,
    3,
    figsize=(19, 5.8),
    sharey=False,
)

for axis, true_class in zip(
    axes,
    DIFFICULT_CLASSES,
):
    subset = long_confusions.loc[
        long_confusions["true_class"]
        == true_class
    ]

    sns.barplot(
        data=subset,
        x="model",
        y="observations",
        hue="prediction_group",
        order=model_order,
        hue_order=DIFFICULT_CLASSES + ["other"],
        ax=axis,
    )

    axis.set_title(
        f"True class: {true_class}"
    )
    axis.set_xlabel("")
    axis.set_ylabel(
        "Held-out observations"
    )
    axis.tick_params(
        axis="x",
        rotation=35,
    )
    axis.legend(
        title="Predicted",
        fontsize=7,
        title_fontsize=8,
    )

figure.suptitle(
    "Selected-Model Difficult-Class Prediction Patterns",
    fontweight="bold",
)

figure.tight_layout()

figure.savefig(
    FIGURES_DIR
    / "02_all_model_confusion_patterns.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    FIGURES_DIR
    / "02_all_model_confusion_patterns.pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

provenance = {
    "analysis": (
        "Difficult-class comparison across selected "
        "deep-learning and classical models"
    ),
    "random_forest_included": True,
    "models": {
        model: {
            "run_dir": str(
                selected["run_dir"]
            ),
            "evaluation_dir": str(
                selected["evaluation_dir"]
            ),
            "selection_label": selected[
                "selection_label"
            ],
            "selection_value": selected[
                "selection_value"
            ],
        }
        for model, selected in selected_runs.items()
    },
    "difficult_classes": DIFFICULT_CLASSES,
    "selection_role": "post-hoc diagnostic only",
}

(
    OUTPUT_DIR / "provenance.json"
).write_text(
    json.dumps(
        provenance,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nDifficult-class metrics:")

display(
    difficult_metrics[
        [
            "model",
            "model_family",
            "class",
            "precision",
            "recall",
            "f1_score",
            "support",
        ]
    ].sort_values(
        ["class", "f1_score"],
        ascending=[True, False],
    )
)

print("\nOverall held-out metrics:")

display(
    overall_metrics[
        [
            "model",
            "model_family",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
        ]
    ].sort_values(
        "macro_f1",
        ascending=False,
    )
)

print(
    f"\nSaved outputs to:\n{OUTPUT_DIR}"
)

### Real-Data Coverage and Feature-Space Separability

This analysis reconstructs the exact deduplicated corpus and deterministic split using the archived split indices. The dataset hashes are checked against the selected run before analysis. The saved training-fitted imputer and scaler are reused unchanged.

Four related questions are examined:

1. How many real observations of each class occur in the training, validation and held-out partitions?
2. How many post-SMOTE training observations are real and how many are synthetic?
3. Which retained predictors distinguish each difficult-class pair in the real training data?
4. Do validation observations have same-class or different-class nearest neighbours in the real training partition?

The pairwise statistical tests use real training observations only. Benjamini–Hochberg adjustment controls the false-discovery rate across the 244 predictors. A logistic-regression probe is used only to measure pairwise linear separability; it is not introduced as a comparative thesis model.

The PCA plot is descriptive only. PCA is fitted on the real training partition and then applied unchanged to validation observations. It is not used for training or feature selection.

In [ ]:
from pathlib import Path
import hashlib
import json

from google.colab import drive
import joblib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.neighbors import NearestNeighbors

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
RUNS_ROOT = PROJECT_ROOT / "Models" / "Tuned" / "CNN_BiLSTM"
TRAIN_CSV = PROJECT_ROOT / "iot_device_train.csv"
TEST_CSV = PROJECT_ROOT / "iot_device_test.csv"
LABEL_COLUMN = "device_category"
DIFFICULT_CLASSES = ["lights", "socket", "water_sensor"]
PAIRWISE_COMPARISONS = [
    ("lights", "socket"),
    ("lights", "water_sensor"),
    ("socket", "water_sensor"),
]


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def benjamini_hochberg(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(p_values.shape, np.nan, dtype=float)
    valid = np.isfinite(p_values)

    if not np.any(valid):
        return adjusted

    valid_values = p_values[valid]
    order = np.argsort(valid_values)
    ranked = valid_values[order]
    count = len(ranked)

    corrected = ranked * count / np.arange(1, count + 1)
    corrected = np.minimum.accumulate(corrected[::-1])[::-1]
    corrected = np.clip(corrected, 0, 1)

    valid_adjusted = np.empty(count, dtype=float)
    valid_adjusted[order] = corrected
    adjusted[valid] = valid_adjusted
    return adjusted


def select_cnn_bilstm_run(required_paths):
    candidates = []

    if not RUNS_ROOT.exists():
        raise FileNotFoundError(f"Run root does not exist: {RUNS_ROOT}")

    for run_dir in RUNS_ROOT.iterdir():
        if not run_dir.is_dir():
            continue
        if not all((run_dir / path).exists() for path in required_paths):
            continue

        metadata = {}
        metadata_path = run_dir / "run_metadata.json"
        if metadata_path.exists():
            try:
                metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            except Exception:
                metadata = {}

        selection = metadata.get("selection", {})
        candidates.append(
            {
                "run_dir": run_dir,
                "modified": run_dir.stat().st_mtime,
                "validation_accuracy": selection.get("champion_validation_accuracy"),
                "champion_seed": selection.get("champion_seed"),
            }
        )

    candidates.sort(key=lambda item: item["modified"], reverse=True)

    if not candidates:
        raise FileNotFoundError("No compatible CNN-BiLSTM run was found.")

    print("=" * 100)
    print("SELECT CNN-BILSTM RUN FOR DATA-SPACE ANALYSIS")
    print("=" * 100)

    for number, candidate in enumerate(candidates, start=1):
        validation = candidate["validation_accuracy"]
        validation_text = "unknown" if validation is None else f"{validation * 100:.3f}%"
        print(
            f"[{number}] validation accuracy={validation_text} | "
            f"champion seed={candidate['champion_seed']}\n"
            f"    {candidate['run_dir']}"
        )

    while True:
        response = input(f"Select 1-{len(candidates)} (Enter = newest): ").strip()
        if response == "":
            return candidates[0]["run_dir"]
        if response.isdigit() and 1 <= int(response) <= len(candidates):
            return candidates[int(response) - 1]["run_dir"]
        print("Invalid selection.")


RUN_DIR = select_cnn_bilstm_run(
    [
        "dataset_manifest.json",
        "split_indices.npz",
        "feature_names.npy",
        "device_classes.npy",
        "imputer.joblib",
        "scaler.joblib",
        "label_encoder.joblib",
        "cnn_bilstm_dataset.npz",
    ]
)

OUTPUT_DIR = (
    RUN_DIR
    / "Post_Hoc_Difficult_Class_Error_Analysis"
    / "04_Data_Space_Analysis"
)
FIGURES_DIR = OUTPUT_DIR / "Figures"
TABLES_DIR = OUTPUT_DIR / "Tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

if not TRAIN_CSV.exists() or not TEST_CSV.exists():
    raise FileNotFoundError(
        "The publisher CSV files must exist at:\n"
        f"  {TRAIN_CSV}\n"
        f"  {TEST_CSV}"
    )

manifest = json.loads(
    (RUN_DIR / "dataset_manifest.json").read_text(encoding="utf-8")
)

actual_train_hash = sha256_file(TRAIN_CSV)
actual_test_hash = sha256_file(TEST_CSV)
expected_train_hash = manifest["publisher_train"]["sha256"]
expected_test_hash = manifest["publisher_test"]["sha256"]

if actual_train_hash != expected_train_hash:
    raise RuntimeError(
        "The current training CSV does not match the dataset used by the selected run."
    )

if actual_test_hash != expected_test_hash:
    raise RuntimeError(
        "The current testing CSV does not match the dataset used by the selected run."
    )

publisher_train = pd.read_csv(TRAIN_CSV)
publisher_test = pd.read_csv(TEST_CSV)

combined = (
    pd.concat([publisher_train, publisher_test], ignore_index=True)
    .drop_duplicates()
    .reset_index(drop=True)
)

if len(combined) != 1877:
    raise RuntimeError(
        f"Expected 1,877 deduplicated observations; found {len(combined)}."
    )

feature_names = np.load(
    RUN_DIR / "feature_names.npy",
    allow_pickle=True,
).astype(str)
device_classes = np.load(
    RUN_DIR / "device_classes.npy",
    allow_pickle=True,
).astype(str)

if len(feature_names) != 244 or len(device_classes) != 10:
    raise RuntimeError(
        f"Unexpected artifact sizes: features={len(feature_names)}, "
        f"classes={len(device_classes)}."
    )

label_encoder = joblib.load(RUN_DIR / "label_encoder.joblib")
imputer = joblib.load(RUN_DIR / "imputer.joblib")
scaler = joblib.load(RUN_DIR / "scaler.joblib")

if not np.array_equal(
    np.asarray(label_encoder.classes_).astype(str),
    device_classes,
):
    raise RuntimeError("Label-encoder classes do not match device_classes.npy.")

split_indices = np.load(RUN_DIR / "split_indices.npz")
train_indices = split_indices["train"].astype(int)
validation_indices = split_indices["validation"].astype(int)
test_indices = split_indices["test"].astype(int)

if (
    len(train_indices),
    len(validation_indices),
    len(test_indices),
) != (1200, 301, 376):
    raise RuntimeError(
        "Unexpected split sizes: "
        f"{len(train_indices)}, {len(validation_indices)}, {len(test_indices)}"
    )

X_all = (
    combined.drop(columns=LABEL_COLUMN)
    .replace([np.inf, -np.inf], np.nan)
)
y_all_text = combined[LABEL_COLUMN].astype(str).to_numpy()
y_all = label_encoder.transform(y_all_text)

missing_features = sorted(set(feature_names) - set(X_all.columns))
if missing_features:
    raise RuntimeError(
        f"Saved retained predictors are missing from the CSV files: "
        f"{missing_features[:10]}"
    )

X_train_raw = X_all.iloc[train_indices][feature_names]
X_validation_raw = X_all.iloc[validation_indices][feature_names]
X_test_raw = X_all.iloc[test_indices][feature_names]

y_train = y_all[train_indices]
y_validation = y_all[validation_indices]
y_test = y_all[test_indices]

X_train_scaled = scaler.transform(
    imputer.transform(X_train_raw)
).astype(np.float32)
X_validation_scaled = scaler.transform(
    imputer.transform(X_validation_raw)
).astype(np.float32)
X_test_scaled = scaler.transform(
    imputer.transform(X_test_raw)
).astype(np.float32)

dataset = np.load(
    RUN_DIR / "cnn_bilstm_dataset.npz",
    allow_pickle=True,
)
X_train_balanced = dataset["X_train"].reshape(
    len(dataset["X_train"]),
    -1,
).astype(np.float32)
y_train_balanced = np.argmax(dataset["y_train"], axis=1).astype(int)

if X_train_balanced.shape != (1280, 244):
    raise RuntimeError(
        f"Unexpected post-SMOTE training shape: {X_train_balanced.shape}"
    )

partition_rows = []

for partition_name, labels in [
    ("training_real", y_train),
    ("validation_real", y_validation),
    ("held_out_test_real", y_test),
    ("training_after_smote", y_train_balanced),
]:
    counts = pd.Series(
        device_classes[labels],
        name="class",
    ).value_counts()

    for class_name in device_classes:
        partition_rows.append(
            {
                "partition": partition_name,
                "class": class_name,
                "observations": int(counts.get(class_name, 0)),
            }
        )

class_coverage = pd.DataFrame(partition_rows)
class_coverage.to_csv(
    TABLES_DIR / "class_coverage_by_partition.csv",
    index=False,
)

from collections import defaultdict, deque

smote_rows = []

for class_index, class_name in enumerate(device_classes):
    real_class = np.ascontiguousarray(
        X_train_scaled[y_train == class_index],
        dtype=np.float32,
    )

    balanced_class = np.ascontiguousarray(
        X_train_balanced[y_train_balanced == class_index],
        dtype=np.float32,
    )

    real_count = len(real_class)
    balanced_count = len(balanced_class)
    expected_synthetic_count = balanced_count - real_count

    if expected_synthetic_count < 0:
        raise RuntimeError(
            f"Post-SMOTE class {class_name} contains fewer observations "
            "than the real training class."
        )

    balanced_row_lookup = defaultdict(deque)

    for balanced_index, balanced_row in enumerate(balanced_class):
        balanced_row_lookup[
            balanced_row.tobytes()
        ].append(balanced_index)

    original_mask = np.zeros(
        balanced_count,
        dtype=bool,
    )

    unmatched_original_rows = 0

    for real_row in real_class:
        row_key = real_row.tobytes()
        available_indices = balanced_row_lookup[row_key]

        if not available_indices:
            unmatched_original_rows += 1
            continue

        matched_index = available_indices.popleft()
        original_mask[matched_index] = True

    if unmatched_original_rows != 0:
        raise RuntimeError(
            f"Could not match {unmatched_original_rows} original "
            f"{class_name} rows inside the archived post-SMOTE dataset. "
            "Confirm that the selected run and publisher datasets are identical."
        )

    identified_original_count = int(
        np.sum(original_mask)
    )

    if identified_original_count != real_count:
        raise RuntimeError(
            f"Original-row recovery failed for {class_name}: "
            f"expected {real_count}, recovered "
            f"{identified_original_count}."
        )

    synthetic_mask = ~original_mask
    synthetic_class = balanced_class[synthetic_mask]
    synthetic_count = len(synthetic_class)

    if synthetic_count != expected_synthetic_count:
        raise RuntimeError(
            f"Synthetic-row recovery failed for {class_name}: "
            f"expected {expected_synthetic_count}, recovered "
            f"{synthetic_count}."
        )

    if synthetic_count > 0:
        same_class_neighbours = NearestNeighbors(
            n_neighbors=1,
            metric="euclidean",
        ).fit(real_class)

        same_distances = (
            same_class_neighbours
            .kneighbors(
                synthetic_class,
                return_distance=True,
            )[0][:, 0]
        )

        other_real = X_train_scaled[
            y_train != class_index
        ]

        other_class_neighbours = NearestNeighbors(
            n_neighbors=1,
            metric="euclidean",
        ).fit(other_real)

        other_distances = (
            other_class_neighbours
            .kneighbors(
                synthetic_class,
                return_distance=True,
            )[0][:, 0]
        )

        synthetic_closer_to_other = int(
            np.sum(
                other_distances
                < same_distances
            )
        )

        closer_fraction = (
            synthetic_closer_to_other
            / synthetic_count
        )

        median_same_distance = float(
            np.median(same_distances)
        )

        median_other_distance = float(
            np.median(other_distances)
        )

        minimum_same_distance = float(
            np.min(same_distances)
        )

        maximum_same_distance = float(
            np.max(same_distances)
        )
    else:
        synthetic_closer_to_other = 0
        closer_fraction = 0.0
        median_same_distance = 0.0
        median_other_distance = 0.0
        minimum_same_distance = 0.0
        maximum_same_distance = 0.0

    smote_rows.append(
        {
            "class": class_name,
            "real_training_observations": real_count,
            "post_smote_observations": balanced_count,
            "identified_original_rows": identified_original_count,
            "identified_synthetic_rows": synthetic_count,
            "identification_method": (
                "exact float32 row multiset matching after shuffle"
            ),
            "synthetic_rows_closer_to_other_class_than_same_class": (
                synthetic_closer_to_other
            ),
            "synthetic_closer_to_other_fraction": (
                closer_fraction
            ),
            "minimum_synthetic_distance_to_same_class_real": (
                minimum_same_distance
            ),
            "median_synthetic_distance_to_same_class_real": (
                median_same_distance
            ),
            "maximum_synthetic_distance_to_same_class_real": (
                maximum_same_distance
            ),
            "median_synthetic_distance_to_other_class_real": (
                median_other_distance
            ),
        }
    )

smote_summary = pd.DataFrame(smote_rows)

expected_original_total = len(X_train_scaled)
expected_synthetic_total = (
    len(X_train_balanced)
    - len(X_train_scaled)
)

recovered_original_total = int(
    smote_summary[
        "identified_original_rows"
    ].sum()
)

recovered_synthetic_total = int(
    smote_summary[
        "identified_synthetic_rows"
    ].sum()
)

if recovered_original_total != expected_original_total:
    raise RuntimeError(
        "Incorrect total number of original rows recovered: "
        f"expected {expected_original_total}, "
        f"recovered {recovered_original_total}."
    )

if recovered_synthetic_total != expected_synthetic_total:
    raise RuntimeError(
        "Incorrect total number of synthetic rows recovered: "
        f"expected {expected_synthetic_total}, "
        f"recovered {recovered_synthetic_total}."
    )

smote_summary.to_csv(
    TABLES_DIR
    / "smote_real_and_synthetic_summary.csv",
    index=False,
)

print("\nSMOTE row-identification verification:")
print(
    f"Original rows recovered: "
    f"{recovered_original_total:,}/"
    f"{expected_original_total:,}"
)
print(
    f"Synthetic rows recovered: "
    f"{recovered_synthetic_total:,}/"
    f"{expected_synthetic_total:,}"
)
display(smote_summary)

pairwise_feature_frames = []
pairwise_probe_rows = []

for class_a, class_b in PAIRWISE_COMPARISONS:
    index_a = int(np.where(device_classes == class_a)[0][0])
    index_b = int(np.where(device_classes == class_b)[0][0])

    training_mask = np.isin(y_train, [index_a, index_b])
    validation_mask = np.isin(y_validation, [index_a, index_b])

    X_pair_train = X_train_scaled[training_mask]
    X_pair_validation = X_validation_scaled[validation_mask]
    y_pair_train_original = y_train[training_mask]
    y_pair_validation_original = y_validation[validation_mask]

    y_pair_train = (y_pair_train_original == index_b).astype(int)
    y_pair_validation = (
        y_pair_validation_original == index_b
    ).astype(int)

    mutual_information = mutual_info_classif(
        X_pair_train,
        y_pair_train,
        discrete_features=False,
        random_state=42,
    )

    feature_rows = []

    for feature_index, feature_name in enumerate(feature_names):
        values_a = X_train_scaled[
            y_train == index_a,
            feature_index,
        ]
        values_b = X_train_scaled[
            y_train == index_b,
            feature_index,
        ]

        try:
            test = mannwhitneyu(
                values_a,
                values_b,
                alternative="two-sided",
                method="auto",
            )
            u_statistic = float(test.statistic)
            p_value = float(test.pvalue)
        except Exception:
            u_statistic = np.nan
            p_value = np.nan

        denominator = len(values_a) * len(values_b)
        rank_biserial = (
            2.0 * u_statistic / denominator - 1.0
            if denominator > 0 and np.isfinite(u_statistic)
            else np.nan
        )

        feature_rows.append(
            {
                "comparison": f"{class_a}_vs_{class_b}",
                "class_a": class_a,
                "class_b": class_b,
                "feature_index": feature_index,
                "feature": feature_name,
                "median_class_a": float(np.median(values_a)),
                "median_class_b": float(np.median(values_b)),
                "median_difference_a_minus_b": float(
                    np.median(values_a) - np.median(values_b)
                ),
                "mann_whitney_u": u_statistic,
                "raw_p_value": p_value,
                "rank_biserial_effect": rank_biserial,
                "absolute_rank_biserial_effect": abs(rank_biserial),
                "mutual_information": float(
                    mutual_information[feature_index]
                ),
            }
        )

    feature_frame = pd.DataFrame(feature_rows)
    feature_frame["bh_adjusted_p_value"] = benjamini_hochberg(
        feature_frame["raw_p_value"]
    )
    feature_frame["significant_bh_0_05"] = (
        feature_frame["bh_adjusted_p_value"] < 0.05
    )
    feature_frame = feature_frame.sort_values(
        [
            "absolute_rank_biserial_effect",
            "mutual_information",
        ],
        ascending=False,
    ).reset_index(drop=True)
    feature_frame.insert(
        0,
        "effect_rank",
        np.arange(1, len(feature_frame) + 1),
    )

    pairwise_feature_frames.append(feature_frame)
    feature_frame.to_csv(
        TABLES_DIR / f"pairwise_features_{class_a}_vs_{class_b}.csv",
        index=False,
    )

    linear_probe = LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=42,
    )
    linear_probe.fit(X_pair_train, y_pair_train)

    validation_probability = linear_probe.predict_proba(
        X_pair_validation
    )[:, 1]
    validation_prediction = (
        validation_probability >= 0.5
    ).astype(int)

    pairwise_probe_rows.append(
        {
            "comparison": f"{class_a}_vs_{class_b}",
            "training_observations": int(len(y_pair_train)),
            "validation_observations": int(len(y_pair_validation)),
            "validation_accuracy": accuracy_score(
                y_pair_validation,
                validation_prediction,
            ),
            "validation_balanced_accuracy": balanced_accuracy_score(
                y_pair_validation,
                validation_prediction,
            ),
            "validation_macro_f1": f1_score(
                y_pair_validation,
                validation_prediction,
                average="macro",
            ),
            "validation_roc_auc": roc_auc_score(
                y_pair_validation,
                validation_probability,
            ),
            "role": "diagnostic linear-separability probe only",
        }
    )

pairwise_features = pd.concat(
    pairwise_feature_frames,
    ignore_index=True,
)
pairwise_probe_summary = pd.DataFrame(pairwise_probe_rows)

pairwise_features.to_csv(
    TABLES_DIR / "all_pairwise_feature_results.csv",
    index=False,
)
pairwise_probe_summary.to_csv(
    TABLES_DIR / "pairwise_linear_probe_summary.csv",
    index=False,
)

training_neighbours = NearestNeighbors(
    n_neighbors=5,
    metric="euclidean",
).fit(X_train_scaled)

difficult_validation_mask = np.isin(
    device_classes[y_validation],
    DIFFICULT_CLASSES,
)
validation_query_indices = np.where(difficult_validation_mask)[0]
query_features = X_validation_scaled[difficult_validation_mask]

neighbour_distances, neighbour_positions = (
    training_neighbours.kneighbors(
        query_features,
        return_distance=True,
    )
)

neighbour_rows = []

for query_number, validation_position in enumerate(
    validation_query_indices
):
    true_index = y_validation[validation_position]
    neighbour_labels = y_train[
        neighbour_positions[query_number]
    ]

    row = {
        "validation_position": int(validation_position),
        "combined_index": int(
            validation_indices[validation_position]
        ),
        "true_class": device_classes[true_index],
        "nearest_neighbour_class": device_classes[
            neighbour_labels[0]
        ],
        "nearest_neighbour_correct": bool(
            neighbour_labels[0] == true_index
        ),
        "nearest_neighbour_distance": float(
            neighbour_distances[query_number, 0]
        ),
        "same_class_fraction_among_five": float(
            np.mean(neighbour_labels == true_index)
        ),
    }

    for neighbour_number in range(5):
        row[f"neighbour_{neighbour_number + 1}_class"] = (
            device_classes[
                neighbour_labels[neighbour_number]
            ]
        )
        row[f"neighbour_{neighbour_number + 1}_distance"] = float(
            neighbour_distances[
                query_number,
                neighbour_number,
            ]
        )

    neighbour_rows.append(row)

neighbour_audit = pd.DataFrame(neighbour_rows)
neighbour_summary = (
    neighbour_audit.groupby("true_class", as_index=False)
    .agg(
        validation_observations=("validation_position", "count"),
        nearest_neighbour_accuracy=("nearest_neighbour_correct", "mean"),
        mean_same_class_fraction_among_five=(
            "same_class_fraction_among_five",
            "mean",
        ),
        median_nearest_neighbour_distance=(
            "nearest_neighbour_distance",
            "median",
        ),
    )
)

neighbour_audit.to_csv(
    TABLES_DIR / "difficult_class_validation_neighbours.csv",
    index=False,
)
neighbour_summary.to_csv(
    TABLES_DIR / "difficult_class_neighbour_summary.csv",
    index=False,
)

feature_hashes = pd.util.hash_pandas_object(
    X_all[feature_names],
    index=False,
).astype(str)

feature_label_audit = pd.DataFrame(
    {
        "combined_index": np.arange(len(combined)),
        "feature_hash": feature_hashes,
        "class": y_all_text,
    }
)

conflicting_hashes = (
    feature_label_audit.groupby("feature_hash")["class"]
    .nunique()
)
conflicting_hashes = conflicting_hashes.loc[
    conflicting_hashes > 1
].index

exact_feature_conflicts = feature_label_audit.loc[
    feature_label_audit["feature_hash"].isin(conflicting_hashes)
].sort_values(
    ["feature_hash", "class", "combined_index"]
)

exact_feature_conflicts.to_csv(
    TABLES_DIR / "exact_feature_vectors_with_conflicting_labels.csv",
    index=False,
)

coverage_plot = class_coverage.loc[
    class_coverage["partition"].isin(
        ["training_real", "training_after_smote"]
    )
].copy()

figure, axis = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=coverage_plot,
    x="class",
    y="observations",
    hue="partition",
    ax=axis,
)
axis.set_title("Real Training Coverage and Post-SMOTE Coverage")
axis.set_xlabel("Device class")
axis.set_ylabel("Training observations")
axis.tick_params(axis="x", rotation=40)
axis.legend(title="Partition")
figure.tight_layout()
figure.savefig(
    FIGURES_DIR / "01_real_and_post_smote_class_coverage.png",
    dpi=300,
    bbox_inches="tight",
)
figure.savefig(
    FIGURES_DIR / "01_real_and_post_smote_class_coverage.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(figure)

figure, axes = plt.subplots(
    1,
    len(PAIRWISE_COMPARISONS),
    figsize=(19, 7),
)

for axis, (class_a, class_b), frame in zip(
    axes,
    PAIRWISE_COMPARISONS,
    pairwise_feature_frames,
):
    top_features = (
        frame.head(12)
        .sort_values("rank_biserial_effect")
    )

    colours = [
        "#4E79A7" if value < 0 else "#E15759"
        for value in top_features["rank_biserial_effect"]
    ]

    axis.barh(
        top_features["feature"],
        top_features["rank_biserial_effect"],
        color=colours,
    )
    axis.axvline(0, color="black", linewidth=0.8)
    axis.set_title(f"{class_a} vs {class_b}")
    axis.set_xlabel(
        "Rank-biserial effect\n"
        f"positive = larger for {class_a}"
    )
    axis.set_ylabel("")

figure.suptitle(
    "Strongest Training-Only Pairwise Feature Differences",
    fontweight="bold",
)
figure.tight_layout()
figure.savefig(
    FIGURES_DIR / "02_pairwise_feature_effects.png",
    dpi=300,
    bbox_inches="tight",
)
figure.savefig(
    FIGURES_DIR / "02_pairwise_feature_effects.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(figure)

figure, axis = plt.subplots(figsize=(9, 5.5))
sns.boxplot(
    data=neighbour_audit,
    x="true_class",
    y="same_class_fraction_among_five",
    order=DIFFICULT_CLASSES,
    ax=axis,
)
sns.stripplot(
    data=neighbour_audit,
    x="true_class",
    y="same_class_fraction_among_five",
    order=DIFFICULT_CLASSES,
    color="black",
    alpha=0.45,
    size=4,
    ax=axis,
)
axis.set_title(
    "Same-Class Representation Among Five Nearest Real Training Neighbours"
)
axis.set_xlabel("Validation observation true class")
axis.set_ylabel("Same-class fraction")
axis.set_ylim(-0.05, 1.05)
figure.tight_layout()
figure.savefig(
    FIGURES_DIR / "03_difficult_class_neighbour_overlap.png",
    dpi=300,
    bbox_inches="tight",
)
figure.savefig(
    FIGURES_DIR / "03_difficult_class_neighbour_overlap.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(figure)

pca = PCA(n_components=2, random_state=42)
training_projection = pca.fit_transform(X_train_scaled)
validation_projection = pca.transform(X_validation_scaled)

pca_rows = []

for partition_name, projection, labels in [
    ("Real training", training_projection, y_train),
    ("Real validation", validation_projection, y_validation),
]:
    for class_name in DIFFICULT_CLASSES:
        class_index = int(
            np.where(device_classes == class_name)[0][0]
        )
        mask = labels == class_index

        for pc1, pc2 in projection[mask]:
            pca_rows.append(
                {
                    "partition": partition_name,
                    "class": class_name,
                    "PC1": pc1,
                    "PC2": pc2,
                }
            )

pca_frame = pd.DataFrame(pca_rows)
pca_frame.to_csv(
    TABLES_DIR / "difficult_class_train_validation_pca_coordinates.csv",
    index=False,
)

class_colours = {
    "lights": "#F28E2B",
    "socket": "#4E79A7",
    "water_sensor": "#59A14F",
}

figure, axis = plt.subplots(figsize=(10, 7))

for class_name in DIFFICULT_CLASSES:
    training_subset = pca_frame.loc[
        (pca_frame["class"] == class_name)
        & (pca_frame["partition"] == "Real training")
    ]
    validation_subset = pca_frame.loc[
        (pca_frame["class"] == class_name)
        & (pca_frame["partition"] == "Real validation")
    ]

    axis.scatter(
        training_subset["PC1"],
        training_subset["PC2"],
        s=28,
        alpha=0.35,
        color=class_colours[class_name],
        marker="o",
    )
    axis.scatter(
        validation_subset["PC1"],
        validation_subset["PC2"],
        s=55,
        alpha=0.9,
        color=class_colours[class_name],
        marker="x",
        linewidth=1.5,
    )

class_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        markerfacecolor=class_colours[class_name],
        markeredgecolor=class_colours[class_name],
        label=class_name,
    )
    for class_name in DIFFICULT_CLASSES
]
partition_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        color="black",
        label="Real training",
    ),
    Line2D(
        [0],
        [0],
        marker="x",
        linestyle="",
        color="black",
        label="Real validation",
    ),
]

first_legend = axis.legend(
    handles=class_handles,
    title="Class",
    loc="upper right",
)
axis.add_artist(first_legend)
axis.legend(
    handles=partition_handles,
    title="Partition",
    loc="lower right",
)

axis.set_title(
    "Training-Fitted PCA Projection of the Difficult Classes"
)
axis.set_xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0] * 100:.2f}% variance)"
)
axis.set_ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1] * 100:.2f}% variance)"
)
axis.grid(True, linestyle="--", alpha=0.25)
figure.tight_layout()
figure.savefig(
    FIGURES_DIR / "04_difficult_class_train_validation_pca.png",
    dpi=300,
    bbox_inches="tight",
)
figure.savefig(
    FIGURES_DIR / "04_difficult_class_train_validation_pca.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(figure)

provenance = {
    "analysis": "Real-data coverage, SMOTE, pairwise separability and neighbour overlap",
    "selected_run": str(RUN_DIR),
    "publisher_train": str(TRAIN_CSV),
    "publisher_test": str(TEST_CSV),
    "publisher_train_sha256": actual_train_hash,
    "publisher_test_sha256": actual_test_hash,
    "training_rows": int(len(train_indices)),
    "validation_rows": int(len(validation_indices)),
    "held_out_rows": int(len(test_indices)),
    "retained_features": int(len(feature_names)),
    "pairwise_probe_role": "diagnostic only; not a comparative thesis model",
    "test_data_used_for_pairwise_feature_analysis": False,
    "test_data_used_for_neighbour_probe": False,
}

(OUTPUT_DIR / "provenance.json").write_text(
    json.dumps(provenance, indent=2),
    encoding="utf-8",
)

print("\nClass coverage:")
display(
    class_coverage.loc[
        class_coverage["class"].isin(DIFFICULT_CLASSES)
    ]
)

print("\nSMOTE audit:")
display(
    smote_summary.loc[
        smote_summary["class"].isin(DIFFICULT_CLASSES)
    ]
)

print("\nPairwise validation separability probes:")
display(pairwise_probe_summary)

print("\nNearest-neighbour summary:")
display(neighbour_summary)

print(
    "\nExact feature-vector groups containing conflicting labels:",
    len(conflicting_hashes),
)

print(f"\nSaved outputs to:\n{OUTPUT_DIR}")

### Contrastive SHAP Analysis of Difficult-Class Errors

Global SHAP importance does not directly explain why one class is confused with another. This analysis therefore compares SHAP contributions for specific confusion directions, such as a true socket observation being classified as lights.

For a true class \(a\) and an incorrectly predicted class \(b\), the contrastive attribution for feature \(j\) is:

\[
\Delta\phi_j = \phi_{j,b} - \phi_{j,a}.
\]

A positive contrast indicates that the feature contribution favours class \(b\) over the true class \(a\). The analysis compares this contrast for misclassified \(a\rightarrow b\) observations against correctly classified observations from class \(a\).

The reported difference is:

\[
D_j =
\mathbb{E}[\Delta\phi_j \mid a\rightarrow b]
-
\mathbb{E}[\Delta\phi_j \mid a\rightarrow a].
\]

Positive values indicate features that favour the incorrect class more strongly in the errors than in correctly classified observations. Negative values indicate features that favour the true class more strongly in correct cases.

Kernel SHAP and Partition SHAP are analysed independently. Agreement between the two methods provides stronger evidence that a pattern is not specific to one attribution procedure. These explanations remain associational and do not establish causal relationships.

In [ ]:
from pathlib import Path
import json

from google.colab import drive
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Honours_Project")
RUNS_ROOT = PROJECT_ROOT / "Models" / "Tuned" / "CNN_BiLSTM"
DIFFICULT_CLASSES = ["lights", "socket", "water_sensor"]


def select_cnn_bilstm_run(required_paths):
    candidates = []

    if not RUNS_ROOT.exists():
        raise FileNotFoundError(f"Run root does not exist: {RUNS_ROOT}")

    for run_dir in RUNS_ROOT.iterdir():
        if not run_dir.is_dir():
            continue
        if not all((run_dir / path).exists() for path in required_paths):
            continue

        metadata = {}
        metadata_path = run_dir / "run_metadata.json"
        if metadata_path.exists():
            try:
                metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            except Exception:
                metadata = {}

        selection = metadata.get("selection", {})
        candidates.append(
            {
                "run_dir": run_dir,
                "modified": run_dir.stat().st_mtime,
                "validation_accuracy": selection.get("champion_validation_accuracy"),
                "champion_seed": selection.get("champion_seed"),
            }
        )

    candidates.sort(key=lambda item: item["modified"], reverse=True)

    if not candidates:
        raise FileNotFoundError(
            "No compatible CNN-BiLSTM run containing complete full-test "
            "Kernel and Partition SHAP tensors was found."
        )

    print("=" * 100)
    print("SELECT CNN-BILSTM RUN FOR CONTRASTIVE SHAP ERROR ANALYSIS")
    print("=" * 100)

    for number, candidate in enumerate(candidates, start=1):
        validation = candidate["validation_accuracy"]
        validation_text = "unknown" if validation is None else f"{validation * 100:.3f}%"
        print(
            f"[{number}] validation accuracy={validation_text} | "
            f"champion seed={candidate['champion_seed']}\n"
            f"    {candidate['run_dir']}"
        )

    while True:
        response = input(f"Select 1-{len(candidates)} (Enter = newest): ").strip()
        if response == "":
            return candidates[0]["run_dir"]
        if response.isdigit() and 1 <= int(response) <= len(candidates):
            return candidates[int(response) - 1]["run_dir"]
        print("Invalid selection.")


RUN_DIR = select_cnn_bilstm_run(
    [
        "y_test_1d.npy",
        "device_classes.npy",
        "feature_names.npy",
        "Test_Evaluation/best_validation_cnn_bilstm/prediction_probabilities.npy",
        "SHAP_Final/Full376/Kernel_SHAP_Full376_4096.npy",
        "SHAP_Final/Full376/Partition_SHAP_Full376_4096.npy",
        "SHAP_Final/Common/Feature_Family_Mapping.csv",
    ]
)

SHAP_ROOT = RUN_DIR / "SHAP_Final"
FULL376_DIR = SHAP_ROOT / "Full376"
COMMON_DIR = SHAP_ROOT / "Common"
EVALUATION_DIR = RUN_DIR / "Test_Evaluation" / "best_validation_cnn_bilstm"

OUTPUT_DIR = (
    RUN_DIR
    / "Post_Hoc_Difficult_Class_Error_Analysis"
    / "05_Contrastive_SHAP_Error_Analysis"
)
FIGURES_DIR = OUTPUT_DIR / "Figures"
TABLES_DIR = OUTPUT_DIR / "Tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

y_true = np.load(RUN_DIR / "y_test_1d.npy").astype(int)
device_classes = np.load(
    RUN_DIR / "device_classes.npy",
    allow_pickle=True,
).astype(str)
feature_names = np.load(
    RUN_DIR / "feature_names.npy",
    allow_pickle=True,
).astype(str)
probabilities = np.load(
    EVALUATION_DIR / "prediction_probabilities.npy"
).astype(float)
predictions = np.argmax(probabilities, axis=1)

family_mapping = pd.read_csv(
    COMMON_DIR / "Feature_Family_Mapping.csv"
).sort_values("feature_index")

if not np.array_equal(
    family_mapping["feature"].astype(str).to_numpy(),
    feature_names,
):
    raise RuntimeError(
        "Feature-family mapping does not match feature_names.npy."
    )


def normalise_shap_tensor(values, method_name):
    values = np.asarray(values)

    expected_shape = (
        len(y_true),
        len(feature_names),
        len(device_classes),
    )

    if values.shape == expected_shape:
        return values.astype(np.float64)

    if values.shape == (
        len(device_classes),
        len(y_true),
        len(feature_names),
    ):
        return np.transpose(values, (1, 2, 0)).astype(np.float64)

    if values.shape == (
        len(y_true),
        len(device_classes),
        len(feature_names),
    ):
        return np.transpose(values, (0, 2, 1)).astype(np.float64)

    raise RuntimeError(
        f"Unexpected {method_name} SHAP tensor shape: {values.shape}. "
        f"Expected {expected_shape}."
    )


kernel_shap = normalise_shap_tensor(
    np.load(FULL376_DIR / "Kernel_SHAP_Full376_4096.npy"),
    "Kernel",
)
partition_shap = normalise_shap_tensor(
    np.load(FULL376_DIR / "Partition_SHAP_Full376_4096.npy"),
    "Partition",
)

method_tensors = {
    "Kernel SHAP": kernel_shap,
    "Partition SHAP": partition_shap,
}

class_indices = {
    class_name: int(np.where(device_classes == class_name)[0][0])
    for class_name in DIFFICULT_CLASSES
}

feature_result_frames = []
family_result_frames = []
comparison_records = []

for method_name, shap_tensor in method_tensors.items():
    method_key = method_name.lower().replace(" ", "_")

    for actual_class in DIFFICULT_CLASSES:
        actual_index = class_indices[actual_class]
        correct_mask = (
            (y_true == actual_index)
            & (predictions == actual_index)
        )

        if np.sum(correct_mask) == 0:
            continue

        for predicted_class in DIFFICULT_CLASSES:
            if predicted_class == actual_class:
                continue

            predicted_index = class_indices[predicted_class]
            error_mask = (
                (y_true == actual_index)
                & (predictions == predicted_index)
            )

            error_count = int(np.sum(error_mask))
            correct_count = int(np.sum(correct_mask))

            if error_count == 0:
                continue

            error_contrast = (
                shap_tensor[
                    error_mask,
                    :,
                    predicted_index,
                ]
                - shap_tensor[
                    error_mask,
                    :,
                    actual_index,
                ]
            )

            correct_contrast = (
                shap_tensor[
                    correct_mask,
                    :,
                    predicted_index,
                ]
                - shap_tensor[
                    correct_mask,
                    :,
                    actual_index,
                ]
            )

            mean_error_contrast = np.mean(
                error_contrast,
                axis=0,
            )
            mean_correct_contrast = np.mean(
                correct_contrast,
                axis=0,
            )
            contrast_difference = (
                mean_error_contrast
                - mean_correct_contrast
            )

            comparison_name = (
                f"{actual_class}_to_{predicted_class}"
            )

            feature_frame = pd.DataFrame(
                {
                    "method": method_name,
                    "comparison": comparison_name,
                    "actual_class": actual_class,
                    "predicted_class": predicted_class,
                    "error_observations": error_count,
                    "correct_actual_class_observations": correct_count,
                    "feature_index": np.arange(len(feature_names)),
                    "feature": feature_names,
                    "feature_family": family_mapping[
                        "feature_family"
                    ].to_numpy(),
                    "mean_error_predicted_minus_true_shap": mean_error_contrast,
                    "mean_correct_predicted_minus_true_shap": mean_correct_contrast,
                    "error_minus_correct_contrast": contrast_difference,
                    "absolute_error_minus_correct_contrast": np.abs(
                        contrast_difference
                    ),
                }
            ).sort_values(
                "absolute_error_minus_correct_contrast",
                ascending=False,
            ).reset_index(drop=True)

            feature_frame.insert(
                0,
                "absolute_contrast_rank",
                np.arange(1, len(feature_frame) + 1),
            )

            feature_result_frames.append(feature_frame)

            feature_frame.to_csv(
                TABLES_DIR
                / f"{method_key}_{comparison_name}_feature_contrasts.csv",
                index=False,
            )

            family_frame = (
                feature_frame.groupby(
                    "feature_family",
                    as_index=False,
                )
                .agg(
                    member_features=("feature", "count"),
                    signed_contrast_difference=(
                        "error_minus_correct_contrast",
                        "sum",
                    ),
                    total_absolute_contrast_difference=(
                        "absolute_error_minus_correct_contrast",
                        "sum",
                    ),
                    mean_absolute_contrast_difference=(
                        "absolute_error_minus_correct_contrast",
                        "mean",
                    ),
                )
                .sort_values(
                    "total_absolute_contrast_difference",
                    ascending=False,
                )
                .reset_index(drop=True)
            )

            family_frame.insert(
                0,
                "family_rank",
                np.arange(1, len(family_frame) + 1),
            )
            family_frame.insert(0, "comparison", comparison_name)
            family_frame.insert(0, "method", method_name)
            family_frame["actual_class"] = actual_class
            family_frame["predicted_class"] = predicted_class
            family_frame["error_observations"] = error_count
            family_frame["correct_actual_class_observations"] = correct_count

            family_result_frames.append(family_frame)

            family_frame.to_csv(
                TABLES_DIR
                / f"{method_key}_{comparison_name}_family_contrasts.csv",
                index=False,
            )

            top_features = (
                feature_frame.head(15)
                .sort_values("error_minus_correct_contrast")
            )

            colours = [
                "#4E79A7" if value < 0 else "#E15759"
                for value in top_features[
                    "error_minus_correct_contrast"
                ]
            ]

            figure, axis = plt.subplots(figsize=(10, 7))
            axis.barh(
                top_features["feature"],
                top_features["error_minus_correct_contrast"],
                color=colours,
            )
            axis.axvline(0, color="black", linewidth=0.8)
            axis.set_title(
                f"{method_name}: {actual_class} → {predicted_class}\n"
                f"{error_count} errors versus {correct_count} correct "
                f"{actual_class} cases"
            )
            axis.set_xlabel(
                "Error-group contrast minus correct-group contrast"
            )
            axis.set_ylabel("Feature")
            figure.tight_layout()
            figure.savefig(
                FIGURES_DIR
                / f"{method_key}_{comparison_name}_top_contrasts.png",
                dpi=300,
                bbox_inches="tight",
            )
            figure.savefig(
                FIGURES_DIR
                / f"{method_key}_{comparison_name}_top_contrasts.pdf",
                bbox_inches="tight",
            )
            plt.show()
            plt.close(figure)

all_feature_results = pd.concat(
    feature_result_frames,
    ignore_index=True,
)
all_family_results = pd.concat(
    family_result_frames,
    ignore_index=True,
)

all_feature_results.to_csv(
    TABLES_DIR / "all_contrastive_shap_feature_results.csv",
    index=False,
)
all_family_results.to_csv(
    TABLES_DIR / "all_contrastive_shap_family_results.csv",
    index=False,
)

comparisons = sorted(
    all_feature_results["comparison"].unique()
)

for comparison in comparisons:
    kernel_frame = (
        all_feature_results.loc[
            (all_feature_results["method"] == "Kernel SHAP")
            & (all_feature_results["comparison"] == comparison),
            [
                "feature",
                "error_minus_correct_contrast",
                "absolute_error_minus_correct_contrast",
            ],
        ]
        .rename(
            columns={
                "error_minus_correct_contrast": "kernel_signed_contrast",
                "absolute_error_minus_correct_contrast": "kernel_absolute_contrast",
            }
        )
    )

    partition_frame = (
        all_feature_results.loc[
            (all_feature_results["method"] == "Partition SHAP")
            & (all_feature_results["comparison"] == comparison),
            [
                "feature",
                "error_minus_correct_contrast",
                "absolute_error_minus_correct_contrast",
            ],
        ]
        .rename(
            columns={
                "error_minus_correct_contrast": "partition_signed_contrast",
                "absolute_error_minus_correct_contrast": "partition_absolute_contrast",
            }
        )
    )

    merged = kernel_frame.merge(
        partition_frame,
        on="feature",
        how="inner",
    )

    signed_correlation = spearmanr(
        merged["kernel_signed_contrast"],
        merged["partition_signed_contrast"],
    )
    absolute_correlation = spearmanr(
        merged["kernel_absolute_contrast"],
        merged["partition_absolute_contrast"],
    )

    kernel_top20 = set(
        merged.nlargest(
            20,
            "kernel_absolute_contrast",
        )["feature"]
    )
    partition_top20 = set(
        merged.nlargest(
            20,
            "partition_absolute_contrast",
        )["feature"]
    )

    comparison_records.append(
        {
            "comparison": comparison,
            "all_feature_signed_spearman": float(
                signed_correlation.statistic
            ),
            "all_feature_signed_spearman_p": float(
                signed_correlation.pvalue
            ),
            "all_feature_absolute_spearman": float(
                absolute_correlation.statistic
            ),
            "all_feature_absolute_spearman_p": float(
                absolute_correlation.pvalue
            ),
            "top20_overlap_count": len(
                kernel_top20 & partition_top20
            ),
            "top20_jaccard": (
                len(kernel_top20 & partition_top20)
                / len(kernel_top20 | partition_top20)
            ),
        }
    )

cross_method_agreement = pd.DataFrame(comparison_records)
cross_method_agreement.to_csv(
    TABLES_DIR / "contrastive_shap_cross_method_agreement.csv",
    index=False,
)

provenance = {
    "analysis": "Contrastive SHAP difficult-class error analysis",
    "selected_run": str(RUN_DIR),
    "kernel_source": str(
        FULL376_DIR / "Kernel_SHAP_Full376_4096.npy"
    ),
    "partition_source": str(
        FULL376_DIR / "Partition_SHAP_Full376_4096.npy"
    ),
    "probability_source": str(
        EVALUATION_DIR / "prediction_probabilities.npy"
    ),
    "test_rows": int(len(y_true)),
    "features": int(len(feature_names)),
    "classes": int(len(device_classes)),
    "difficult_classes": DIFFICULT_CLASSES,
    "selection_role": "post-hoc explanation only",
}

(OUTPUT_DIR / "provenance.json").write_text(
    json.dumps(provenance, indent=2),
    encoding="utf-8",
)

print("\nCross-method agreement for each confusion direction:")
display(cross_method_agreement)

print("\nHighest-ranked Kernel SHAP contrastive features:")
display(
    all_feature_results.loc[
        all_feature_results["method"] == "Kernel SHAP"
    ].groupby(
        "comparison",
        group_keys=False,
    ).head(10)
)

print("\nHighest-ranked Partition SHAP contrastive features:")
display(
    all_feature_results.loc[
        all_feature_results["method"] == "Partition SHAP"
    ].groupby(
        "comparison",
        group_keys=False,
    ).head(10)
)

print(f"\nSaved outputs to:\n{OUTPUT_DIR}")

### Interpretation Criteria

The difficult-class findings are interpreted using the following decision rules:

- **Repeated errors across all five seeds:** indicate persistent ambiguity rather than an isolated unfavourable initialisation.
- **High-confidence errors:** suggest substantial feature overlap, label ambiguity or missing discriminatory information.
- **Low-confidence errors:** indicate observations close to a learned decision boundary.
- **Strong pairwise feature effects and strong validation probe performance:** indicate that the retained predictors contain discriminatory information that the selected architecture may not be using effectively.
- **Weak pairwise effects and weak probe performance:** indicate that the present feature representation may not contain sufficient information to distinguish the classes reliably.
- **Low same-class nearest-neighbour proportions:** indicate local overlap between class distributions.
- **Synthetic observations closer to another class than to their own real class:** suggest that SMOTE may be interpolating through ambiguous regions of the feature space.
- **Agreement between Kernel SHAP and Partition SHAP:** strengthens the evidence for a feature or feature-family interpretation.
- **Disagreement between SHAP methods:** is retained as attribution uncertainty and is not resolved by selecting the more favourable result.

The post-hoc results must not be used to select a replacement model using the existing held-out test set. Any corrective experiment must be developed using training and validation evidence, repeated across the same five seeds, and evaluated using a predeclared protocol.